[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap09/cap09_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)

# Aprendizado Profundo para Visão Computacional

Os capítulos anteriores estabeleceram os fundamentos da Visão Computacional (VC) por meio de métodos clássicos de extração e representação de características. No **Capítulo 7**, descritores como *Local Binary Patterns* (*LBP*) e *Histogram of Oriented Gradients* (*HOG*) mostraram como texturas e formas podem ser codificadas por descritores projetados manualmente. No **Capítulo 8**, algoritmos como *Oriented FAST and Rotated BRIEF* (*ORB*) e o detector *Haar Cascade* estenderam esse princípio a tarefas de correspondência, detecção e reconhecimento de objetos.

Essas técnicas permanecem relevantes por sua interpretabilidade e eficiência computacional, mas dependem de uma etapa prévia de definição manual de descritores, denominada **engenharia de características** (*feature engineering*). Essa dependência limita a adaptação do modelo a cenários para os quais o descritor não foi projetado.

O **Aprendizado Profundo** (*Deep Learning*) propõe uma alternativa: em vez de especificar manualmente as características relevantes, o modelo aprende automaticamente representações a partir dos dados durante o treinamento — processo conhecido como **aprendizado de representações** (*representation learning*) [@goodfellow2016deep]. Na VC, essa estratégia é implementada principalmente pelas **Redes Neurais Convolucionais** (*Convolutional Neural Networks* — CNNs), nas quais os filtros convolucionais deixam de ter coeficientes fixos e passam a ser ajustados por algoritmos de otimização.

Ainda que representem uma mudança na construção de sistemas de reconhecimento de padrões, as CNNs preservam conceitos já estudados neste livro: a convolução, apresentada no **Capítulo 3**, permanece a operação responsável pela extração local de características, agora aplicada com coeficientes aprendidos em vez de projetados.

Vale ressaltar que o objetivo deste capítulo não é explorar exaustivamente a teoria do Aprendizado Profundo, mas sim oferecer uma visão geral de seus fundamentos e demonstrar como essas arquiteturas são aplicadas no contexto da VC. Leitores interessados em um aprofundamento teórico e conceitual na área devem recorrer a referências especializadas da literatura, como @goodfellow2016deep e @lecun2015deep.


## Objetivos do Capítulo

Ao final deste capítulo, o estudante deverá ser capaz de:

- Relacionar a convolução aprendida pelas CNNs com a convolução de *kernels* fixos apresentada no **Capítulo 3**;
- Descrever a arquitetura básica de uma CNN e a função de suas camadas principais;
- Implementar, treinar e avaliar modelos de CNN para classificação de imagens;
- Aplicar **transferência de aprendizado** (*transfer learning*) para adaptar modelos pré-treinados a novos problemas;
- Utilizar modelos pré-treinados em tarefas de classificação, detecção de objetos e segmentação;
- Implementar, treinar e avaliar uma arquitetura *U-Net* para segmentação semântica, comparando-a a abordagens clássicas;
- Preparar conjuntos de dados anotados e integrá-los a um *pipeline* de treinamento por meio de plataformas como o **Roboflow**;
- Integrar geometria computacional e Aprendizado Profundo em aplicações de realidade aumentada e fotogrametria.

A @fig-09-infografo sintetiza a organização dos conceitos estudados neste capítulo e as relações entre eles.

::: {#fig-09-infografo}
![](imagens/fig-09-infografo.png){width=100% fig-align="center"}

Visão geral dos principais conceitos abordados neste capítulo. **Fonte:** elaborado com auxílio do *Gemini Notebook* [@notebooklm2025].
:::

## Panorama: Classificação, Detecção e Segmentação

As tarefas de VC diferenciam-se, sobretudo, pela informação produzida como saída. A **classificação** atribui um único rótulo à imagem inteira; a **detecção de objetos** localiza e identifica os objetos presentes na cena; a **segmentação** associa uma classe a cada pixel e, em algumas abordagens, distingue diferentes instâncias de uma mesma categoria.

A @tbl-panorama-tarefas resume as tarefas estudadas ao longo do livro, indicando a pergunta que cada uma responde e a granularidade da informação produzida.

| Tarefa | Pergunta respondida | Granularidade da saída | Capítulo |
|-----------------------------|-----------------------------------------------|-------------------------------------------------------------|:--------:|
| **Classificação** | "Qual é a classe desta imagem?" | Um único rótulo para a imagem inteira | 7 e 9 |
| **Detecção de objetos** | "Quais objetos existem e onde estão?" | Classe e caixa delimitadora (*bounding box*) para cada objeto | 8 e 9 |
| **Segmentação semântica** | "A que classe pertence cada pixel?" | Um rótulo de classe para cada pixel | 8 e 9 |
| **Segmentação de instâncias** | "Quais pixels pertencem a cada objeto?" | Um rótulo por pixel para cada instância | 8 e 9 |
| **Segmentação panóptica** | "Qual é a classe e a identidade de cada objeto?" | Classe e identificador de instância para cada pixel | 8 e 9 |

: Comparativo entre as principais tarefas de VC segundo a granularidade da informação produzida. {#tbl-panorama-tarefas}

Essas tarefas representam níveis crescentes de interpretação da imagem: a classificação descreve a cena de forma global, a detecção acrescenta a localização dos objetos e a segmentação produz uma representação espacial detalhada, permitindo analisar cada região individualmente. Este capítulo concentra-se, primeiro, na classificação por CNNs, para em seguida estender os mesmos princípios à detecção e à segmentação.


## Configuração do Ambiente

Os exemplos deste capítulo utilizam o **PyTorch**, um *framework* amplamente empregado no desenvolvimento e no treinamento de modelos de Aprendizado Profundo. O código a seguir verifica a disponibilidade das bibliotecas necessárias e instala, automaticamente, aquelas ainda não presentes no ambiente de execução.

Caso o **PyTorch** não esteja instalado, é selecionada automaticamente uma versão compatível com o hardware disponível: a versão com suporte a **CUDA**, se houver GPU NVIDIA disponível, ou a versão para execução em CPU, caso contrário.

Em seguida, o ambiente é inicializado com a importação das bibliotecas utilizadas ao longo do capítulo, a definição de uma semente aleatória para favorecer a reprodutibilidade dos experimentos e a obtenção do arquivo `morph.py` — a biblioteca didática de processamento morfológico já utilizada em capítulos anteriores —, caso ele ainda não esteja disponível no diretório de trabalho.

In [5]:
#| quarto-raw: true
import contextlib, importlib, importlib.metadata, importlib.util
import io, os, shutil, subprocess, sys, urllib.request

url = ("https://raw.githubusercontent.com/fzampirolli/"
       "pdi-vc/master/morph/config.py")
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
with contextlib.redirect_stdout(io.StringIO()):
    config.setup()
from morph import mm


def setup_cap09():
    """Instala libs ausentes p/ este capítulo (PyTorch, detecção/
    segmentação, viz de rede). Versão do OpenCV já garantida por
    config.setup()."""
    pkgs = {
        "skimage": "scikit-image", "numpy": "numpy",
        "sklearn": "scikit-learn", "matplotlib": "matplotlib",
        "torchviz": "torchviz", "ultralytics": "ultralytics",
        "roboflow": "roboflow",
    }
    for mod, pkg in pkgs.items():
        if importlib.util.find_spec(mod) is None:
            r = subprocess.run([sys.executable, "-m", "pip",
                                 "install", "-q", pkg])
            if r.returncode != 0:
                print(f"[AVISO] Falha ao instalar {pkg} ({mod}).")

    if importlib.util.find_spec("torch") is None:
        args = (["torch", "torchvision"] if shutil.which("nvidia-smi")
                 else ["--index-url",
                       "https://download.pytorch.org/whl/cpu",
                       "torch", "torchvision"])
        r = subprocess.run([sys.executable, "-m", "pip",
                             "install", "-q", *args])
        if r.returncode != 0:
            print("[AVISO] Falha ao instalar PyTorch/torchvision.")

    if not shutil.which("dot") and shutil.which("apt-get"):
        subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz"],
                        check=False)


setup_cap09()

import cv2, numpy as np, torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from skimage import data as skdata
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import OxfordIIITPet
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights)
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.transforms.functional import to_tensor
from torchviz import make_dot
from ultralytics import YOLO

torch.manual_seed(42)
FLAG_LIMPAR_DADOS = False
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
ver = importlib.metadata.version("ultralytics")
print(f"✅ Ambiente pronto. OpenCV {cv2.__version__} | "
      f"morph {getattr(mm, '__version__', 'local_file')} | "
      f"PyTorch {torch.__version__} | Ultralytics {ver} | {device}{gpu}")

✅ Ambiente pronto. OpenCV 5.0.0 | morph local_file | PyTorch 2.12.1+cpu | Ultralytics 8.4.87 | cpu


## Fundamentos de Aprendizado Profundo para VC

As CNNs são a principal arquitetura de Aprendizado Profundo aplicada à análise de imagens. Seu funcionamento baseia-se na composição de operações convolucionais organizadas em camadas sucessivas, nas quais os filtros aprendidos durante o treinamento transformam a imagem em representações progressivamente mais abstratas. Nesta seção, são apresentados os conceitos fundamentais que conectam a convolução espacial estudada anteriormente aos modelos modernos de VC, incluindo a extração hierárquica de características, o processo de treinamento e a utilização de modelos pré-treinados.

### Da Convolução Fixa à Convolução Aprendida

O **Capítulo 3** apresentou a convolução espacial com *kernels* fixos, como os operadores de Sobel, projetados para realçar características específicas de uma imagem. Nos **Capítulos 7** e **8**, o mesmo princípio sustentou descritores como *HOG*, *LBP* e *ORB*, além do detector *Haar Cascade*: em todos esses casos, os filtros são definidos antes da execução do algoritmo e permanecem inalterados durante o processamento.

A @fig-09-sim-09-convolucao retoma o funcionamento da convolução espacial: o simulador permite selecionar diferentes *kernels* e acompanhar o deslocamento da janela de convolução sobre uma imagem. Em cada posição, os coeficientes do *kernel* combinam-se com a vizinhança local da imagem — denominada **campo receptivo** (*receptive field*) — para produzir um valor do **mapa de características** (*feature map*), ilustrando também o compartilhamento de pesos (*weight sharing*)


As CNNs preservam essa operação, mas substituem *kernels* fixos por **filtros aprendidos**: em vez de coeficientes definidos previamente, a rede ajusta esses valores durante o treinamento a partir de exemplos rotulados, buscando minimizar uma **função de perda** (*loss function*), que mede a diferença entre as previsões do modelo e as respostas esperadas.

A diferença essencial entre os métodos clássicos e as CNNs, portanto, não está na operação de convolução em si, mas na forma como os filtros são obtidos: enquanto os primeiros utilizam filtros projetados manualmente, as CNNs aprendem, a partir dos dados de treinamento, representações adequadas à tarefa.

In [ ]:
#| label: fig-09-sim-09-convolucao
#| fig-cap: "Simulador interativo de convolução 2D clássica: escolha entre as três imagens sintéticas de entrada 12×12 (casa, rosto feliz ou triste) e um filtro 3×3 (Sobel V, Sobel H, Nitidez ou Identidade) e avance passo a passo para observar como os produtos internos locais do campo receptivo constroem o mapa de características célula a célula."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-convolucao" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-convolucao .cap09conv_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-convolucao .cap09conv_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-convolucao .cap09conv_btn {
      font-size:11px; font-weight:600; padding:6px 10px; border-radius:6px;
      border:1px solid #E4DCC8; background:#FFF; color:#26241D; cursor:pointer;
      transition:all .15s ease;
    }
    #sim-09-convolucao .cap09conv_btn:hover { background:#F1EAD7; }
    #sim-09-convolucao .cap09conv_modebtn {
      flex:1; text-align:center; padding:5px 8px; font-size:11px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:8px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-convolucao .cap09conv_modebtn.cap09conv_active { background:#26241D; color:#FBF7EE; }
    #sim-09-convolucao .cap09conv_btn.cap09conv_active { background:#26241D !important; color:#FBF7EE !important; border-color:#26241D !important; }
    #sim-09-convolucao .cap09conv_btn_primary { background:#2F6F9F; color:#FFF; border-color:#2F6F9F; }
    #sim-09-convolucao .cap09conv_btn_primary:hover { background:#245880; }
    #sim-09-convolucao .cap09conv_btn_success { background:#1E8F6F; color:#FFF; border-color:#1E8F6F; }
    #sim-09-convolucao .cap09conv_btn_success:hover { background:#166e55; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">🎯 Simulador: Operação de Convolução 2D Clássica</span>
    <span class="cap09conv_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Entrada 12×12 · Kernel 3×3 · Stride 1</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Painel de Seleção de Imagens e Kernels -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:12px;">
      <div style="flex:1;min-width:230px;">
        <div class="cap09conv_grouplabel">IMAGEM DE ENTRADA (12×12)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09conv_btnCasa" class="cap09conv_modebtn cap09conv_active">🏠 Casa</button>
          <button id="cap09conv_btnFeliz" class="cap09conv_modebtn">😊 Feliz</button>
          <button id="cap09conv_btnTriste" class="cap09conv_modebtn">😢 Triste</button>
        </div>
      </div>

      <div style="flex:1;min-width:280px;">
        <div class="cap09conv_grouplabel">FILTRO (KERNEL 3×3)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09conv_btnSobelV" class="cap09conv_btn cap09conv_active">📐 Sobel V</button>
          <button id="cap09conv_btnSobelH" class="cap09conv_btn">📏 Sobel H</button>
          <button id="cap09conv_btnSharpen" class="cap09conv_btn">✨ Nitidez</button>
          <button id="cap09conv_btnIdentidade" class="cap09conv_btn">🎯 Identidade</button>
        </div>
      </div>
    </div>

    <!-- Controles do Passo a Passo -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-bottom:12px;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;justify-content:space-between;">
        <div style="display:flex;gap:6px;align-items:center;">
          <button id="cap09conv_btnPasso" class="cap09conv_btn cap09conv_btn_success">▶ Avançar Passo</button>
          <button id="cap09conv_btnTudo" class="cap09conv_btn cap09conv_btn_primary">⏭ Calcular Tudo</button>
          <button id="cap09conv_btnReset" class="cap09conv_btn">↺ Resetar</button>
        </div>
        <span class="cap09conv_mono" style="font-size:11px;color:#5b5647;">Posição Atual: <b id="cap09conv_posTxt" style="color:#2F6F9F;">(0, 0)</b> [Saída 10×10]</span>
      </div>
    </div>

    <!-- Área Gráfica: Entrada, Kernel e Saída -->
    <div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Entrada (12×12)</div>
        <canvas id="cap09conv_canvasEntrada" style="width:240px;height:240px;"></canvas>
      </div>

      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Kernel (3×3)</div>
        <canvas id="cap09conv_canvasKernel" style="width:105px;height:105px;"></canvas>
      </div>

      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Mapa de Saída (10×10)</div>
        <canvas id="cap09conv_canvasSaida" style="width:200px;height:200px;"></canvas>
      </div>
    </div>

    <!-- Terminal de Cálculo em Tempo Real -->
    <div id="cap09conv_calcTxt" class="cap09conv_mono" style="text-align:center;font-size:11px;margin-top:12px;color:#7EE7C6;background:#1B2430;padding:8px 10px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;overflow-x:auto;">
      ∑ (xᵢ × wᵢ) = cálculo da posição atual...
    </div>

  </div>
</div>

<script>
(function(){
  var cap09conv_IMAGENS = {
    casa: [
      "............", "....XXXX....", "...XXXXXX...", "..XXXXXXXX..",
      ".XXXXXXXXXX.", ".XXXXXXXXXX.", ".XXXXXXXXXX.", ".XXXXXXXXXX.",
      ".XXXX..XXXX.", ".XXXX..XXXX.", ".XXXX..XXXX.", "............"
    ],
    feliz: [
      "............", "....XXXX....", "..XXXXXXXX..", ".XXXXXXXXXX.",
      ".XXXXXXXXXX.", ".XXX.XX.XXX.", ".XXXXXXXXXX.", ".XX......XX.",
      ".XXX....XXX.", ".XXXXXXXXXX.", "..XXXXXXXX..", "............"
    ],
    triste: [
      "............", "....XXXX....", "..XXXXXXXX..", ".XXXXXXXXXX.",
      ".XXXXXXXXXX.", ".XXX.XX.XXX.", ".XXXXXXXXXX.", ".XXX....XXX.",
      ".XX......XX.", ".XXXXXXXXXX.", "..XXXXXXXX..", "............"
    ]
  };

  var cap09conv_KERNELS = {
    sobelV:     { nome: "Sobel Vertical", k: [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]] },
    sobelH:     { nome: "Sobel Horizontal", k: [[-1, -2, -1], [0, 0, 0], [1, 2, 1]] },
    sharpen:    { nome: "Nitidez (Sharpen)", k: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]] },
    identidade: { nome: "Identidade", k: [[0, 0, 0], [0, 1, 0], [0, 0, 0]] }
  };

  function cap09conv_converterLinhas(cap09conv_linhas){
    return cap09conv_linhas.map(function(cap09conv_l){
      var cap09conv_res = [];
      for(var cap09conv_i=0; cap09conv_i<cap09conv_l.length; cap09conv_i++) cap09conv_res.push(cap09conv_l[cap09conv_i]==='X' ? 1.0 : 0.0);
      return cap09conv_res;
    });
  }

  function cap09conv_init(cap09conv_root){
    if(!cap09conv_root || cap09conv_root.dataset.initConv) return;
    cap09conv_root.dataset.initConv = "1";

    var cap09conv_imgAtualId = "casa";
    var cap09conv_kernelAtualKey = "sobelV";

    var cap09conv_imgEntradaBase = cap09conv_converterLinhas(cap09conv_IMAGENS[cap09conv_imgAtualId]);
    var cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
    var cap09conv_pos = {r:0, c:0};

    var cap09conv_cvsEnt = cap09conv_root.querySelector('#cap09conv_canvasEntrada');
    var cap09conv_cvsKer = cap09conv_root.querySelector('#cap09conv_canvasKernel');
    var cap09conv_cvsSai = cap09conv_root.querySelector('#cap09conv_canvasSaida');

    var cap09conv_ctxEnt = cap09conv_cvsEnt.getContext('2d');
    var cap09conv_ctxKer = cap09conv_cvsKer.getContext('2d');
    var cap09conv_ctxSai = cap09conv_cvsSai.getContext('2d');

    function cap09conv_prepararCanvas(cap09conv_canvas, cap09conv_ctx, cap09conv_cssW, cap09conv_cssH){
      var cap09conv_dpr = window.devicePixelRatio || 1;
      cap09conv_canvas.width = cap09conv_cssW * cap09conv_dpr;
      cap09conv_canvas.height = cap09conv_cssH * cap09conv_dpr;
      cap09conv_ctx.scale(cap09conv_dpr, cap09conv_dpr);
    }
    cap09conv_prepararCanvas(cap09conv_cvsEnt, cap09conv_ctxEnt, 240, 240);
    cap09conv_prepararCanvas(cap09conv_cvsKer, cap09conv_ctxKer, 105, 105);
    cap09conv_prepararCanvas(cap09conv_cvsSai, cap09conv_ctxSai, 200, 200);

    var cap09conv_posTxt  = cap09conv_root.querySelector('#cap09conv_posTxt');
    var cap09conv_calcTxt = cap09conv_root.querySelector('#cap09conv_calcTxt');

    var cap09conv_btnCasa   = cap09conv_root.querySelector('#cap09conv_btnCasa');
    var cap09conv_btnFeliz  = cap09conv_root.querySelector('#cap09conv_btnFeliz');
    var cap09conv_btnTriste = cap09conv_root.querySelector('#cap09conv_btnTriste');

    var cap09conv_btnSobelV     = cap09conv_root.querySelector('#cap09conv_btnSobelV');
    var cap09conv_btnSobelH     = cap09conv_root.querySelector('#cap09conv_btnSobelH');
    var cap09conv_btnSharpen    = cap09conv_root.querySelector('#cap09conv_btnSharpen');
    var cap09conv_btnIdentidade = cap09conv_root.querySelector('#cap09conv_btnIdentidade');

    function cap09conv_desenharEntrada(){
      var cap09conv_tam = 20;
      cap09conv_ctxEnt.clearRect(0,0,240,240);
      for (var cap09conv_r=0; cap09conv_r<12; cap09conv_r++){
        for (var cap09conv_c=0; cap09conv_c<12; cap09conv_c++){
          var cap09conv_val = cap09conv_imgEntradaBase[cap09conv_r][cap09conv_c];
          var cap09conv_g = Math.round(cap09conv_val * 255);
          cap09conv_ctxEnt.fillStyle = "rgb(" + cap09conv_g + "," + cap09conv_g + "," + cap09conv_g + ")";
          cap09conv_ctxEnt.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
          cap09conv_ctxEnt.strokeStyle = "#E4DCC8";
          cap09conv_ctxEnt.lineWidth = 0.8;
          cap09conv_ctxEnt.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

          cap09conv_ctxEnt.fillStyle = cap09conv_g > 128 ? "#111827" : "#FFFFFF";
          cap09conv_ctxEnt.font = "600 8px 'JetBrains Mono', monospace";
          cap09conv_ctxEnt.textAlign = "center";
          cap09conv_ctxEnt.textBaseline = "middle";
          cap09conv_ctxEnt.fillText(cap09conv_val.toFixed(0), cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
        }
      }

      if (cap09conv_pos.r < 10) {
        cap09conv_ctxEnt.strokeStyle = "#C1443A";
        cap09conv_ctxEnt.lineWidth = 2.5;
        cap09conv_ctxEnt.strokeRect(cap09conv_pos.c*cap09conv_tam, cap09conv_pos.r*cap09conv_tam, cap09conv_tam*3, cap09conv_tam*3);
      }
    }

    function cap09conv_desenharKernel(){
      var cap09conv_tam = 35;
      cap09conv_ctxKer.clearRect(0,0,105,105);
      var cap09conv_k = cap09conv_KERNELS[cap09conv_kernelAtualKey].k;
      for (var cap09conv_i=0; cap09conv_i<3; cap09conv_i++){
        for (var cap09conv_j=0; cap09conv_j<3; cap09conv_j++){
          var cap09conv_val = cap09conv_k[cap09conv_i][cap09conv_j];
          cap09conv_ctxKer.fillStyle = cap09conv_val > 0 ? "#E6F4EA" : (cap09conv_val < 0 ? "#FCE8E6" : "#FAFAF7");
          cap09conv_ctxKer.fillRect(cap09conv_j*cap09conv_tam, cap09conv_i*cap09conv_tam, cap09conv_tam, cap09conv_tam);
          cap09conv_ctxKer.strokeStyle = "#E4DCC8";
          cap09conv_ctxKer.strokeRect(cap09conv_j*cap09conv_tam, cap09conv_i*cap09conv_tam, cap09conv_tam, cap09conv_tam);

          cap09conv_ctxKer.fillStyle = cap09conv_val > 0 ? "#1E8F6F" : (cap09conv_val < 0 ? "#C1443A" : "#8A8371");
          cap09conv_ctxKer.font = "600 10px 'JetBrains Mono', monospace";
          cap09conv_ctxKer.textAlign = "center";
          cap09conv_ctxKer.textBaseline = "middle";
          cap09conv_ctxKer.fillText(cap09conv_val.toString(), cap09conv_j*cap09conv_tam + cap09conv_tam/2, cap09conv_i*cap09conv_tam + cap09conv_tam/2);
        }
      }
    }

    function cap09conv_desenharSaida(){
      var cap09conv_tam = 20;
      cap09conv_ctxSai.clearRect(0,0,200,200);
      for (var cap09conv_r=0; cap09conv_r<10; cap09conv_r++){
        for (var cap09conv_c=0; cap09conv_c<10; cap09conv_c++){
          var cap09conv_v = cap09conv_saida[cap09conv_r][cap09conv_c];
          if (cap09conv_v === null) {
            cap09conv_ctxSai.fillStyle = "#F7F5EE";
            cap09conv_ctxSai.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
            cap09conv_ctxSai.strokeStyle = "#E4DCC8";
            cap09conv_ctxSai.lineWidth = 0.8;
            cap09conv_ctxSai.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

            cap09conv_ctxSai.fillStyle = "#B8AE94";
            cap09conv_ctxSai.font = "700 8px 'JetBrains Mono', monospace";
            cap09conv_ctxSai.textAlign = "center";
            cap09conv_ctxSai.textBaseline = "middle";
            cap09conv_ctxSai.fillText("·", cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
          } else {
            var cap09conv_normV = Math.max(0, Math.min(1, (cap09conv_v + 2.0) / 4.0));
            var cap09conv_g = Math.round(cap09conv_normV * 255);
            cap09conv_ctxSai.fillStyle = "rgb(" + cap09conv_g + "," + cap09conv_g + "," + cap09conv_g + ")";
            cap09conv_ctxSai.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
            cap09conv_ctxSai.strokeStyle = "#E4DCC8";
            cap09conv_ctxSai.lineWidth = 0.8;
            cap09conv_ctxSai.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

            cap09conv_ctxSai.fillStyle = cap09conv_g > 128 ? "#111827" : "#FFFFFF";
            cap09conv_ctxSai.font = "600 7.5px 'JetBrains Mono', monospace";
            cap09conv_ctxSai.textAlign = "center";
            cap09conv_ctxSai.textBaseline = "middle";
            cap09conv_ctxSai.fillText(cap09conv_v.toFixed(1), cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
          }
        }
      }

      if (cap09conv_pos.r < 10){
        cap09conv_ctxSai.strokeStyle = "#2F6F9F";
        cap09conv_ctxSai.lineWidth = 2;
        cap09conv_ctxSai.strokeRect(cap09conv_pos.c*cap09conv_tam, cap09conv_pos.r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
      }
    }

    function cap09conv_calcularPosicao(cap09conv_r, cap09conv_c){
      var cap09conv_k = cap09conv_KERNELS[cap09conv_kernelAtualKey].k;
      var cap09conv_soma = 0;
      var cap09conv_termos = [];
      for (var cap09conv_i=0; cap09conv_i<3; cap09conv_i++){
        for (var cap09conv_j=0; cap09conv_j<3; cap09conv_j++){
          var cap09conv_valEnt = cap09conv_imgEntradaBase[cap09conv_r+cap09conv_i][cap09conv_c+cap09conv_j];
          var cap09conv_valKer = cap09conv_k[cap09conv_i][cap09conv_j];
          var cap09conv_prod = cap09conv_valEnt * cap09conv_valKer;
          cap09conv_soma += cap09conv_prod;
          cap09conv_termos.push(cap09conv_prod >= 0 ? cap09conv_prod.toFixed(0) : '(' + cap09conv_prod.toFixed(0) + ')');
        }
      }
      return { valFinal: cap09conv_soma, expressao: cap09conv_termos.join(" + ") };
    }

    function cap09conv_atualizarCalculoTexto(){
      if (cap09conv_pos.r >= 10) {
        cap09conv_calcTxt.textContent = "Convolução Concluída! Todos os 100 pixels do mapa de saída foram gerados.";
        return;
      }
      var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
      cap09conv_calcTxt.textContent = 'Posição (' + cap09conv_pos.r + ',' + cap09conv_pos.c + ') → z = ' + cap09conv_obj.expressao + ' = ' + cap09conv_obj.valFinal.toFixed(2);
    }

    function cap09conv_avancarPasso(){
      if (cap09conv_pos.r >= 10) return;
      var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
      cap09conv_saida[cap09conv_pos.r][cap09conv_pos.c] = cap09conv_obj.valFinal;
      cap09conv_pos.c++;
      if (cap09conv_pos.c >= 10){ cap09conv_pos.c = 0; cap09conv_pos.r++; }
      cap09conv_render();
    }

    function cap09conv_calcularTudo(){
      while (cap09conv_pos.r < 10) {
        var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
        cap09conv_saida[cap09conv_pos.r][cap09conv_pos.c] = cap09conv_obj.valFinal;
        cap09conv_pos.c++;
        if (cap09conv_pos.c >= 10){ cap09conv_pos.c = 0; cap09conv_pos.r++; }
      }
      cap09conv_render();
    }

    function cap09conv_resetar(){
      cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
      cap09conv_pos = {r:0, c:0};
      cap09conv_render();
    }

    function cap09conv_resetarECalcular(){
      cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
      cap09conv_pos = {r:0, c:0};
      cap09conv_calcularTudo();
    }

    function cap09conv_render(){
      cap09conv_desenharEntrada();
      cap09conv_desenharKernel();
      cap09conv_desenharSaida();
      cap09conv_posTxt.textContent = cap09conv_pos.r < 10 ? '(' + cap09conv_pos.r + ', ' + cap09conv_pos.c + ')' : 'concluído';
      cap09conv_atualizarCalculoTexto();
    }

    function cap09conv_trocarImagem(cap09conv_id, cap09conv_btn){
      [cap09conv_btnCasa, cap09conv_btnFeliz, cap09conv_btnTriste].forEach(function(b){ b.classList.remove('cap09conv_active'); });
      cap09conv_btn.classList.add('cap09conv_active');
      cap09conv_imgAtualId = cap09conv_id;
      cap09conv_imgEntradaBase = cap09conv_converterLinhas(cap09conv_IMAGENS[cap09conv_id]);
      cap09conv_resetarECalcular();
    }

    function cap09conv_trocarKernel(cap09conv_key, cap09conv_btn){
      [cap09conv_btnSobelV, cap09conv_btnSobelH, cap09conv_btnSharpen, cap09conv_btnIdentidade].forEach(function(b){ b.classList.remove('cap09conv_active'); });
      cap09conv_btn.classList.add('cap09conv_active');
      cap09conv_kernelAtualKey = cap09conv_key;
      cap09conv_resetarECalcular();
    }

    cap09conv_btnCasa.addEventListener('click', function(){ cap09conv_trocarImagem("casa", cap09conv_btnCasa); });
    cap09conv_btnFeliz.addEventListener('click', function(){ cap09conv_trocarImagem("feliz", cap09conv_btnFeliz); });
    cap09conv_btnTriste.addEventListener('click', function(){ cap09conv_trocarImagem("triste", cap09conv_btnTriste); });

    cap09conv_btnSobelV.addEventListener('click', function(){ cap09conv_trocarKernel("sobelV", cap09conv_btnSobelV); });
    cap09conv_btnSobelH.addEventListener('click', function(){ cap09conv_trocarKernel("sobelH", cap09conv_btnSobelH); });
    cap09conv_btnSharpen.addEventListener('click', function(){ cap09conv_trocarKernel("sharpen", cap09conv_btnSharpen); });
    cap09conv_btnIdentidade.addEventListener('click', function(){ cap09conv_trocarKernel("identidade", cap09conv_btnIdentidade); });

    cap09conv_root.querySelector('#cap09conv_btnPasso').addEventListener('click', cap09conv_avancarPasso);
    cap09conv_root.querySelector('#cap09conv_btnTudo').addEventListener('click', cap09conv_calcularTudo);
    cap09conv_root.querySelector('#cap09conv_btnReset').addEventListener('click', cap09conv_resetar);

    cap09conv_resetarECalcular();
  
  }

  function cap09conv_tryInit(){
    var cap09conv_root = document.getElementById('sim-09-convolucao');
    if(cap09conv_root) cap09conv_init(cap09conv_root); else setTimeout(cap09conv_tryInit, 200);
  }
  cap09conv_tryInit();
})();
</script>
''')

Para compreender como esse aprendizado ocorre, é necessário estudar a unidade básica de processamento das redes neurais: o **neurônio artificial**.

### Neurônio Artificial

O **neurônio artificial** (*artificial neuron*) é a unidade fundamental de processamento de uma rede neural. Seu primeiro modelo matemático — um conjunto de entradas combinadas e comparadas a um limiar — foi proposto por @mcculloch1943logical, ainda sem qualquer mecanismo de aprendizado. O **Perceptron** [@rosenblatt1958perceptron] avançou sobre essa formulação ao introduzir uma regra de ajuste dos pesos a partir de exemplos, tornando-se o primeiro modelo de neurônio artificial capaz de aprender e a base das arquiteturas modernas de **Aprendizado Profundo** (*Deep Learning*). O termo Aprendizado Profundo refere-se ao uso de redes com múltiplas camadas de processamento, capazes de aprender representações hierárquicas dos dados: as primeiras camadas aprendem características simples, como bordas e texturas, e as camadas mais profundas combinam progressivamente essas representações para identificar estruturas e objetos mais complexos.

Cada neurônio recebe um conjunto de entradas, calcula uma combinação linear desses valores e aplica uma **função de ativação** (*activation function*), produzindo um único valor de saída. Matematicamente, a combinação linear é dada por

$$
z=\sum_{i=1}^{n}w_i x_i+b,
$$

em que $x_i$ representam as entradas, $w_i$ os pesos associados a cada entrada e $b$ o **viés** (*bias*). A saída do neurônio é obtida pela aplicação da função de ativação:

$$
y=f(z).
$$

Nas CNNs, esse princípio assume formas distintas conforme a camada. Nas **camadas convolucionais** (*convolutional layers*), cada neurônio processa apenas uma pequena região da entrada, denominada **campo receptivo** (*receptive field*), preservando a organização espacial da imagem. Nas **camadas totalmente conectadas** (*fully connected layers*), cada neurônio recebe todas as saídas da camada anterior, combinando as características extraídas para produzir a saída final da rede, como a classe atribuída à imagem.

A @fig-09-sim-09-neuronio ilustra o funcionamento de um neurônio artificial: o simulador permite modificar as entradas ($x_1$ e $x_2$), os pesos ($w_1$ e $w_2$), o viés ($b$) e a função de ativação, observando em tempo real o cálculo da combinação linear e da saída correspondente.

In [ ]:
#| label: fig-09-sim-09-neuronio
#| fig-cap: "Simulador interativo do Neurônio Artificial em contexto de CNN: alterne entre um neurônio de camada convolucional (onde x_i são intensidades de pixel em um campo receptivo e w_i são pesos do *kernel*) e um neurônio de camada totalmente conectada, ajustando entradas, pesos, viés e função de ativação para visualizar o cálculo de z e da saída y em tempo real."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-neuronio" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-neuronio .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-neuronio .cn-modebtn {
      flex:1; text-align:center; padding:8px 10px; font-size:12px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:10px;
      transition:background .15s ease, color .15s ease;
    }
    #sim-09-neuronio .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-neuronio input[type=range] { accent-color:#2F6F9F; min-width:0; }
    #sim-09-neuronio .cn-tag {
      font-size:10px; font-weight:700; color:#8A8371; width:16px; text-align:center; flex-shrink:0;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulador: Neurônio Artificial em uma CNN</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">y = f(∑ wᵢxᵢ + b)</span>
  </div>

  <div style="padding:18px 20px;background:#FFFFFF;overflow:auto">

    <!-- Alternador de contexto: camada convolucional vs. totalmente conectada -->
    <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:12px;padding:4px;margin-bottom:14px;max-width:480px;margin-left:auto;margin-right:auto;">
      <button id="cap09neuronio_modeConv" class="cn-modebtn active">🧩 Camada Convolucional</button>
      <button id="cap09neuronio_modeFC" class="cn-modebtn">🔗 Camada Totalmente Conectada</button>
    </div>

    <!-- Painel de Controles -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));gap:12px;align-items:start;">

        <div>
          <label id="cap09neuronio_lblX1" style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Pixel x₁ (normalizado) / Peso do kernel w₁</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">x₁</span>
            <input type="range" id="cap09neuronio_x1" min="-3" max="3" step="0.1" value="1.0" style="flex:1;">
            <span class="cn-tag cn-mono">w₁</span>
            <input type="range" id="cap09neuronio_w1" min="-3" max="3" step="0.1" value="0.8" style="flex:1;">
          </div>
        </div>

        <div>
          <label id="cap09neuronio_lblX2" style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Pixel x₂ (normalizado) / Peso do kernel w₂</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">x₂</span>
            <input type="range" id="cap09neuronio_x2" min="-3" max="3" step="0.1" value="-1.5" style="flex:1;">
            <span class="cn-tag cn-mono">w₂</span>
            <input type="range" id="cap09neuronio_w2" min="-3" max="3" step="0.1" value="0.5" style="flex:1;">
          </div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Viés (b) e Função f(z)</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">b</span>
            <input type="range" id="cap09neuronio_bias" min="-3" max="3" step="0.1" value="0.2" style="flex:1;">
            <select id="cap09neuronio_func" style="font-size:11px;padding:5px 6px;border-radius:6px;border:1px solid #ccc;background:#fff;">
              <option value="relu">ReLU</option>
              <option value="sigmoid">Sigmoide</option>
              <option value="step">Degrau (Step)</option>
              <option value="identity">Identidade</option>
            </select>
          </div>
        </div>

      </div>
      <div id="cap09neuronio_notaEscala" style="font-size:10.5px;color:#8A8371;margin-top:10px;padding-top:8px;border-top:1px dashed #E9E3D3;">
        Os valores de x variam de -3 a 3 porque, em uma CNN, os pixels (0–255) são <b>normalizados</b> antes de entrar na rede. O desenho ao lado traduz esse valor normalizado de volta em um tom de cinza, só para dar intuição visual — os números que valem para a conta são os das barras.
      </div>
    </div>

    <!-- Área Gráfica: Esquema do Neurônio + Curva -->
    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div id="cap09neuronio_diagTitle" style="font-size:11px;font-weight:600;text-align:center;margin-bottom:6px;color:#4b5563;">Campo Receptivo → Convolução → Mapa de Características</div>
        <canvas id="cap09neuronio_canvasEsquema" style="width:320px;height:230px;"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:6px;color:#4b5563;">Ativação em f(z)</div>
        <canvas id="cap09neuronio_canvasCurva" style="width:260px;height:230px;"></canvas>
      </div>
    </div>

    <!-- Legenda contextual -->
    <div id="cap09neuronio_caption" style="font-size:11.5px;line-height:1.5;color:#5b5647;background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:10px 12px;margin-top:14px;">
      <b>x₁, x₂</b> = intensidade dos pixels no campo receptivo · <b>w₁, w₂</b> = pesos do kernel (filtro) · <b>z</b> = resultado da convolução nesta posição · <b>y</b> = valor do pixel produzido no mapa de características, após a ativação.
    </div>

    <!-- Status numérico estilo terminal -->
    <div id="cap09neuronio_valTxt" class="cn-mono" style="text-align:center;font-size:12px;margin-top:12px;color:#7EE7C6;background:#1B2430;padding:10px 12px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;">
      z = (1.00 × 0.80) + (-1.50 × 0.50) + 0.20 = 0.25 → y = 0.25
    </div>

  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var canvasEsquema = root.querySelector('#cap09neuronio_canvasEsquema');
    var canvasCurva   = root.querySelector('#cap09neuronio_canvasCurva');
    var ctxEsquema = canvasEsquema.getContext('2d');
    var ctxCurva   = canvasCurva.getContext('2d');

    // Escala para telas de alta resolução (retina)
    function prepararCanvas(canvas, ctx, cssW, cssH){
      var dpr = window.devicePixelRatio || 1;
      canvas.width = cssW * dpr;
      canvas.height = cssH * dpr;
      ctx.scale(dpr, dpr);
    }
    prepararCanvas(canvasEsquema, ctxEsquema, 320, 230);
    prepararCanvas(canvasCurva, ctxCurva, 260, 230);

    var inX1 = root.querySelector('#cap09neuronio_x1');
    var inW1 = root.querySelector('#cap09neuronio_w1');
    var inX2 = root.querySelector('#cap09neuronio_x2');
    var inW2 = root.querySelector('#cap09neuronio_w2');
    var inB  = root.querySelector('#cap09neuronio_bias');
    var selF = root.querySelector('#cap09neuronio_func');
    var valTxt = root.querySelector('#cap09neuronio_valTxt');
    var caption = root.querySelector('#cap09neuronio_caption');
    var diagTitle = root.querySelector('#cap09neuronio_diagTitle');
    var lblX1 = root.querySelector('#cap09neuronio_lblX1');
    var lblX2 = root.querySelector('#cap09neuronio_lblX2');
    var notaEscala = root.querySelector('#cap09neuronio_notaEscala');
    var btnConv = root.querySelector('#cap09neuronio_modeConv');
    var btnFC   = root.querySelector('#cap09neuronio_modeFC');

    var modo = 'conv'; // 'conv' | 'fc'

    var CORES = {
      pos: "#1E8F6F",      // peso/conexão positiva
      neg: "#C1443A",      // peso/conexão negativa
      bias: "#C08A2E",     // viés
      saida: "#2F5FA8",    // sinal de saída
      texto: "#26241D",
      textoSuave: "#6b7280",
      grade: "#EFEAdd"
    };

    function calcularAtivacao(z, tipo){
      if (tipo === 'relu') return Math.max(0, z);
      if (tipo === 'sigmoid') return 1 / (1 + Math.exp(-z));
      if (tipo === 'step') return z >= 0 ? 1 : 0;
      return z; // identity
    }

    // Converte um valor de entrada (-3..3) em um tom de cinza 0-255 (intensidade de pixel)
    function valorParaCinza(v){
      var t = Math.max(0, Math.min(1, (v + 3) / 6));
      return Math.round(t * 255);
    }
    // Converte a saída y (que pode ter faixas diferentes conforme f) em cinza 0-255
    function saidaParaCinza(y, tipo){
      var t;
      if (tipo === 'sigmoid' || tipo === 'step') t = y;
      else t = Math.max(0, Math.min(1, (y + 3) / 6));
      return Math.round(Math.max(0, Math.min(1, t)) * 255);
    }

    function desenharPixel(ctx, cx, cy, lado, cinza, corBorda){
      var g = "rgb(" + cinza + "," + cinza + "," + cinza + ")";
      ctx.fillStyle = g;
      ctx.fillRect(cx - lado/2, cy - lado/2, lado, lado);
      ctx.strokeStyle = corBorda || "#9ca3af";
      ctx.lineWidth = 1.2;
      ctx.strokeRect(cx - lado/2, cy - lado/2, lado, lado);
    }

    function chipPeso(ctx, cx, cy, valor){
      var cor = valor >= 0 ? CORES.pos : CORES.neg;
      ctx.font = "600 10px 'JetBrains Mono', monospace";
      var texto = (valor>=0?"+":"") + valor.toFixed(1);
      var w = ctx.measureText(texto).width + 10;
      ctx.fillStyle = cor;
      roundRect(ctx, cx - w/2, cy - 9, w, 18, 9);
      ctx.fill();
      ctx.fillStyle = "#fff";
      ctx.textAlign = "center";
      ctx.textBaseline = "middle";
      ctx.fillText(texto, cx, cy+1);
    }

    function roundRect(ctx, x, y, w, h, r){
      ctx.beginPath();
      ctx.moveTo(x+r, y);
      ctx.arcTo(x+w, y, x+w, y+h, r);
      ctx.arcTo(x+w, y+h, x, y+h, r);
      ctx.arcTo(x, y+h, x, y, r);
      ctx.arcTo(x, y, x+w, y, r);
      ctx.closePath();
    }

    function seta(ctx, x1,y1,x2,y2,cor){
      ctx.strokeStyle = cor; ctx.lineWidth = 2;
      ctx.beginPath(); ctx.moveTo(x1,y1); ctx.lineTo(x2,y2); ctx.stroke();
      var ang = Math.atan2(y2-y1, x2-x1);
      ctx.fillStyle = cor;
      ctx.beginPath();
      ctx.moveTo(x2,y2);
      ctx.lineTo(x2 - 7*Math.cos(ang-0.4), y2 - 7*Math.sin(ang-0.4));
      ctx.lineTo(x2 - 7*Math.cos(ang+0.4), y2 - 7*Math.sin(ang+0.4));
      ctx.closePath(); ctx.fill();
    }

    // ---------- MODO CONVOLUÇÃO ----------
    function desenharConvolucao(x1, w1, x2, w2, b, z, y, tipo){
      var W=320, H=230;
      ctxEsquema.clearRect(0,0,W,H);
      ctxEsquema.textAlign = "center"; ctxEsquema.textBaseline = "middle";

      var cinzaX1 = valorParaCinza(x1), cinzaX2 = valorParaCinza(x2);
      var cinzaY  = saidaParaCinza(y, tipo);

      // Campo receptivo (patch de imagem de entrada) — dois pixels empilhados
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Campo receptivo", 48, 14);

      desenharPixel(ctxEsquema, 48, 48, 44, cinzaX1, "#9ca3af");
      desenharPixel(ctxEsquema, 48, 118, 44, cinzaX2, "#9ca3af");
      ctxEsquema.fillStyle = cinzaX1 > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("x₁=" + x1.toFixed(1), 48, 48);
      ctxEsquema.fillStyle = cinzaX2 > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.fillText("x₂=" + x2.toFixed(1), 48, 118);

      // CORREÇÃO AQUI: As setas agora param na borda esquerda do bloco do kernel (x=118)
      seta(ctxEsquema, 70, 48, 118, 72, w1 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 70, 118, 118, 96, w2 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 140, 45, 140, 62, b >= 0 ? CORES.bias : CORES.neg);

      // CORREÇÃO AQUI: Chips dos pesos centralizados no meio da seta (x=94)
      chipPeso(ctxEsquema, 94, 60, w1);
      chipPeso(ctxEsquema, 94, 107, w2);

      // Viés
      ctxEsquema.fillStyle = "#FDF0DA";
      ctxEsquema.beginPath(); ctxEsquema.arc(140, 26, 19, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.bias; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#8a5a12"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("b=" + b.toFixed(1), 140, 26);

      // Nó de convolução (kernel * patch) — ocupa de x=118 a x=162
      ctxEsquema.fillStyle = "#DCE8F5";
      roundRect(ctxEsquema, 118, 62, 44, 44, 10); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#2F5FA8"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#1e3a5f"; ctxEsquema.font = "600 12px Inter, sans-serif";
      ctxEsquema.fillText("⊛", 140, 78);
      ctxEsquema.font = "9px Inter, sans-serif";
      ctxEsquema.fillText("kernel", 140, 94);

      // Seta para o mapa de características
      seta(ctxEsquema, 162, 84, 202, 84, CORES.saida);

      // Mapa de características (mini tira com o pixel de saída em destaque)
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Mapa de características", 265, 14);

      var vizinhos = [190, 190];
      desenharPixel(ctxEsquema, 224, 84, 26, vizinhos[0], "#d1d5db");
      ctxEsquema.save();
      desenharPixel(ctxEsquema, 265, 84, 40, cinzaY, "#2F5FA8");
      ctxEsquema.lineWidth = 2.4; ctxEsquema.strokeStyle = CORES.saida;
      ctxEsquema.strokeRect(265-21, 84-21, 42, 42);
      ctxEsquema.fillStyle = cinzaY > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("y=" + y.toFixed(1), 265, 84);
      ctxEsquema.restore();
      desenharPixel(ctxEsquema, 306, 84, 26, vizinhos[1], "#d1d5db");

      ctxEsquema.fillStyle = "#9ca3af"; ctxEsquema.font = "8px Inter, sans-serif";
      ctxEsquema.fillText("(o kernel desliza →)", 265, 212);
    }

    // ---------- MODO TOTALMENTE CONECTADA ----------
    function desenharFC(x1, w1, x2, w2, b, z, y){
      var W=320, H=230;
      ctxEsquema.clearRect(0,0,W,H);
      ctxEsquema.textAlign = "center"; ctxEsquema.textBaseline = "middle";

      seta(ctxEsquema, 55, 60, 155, 118, w1 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 55, 176, 155, 118, w2 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 170, 45, 170, 100, b  >= 0 ? CORES.bias : CORES.neg);
      seta(ctxEsquema, 195, 118, 260, 118, CORES.saida);

      // Entrada x1
      ctxEsquema.fillStyle = "#EDEDE7"; ctxEsquema.beginPath(); ctxEsquema.arc(45, 60, 24, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#9ca3af"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = CORES.texto; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("x₁=" + x1.toFixed(1), 45, 60);

      // Entrada x2
      ctxEsquema.fillStyle = "#EDEDE7"; ctxEsquema.beginPath(); ctxEsquema.arc(45, 176, 24, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#9ca3af"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = CORES.texto;
      ctxEsquema.fillText("x₂=" + x2.toFixed(1), 45, 176);

      chipPeso(ctxEsquema, 108, 92, w1);
      chipPeso(ctxEsquema, 108, 148, w2);

      // Viés
      ctxEsquema.fillStyle = "#FDF0DA"; ctxEsquema.beginPath(); ctxEsquema.arc(170, 30, 20, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.bias; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#8a5a12"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("b=" + b.toFixed(1), 170, 30);

      // Soma / ativação
      ctxEsquema.fillStyle = "#DCE8F5"; ctxEsquema.beginPath(); ctxEsquema.arc(172, 118, 28, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#2F5FA8"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#1e3a5f"; ctxEsquema.font = "12px Inter, sans-serif";
      ctxEsquema.fillText("∑, f", 172, 118);

      // Saída
      ctxEsquema.fillStyle = "#DCEEE6"; ctxEsquema.beginPath(); ctxEsquema.arc(280, 118, 26, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.pos; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#14532d"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("y=" + y.toFixed(2), 280, 118);

      ctxEsquema.fillStyle = CORES.textoSuave; ctxEsquema.font = "9px Inter, sans-serif";
      ctxEsquema.fillText("neurônio da camada totalmente conectada", 172, 210);
    }

    function desenharCurva(z, y, tipo){
      var W = 260, H = 230;
      var origemX = 130, origemY = 165;
      var escalaX = 20, escalaY = 40;

      ctxCurva.clearRect(0,0,W,H);

      // Grade sutil
      ctxCurva.strokeStyle = CORES.grade; ctxCurva.lineWidth = 1;
      for (var gx = 10; gx <= W-10; gx += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(gx,10); ctxCurva.lineTo(gx,H-10); ctxCurva.stroke(); }
      for (var gy = 10; gy <= H-10; gy += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(10,gy); ctxCurva.lineTo(W-10,gy); ctxCurva.stroke(); }

      // Eixos
      ctxCurva.strokeStyle = "#9ca3af"; ctxCurva.lineWidth = 1.3;
      ctxCurva.beginPath();
      ctxCurva.moveTo(10, origemY); ctxCurva.lineTo(W-10, origemY);
      ctxCurva.moveTo(origemX, 10); ctxCurva.lineTo(origemX, H-10);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280"; ctxCurva.font = "10px Inter, sans-serif"; ctxCurva.textAlign = "left";
      ctxCurva.fillText("z", W-18, origemY-6);
      ctxCurva.fillText("y", origemX+6, 16);

      // Curva da função
      ctxCurva.strokeStyle = "#2F5FA8"; ctxCurva.lineWidth = 2.4;
      ctxCurva.beginPath();
      var primeiro = true;
      for (var px = -5; px <= 5; px += 0.1) {
        var valY = calcularAtivacao(px, tipo);
        var cx = origemX + px * escalaX;
        var cy = origemY - valY * escalaY;
        if (primeiro) { ctxCurva.moveTo(cx, cy); primeiro = false; }
        else { ctxCurva.lineTo(cx, cy); }
      }
      ctxCurva.stroke();

      // Ponto atual
      var ptX = Math.max(10, Math.min(W-10, origemX + z * escalaX));
      var ptY = Math.max(10, Math.min(H-10, origemY - y * escalaY));

      ctxCurva.strokeStyle = "#C1443A"; ctxCurva.setLineDash([3,3]); ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(ptX, origemY); ctxCurva.lineTo(ptX, ptY); ctxCurva.lineTo(origemX, ptY);
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);

      ctxCurva.fillStyle = "#C1443A";
      ctxCurva.beginPath(); ctxCurva.arc(ptX, ptY, 5, 0, 2*Math.PI); ctxCurva.fill();
      ctxCurva.strokeStyle = "#fff"; ctxCurva.lineWidth = 1.5; ctxCurva.stroke();
    }

    function atualizarLegendas(){
      if (modo === 'conv'){
        lblX1.textContent = "Pixel x₁ (normalizado) / Peso do kernel w₁";
        lblX2.textContent = "Pixel x₂ (normalizado) / Peso do kernel w₂";
        diagTitle.textContent = "Campo Receptivo → Convolução → Mapa de Características";
        caption.innerHTML = "<b>🧩 Preservação Espacial:</b> Na camada convolucional, a operação ocorre localmente via campo receptivo. O resultado (y) mantém uma posição bem definida no mapa de características 2D, preservando a vizinhança e a estrutura geométrica dos pixels.";
        notaEscala.innerHTML = "Os valores de x variam de -3 a 3 porque, em uma CNN, os pixels (0–255) são <b>normalizados</b> antes de entrar na rede. O desenho ao lado traduz esse valor normalizado de volta em um tom de cinza, só para dar intuição visual — os números que valem para a conta são os das barras.";
      } else {
        lblX1.textContent = "Atributo x₁ (feature) / Peso w₁";
        lblX2.textContent = "Atributo x₂ (feature) / Peso w₂";
        diagTitle.textContent = "Vetor de Atributos → Neurônio → Saída";
        caption.innerHTML = "<b>🔗 Perda da Informação Espacial:</b> Na camada totalmente conectada, os mapas de características são achatados (flatten) em um vetor 1D. Como o neurônio se conecta a todas as entradas indiferenciadamente, a noção de 'vizinho de cima/lado' é destruída em prol de uma decisão global.";
        notaEscala.innerHTML = "Aqui x₁, x₂ representam atributos já extraídos (não pixels), tipicamente padronizados para uma faixa pequena como esta antes de entrarem na camada.";
      }
    }

    function atualizar(){
      var x1 = parseFloat(inX1.value);
      var w1 = parseFloat(inW1.value);
      var x2 = parseFloat(inX2.value);
      var w2 = parseFloat(inW2.value);
      var b  = parseFloat(inB.value);
      var tipo = selF.value;

      var z = (x1 * w1) + (x2 * w2) + b;
      var y = calcularAtivacao(z, tipo);

      if (modo === 'conv') desenharConvolucao(x1, w1, x2, w2, b, z, y, tipo);
      else desenharFC(x1, w1, x2, w2, b, z, y);

      desenharCurva(z, y, tipo);

      valTxt.textContent = 'z = (' + x1.toFixed(2) + ' × ' + w1.toFixed(2) + ') + (' +
                           x2.toFixed(2) + ' × ' + w2.toFixed(2) + ') + (' + b.toFixed(2) +
                           ') = ' + z.toFixed(2) + '  →  y = ' + selF.options[selF.selectedIndex].text + '(z) = ' + y.toFixed(2);
    }

    function definirModo(novoModo){
      modo = novoModo;
      btnConv.classList.toggle('active', modo === 'conv');
      btnFC.classList.toggle('active', modo === 'fc');
      atualizarLegendas();
      atualizar();
    }

    btnConv.addEventListener('click', function(){ definirModo('conv'); });
    btnFC.addEventListener('click', function(){ definirModo('fc'); });

    inX1.addEventListener('input', atualizar);
    inW1.addEventListener('input', atualizar);
    inX2.addEventListener('input', atualizar);
    inW2.addEventListener('input', atualizar);
    inB.addEventListener('input', atualizar);
    selF.addEventListener('change', atualizar);

    atualizarLegendas();
    atualizar();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-neuronio');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

Em uma CNN, milhares de neurônios organizam-se em camadas com funções específicas: as primeiras são responsáveis pela extração de características por meio da convolução, e as últimas realizam a classificação a partir das características aprendidas.

### Camada Convolucional

A **camada convolucional** (*convolutional layer*) é responsável pela extração de características da imagem. Cada filtro gera um **mapa de características** (*feature map*), cuja intensidade em cada posição indica a resposta do filtro à região correspondente da entrada.

A operação realizada segue o mesmo princípio de deslocamento e combinação local apresentado no **Capítulo 3** para a convolução espacial. Considerando um *kernel* $K$ de dimensão $k \times k$, o valor produzido na posição $(i,j)$ é dado por

$$
F(i,j)=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1}K(u,v)\,I(i+u,j+v).
$$

Vale registrar uma distinção terminológica: a expressão acima corresponde, formalmente, a uma **correlação cruzada** (*cross-correlation*), e não à convolução matemática estrita, que exige a reflexão do *kernel* antes da combinação. A maioria dos *frameworks* de Aprendizado Profundo, incluindo o **PyTorch**, implementa essa operação sem reflexão e a designa, por convenção, como convolução — convenção adotada também neste capítulo. Essa diferença não tem efeito prático sobre o treinamento, uma vez que os coeficientes do *kernel* são aprendidos e não impostos previamente.

A principal diferença em relação aos métodos clássicos está, assim, na obtenção do *kernel* $K$: em filtros tradicionais, seus coeficientes são definidos manualmente para destacar características específicas da imagem; nas CNNs, os coeficientes são inicializados automaticamente e ajustados durante o treinamento por meio da **retropropagação do erro** (*backpropagation*), o que torna cada filtro especializado em identificar padrões relevantes para a tarefa em estudo.

Dois conceitos caracterizam essa camada:

- **Compartilhamento de pesos** (*weight sharing*): o mesmo filtro é aplicado em todas as posições da imagem, reduzindo significativamente o número de parâmetros do modelo.
- **Campo receptivo** (*receptive field*): cada neurônio convolucional processa apenas uma pequena vizinhança da imagem, preservando a estrutura espacial dos dados.

Ao empilhar várias camadas convolucionais, a rede aprende uma **hierarquia de características**: as primeiras camadas tendem a detectar padrões simples, como bordas e texturas, e as camadas mais profundas combinam essas informações para representar estruturas progressivamente mais complexas. Após a convolução, o mapa de características é submetido a uma função de ativação, introduzindo não linearidade no modelo e ampliando sua capacidade de representar relações complexas entre as variáveis de entrada.

A @fig-09-sim-09-camada-conv apresenta essa camada de forma interativa.

In [ ]:
#| label: fig-09-sim-09-camada-conv
#| fig-cap: "Simulador interativo da Camada Convolucional: escolha entre três imagens de entrada 12×12 (casa, rosto feliz ou rosto triste) para observar como os mesmos *kernels* fixos reagem a diferentes bordas e formas. Navegue pelo campo receptivo com os botões ou reprodução automática, ajuste a função de ativação e alterne o zero-padding, acompanhando o mapa de características sendo revelado célula a célula, com o cálculo detalhado termo a termo e a fórmula da dimensão de saída em tempo real."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-camada-conv" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-camada-conv .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-camada-conv .cn-grouplabel {
      font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px;
    }
    #sim-09-camada-conv .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-camada-conv .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-camada-conv input[type=range] { accent-color:#2F6F9F; min-width:0; }
    #sim-09-camada-conv .cn-navbtn {
      width:26px;height:26px;border-radius:8px;border:1px solid #E4DCC8;background:#FAFAF7;
      color:#26241D;font-size:12px;cursor:pointer;display:flex;align-items:center;justify-content:center;
      transition:background .15s ease; flex-shrink:0;
    }
    #sim-09-camada-conv .cn-navbtn:hover { background:#F1EAD7; }
    #sim-09-camada-conv .cn-playbtn {
      padding:0 10px;height:26px;border-radius:8px;border:1px solid #2F6F9F;background:#EAF2FA;
      color:#2F6F9F;font-size:10.5px;font-weight:700;cursor:pointer;white-space:nowrap;flex-shrink:0;
    }
    #sim-09-camada-conv .cn-playbtn:hover { background:#DCEEFB; }
    #sim-09-camada-conv .cn-prodcell {
      border-radius:6px;padding:3px 2px;text-align:center;border:1px solid #e5e7eb;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulador: Operação de Convolução & Mapa de Características</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">F(i,j) = f(∑ K(u,v) · I(i+u, j+v))</span>
  </div>

  <div style="padding:12px 14px;background:#FFFFFF;overflow:auto">

    <!-- Seletores: Imagem de Entrada + Kernel, lado a lado para compactar -->
    <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px;">
      <div style="flex:1;min-width:230px;">
        <div class="cn-grouplabel">IMAGEM DE ENTRADA</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09camdconvimg_btnImgCasa" class="cn-modebtn active">🏠 Casa</button>
          <button id="cap09camdconvimg_btnImgFeliz" class="cn-modebtn">😊 Feliz</button>
          <button id="cap09camdconvimg_btnImgTriste" class="cn-modebtn">😢 Triste</button>
        </div>
      </div>
      <div style="flex:1;min-width:280px;">
        <div class="cn-grouplabel">KERNEL (FILTRO FIXO)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09camdconvimg_btnSobelV" class="cn-modebtn active">📐 Vertical</button>
          <button id="cap09camdconvimg_btnSobelH" class="cn-modebtn">📏 Horizontal</button>
          <button id="cap09camdconvimg_btnSharpen" class="cn-modebtn">✨ Nitidez</button>
          <button id="cap09camdconvimg_btnIdentity" class="cn-modebtn">🎯 Identidade</button>
        </div>
      </div>
    </div>

    <!-- Painel de Controles -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-bottom:10px;">
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));gap:10px;align-items:start;">

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Posição do Campo Receptivo</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <button id="cap09camdconvimg_btnAnterior" class="cn-navbtn" title="Passo anterior">◀</button>
            <input type="range" id="cap09camdconvimg_step" min="0" max="8" step="1" value="0" style="flex:1;">
            <button id="cap09camdconvimg_btnProximo" class="cn-navbtn" title="Próximo passo">▶</button>
            <button id="cap09camdconvimg_btnPlay" class="cn-playbtn">⏵ Auto</button>
            <button id="cap09camdconvimg_btnReiniciar" class="cn-navbtn" title="Reiniciar varredura">↺</button>
          </div>
          <div id="cap09camdconvimg_posLabel" class="cn-mono" style="font-size:10px;color:#8A8371;margin-top:4px;"></div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Função de Ativação f(z)</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <select id="cap09camdconvimg_func" style="font-size:11px;padding:5px 6px;border-radius:6px;border:1px solid #ccc;background:#fff;width:100%;">
              <option value="relu">ReLU</option>
              <option value="identity">Identidade (Linear)</option>
              <option value="sigmoid">Sigmoide</option>
            </select>
          </div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Preenchimento (Padding)</label>
          <div style="display:flex;gap:6px;align-items:center;padding-top:3px;">
            <input type="checkbox" id="cap09camdconvimg_padding" style="accent-color:#2F6F9F;">
            <span style="font-size:11px;color:#374151;font-weight:500;">Zero-Padding (p = 1)</span>
          </div>
          <div style="display:flex;gap:8px;align-items:center;margin-top:6px;font-size:9.5px;color:#6b7280;flex-wrap:wrap;">
            <span><span style="display:inline-block;width:10px;height:10px;background:#555;border:1px solid #999;vertical-align:middle;"></span> pixel real</span>
            <span><span style="display:inline-block;width:10px;height:10px;background:#111;border:1px dashed #C98A2E;vertical-align:middle;"></span> margem fixa da imagem (0)</span>
            <span><span style="display:inline-block;width:10px;height:10px;background:#DCEEFB;border:1px dashed #8AB4D8;vertical-align:middle;"></span> padding do algoritmo (0)</span>
          </div>
        </div>

      </div>
      <div id="cap09camdconvimg_notaEscala" style="font-size:10.5px;color:#8A8371;margin-top:8px;padding-top:6px;border-top:1px dashed #E9E3D3;">
        O mesmo kernel desliza sobre toda a imagem reutilizando seus coeficientes (<b>compartilhamento de pesos</b>). Cada imagem já vem cercada por uma margem fixa de 1 pixel de fundo (zeros, contorno tracejado âmbar), isolando a forma nos quatro lados. Escolha uma imagem e um kernel fixo acima, depois use ◀ ▶ ou "Auto" para percorrer o campo receptivo — o <b>Feature Map</b> à direita é preenchido célula a célula, na mesma ordem em que a convolução é calculada (as células ainda não visitadas aparecem como "···").
      </div>
      <div id="cap09camdconvimg_notaFormula" class="cn-mono" style="font-size:10.5px;color:#2F6F9F;margin-top:5px;"></div>
      <div id="cap09camdconvimg_notaOffset" style="font-size:10.5px;color:#8A8371;margin-top:4px;">
        📌 A saída F(i,j) vem do campo receptivo entre (i,j) e (i+2,j+2); seu centro real é (i+1,j+1) — <b>1 linha e 1 coluna abaixo/à direita</b> do índice usado para rotular a célula, sempre nas duas direções. Esse deslocamento só fica visível no eixo em que o kernel diferencia a imagem (por isso o Sobel V parece deslocar só para o lado, e o Sobel H, só para baixo).
      </div>
    </div>

    <!-- Área Gráfica: Esquema da Convolução + Curva de Ativação -->
    <div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <canvas id="cap09camdconvimg_canvasEsquema" style="width:580px;height:260px;"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Ativação em f(z)</div>
        <canvas id="cap09camdconvimg_canvasCurva" style="width:200px;height:190px;"></canvas>
      </div>
    </div>

    <!-- Painel de Cálculo Detalhado -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-top:10px;">
      <div style="font-size:11px;font-weight:600;color:#4b5563;margin-bottom:8px;">🔍 Cálculo Detalhado no Campo Receptivo Atual</div>
      <div style="display:flex;gap:18px;align-items:center;flex-wrap:wrap;">
        <div id="cap09camdconvimg_gradeProdutos" style="display:grid;grid-template-columns:repeat(3,44px);gap:3px;"></div>
        <div id="cap09camdconvimg_expressaoSoma" style="font-size:11px;color:#374151;line-height:1.6;"></div>
      </div>
    </div>

    <!-- Legenda contextual (dinâmica: imagem escolhida + kernel escolhido) -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin-top:10px;">
      <div id="cap09camdconvimg_legendaImagem" style="flex:1;min-width:250px;font-size:11px;line-height:1.5;color:#5b5647;background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:8px 10px;"></div>
      <div id="cap09camdconvimg_legendaKernel" style="flex:1;min-width:250px;font-size:11px;line-height:1.5;color:#5b5647;background:#F1F6FB;border:1px solid #DCEEFB;border-radius:10px;padding:8px 10px;"></div>
    </div>

    <!-- Status numérico estilo terminal -->
    <div id="cap09camdconvimg_valTxt" class="cn-mono" style="text-align:center;font-size:11px;margin-top:10px;color:#7EE7C6;background:#1B2430;padding:8px 10px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;overflow-x:auto;">
      z = 0.00 → y = ReLU(z) = 0.00
    </div>

  </div>
</div>

<script>
(function(){
  function initCamadaConvImg(root){
    if(!root || root.dataset.initConvImg) return;
    root.dataset.initConvImg = "1";

    var canvasEsquema = root.querySelector('#cap09camdconvimg_canvasEsquema');
    var canvasCurva   = root.querySelector('#cap09camdconvimg_canvasCurva');
    var ctxEsquema = canvasEsquema.getContext('2d');
    var ctxCurva   = canvasCurva.getContext('2d');

    function prepararCanvas(canvas, ctx, cssW, cssH){
      var dpr = window.devicePixelRatio || 1;
      canvas.width = cssW * dpr;
      canvas.height = cssH * dpr;
      ctx.scale(dpr, dpr);
    }
    prepararCanvas(canvasEsquema, ctxEsquema, 580, 260);
    prepararCanvas(canvasCurva, ctxCurva, 200, 190);

    var inStep   = root.querySelector('#cap09camdconvimg_step');
    var selF     = root.querySelector('#cap09camdconvimg_func');
    var chkP     = root.querySelector('#cap09camdconvimg_padding');
    var valTxt   = root.querySelector('#cap09camdconvimg_valTxt');
    var posLabel = root.querySelector('#cap09camdconvimg_posLabel');
    var notaFormula = root.querySelector('#cap09camdconvimg_notaFormula');
    var legendaKernel = root.querySelector('#cap09camdconvimg_legendaKernel');
    var legendaImagem = root.querySelector('#cap09camdconvimg_legendaImagem');
    var gradeProdutos = root.querySelector('#cap09camdconvimg_gradeProdutos');
    var expressaoSoma = root.querySelector('#cap09camdconvimg_expressaoSoma');

    var btnSobelV   = root.querySelector('#cap09camdconvimg_btnSobelV');
    var btnSobelH   = root.querySelector('#cap09camdconvimg_btnSobelH');
    var btnSharpen  = root.querySelector('#cap09camdconvimg_btnSharpen');
    var btnIdentity = root.querySelector('#cap09camdconvimg_btnIdentity');

    var btnImgCasa   = root.querySelector('#cap09camdconvimg_btnImgCasa');
    var btnImgFeliz  = root.querySelector('#cap09camdconvimg_btnImgFeliz');
    var btnImgTriste = root.querySelector('#cap09camdconvimg_btnImgTriste');

    var btnAnterior  = root.querySelector('#cap09camdconvimg_btnAnterior');
    var btnProximo   = root.querySelector('#cap09camdconvimg_btnProximo');
    var btnPlay      = root.querySelector('#cap09camdconvimg_btnPlay');
    var btnReiniciar = root.querySelector('#cap09camdconvimg_btnReiniciar');

    var CORES = {
      pos: "#1E8F6F",
      neg: "#C1443A",
      saida: "#2F5FA8",
      textoSuave: "#6b7280",
      padding: "#DCEEFB",
      paddingBorda: "#8AB4D8",
      paddingTexto: "#2F6F9F",
      margemBase: "#C98A2E"
    };

    var KERNELS = {
      sobelV: [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
      sobelH: [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
      sharpen: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
      identity: [[0, 0, 0], [0, 1, 0], [0, 0, 0]]
    };

    var KERNEL_INFO = {
      sobelV: { emoji: "📐", nome: "Sobel Vertical", desc: "responde fortemente a mudanças bruscas de intensidade na direção horizontal — por isso realça <b>bordas verticais</b> da imagem." },
      sobelH: { emoji: "📏", nome: "Sobel Horizontal", desc: "responde a mudanças bruscas de intensidade na direção vertical — por isso realça <b>bordas horizontais</b> da imagem." },
      sharpen: { emoji: "✨", nome: "Nitidez (Sharpen)", desc: "amplifica o pixel central em relação aos vizinhos, aumentando o contraste local e destacando detalhes finos." },
      identity: { emoji: "🎯", nome: "Identidade", desc: "reproduz o valor original do pixel central sem alterá-lo — útil como referência de que a convolução não introduz distorção por si só." }
    };

    // Três imagens de entrada 12×12 desenhadas como "arte ASCII": 'X' = pixel
    // aceso (1.0), '.' = pixel apagado (0.0). Todas com o mesmo tamanho fixo,
    // para que o simulador continue mostrando apenas a operação de convolução
    // (sem qualquer etapa de classificação).
    var IMAGENS = {
      casa: {
        emoji: "🏠",
        nome: "Casa",
        desc: "combina bordas diagonais no telhado, bordas verticais retas nas paredes e uma porta recortada no centro. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "...XXXXXX...",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXX..XXXX.",
          ".XXXX..XXXX.",
          ".XXXX..XXXX.",
          "............"
        ]
      },
      feliz: {
        emoji: "😊",
        nome: "Rosto Feliz",
        desc: "um contorno arredondado com dois olhos e uma boca que se abre mais na parte de cima e se fecha em direção ao queixo. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXX.XX.XXX.",
          ".XXXXXXXXXX.",
          ".XX......XX.",
          ".XXX....XXX.",
          ".XXXXXXXXXX.",
          "..XXXXXXXX..",
          "............"
        ]
      },
      triste: {
        emoji: "😢",
        nome: "Rosto Triste",
        desc: "mesmo contorno arredondado do rosto feliz, mas com a boca invertida: mais estreita perto do nariz e mais larga perto do queixo, simulando cantos da boca virados para baixo. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXX.XX.XXX.",
          ".XXXXXXXXXX.",
          ".XXX....XXX.",
          ".XX......XX.",
          ".XXXXXXXXXX.",
          "..XXXXXXXX..",
          "............"
        ]
      }
    };

    var kernelAtual = KERNELS.sobelV;
    var kernelAtualId = "sobelV";
    var imagemAtualId = "casa";
    var imgEntradaBase = null;
    var N_BASE = 12;
    var autoplayInterval = null;

    function converterLinhasParaMatriz(linhas){
      return linhas.map(function(linha){
        var pixels = [];
        for (var i = 0; i < linha.length; i++){
          pixels.push(linha[i] === 'X' ? 1.0 : 0.0);
        }
        return pixels;
      });
    }

    function calcularAtivacao(z, tipo){
      if (tipo === 'relu') return Math.max(0, z);
      if (tipo === 'sigmoid') return 1 / (1 + Math.exp(-z));
      return z;
    }

    // Retorna a matriz de entrada (com ou sem padding) e um mapa booleano
    // indicando quais células são padding artificial (para não confundi-las
    // com pixels reais de valor 0).
    function obterMatrizEntrada(comPadding){
      var dim = comPadding ? N_BASE + 2 : N_BASE;
      var matriz = [], mapaPad = [];
      for (var r = 0; r < dim; r++){
        var linhaVal = [], linhaPad = [];
        for (var c = 0; c < dim; c++){
          if (comPadding && (r === 0 || r === dim - 1 || c === 0 || c === dim - 1)){
            linhaVal.push(0.0);
            linhaPad.push(true);
          } else {
            var ri = comPadding ? r - 1 : r;
            var ci = comPadding ? c - 1 : c;
            linhaVal.push(imgEntradaBase[ri][ci]);
            linhaPad.push(false);
          }
        }
        matriz.push(linhaVal);
        mapaPad.push(linhaPad);
      }
      return { matriz: matriz, pad: mapaPad };
    }

    function desenharEsquema(passoIdx, tipoFunc, comPadding){
      ctxEsquema.clearRect(0, 0, 580, 260);

      var entrada = obterMatrizEntrada(comPadding);
      var img = entrada.matriz, mapaPad = entrada.pad;
      var dimImg = img.length;
      var dimOut = dimImg - 3 + 1;

      var maxSteps = (dimOut * dimOut) - 1;
      inStep.max = maxSteps;
      if (passoIdx > maxSteps) {
        passoIdx = maxSteps;
        inStep.value = maxSteps;
      }
      var rowOut = Math.floor(passoIdx / dimOut);
      var colOut = passoIdx % dimOut;

      var startX = 30, startY = 46;
      var cellSize = comPadding ? 12.5 : 14.5;

      ctxEsquema.textAlign = "center";
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Entrada I (" + dimImg + "×" + dimImg + ")", startX + (dimImg * cellSize) / 2, startY - 16);

      ctxEsquema.font = "7px 'JetBrains Mono', monospace";
      for (var c0 = 0; c0 < dimImg; c0++){
        ctxEsquema.fillText(String(c0), startX + c0 * cellSize + cellSize / 2, startY - 5);
      }
      ctxEsquema.textAlign = "right";
      for (var r0 = 0; r0 < dimImg; r0++){
        ctxEsquema.fillText(String(r0), startX - 5, startY + r0 * cellSize + cellSize / 2 + 3);
      }
      ctxEsquema.textAlign = "center";

      for (var r = 0; r < dimImg; r++) {
        for (var c = 0; c < dimImg; c++) {
          var val = img[r][c];
          var x = startX + c * cellSize, y = startY + r * cellSize;

          if (mapaPad[r][c]) {
            ctxEsquema.fillStyle = CORES.padding;
            ctxEsquema.fillRect(x, y, cellSize, cellSize);
            ctxEsquema.strokeStyle = CORES.paddingBorda;
            ctxEsquema.setLineDash([2, 2]);
            ctxEsquema.lineWidth = 1;
            ctxEsquema.strokeRect(x, y, cellSize, cellSize);
            ctxEsquema.setLineDash([]);
            ctxEsquema.fillStyle = CORES.paddingTexto;
          } else {
            var g = Math.round(val * 255);
            ctxEsquema.fillStyle = "rgb(" + g + "," + g + "," + g + ")";
            ctxEsquema.fillRect(x, y, cellSize, cellSize);

            // Distingue a margem fixa da própria imagem (1px de zeros nos
            // quatro lados, embutida em imgEntradaBase) do padding opcional
            // do algoritmo: mesmo contorno tracejado, mas em âmbar.
            var riBase = comPadding ? r - 1 : r;
            var ciBase = comPadding ? c - 1 : c;
            var ehMargemBase = (riBase === 0 || riBase === N_BASE - 1 || ciBase === 0 || ciBase === N_BASE - 1);

            if (ehMargemBase) {
              ctxEsquema.strokeStyle = CORES.margemBase;
              ctxEsquema.setLineDash([2, 2]);
              ctxEsquema.lineWidth = 1;
              ctxEsquema.strokeRect(x, y, cellSize, cellSize);
              ctxEsquema.setLineDash([]);
            } else {
              ctxEsquema.strokeStyle = "#d1d5db";
              ctxEsquema.lineWidth = 1;
              ctxEsquema.strokeRect(x, y, cellSize, cellSize);
            }
            ctxEsquema.fillStyle = g > 140 ? "#374151" : "#f3f4f6";
          }
          ctxEsquema.font = "600 7px 'JetBrains Mono', monospace";
          ctxEsquema.fillText(val.toFixed(0), x + cellSize / 2, y + cellSize / 2 + 2.5);
        }
      }

      // Destacar Campo Receptivo
      var krX = startX + colOut * cellSize;
      var krY = startY + rowOut * cellSize;
      ctxEsquema.strokeStyle = "#C1443A";
      ctxEsquema.lineWidth = 2.2;
      ctxEsquema.strokeRect(krX, krY, 3 * cellSize, 3 * cellSize);

      // Calcular z, y e os 9 termos do produto no ponto atual
      var termos = [];
      var z = 0;
      for (var kr = 0; kr < 3; kr++) {
        for (var kc = 0; kc < 3; kc++) {
          var iv = img[rowOut + kr][colOut + kc];
          var kv = kernelAtual[kr][kc];
          var prod = iv * kv;
          termos.push({ i: iv, k: kv, p: prod });
          z += prod;
        }
      }
      var y = calcularAtivacao(z, tipoFunc);

      // Desenhar Kernel (K)
      var kCell = 17;
      var kStartX = startX + (dimImg * cellSize) + 16;
      var kStartY = startY + (dimImg * cellSize) / 2 - (3 * kCell) / 2;

      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Kernel K", kStartX + (3 * kCell) / 2, kStartY - 9);

      for (var kr2 = 0; kr2 < 3; kr2++) {
        for (var kc2 = 0; kc2 < 3; kc2++) {
          var kv2 = kernelAtual[kr2][kc2];
          var kx = kStartX + kc2 * kCell, ky = kStartY + kr2 * kCell;
          ctxEsquema.fillStyle = kv2 > 0 ? "#E6F4EA" : (kv2 < 0 ? "#FCE8E6" : "#F3F4F6");
          ctxEsquema.fillRect(kx, ky, kCell, kCell);
          ctxEsquema.strokeStyle = "#9ca3af";
          ctxEsquema.strokeRect(kx, ky, kCell, kCell);

          ctxEsquema.fillStyle = kv2 > 0 ? CORES.pos : (kv2 < 0 ? CORES.neg : "#374151");
          ctxEsquema.font = "600 9px 'JetBrains Mono', monospace";
          ctxEsquema.fillText(String(kv2), kx + kCell / 2, ky + kCell / 2 + 3);
        }
      }

      // Desenhar Feature Map (F) — revelado progressivamente, na mesma ordem
      // (varredura linha a linha) em que a convolução realmente é calculada.
      // Células além do passo atual ainda não foram "computadas" e aparecem
      // como pendentes ("···"), reforçando que o mapa é construído aos poucos.
      var outCell = comPadding ? 13.5 : 16;
      var outStartX = kStartX + 3 * kCell + 52;
      var outStartY = startY + (dimImg * cellSize) / 2 - (dimOut * outCell) / 2;

      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Feature Map F (" + dimOut + "×" + dimOut + ")", outStartX + (dimOut * outCell) / 2, outStartY - 16);

      ctxEsquema.font = "7px 'JetBrains Mono', monospace";
      for (var oc0 = 0; oc0 < dimOut; oc0++){
        ctxEsquema.fillText(String(oc0), outStartX + oc0 * outCell + outCell / 2, outStartY - 5);
      }
      ctxEsquema.textAlign = "right";
      for (var or0 = 0; or0 < dimOut; or0++){
        ctxEsquema.fillText(String(or0), outStartX - 5, outStartY + or0 * outCell + outCell / 2 + 3);
      }
      ctxEsquema.textAlign = "center";

      for (var orr = 0; orr < dimOut; orr++) {
        for (var occ = 0; occ < dimOut; occ++) {
          var linIdx = orr * dimOut + occ;
          var jaCalculado = linIdx <= passoIdx;
          var cx = outStartX + occ * outCell;
          var cy = outStartY + orr * outCell;
          var isAtual = (orr === rowOut && occ === colOut);

          if (!jaCalculado) {
            // Célula ainda pendente: ainda não "visitada" pela varredura.
            ctxEsquema.fillStyle = "#F7F5EE";
            ctxEsquema.fillRect(cx, cy, outCell, outCell);
            ctxEsquema.strokeStyle = "#d9d2bd";
            ctxEsquema.setLineDash([2, 2]);
            ctxEsquema.lineWidth = 1;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);
            ctxEsquema.setLineDash([]);
            ctxEsquema.fillStyle = "#b8ae94";
            ctxEsquema.font = "700 8px 'JetBrains Mono', monospace";
            ctxEsquema.fillText("·", cx + outCell / 2, cy + outCell / 2 + 2.5);
          } else {
            var oz = 0;
            for (var kr3 = 0; kr3 < 3; kr3++) {
              for (var kc3 = 0; kc3 < 3; kc3++) {
                oz += img[orr + kr3][occ + kc3] * kernelAtual[kr3][kc3];
              }
            }
            var oy = calcularAtivacao(oz, tipoFunc);
            var normY = tipoFunc === 'sigmoid' ? oy : Math.max(0, Math.min(1, (oy + 2) / 4));
            var og = Math.round(normY * 255);

            ctxEsquema.fillStyle = "rgb(" + og + "," + og + "," + og + ")";
            ctxEsquema.fillRect(cx, cy, outCell, outCell);
            ctxEsquema.strokeStyle = isAtual ? CORES.saida : "#d1d5db";
            ctxEsquema.lineWidth = isAtual ? 2.4 : 1;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);

            ctxEsquema.fillStyle = og > 140 ? "#374151" : "#f3f4f6";
            ctxEsquema.font = "600 6.5px 'JetBrains Mono', monospace";
            ctxEsquema.fillText(oy.toFixed(1), cx + outCell / 2, cy + outCell / 2 + 2.2);
          }

          if (isAtual) {
            ctxEsquema.strokeStyle = CORES.saida;
            ctxEsquema.lineWidth = 2.4;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);
          }
        }
      }

      return { z: z, y: y, posR: rowOut, posC: colOut, dimImg: dimImg, dimOut: dimOut, termos: termos, p: comPadding ? 1 : 0 };
    }

    function desenharCurva(z, y, tipo){
      var W = 200, H = 190;
      var origemX = 100, origemY = 135;
      var escalaX = 17, escalaY = 32;

      ctxCurva.clearRect(0,0,W,H);

      ctxCurva.strokeStyle = "#EFEAdd"; ctxCurva.lineWidth = 1;
      for (var gx = 10; gx <= W-10; gx += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(gx,10); ctxCurva.lineTo(gx,H-10); ctxCurva.stroke(); }
      for (var gy = 10; gy <= H-10; gy += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(10,gy); ctxCurva.lineTo(W-10,gy); ctxCurva.stroke(); }

      ctxCurva.strokeStyle = "#9ca3af"; ctxCurva.lineWidth = 1.2;
      ctxCurva.beginPath();
      ctxCurva.moveTo(10, origemY); ctxCurva.lineTo(W-10, origemY);
      ctxCurva.moveTo(origemX, 10); ctxCurva.lineTo(origemX, H-10);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280"; ctxCurva.font = "9.5px Inter, sans-serif"; ctxCurva.textAlign = "left";
      ctxCurva.fillText("z", W-16, origemY-5);
      ctxCurva.fillText("y", origemX+5, 14);

      ctxCurva.strokeStyle = "#2F5FA8"; ctxCurva.lineWidth = 2.2;
      ctxCurva.beginPath();
      var primeiro = true;
      for (var px = -5; px <= 5; px += 0.1) {
        var valY = calcularAtivacao(px, tipo);
        var cx = origemX + px * escalaX;
        var cy = origemY - valY * escalaY;
        if (primeiro) { ctxCurva.moveTo(cx, cy); primeiro = false; }
        else { ctxCurva.lineTo(cx, cy); }
      }
      ctxCurva.stroke();

      var ptX = Math.max(10, Math.min(W-10, origemX + z * escalaX));
      var ptY = Math.max(10, Math.min(H-10, origemY - y * escalaY));

      ctxCurva.strokeStyle = "#C1443A"; ctxCurva.setLineDash([3,3]); ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(ptX, origemY); ctxCurva.lineTo(ptX, ptY); ctxCurva.lineTo(origemX, ptY);
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);

      ctxCurva.fillStyle = "#C1443A";
      ctxCurva.beginPath(); ctxCurva.arc(ptX, ptY, 4.5, 0, 2*Math.PI); ctxCurva.fill();
      ctxCurva.strokeStyle = "#fff"; ctxCurva.lineWidth = 1.4; ctxCurva.stroke();
    }

    function atualizarPainelCalculo(res){
      var htmlGrade = '';
      for (var idx = 0; idx < res.termos.length; idx++) {
        var t = res.termos[idx];
        var corFundo = t.p > 0 ? "#E6F4EA" : (t.p < 0 ? "#FCE8E6" : "#F3F4F6");
        var corTxt = t.p > 0 ? CORES.pos : (t.p < 0 ? CORES.neg : "#374151");
        htmlGrade += '<div class="cn-prodcell" style="background:' + corFundo + ';">' +
          '<div class="cn-mono" style="font-size:9px;color:#6b7280;">' + t.i.toFixed(0) + '×' + t.k + '</div>' +
          '<div class="cn-mono" style="font-size:11px;font-weight:700;color:' + corTxt + ';">' + t.p.toFixed(0) + '</div></div>';
      }
      gradeProdutos.innerHTML = htmlGrade;

      var partes = res.termos.map(function(t){
        return t.p >= 0 ? t.p.toFixed(0) : '(' + t.p.toFixed(0) + ')';
      });
      var nomeFunc = selF.options[selF.selectedIndex].text;
      var htmlExpr = '<div class="cn-mono">z = ' + partes.join(' + ') + ' = <b>' + res.z.toFixed(2) + '</b></div>' +
        '<div class="cn-mono" style="margin-top:4px;">y = ' + nomeFunc + '(z) = <b>' + res.y.toFixed(2) + '</b></div>';
      if (res.p) {
        htmlExpr += '<div style="margin-top:6px;color:#2F6F9F;font-size:10.5px;">💡 Termos com fundo azul tracejado no diagrama vêm de <b>padding</b> — zeros adicionados artificialmente na borda, que não fazem parte da imagem original.</div>';
      }
      expressaoSoma.innerHTML = htmlExpr;
    }

    function atualizar(){
      var passoIdx = parseInt(inStep.value);
      var tipo = selF.value;
      var comPadding = chkP.checked;

      var res = desenharEsquema(passoIdx, tipo, comPadding);
      desenharCurva(res.z, res.y, tipo);
      atualizarPainelCalculo(res);

      var maxSteps = res.dimOut * res.dimOut - 1;
      var passoAtual = res.posR * res.dimOut + res.posC;
      posLabel.textContent = 'Passo ' + (passoAtual + 1) + ' de ' + (maxSteps + 1) +
        '  •  posição (i=' + res.posR + ', j=' + res.posC + ')';

      var nomeFunc = selF.options[selF.selectedIndex].text;
      valTxt.textContent = 'Posição (' + res.posR + ',' + res.posC + '): z = ' + res.z.toFixed(2) +
                           '  →  y = ' + nomeFunc + '(z) = ' + res.y.toFixed(2);

      notaFormula.textContent = '📏 Dimensão da saída: n_saída = (n + 2p − k)/s + 1 = (' + N_BASE + ' + 2×' + res.p + ' − 3)/1 + 1 = ' + res.dimOut;

      var infoKernel = KERNEL_INFO[kernelAtualId];
      legendaKernel.innerHTML = '<b>' + infoKernel.emoji + ' ' + infoKernel.nome + ':</b> este filtro ' + infoKernel.desc;

      var infoImagem = IMAGENS[imagemAtualId];
      legendaImagem.innerHTML = '<b>' + infoImagem.emoji + ' ' + infoImagem.nome + ':</b> ' + infoImagem.desc;
    }

    function pararAutoplay(){
      if (autoplayInterval) {
        clearInterval(autoplayInterval);
        autoplayInterval = null;
        btnPlay.textContent = '⏵ Auto';
      }
    }

    function setKernel(k, id, btn){
      [btnSobelV, btnSobelH, btnSharpen, btnIdentity].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      kernelAtual = k;
      kernelAtualId = id;
      pararAutoplay();
      atualizar();
    }

    function setImagem(id, btn){
      [btnImgCasa, btnImgFeliz, btnImgTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      imagemAtualId = id;
      imgEntradaBase = converterLinhasParaMatriz(IMAGENS[id].linhas);
      pararAutoplay();
      atualizar();
    }

    btnSobelV.addEventListener('click', function(){ setKernel(KERNELS.sobelV, 'sobelV', btnSobelV); });
    btnSobelH.addEventListener('click', function(){ setKernel(KERNELS.sobelH, 'sobelH', btnSobelH); });
    btnSharpen.addEventListener('click', function(){ setKernel(KERNELS.sharpen, 'sharpen', btnSharpen); });
    btnIdentity.addEventListener('click', function(){ setKernel(KERNELS.identity, 'identity', btnIdentity); });

    btnImgCasa.addEventListener('click', function(){ setImagem('casa', btnImgCasa); });
    btnImgFeliz.addEventListener('click', function(){ setImagem('feliz', btnImgFeliz); });
    btnImgTriste.addEventListener('click', function(){ setImagem('triste', btnImgTriste); });

    inStep.addEventListener('input', function(){ pararAutoplay(); atualizar(); });
    selF.addEventListener('change', function(){ pararAutoplay(); atualizar(); });
    chkP.addEventListener('change', function(){ pararAutoplay(); atualizar(); });

    btnAnterior.addEventListener('click', function(){
      pararAutoplay();
      var max = parseInt(inStep.max);
      var v = parseInt(inStep.value) - 1;
      inStep.value = v < 0 ? max : v;
      atualizar();
    });
    btnProximo.addEventListener('click', function(){
      pararAutoplay();
      var max = parseInt(inStep.max);
      var v = parseInt(inStep.value) + 1;
      inStep.value = v > max ? 0 : v;
      atualizar();
    });
    btnPlay.addEventListener('click', function(){
      if (autoplayInterval) { pararAutoplay(); return; }
      btnPlay.textContent = '⏸ Pausar';
      autoplayInterval = setInterval(function(){
        var max = parseInt(inStep.max);
        var v = parseInt(inStep.value) + 1;
        if (v > max) { pararAutoplay(); v = max; }
        inStep.value = v;
        atualizar();
      }, 130);
    });
    btnReiniciar.addEventListener('click', function(){
      pararAutoplay();
      inStep.value = 0;
      atualizar();
    });

    imgEntradaBase = converterLinhasParaMatriz(IMAGENS[imagemAtualId].linhas);
    atualizar();
  }

  function tryInitCamadaConvImg(){
    var root = document.getElementById('sim-09-camada-conv');
    if(root) initCamadaConvImg(root); else setTimeout(tryInitCamadaConvImg, 200);
  }
  tryInitCamadaConvImg();
})();
</script>
''')

### Função de Ativação

A convolução é uma operação linear. Para que a rede possa modelar relações não lineares entre entradas e saídas, aplica-se uma **função de ativação** (*activation function*) após cada camada convolucional.

A função mais utilizada em CNNs é a **ReLU** (*Rectified Linear Unit*), definida por

$$
\mathrm{ReLU}(x)=\max(0,x).
$$

Essa função preserva os valores positivos e substitui por zero os valores negativos, introduzindo não linearidade no modelo e favorecendo o treinamento de redes profundas com baixo custo computacional.

A @fig-09-sim-09-relu ilustra o funcionamento da **ReLU** aplicada tanto a valores individuais quanto a um mapa de características, permitindo comparar a saída antes e depois da ativação.


In [ ]:
#| label: fig-09-sim-09-relu
#| fig-cap: "Simulador interativo da função de ativação ReLU: arraste o controle para ver como valores negativos são zerados e valores positivos são preservados, tanto na curva quanto em um mapa de características real."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-relu" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>⚡ Simulador: Função de Ativação ReLU</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">ReLU(x) = max(0, x)</span>
  </div>
  <div style="padding:20px;background:white;overflow:auto">

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:10px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:600;color:#374151;">x =</span>
        <input type="range" id="cap09relu_slider" min="-5" max="5" step="0.1" value="-2.5" style="flex:1;min-width:160px;">
        <span id="cap09relu_valTxt" style="font-family:monospace;font-size:12px;min-width:190px;color:#374151;">x = -2.50  →  ReLU(x) = 0.00</span>
      </div>
    </div>

    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Curva da função ReLU</div>
        <canvas id="cap09relu_canvasCurva" width="280" height="220"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Mapa de características: antes / depois</div>
        <canvas id="cap09relu_canvasMapa" width="260" height="220"></canvas>
        <div style="display:flex;gap:8px;justify-content:center;margin-top:8px;">
          <button id="cap09relu_btnAplicar" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #16a34a;background:#16a34a;color:#fff;cursor:pointer;">Aplicar ReLU ao mapa</button>
          <button id="cap09relu_btnReset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">↺ Resetar</button>
        </div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  var cap09relu_MAPA = [
    [ 1.2, -0.8,  3.4, -2.1, 0.5],
    [-1.5,  2.7, -0.3,  1.1, -4.0],
    [ 0.9, -2.9,  4.8, -0.6,  2.2],
    [-3.3,  0.2, -1.1,  3.9, -0.4],
    [ 2.0, -1.7,  0.8, -2.6,  1.4]
  ];

  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var aplicado = false;

    var ctxCurva = root.querySelector('#cap09relu_canvasCurva').getContext('2d');
    var ctxMapa  = root.querySelector('#cap09relu_canvasMapa').getContext('2d');
    var slider   = root.querySelector('#cap09relu_slider');
    var valTxt   = root.querySelector('#cap09relu_valTxt');

    var W = 280, H = 220;
    var origemX = 40, origemY = H - 30;
    var escala = 22;

    function xParaPixel(x){ return origemX + x*escala; }
    function yParaPixel(y){ return origemY - y*escala; }

    function desenharCurva(x){
      ctxCurva.clearRect(0,0,W,H);

      ctxCurva.strokeStyle = "#9ca3af";
      ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(0, origemY); ctxCurva.lineTo(W, origemY);
      ctxCurva.moveTo(origemX, 0); ctxCurva.lineTo(origemX, H);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280";
      ctxCurva.font = "10px sans-serif";
      ctxCurva.fillText("x", W-12, origemY-4);
      ctxCurva.fillText("ReLU(x)", origemX+4, 10);

      ctxCurva.strokeStyle = "#4f46e5";
      ctxCurva.lineWidth = 2.5;
      ctxCurva.beginPath();
      ctxCurva.moveTo(xParaPixel(-5), yParaPixel(0));
      ctxCurva.lineTo(xParaPixel(0), yParaPixel(0));
      ctxCurva.lineTo(xParaPixel(5), yParaPixel(5));
      ctxCurva.stroke();

      var y = Math.max(0, x);
      ctxCurva.fillStyle = "#dc2626";
      ctxCurva.beginPath();
      ctxCurva.arc(xParaPixel(x), yParaPixel(y), 5, 0, 2*Math.PI);
      ctxCurva.fill();

      ctxCurva.strokeStyle = "#fca5a5";
      ctxCurva.setLineDash([3,3]);
      ctxCurva.beginPath();
      ctxCurva.moveTo(xParaPixel(x), origemY);
      ctxCurva.lineTo(xParaPixel(x), yParaPixel(y));
      ctxCurva.lineTo(origemX, yParaPixel(y));
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);
    }

    function corValor(v, apl){
      if (apl && v < 0) v = 0;
      if (v < 0){
        var inten = Math.min(1, Math.abs(v)/5);
        var c = Math.round(255 - inten*180);
        return 'rgb('+c+','+c+',255)';
      } else {
        var inten2 = Math.min(1, v/5);
        var c2 = Math.round(255 - inten2*200);
        return 'rgb('+c2+',255,'+c2+')';
      }
    }

    function desenharMapa(){
      var tam = 42, offX = 20, offY = 10;
      ctxMapa.clearRect(0,0,260,220);
      for (var r=0;r<5;r++){
        for (var c=0;c<5;c++){
          var vOrig = cap09relu_MAPA[r][c];
          var v = aplicado ? Math.max(0, vOrig) : vOrig;
          ctxMapa.fillStyle = corValor(vOrig, aplicado);
          ctxMapa.fillRect(offX+c*tam, offY+r*tam, tam, tam);
          ctxMapa.strokeStyle = "#d1d5db";
          ctxMapa.strokeRect(offX+c*tam, offY+r*tam, tam, tam);
          ctxMapa.fillStyle = "#1f2937";
          ctxMapa.font = "10px monospace";
          ctxMapa.textAlign = "center";
          ctxMapa.fillText(v.toFixed(1), offX+c*tam+tam/2, offY+r*tam+tam/2+4);
        }
      }
      ctxMapa.fillStyle = "#6b7280";
      ctxMapa.font = "10px sans-serif";
      ctxMapa.textAlign = "left";
      ctxMapa.fillText(aplicado ? "Depois da ReLU (negativos → 0)" : "Antes da ReLU (valores brutos da convolução)", offX, 215);
    }

    function atualizarSlider(){
      var x = parseFloat(slider.value);
      var y = Math.max(0, x);
      valTxt.textContent = 'x = ' + x.toFixed(2) + '  →  ReLU(x) = ' + y.toFixed(2);
      desenharCurva(x);
    }

    slider.addEventListener('input', atualizarSlider);

    root.querySelector('#cap09relu_btnAplicar').addEventListener('click', function(){
      aplicado = true;
      desenharMapa();
    });
    root.querySelector('#cap09relu_btnReset').addEventListener('click', function(){
      aplicado = false;
      desenharMapa();
    });

    atualizarSlider();
    desenharMapa();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-relu');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


Os mapas de características resultantes da convolução e da ativação preservam a estrutura espacial da imagem. Em muitas arquiteturas, a etapa seguinte reduz sua resolução por meio de uma operação de *pooling*.

### *Pooling*

A camada de ***pooling*** reduz a resolução espacial dos mapas de características, preservando as informações mais relevantes para as etapas seguintes do processamento. A operação mais utilizada é o ***max-pooling***, que seleciona o maior valor em cada janela da imagem:

$$
P(i,j)=\max_{(u,v)\in\text{janela}(i,j)}F(u,v).
$$

Essa redução diminui o custo computacional das camadas subsequentes e torna a representação mais robusta a pequenas variações na posição dos padrões presentes na imagem.

A @fig-09-sim-09-pooling apresenta essa operação sobre um mapa de características de 8×8 pixels, reduzido para 4×4 por janelas de 2×2 com passo igual a 2, alternando entre **max-pooling** e **average-pooling** — que calcula, em vez do máximo, a média dos valores da janela correspondente.

In [ ]:
#| label: fig-09-sim-09-pooling
#| fig-cap: "Simulador interativo de *pooling*: escolha entre *max-pooling* e *average-pooling* e avance passo a passo para observar a redução da resolução espacial do mapa de características."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-pooling" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🔻 Simulador: Pooling</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">janela 2×2, stride 2</span>
  </div>
  <div style="padding:20px;background:white;overflow:auto">

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin-bottom:10px;">
        <span style="font-size:11px;font-weight:600;color:#374151;">Tipo:</span>
        <button id="cap09pool_btnMax" class="cap09pool_active" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #4f46e5;background:#4f46e5;color:#fff;cursor:pointer;">Max-pooling</button>
        <button id="cap09pool_btnAvg" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">Average-pooling</button>
      </div>
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <button id="cap09pool_btnPasso" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #16a34a;background:#16a34a;color:#fff;cursor:pointer;">▶ Avançar 1 Passo</button>
        <button id="cap09pool_btnTudo" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">⏭ Calcular Tudo</button>
        <button id="cap09pool_btnReset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">↺ Resetar</button>
        <span style="font-size:11px;color:#6b7280;">Janela atual: <b id="cap09pool_posTxt">(0, 0)</b> de 4×4</span>
      </div>
      <div id="cap09pool_explicacao" style="font-size:10.5px;color:#6b7280;margin-top:8px;line-height:1.4;">O <b>max-pooling</b> mantém apenas o maior valor de cada janela 2×2, reduzindo a resolução espacial pela metade e preservando as respostas mais fortes do mapa de características.</div>
    </div>

    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Mapa de entrada (8×8) — janela atual destacada</div>
        <canvas id="cap09pool_canvasEntrada" width="240" height="240"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Mapa reduzido (4×4)</div>
        <canvas id="cap09pool_canvasSaida" width="160" height="160"></canvas>
      </div>
    </div>

  </div>
</div>

<style>
  #sim-09-pooling button.cap09pool_active { background: #4f46e5 !important; color: #fff !important; border-color: #4f46e5 !important; }
</style>

<script>
(function(){
  var cap09pool_ENTRADA = [
    [1, 3, 2, 8,  5, 1, 0, 2],
    [4, 6, 1, 2,  3, 9, 1, 0],
    [0, 1, 9, 3,  1, 2, 8, 4],
    [2, 5, 4, 7,  0, 1, 3, 6],
    [3, 8, 1, 0,  6, 2, 5, 1],
    [1, 2, 6, 4,  9, 0, 2, 3],
    [7, 0, 3, 1,  2, 8, 1, 4],
    [2, 4, 1, 5,  3, 1, 6, 9]
  ];
  var MAX_GLOBAL = 9;

  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var tipo = "max";
    var saida = Array.from({length:4}, function(){ return Array(4).fill(null); });
    var pos = {r:0, c:0};

    var ctxEnt = root.querySelector('#cap09pool_canvasEntrada').getContext('2d');
    var ctxSai = root.querySelector('#cap09pool_canvasSaida').getContext('2d');
    var posTxt = root.querySelector('#cap09pool_posTxt');
    var explicacao = root.querySelector('#cap09pool_explicacao');

    function corEscala(v, max){
      var inten = Math.min(1, v/max);
      var c = Math.round(245 - inten*160);
      return 'rgb('+c+','+(c+8)+',255)';
    }

    function desenharEntrada(){
      var tam = 30;
      ctxEnt.clearRect(0,0,240,240);
      for (var r=0;r<8;r++){
        for (var c=0;c<8;c++){
          var v = cap09pool_ENTRADA[r][c];
          ctxEnt.fillStyle = corEscala(v, MAX_GLOBAL);
          ctxEnt.fillRect(c*tam, r*tam, tam, tam);
          ctxEnt.strokeStyle = "#e5e7eb";
          ctxEnt.strokeRect(c*tam, r*tam, tam, tam);
          ctxEnt.fillStyle = "#1f2937";
          ctxEnt.font = "11px monospace";
          ctxEnt.textAlign = "center";
          ctxEnt.fillText(v, c*tam+tam/2, r*tam+tam/2+4);
        }
      }
      if (pos.r < 4){
        ctxEnt.strokeStyle = "#dc2626";
        ctxEnt.lineWidth = 3;
        ctxEnt.strokeRect(pos.c*2*tam, pos.r*2*tam, tam*2, tam*2);
        ctxEnt.lineWidth = 1;
      }
    }

    function desenharSaida(){
      var tam = 40;
      ctxSai.clearRect(0,0,160,160);
      for (var r=0;r<4;r++){
        for (var c=0;c<4;c++){
          var v = saida[r][c];
          ctxSai.fillStyle = (v === null) ? "#f3f4f6" : corEscala(v, MAX_GLOBAL);
          ctxSai.fillRect(c*tam, r*tam, tam, tam);
          ctxSai.strokeStyle = "#e5e7eb";
          ctxSai.strokeRect(c*tam, r*tam, tam, tam);
          if (v !== null){
            ctxSai.fillStyle = "#1f2937";
            ctxSai.font = "11px monospace";
            ctxSai.textAlign = "center";
            ctxSai.fillText(v.toFixed(1), c*tam+tam/2, r*tam+tam/2+4);
          }
        }
      }
      if (pos.r < 4){
        ctxSai.strokeStyle = "#dc2626";
        ctxSai.lineWidth = 2;
        ctxSai.strokeRect(pos.c*tam, pos.r*tam, tam, tam);
        ctxSai.lineWidth = 1;
      }
    }

    function calcularJanela(r, c){
      var vals = [];
      for (var i=0;i<2;i++) for (var j=0;j<2;j++) vals.push(cap09pool_ENTRADA[r*2+i][c*2+j]);
      if (tipo === "max") return Math.max.apply(null, vals);
      return vals.reduce(function(a,b){return a+b;},0) / vals.length;
    }

    function avancarPasso(){
      if (pos.r >= 4) return;
      saida[pos.r][pos.c] = calcularJanela(pos.r, pos.c);
      pos.c++;
      if (pos.c >= 4){ pos.c = 0; pos.r++; }
      render();
    }

    function calcularTudo(){
      while (pos.r < 4) avancarPasso();
    }

    function resetar(){
      saida = Array.from({length:4}, function(){ return Array(4).fill(null); });
      pos = {r:0, c:0};
      render();
    }

    function render(){
      desenharEntrada();
      desenharSaida();
      posTxt.textContent = pos.r < 4 ? '(' + pos.r + ', ' + pos.c + ')' : 'concluído';
    }

    function selecionarTipo(t){
      tipo = t;
      root.querySelector('#cap09pool_btnMax').classList.toggle('cap09pool_active', t === "max");
      root.querySelector('#cap09pool_btnAvg').classList.toggle('cap09pool_active', t === "avg");
      explicacao.innerHTML = t === "max"
        ? "O <b>max-pooling</b> mantém apenas o maior valor de cada janela 2×2, reduzindo a resolução espacial pela metade e preservando as respostas mais fortes do mapa de características."
        : "O <b>average-pooling</b> calcula a média dos quatro valores de cada janela 2×2, suavizando a informação em vez de preservar apenas o pico de resposta.";
      resetar();
    }

    root.querySelector('#cap09pool_btnMax').addEventListener('click', function(){ selecionarTipo("max"); });
    root.querySelector('#cap09pool_btnAvg').addEventListener('click', function(){ selecionarTipo("avg"); });
    root.querySelector('#cap09pool_btnPasso').addEventListener('click', avancarPasso);
    root.querySelector('#cap09pool_btnTudo').addEventListener('click', calcularTudo);
    root.querySelector('#cap09pool_btnReset').addEventListener('click', resetar);

    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-pooling');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


Em conjunto, convolução, função de ativação e *pooling* formam o bloco básico utilizado na construção de uma CNN.

### Treinamento de Redes Neurais: Como as CNNs Aprendem

Uma CNN aprende ajustando automaticamente seus parâmetros — os coeficientes dos filtros convolucionais, os pesos das camadas totalmente conectadas e os vieses (*biases*) — a partir de exemplos rotulados. Esse treinamento é iterativo e envolve três etapas: medir o erro produzido pela rede por meio de uma **função de perda** (*loss function*), calcular como esse erro depende de cada parâmetro por meio da **retropropagação** (*backpropagation*) e atualizar os parâmetros com um **algoritmo de otimização** (*optimizer*).

#### Função de Perda (*Loss Function*)

A **função de perda** (*loss function*) quantifica a diferença entre a previsão da rede e a resposta correta, denominada **verdade de referência** (*ground truth*). O resultado é um escalar $L$: quanto menor a perda, mais próxima a previsão está da resposta esperada.

Em problemas de classificação multiclasse, a função mais utilizada é a **Entropia Cruzada** (*Cross-Entropy Loss*), aplicada às probabilidades produzidas pela camada **Softmax**:

$$
L=-\sum_{c=1}^{C} y_c \log(\hat{y}_c),
$$

em que $C$ é o número de classes, $y_c$ é o rótulo real em codificação *one-hot* e $\hat{y}_c$ é a probabilidade prevista para a classe $c$. A perda aproxima-se de zero quando a rede atribui alta probabilidade à classe correta e cresce rapidamente à medida que essa probabilidade diminui.

A @fig-09-sim-09-loss ilustra esse comportamento: o simulador permite selecionar a classe correta e alterar as probabilidades produzidas pela *Softmax*, mostrando em tempo real a variação da função de perda.



In [ ]:
#| label: fig-09-sim-09-loss
#| fig-cap: "Simulador interativo da Função de Perda (*Cross-Entropy*): selecione a classe real da imagem (Casa, Feliz ou Triste) e ajuste as probabilidades estimadas pela Softmax para visualizar o cálculo da penalização escalar e o gráfico do logaritmo negativo em tempo real."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-loss" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-loss .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-loss .cn-grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-loss .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-loss .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-loss input[type=range] { accent-color:#2F6F9F; width:100%; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">📉 Simulador: Função de Perda (Entropia Cruzada)</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">L = -log(ŷ_alvo)</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletores de Rótulo Real (Ground Truth) -->
    <div style="margin-bottom:12px;">
      <div class="cn-grouplabel">CLASSE REAL DA IMAGEM (GROUND TRUTH: y_c = 1)</div>
      <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;max-width:340px;">
        <button id="cap09loss_btnCasa" class="cn-modebtn active">🏠 Casa</button>
        <button id="cap09loss_btnFeliz" class="cn-modebtn">😊 Feliz</button>
        <button id="cap09loss_btnTriste" class="cn-modebtn">😢 Triste</button>
      </div>
    </div>

    <!-- Ajuste de Probabilidades Preditas (Softmax ŷ) -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:12px;margin-bottom:14px;">
      <div class="cn-grouplabel" style="margin-bottom:8px;">PROBABILIDADES ESTIMADAS PELA SOFTMAX (ŷ_c)</div>
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));gap:12px;">
        
        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>🏠 Casa (ŷ_1):</span>
            <span id="cap09loss_txtProbCasa" class="cn-mono" style="color:#2F6F9F;">0.70</span>
          </div>
          <input type="range" id="cap09loss_rangeCasa" min="0.01" max="0.98" step="0.01" value="0.70">
        </div>

        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>😊 Feliz (ŷ_2):</span>
            <span id="cap09loss_txtProbFeliz" class="cn-mono" style="color:#2F6F9F;">0.20</span>
          </div>
          <input type="range" id="cap09loss_rangeFeliz" min="0.01" max="0.98" step="0.01" value="0.20">
        </div>

        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>😢 Triste (ŷ_3):</span>
            <span id="cap09loss_txtProbTriste" class="cn-mono" style="color:#2F6F9F;">0.10</span>
          </div>
          <input type="range" id="cap09loss_rangeTriste" min="0.01" max="0.98" step="0.01" value="0.10">
        </div>

      </div>
    </div>

    <!-- Curva da Função Logarítmica & Resultado -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:18px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#5E5A4A;">Curva de Penalização L = -log(ŷ_alvo)</div>
        <canvas id="cap09loss_canvasCurva" width="260" height="170" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
      </div>

      <div style="flex:1;min-width:240px;">
        <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:12px;margin-bottom:8px;">
          <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:4px;">CÁLCULO DA PERDA:</div>
          <div id="cap09loss_exprCalc" class="cn-mono" style="font-size:11.5px;color:#374151;line-height:1.6;"></div>
          <div style="margin-top:6px;font-size:14px;font-weight:700;color:#C1443A;">
            Perda L = <span id="cap09loss_valTotal" class="cn-mono">0.3567</span>
          </div>
        </div>
        <div id="cap09loss_explicacao" style="font-size:10.5px;color:#8A8371;line-height:1.4;"></div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function initLoss(root){
    if(!root || root.dataset.initLoss) return;
    root.dataset.initLoss = "1";

    var classeAlvo = "casa"; // "casa", "feliz", "triste"
    var probs = { casa: 0.70, feliz: 0.20, triste: 0.10 };

    var btnCasa   = root.querySelector('#cap09loss_btnCasa');
    var btnFeliz  = root.querySelector('#cap09loss_btnFeliz');
    var btnTriste = root.querySelector('#cap09loss_btnTriste');

    var rangeCasa   = root.querySelector('#cap09loss_rangeCasa');
    var rangeFeliz  = root.querySelector('#cap09loss_rangeFeliz');
    var rangeTriste = root.querySelector('#cap09loss_rangeTriste');

    var txtProbCasa   = root.querySelector('#cap09loss_txtProbCasa');
    var txtProbFeliz  = root.querySelector('#cap09loss_txtProbFeliz');
    var txtProbTriste = root.querySelector('#cap09loss_txtProbTriste');

    var exprCalc   = root.querySelector('#cap09loss_exprCalc');
    var valTotal   = root.querySelector('#cap09loss_valTotal');
    var explicacao = root.querySelector('#cap09loss_explicacao');

    var canvas = root.querySelector('#cap09loss_canvasCurva');
    var ctx    = canvas.getContext('2d');

    function normalizarProbs(modificado){
      var somaOutros = 0;
      var chaves = ["casa", "feliz", "triste"];
      chaves.forEach(function(k){ if(k !== modificado) somaOutros += probs[k]; });
      
      var restante = 1.0 - probs[modificado];
      if(somaOutros > 0){
        chaves.forEach(function(k){
          if(k !== modificado) probs[k] = (probs[k] / somaOutros) * restante;
        });
      } else {
        chaves.forEach(function(k){
          if(k !== modificado) probs[k] = restante / 2.0;
        });
      }

      rangeCasa.value   = probs.casa;
      rangeFeliz.value  = probs.feliz;
      rangeTriste.value = probs.triste;

      txtProbCasa.textContent   = probs.casa.toFixed(2);
      txtProbFeliz.textContent  = probs.feliz.toFixed(2);
      txtProbTriste.textContent = probs.triste.toFixed(2);
    }

    function desenharCurvaLog(){
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0,0,W,H);

      // Eixos
      ctx.strokeStyle = "#E4DCC8"; ctx.lineWidth = 1;
      ctx.beginPath();
      ctx.moveTo(30, 10); ctx.lineTo(30, H-20); ctx.lineTo(W-10, H-20);
      ctx.stroke();

      // Curva -log(x)
      ctx.strokeStyle = "#2F6F9F"; ctx.lineWidth = 2;
      ctx.beginPath();
      for(var x=0.002; x<=0.98; x+=0.01){
        var loss = -Math.log(x);
        var cx = 30 + x * (W - 40);
        var cy = (H - 20) - (loss / 4.0) * (H - 30);
        cy = Math.max(10, Math.min(H-20, cy));
        if(x === 0.002) ctx.moveTo(cx, cy); else ctx.lineTo(cx, cy);
      }
      ctx.stroke();

      // Ponto Atual
      var probAlvo = probs[classeAlvo];
      var lossAlvo = -Math.log(probAlvo);
      var ptX = 30 + probAlvo * (W - 40);
      var ptY = (H - 20) - (lossAlvo / 4.0) * (H - 30);
      ptY = Math.max(10, Math.min(H-20, ptY));

      ctx.strokeStyle = "#C1443A"; ctx.setLineDash([3,3]);
      ctx.beginPath();
      ctx.moveTo(ptX, H-20); ctx.lineTo(ptX, ptY); ctx.lineTo(30, ptY);
      ctx.stroke(); ctx.setLineDash([]);

      ctx.fillStyle = "#C1443A";
      ctx.beginPath(); ctx.arc(ptX, ptY, 4.5, 0, 2*Math.PI); ctx.fill();
    }

    function atualizar(){
      desenharCurvaLog();
      var probAlvo = probs[classeAlvo];
      var lossVal  = -Math.log(probAlvo);

      var nomes = { casa: "🏠 Casa", feliz: "😊 Feliz", triste: "😢 Triste" };
      exprCalc.innerHTML = 'L = -log(ŷ_' + classeAlvo + ') = -log(' + probAlvo.toFixed(2) + ')';
      valTotal.textContent = lossVal.toFixed(4);

      if(probAlvo > 0.8) {
        explicacao.innerHTML = "<b>Excelente precisão:</b> A rede atribuiu alta probabilidade à classe correta (" + nomes[classeAlvo] + "), gerando uma perda muito próxima de zero.";
      } else if(probAlvo > 0.4) {
        explicacao.innerHTML = "<b>Incerteza moderada:</b> A probabilidade da classe correta (" + nomes[classeAlvo] + ") é mediana, resultando em uma penalização moderada sobre a rede.";
      } else {
        explicacao.innerHTML = "<b>Erro alto (Confusão):</b> A rede atribuiu baixa probabilidade à classe real (" + nomes[classeAlvo] + "). A função logarítmica penaliza fortemente esse erro, gerando um alto valor de perda $L$.";
      }
    }

    function selecionarClasse(c, btn){
      [btnCasa, btnFeliz, btnTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      classeAlvo = c;
      atualizar();
    }

    btnCasa.addEventListener('click', function(){ selecionarClasse("casa", btnCasa); });
    btnFeliz.addEventListener('click', function(){ selecionarClasse("feliz", btnFeliz); });
    btnTriste.addEventListener('click', function(){ selecionarClasse("triste", btnTriste); });

    rangeCasa.addEventListener('input', function(){ probs.casa = parseFloat(this.value); normalizarProbs("casa"); atualizar(); });
    rangeFeliz.addEventListener('input', function(){ probs.feliz = parseFloat(this.value); normalizarProbs("feliz"); atualizar(); });
    rangeTriste.addEventListener('input', function(){ probs.triste = parseFloat(this.value); normalizarProbs("triste"); atualizar(); });

    atualizar();
  }

  function tryInitLoss(){
    var root = document.getElementById('sim-09-loss');
    if(root) initLoss(root); else setTimeout(tryInitLoss, 200);
  }
  tryInitLoss();
})();
</script>
''')

#### Retropropagação (*Backpropagation*)

Após o cálculo da perda, é necessário determinar como cada parâmetro da rede contribui para esse resultado. Essa etapa é realizada pela **retropropagação** (*backpropagation*), que aplica a **Regra da Cadeia** do cálculo diferencial para obter o gradiente da função de perda em relação a cada parâmetro.

Para um parâmetro $w$, esse gradiente é dado por

$$
\frac{\partial L}{\partial w}.
$$

O gradiente indica como a perda varia em relação a pequenas alterações em $w$: um gradiente positivo indica que aumentar $w$ aumenta a perda, e um gradiente negativo indica o efeito oposto.

A @fig-09-sim-09-backprop apresenta esse processo de forma visual, mostrando a propagação do gradiente da camada de saída até as primeiras camadas convolucionais.


In [ ]:
#| label: fig-09-sim-09-backprop
#| fig-cap: "Simulador interativo de *Backpropagation*: avance os passos da Regra da Cadeia para acompanhar o fluxo do sinal de erro no sentido inverso da rede, observando a computação das derivadas parciais do gradiente em cada camada."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-backprop" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-backprop .cap09backprop_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-backprop .cap09backprop_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-backprop .cap09backprop_navbtn {
      padding:6px 12px; font-size:11px; font-weight:700; border-radius:8px; border:1px solid #E4DCC8;
      background:#FAF6EC; color:#374151; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-backprop .cap09backprop_navbtn:hover { background:#F1EAD7; }
    
    /* Blocos Interativos do Fluxo */
    #sim-09-backprop .cap09backprop_node {
      flex: 1;
      min-width: 90px;
      padding: 10px 6px;
      border-radius: 10px;
      border: 1px solid #E4DCC8;
      background: #FFFFFF;
      text-align: center;
      transition: all 0.25s ease;
      box-shadow: 0 1px 2px rgba(0,0,0,0.02);
      cursor: pointer;
    }
    #sim-09-backprop .cap09backprop_node_title {
      font-size: 11px;
      font-weight: 700;
      color: #374151;
    }
    #sim-09-backprop .cap09backprop_node_sub {
      font-size: 9px;
      font-weight: 600;
      color: #8A8371;
      margin-top: 3px;
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active {
      border: 2px solid #C1443A;
      background: #FCE8E6;
      box-shadow: 0 3px 8px rgba(193,68,58,0.15);
      transform: translateY(-2px);
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active .cap09backprop_node_title {
      color: #C1443A;
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active .cap09backprop_node_sub {
      color: #A8322A;
    }
    
    /* Seta do Fluxo Reverso */
    #sim-09-backprop .cap09backprop_arrow {
      font-size: 14px;
      font-weight: bold;
      color: #D1D5DB;
      transition: color 0.2s ease;
      padding: 0 2px;
    }
    #sim-09-backprop .cap09backprop_arrow.cap09backprop_active_arrow {
      color: #C1443A;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⬅️ Simulador: Retropropagação (Backpropagation)</span>
    <span class="cap09backprop_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">∂L/∂w = (∂L/∂y) · (∂y/∂z) · (∂z/∂w)</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Controles do Passo a Passo do Backprop -->
    <div style="display:flex;gap:10px;align-items:center;justify-content:space-between;margin-bottom:14px;flex-wrap:wrap;">
      <div style="display:flex;gap:6px;">
        <button id="cap09backprop_btnVoltar" class="cap09backprop_navbtn">◀ Voltar Passo</button>
        <button id="cap09backprop_btnAvancar" class="cap09backprop_navbtn" style="border-color:#C1443A;color:#C1443A;background:#FCE8E6;">Passo Inverso (Backprop) ◀</button>
        <button id="cap09backprop_btnReset" class="cap09backprop_navbtn">↺ Reiniciar</button>
      </div>
      <span class="cap09backprop_mono" id="cap09backprop_txtEtapa" style="font-size:11px;color:#2F6F9F;font-weight:700;">Passo 1 de 4: Saída (Loss & Softmax)</span>
    </div>

    <!-- Fluxo Visual das Camadas (Flexbox em alta resolução) -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:16px 12px;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8A8371;margin-bottom:8px;letter-spacing:.3px;text-align:center;">
        DIREÇÃO DA PROPAGAÇÃO DO ERRO (FLUXO REVERSO ⟵)
      </div>
      <div style="display:flex;align-items:center;justify-content:space-between;gap:4px;max-width:620px;margin:0 auto;">
        
        <div id="cap09backprop_node3" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Conv1 Kernels</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂K</div>
        </div>

        <div id="cap09backprop_arrow2" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node2" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Max-Pooling</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂X_pool</div>
        </div>

        <div id="cap09backprop_arrow1" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node1" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Camadas FC</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂W_fc</div>
        </div>

        <div id="cap09backprop_arrow0" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node0" class="cap09backprop_node cap09backprop_active">
          <div class="cap09backprop_node_title">Loss / Softmax</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂y_pred</div>
        </div>

      </div>
    </div>

    <!-- Painel da Regra da Cadeia Detalhada -->
    <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:12px;padding:14px;">
      <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:6px;display:flex;align-items:center;gap:6px;">
        <span>🔗 Regra da Cadeia na Camada Atual:</span>
      </div>
      <div id="cap09backprop_exprCadeia" class="cap09backprop_mono" style="font-size:12px;font-weight:700;color:#C1443A;line-height:1.6;margin-bottom:8px;background:#FFF;padding:8px 10px;border-radius:8px;border:1px solid #E4DCC8;"></div>
      <div id="cap09backprop_descPasso" style="font-size:11.5px;color:#374151;line-height:1.5;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function cap09backprop_init(cap09backprop_root){
    if(!cap09backprop_root || cap09backprop_root.dataset.initBackprop) return;
    cap09backprop_root.dataset.initBackprop = "1";

    var cap09backprop_passoAtual = 0; // 0: Loss/Softmax, 1: Camada FC, 2: Pooling, 3: Conv1 Kernels

    var cap09backprop_btnVoltar  = cap09backprop_root.querySelector('#cap09backprop_btnVoltar');
    var cap09backprop_btnAvancar = cap09backprop_root.querySelector('#cap09backprop_btnAvancar');
    var cap09backprop_btnReset   = cap09backprop_root.querySelector('#cap09backprop_btnReset');

    var cap09backprop_txtEtapa   = cap09backprop_root.querySelector('#cap09backprop_txtEtapa');
    var cap09backprop_exprCadeia = cap09backprop_root.querySelector('#cap09backprop_exprCadeia');
    var cap09backprop_descPasso  = cap09backprop_root.querySelector('#cap09backprop_descPasso');

    var cap09backprop_ETAPAS = [
      {
        nome: "Passo 1 de 4: Saída (Loss & Softmax)",
        expressao: "∂L/∂y_pred = y_pred - y_real = 0.85 - 1.00 = -0.15",
        desc: "O algoritmo de Backpropagation começa no final do pipeline, calculando a derivada direta da função de perda por Entropia Cruzada em relação à probabilidade gerada pela Softmax."
      },
      {
        nome: "Passo 2 de 4: Camadas Densas (Fully Connected)",
        expressao: "∂L/∂w_fc = (∂L/∂y_pred) · (∂y_pred/∂z_fc) = (-0.15) · (0.42) = -0.063",
        desc: "O sinal de erro retropropaga pelas camadas totalmente conectadas através de multiplicadores de matrizes, definindo quanto cada peso denso contribuiu para o desvio final."
      },
      {
        nome: "Passo 3 de 4: Camada de Max-Pooling",
        expressao: "∂L/∂x_pool = (∂L/∂y_fc) · Mútil_max  ➔  Roteado integralmente para a posição do valor máximo",
        desc: "Na camada de Max-Pooling, não há pesos treináveis. O gradiente é repassado sem alteração exatamente para o pixel que forneceu o valor máximo no Forward Pass, enquanto os demais pixels recebem gradiente zero."
      },
      {
        nome: "Passo 4 de 4: Filtros Convolucionais (Conv1 Kernels)",
        expressao: "∂L/∂K(u,v) = ∑ (∂L/∂F) · I(i+u, j+v)  ➔  Gradiente acumulado do filtro 3×3",
        desc: "O erro atinge os coeficientes numéricos dos filtros originais. Como o mesmo kernel foi reutilizado sobre várias regiões da imagem, os gradientes de todas as posições do campo receptivo são somados para atualizar o filtro."
      }
    ];

    function cap09backprop_atualizarUI(){
      var cap09backprop_info = cap09backprop_ETAPAS[cap09backprop_passoAtual];
      cap09backprop_txtEtapa.textContent = cap09backprop_info.nome;
      cap09backprop_exprCadeia.innerHTML = cap09backprop_info.expressao;
      cap09backprop_descPasso.innerHTML  = cap09backprop_info.desc;

      // Atualizar nós ativos
      for(var cap09backprop_i=0; cap09backprop_i<4; cap09backprop_i++){
        var cap09backprop_node = cap09backprop_root.querySelector('#cap09backprop_node' + cap09backprop_i);
        if(cap09backprop_node){
          if(cap09backprop_i === cap09backprop_passoAtual){
            cap09backprop_node.classList.add('cap09backprop_active');
          } else {
            cap09backprop_node.classList.remove('cap09backprop_active');
          }
        }
      }

      // Atualizar setas ativas
      for(var cap09backprop_j=0; cap09backprop_j<3; cap09backprop_j++){
        var cap09backprop_arrow = cap09backprop_root.querySelector('#cap09backprop_arrow' + cap09backprop_j);
        if(cap09backprop_arrow){
          if(cap09backprop_j < cap09backprop_passoAtual){
            cap09backprop_arrow.classList.add('cap09backprop_active_arrow');
          } else {
            cap09backprop_arrow.classList.remove('cap09backprop_active_arrow');
          }
        }
      }
    }

    // Permitir clicar nos nós diretamente
    for(var cap09backprop_k=0; cap09backprop_k<4; cap09backprop_k++){
      (function(idx){
        var cap09backprop_n = cap09backprop_root.querySelector('#cap09backprop_node' + idx);
        if(cap09backprop_n){
          cap09backprop_n.addEventListener('click', function(){
            cap09backprop_passoAtual = idx;
            cap09backprop_atualizarUI();
          });
        }
      })(cap09backprop_k);
    }

    cap09backprop_btnAvancar.addEventListener('click', function(){
      if(cap09backprop_passoAtual < cap09backprop_ETAPAS.length - 1){
        cap09backprop_passoAtual++;
        cap09backprop_atualizarUI();
      }
    });

    cap09backprop_btnVoltar.addEventListener('click', function(){
      if(cap09backprop_passoAtual > 0){
        cap09backprop_passoAtual--;
        cap09backprop_atualizarUI();
      }
    });

    cap09backprop_btnReset.addEventListener('click', function(){
      cap09backprop_passoAtual = 0;
      cap09backprop_atualizarUI();
    });

    cap09backprop_atualizarUI();
  }

  function cap09backprop_tryInit(){
    var cap09backprop_root = document.getElementById('sim-09-backprop');
    if(cap09backprop_root) cap09backprop_init(cap09backprop_root); else setTimeout(cap09backprop_tryInit, 200);
  }
  cap09backprop_tryInit();
})();
</script>
''')

#### Algoritmos de Otimização

Após o cálculo dos gradientes, um **algoritmo de otimização** (*optimizer*) atualiza os parâmetros da rede para reduzir a função de perda. Em redes profundas, essa busca ocorre em um espaço de alta dimensão e, em geral, **não convexo**, o que torna a otimização um problema desafiador.

Para facilitar a compreensão, a @fig-09-sim-09-opt utiliza uma **superfície de perda simplificada**, com um mínimo global, um mínimo local e uma barreira entre essas regiões. O **mínimo global** corresponde ao menor valor da função de perda e representa o melhor conjunto de parâmetros da rede; um **mínimo local** também apresenta baixa perda, mas pode estar distante da melhor solução. Quando a otimização fica retida em um mínimo local, os ajustes de filtros, pesos e vieses tornam-se muito pequenos, e o treinamento para antes de alcançar um modelo com menor erro.

##### Gradiente Descendente Estocástico (*SGD*)

O **Gradiente Descendente Estocástico** (*Stochastic Gradient Descent* — *SGD*) atualiza os parâmetros na direção oposta ao gradiente:

$$
w_{\text{novo}} = w_{\text{atual}} - \eta \frac{\partial L}{\partial w},
$$

em que $\eta$ é a **taxa de aprendizado** (*learning rate*), responsável por controlar o tamanho da atualização. O *SGD* utiliza apenas o gradiente da iteração atual; quando a busca alcança um mínimo local, os gradientes tornam-se muito pequenos e as atualizações praticamente cessam.

##### Otimizadores Adaptativos: *Adam*

O **Adam** (*Adaptive Moment Estimation*) combina estimativas adaptativas dos primeiros e segundos momentos dos gradientes [@kingma2015adam], adaptando a taxa de aprendizado de cada parâmetro individualmente. Essa adaptação favorece, em muitos casos, a superação de mínimos locais que reteriam o *SGD*.

A @fig-09-sim-09-opt compara a trajetória do *SGD* e do *Adam* sobre a mesma superfície de perda não convexa.

In [ ]:
#| label: fig-09-sim-09-opt
#| fig-cap: "Simulador interativo dos Algoritmos de Otimização: compare a trajetória do SGD e do Adam sobre uma superfície de perda não convexa com mapa de calor e curvas de nível. A linha sólida mostra o otimizador selecionado avançando passo a passo; a linha tracejada mostra, para comparação instantânea, o caminho completo que o outro otimizador percorreria a partir do mesmo ponto inicial. Observe como o SGD fica retido no Mínimo Local à direita, enquanto o Adam pode ou não transpor a barreira central dependendo do impulso acumulado e da taxa de aprendizado. Clique em qualquer ponto do mapa para redefinir o ponto inicial dos pesos."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-opt" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-opt .cap09opt_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-opt .cap09opt_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-opt .cap09opt_modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:8px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-opt .cap09opt_modebtn.cap09opt_active { background:#26241D; color:#FBF7EE; }
    #sim-09-opt .cap09opt_playbtn {
      padding:6px 12px; font-size:11px; font-weight:700; border-radius:8px; border:1px solid #2F6F9F;
      background:#EAF2FA; color:#2F6F9F; cursor:pointer; transition:background .15s ease;
    }
    #sim-09-opt .cap09opt_playbtn:hover { background:#DCEEFB; }
    #sim-09-opt .cap09opt_navbtn {
      font-size:11px; font-weight:600; padding:6px 10px; border-radius:8px;
      border:1px solid #E4DCC8; background:#FFF; color:#26241D; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-opt .cap09opt_navbtn:hover { background:#F1EAD7; }
    #sim-09-opt .cap09opt_infobox {
      background:#EAF2FA; border:1px solid #CFE2F3; border-radius:10px; padding:9px 12px;
      font-size:11px; color:#2c4a63; line-height:1.5; margin-bottom:12px;
    }
    #sim-09-opt .cap09opt_legendrow { display:flex; gap:14px; flex-wrap:wrap; align-items:center; margin-top:10px; font-size:10.5px; color:#5E5A4A; }
    #sim-09-opt .cap09opt_legenditem { display:flex; align-items:center; gap:5px; }
    #sim-09-opt .cap09opt_swatch { width:14px; height:3px; border-radius:2px; display:inline-block; }
    #sim-09-opt .cap09opt_dot { width:9px; height:9px; border-radius:50%; display:inline-block; }
    #sim-09-opt .cap09opt_checklbl { display:flex; align-items:center; gap:5px; font-size:10.5px; font-weight:600; color:#374151; cursor:pointer; user-select:none; }
    #sim-09-opt .cap09opt_statgrid { display:grid; grid-template-columns:1fr 1fr; gap:6px 14px; margin-top:6px; }
    #sim-09-opt .cap09opt_statlbl { font-size:9.5px; color:#8A8371; font-weight:700; letter-spacing:.2px; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚡ Simulador: Otimização com Curvas de Nível (SGD vs. Adam)</span>
    <span class="cap09opt_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Relevo Não Convexo: Mínimo Local vs. Global</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Caixa de contexto didático -->

<!-- Texto atualizado na infobox do simulador -->
<div class="cap09opt_infobox">
  💡 <b>Como ler este mapa:</b> a <b>seta amarela</b> aponta na direção de <b>descida</b> (&minus;∇L), que é o sentido oposto ao vetor gradiente (∇L). O otimizador avança nessa direção para reduzir a perda <i>L</i>(<i>w</i><sub>1</sub>, <i>w</i><sub>2</sub>) até atingir as regiões mais profundas (tons mais escuros).
</div>

    <!-- Seletores de Otimizador e Controles -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;align-items:flex-end;margin-bottom:12px;">
      <div style="flex:1;min-width:200px;">
        <div class="cap09opt_grouplabel">ALGORITMO PRINCIPAL (linha sólida)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09opt_btnSGD" class="cap09opt_modebtn cap09opt_active">SGD (Sem Momento)</button>
          <button id="cap09opt_btnAdam" class="cap09opt_modebtn">Adam (Com Momento)</button>
        </div>
      </div>

      <div style="flex:1;min-width:160px;">
        <div class="cap09opt_grouplabel">TAXA DE APRENDIZADO (η)</div>
        <select id="cap09opt_selLR" style="font-size:11px;padding:5px 8px;border-radius:8px;border:1px solid #E4DCC8;background:#FAF6EC;width:100%;font-weight:600;color:#374151;">
          <option value="0.12" selected>0.12 (Alta)</option>
          <option value="0.05">0.05 (Ideal)</option>
          <option value="0.01">0.01 (Lenta)</option>
        </select>
      </div>

      <div style="display:flex;gap:6px;">
        <button id="cap09opt_btnPasso" class="cap09opt_playbtn">▶ Passo</button>
        <button id="cap09opt_btnAuto" class="cap09opt_playbtn">⏵ Executar Auto</button>
        <button id="cap09opt_btnReset" class="cap09opt_navbtn">↺ Resetar</button>
      </div>
    </div>

    <label class="cap09opt_checklbl">
      <input type="checkbox" id="cap09opt_chkComparar" checked style="accent-color:#2F6F9F;cursor:pointer;">
      👻 Mostrar trajetória-fantasma do outro otimizador (comparação instantânea)
    </label>

    <!-- Mapa Topográfico / Superfície da Perda -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:18px;flex-wrap:wrap;align-items:center;justify-content:center;margin-top:12px;">
      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:6px;color:#5E5A4A;">Mapa de Calor da Perda L(w₁, w₂) — clique para escolher o início</div>
        <canvas id="cap09opt_canvasContorno" style="width:320px;height:240px;border:1px solid #E4DCC8;border-radius:10px;cursor:crosshair;box-shadow:0 2px 4px rgba(0,0,0,0.04);"></canvas>

        <div class="cap09opt_legendrow">
          <span class="cap09opt_legenditem"><span class="cap09opt_dot" style="background:#1E8F6F;"></span> Mínimo Global</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_dot" style="background:#C1443A;"></span> Mínimo Local</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" style="background:#F5B301;"></span> Gradiente (↓ descida)</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" id="cap09opt_legSolidLine" style="background:#C1443A;"></span> Trajetória principal</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" id="cap09opt_legGhostLine" style="background:#2F6F9F;opacity:.5;background-image:repeating-linear-gradient(90deg,#2F6F9F 0 4px,transparent 4px 7px);"></span> Fantasma (outro otimizador)</span>
        </div>
      </div>

      <div style="flex:1;min-width:230px;">
        <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:12px;margin-bottom:10px;">
          <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:4px;">ESTADO DA OTIMIZAÇÃO:</div>
          <div class="cap09opt_mono" style="font-size:11px;color:#374151;">w₁ = <span id="cap09opt_txtW1">1.80</span>, w₂ = <span id="cap09opt_txtW2">0.20</span></div>
          <div class="cap09opt_mono" style="font-size:13px;font-weight:700;color:#C1443A;margin-top:4px;">Loss L = <span id="cap09opt_txtLoss">2.450</span></div>

          <div class="cap09opt_statgrid">
            <div>
              <div class="cap09opt_statlbl">PASSO</div>
              <div class="cap09opt_mono" style="font-size:12px;color:#26241D;font-weight:700;" id="cap09opt_txtPasso">0</div>
            </div>
            <div>
              <div class="cap09opt_statlbl">|∇L| (MAGNITUDE)</div>
              <div class="cap09opt_mono" style="font-size:12px;color:#26241D;font-weight:700;" id="cap09opt_txtGrad">0.000</div>
            </div>
          </div>

          <div id="cap09opt_txtStatusRegiao" class="cap09opt_mono" style="font-size:10px;color:#1E8F6F;margin-top:8px;font-weight:600;">Status: Ponto Inicial</div>
        </div>

        <div id="cap09opt_descOpt" style="font-size:11px;color:#6B7280;line-height:1.55;"></div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function cap09opt_hexToRgb(h){
    var v = parseInt(h.slice(1),16);
    return { r:(v>>16)&255, g:(v>>8)&255, b:v&255 };
  }
  function cap09opt_lerp(a,b,t){ return a + (b-a)*t; }
  function cap09opt_lerpColor(c1,c2,t){
    var A=cap09opt_hexToRgb(c1), B=cap09opt_hexToRgb(c2);
    return "rgb(" + Math.round(cap09opt_lerp(A.r,B.r,t)) + "," + Math.round(cap09opt_lerp(A.g,B.g,t)) + "," + Math.round(cap09opt_lerp(A.b,B.b,t)) + ")";
  }
  var cap09opt_STOPS = [
    { l:0.5, c:"#16202B" },
    { l:1.4, c:"#2F6F9F" },
    { l:2.3, c:"#8FB8C9" },
    { l:3.0, c:"#E8DEC4" },
    { l:3.8, c:"#FBF7EE" }
  ];
  function cap09opt_lossToColor(loss){
    for(var i=0;i<cap09opt_STOPS.length-1;i++){
      var a = cap09opt_STOPS[i], b = cap09opt_STOPS[i+1];
      if(loss >= a.l && loss <= b.l){
        var t = (loss - a.l) / (b.l - a.l);
        return cap09opt_lerpColor(a.c, b.c, t);
      }
    }
    return loss < cap09opt_STOPS[0].l ? cap09opt_STOPS[0].c : cap09opt_STOPS[cap09opt_STOPS.length-1].c;
  }

  function cap09opt_init(cap09opt_root){
    if(!cap09opt_root || cap09opt_root.dataset.initOpt) return;
    cap09opt_root.dataset.initOpt = "1";

    var cap09opt_BETA1 = 0.8, cap09opt_BETA2 = 0.99;

    var cap09opt_algoritmo = "sgd";
    var cap09opt_posInicial = { w1: 1.8, w2: 0.2 };
    var cap09opt_posW = { w1: 1.8, w2: 0.2 };
    var cap09opt_trajetoria = [{ w1: 1.8, w2: 0.2 }];
    var cap09opt_trajFantasma = [];
    var cap09opt_passoAtual = 0;

    var cap09opt_m = { w1: 0, w2: 0 };
    var cap09opt_v = { w1: 0, w2: 0 };
    var cap09opt_tStep = 0;
    var cap09opt_autoInterval = null;

    var cap09opt_canvas = cap09opt_root.querySelector('#cap09opt_canvasContorno');
    var cap09opt_ctx    = cap09opt_canvas.getContext('2d');

    function cap09opt_prepararCanvas(cap09opt_cvs, cap09opt_c, cap09opt_cssW, cap09opt_cssH){
      var cap09opt_dpr = window.devicePixelRatio || 1;
      cap09opt_cvs.width = cap09opt_cssW * cap09opt_dpr;
      cap09opt_cvs.height = cap09opt_cssH * cap09opt_dpr;
      cap09opt_c.scale(cap09opt_dpr, cap09opt_dpr);
    }
    cap09opt_prepararCanvas(cap09opt_canvas, cap09opt_ctx, 320, 240);

    var cap09opt_btnSGD    = cap09opt_root.querySelector('#cap09opt_btnSGD');
    var cap09opt_btnAdam   = cap09opt_root.querySelector('#cap09opt_btnAdam');
    var cap09opt_selLR     = cap09opt_root.querySelector('#cap09opt_selLR');
    var cap09opt_chkComp   = cap09opt_root.querySelector('#cap09opt_chkComparar');

    var cap09opt_btnPasso  = cap09opt_root.querySelector('#cap09opt_btnPasso');
    var cap09opt_btnAuto   = cap09opt_root.querySelector('#cap09opt_btnAuto');
    var cap09opt_btnReset  = cap09opt_root.querySelector('#cap09opt_btnReset');

    var cap09opt_txtW1     = cap09opt_root.querySelector('#cap09opt_txtW1');
    var cap09opt_txtW2     = cap09opt_root.querySelector('#cap09opt_txtW2');
    var cap09opt_txtLoss   = cap09opt_root.querySelector('#cap09opt_txtLoss');
    var cap09opt_txtPasso  = cap09opt_root.querySelector('#cap09opt_txtPasso');
    var cap09opt_txtGrad   = cap09opt_root.querySelector('#cap09opt_txtGrad');
    var cap09opt_txtStatus = cap09opt_root.querySelector('#cap09opt_txtStatusRegiao');
    var cap09opt_descOpt   = cap09opt_root.querySelector('#cap09opt_descOpt');
    var cap09opt_legGhost  = cap09opt_root.querySelector('#cap09opt_legGhostLine');
    var cap09opt_legSolid  = cap09opt_root.querySelector('#cap09opt_legSolidLine');

    function cap09opt_wToPx(w1, w2){
      return { x: 160 + w1 * 55, y: 120 - w2 * 45 };
    }
    function cap09opt_pxToW(x, y){
      return { w1: (x - 160) / 55.0, w2: (120 - y) / 45.0 };
    }

    function cap09opt_calcLoss(w1, w2){
      var gGlobal = 3.0 * Math.exp(-((w1 + 1.5)*(w1 + 1.5)*0.8 + w2*w2*1.5));
      var gLocal  = 1.6 * Math.exp(-((w1 - 1.5)*(w1 - 1.5)*1.2 + w2*w2*1.5));
      var parabola = 0.18 * (w1*w1 + w2*w2);
      return 3.5 - gGlobal - gLocal + parabola;
    }

    function cap09opt_calcGrad(w1, w2){
      var eps = 0.001;
      var l0 = cap09opt_calcLoss(w1, w2);
      var dw1 = (cap09opt_calcLoss(w1 + eps, w2) - l0) / eps;
      var dw2 = (cap09opt_calcLoss(w1, w2 + eps) - l0) / eps;
      return { g1: dw1, g2: dw2 };
    }

    function cap09opt_passoGenerico(algo, estado, lr){
      var grad = cap09opt_calcGrad(estado.w1, estado.w2);
      if(algo === "sgd"){
        estado.w1 -= lr * grad.g1;
        estado.w2 -= lr * grad.g2;
      } else {
        estado.t = (estado.t||0) + 1;
        estado.m1 = cap09opt_BETA1 * (estado.m1||0) + (1-cap09opt_BETA1) * grad.g1;
        estado.m2 = cap09opt_BETA1 * (estado.m2||0) + (1-cap09opt_BETA1) * grad.g2;
        estado.v1 = cap09opt_BETA2 * (estado.v1||0) + (1-cap09opt_BETA2) * (grad.g1*grad.g1);
        estado.v2 = cap09opt_BETA2 * (estado.v2||0) + (1-cap09opt_BETA2) * (grad.g2*grad.g2);
        var mHat1 = estado.m1 / (1 - Math.pow(cap09opt_BETA1, estado.t));
        var mHat2 = estado.m2 / (1 - Math.pow(cap09opt_BETA1, estado.t));
        var vHat1 = estado.v1 / (1 - Math.pow(cap09opt_BETA2, estado.t));
        var vHat2 = estado.v2 / (1 - Math.pow(cap09opt_BETA2, estado.t));
        estado.w1 -= (lr / (Math.sqrt(vHat1) + 1e-4)) * mHat1 * 1.5;
        estado.w2 -= (lr / (Math.sqrt(vHat2) + 1e-4)) * mHat2 * 1.5;
      }
      return grad;
    }

    function cap09opt_computarFantasma(){
      var outroAlgo = cap09opt_algoritmo === "sgd" ? "adam" : "sgd";
      var lr = parseFloat(cap09opt_selLR.value);
      var estado = { w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 };
      var caminho = [{ w1: estado.w1, w2: estado.w2 }];
      for(var i=0; i<150; i++){
        var antesW1 = estado.w1, antesW2 = estado.w2;
        cap09opt_passoGenerico(outroAlgo, estado, lr);
        caminho.push({ w1: estado.w1, w2: estado.w2 });
        var delta = Math.hypot(estado.w1-antesW1, estado.w2-antesW2);
        if(delta < 0.0008 && i > 6) break;
      }
      return caminho;
    }

    function cap09opt_desenharSeta(x, y, ang, comprimento, cor){
      var x2 = x + Math.cos(ang) * comprimento;
      var y2 = y + Math.sin(ang) * comprimento;
      cap09opt_ctx.strokeStyle = cor; cap09opt_ctx.fillStyle = cor; cap09opt_ctx.lineWidth = 2;
      cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(x,y); cap09opt_ctx.lineTo(x2,y2); cap09opt_ctx.stroke();
      var cabeca = 6, angSeta = Math.atan2(y2-y, x2-x);
      cap09opt_ctx.beginPath();
      cap09opt_ctx.moveTo(x2, y2);
      cap09opt_ctx.lineTo(x2 - cabeca*Math.cos(angSeta-0.45), y2 - cabeca*Math.sin(angSeta-0.45));
      cap09opt_ctx.lineTo(x2 - cabeca*Math.cos(angSeta+0.45), y2 - cabeca*Math.sin(angSeta+0.45));
      cap09opt_ctx.closePath(); cap09opt_ctx.fill();
    }

    function cap09opt_desenharMapa(){
      var W = 320, H = 240, gridRes = 5;
      cap09opt_ctx.clearRect(0,0,W,H);

      for(var py=0; py<H; py+=gridRes){
        for(var px=0; px<W; px+=gridRes){
          var wC = cap09opt_pxToW(px + gridRes/2, py + gridRes/2);
          var lC = cap09opt_calcLoss(wC.w1, wC.w2);
          cap09opt_ctx.fillStyle = cap09opt_lossToColor(lC);
          cap09opt_ctx.fillRect(px, py, gridRes+0.5, gridRes+0.5);
        }
      }

      var niveisLoss = [0.8, 1.2, 1.6, 2.0, 2.4, 2.8, 3.2, 3.8];
      for(var nIdx=0; nIdx<niveisLoss.length; nIdx++){
        var alvoL = niveisLoss[nIdx];
        cap09opt_ctx.strokeStyle = "rgba(20, 24, 30, 0.18)";
        cap09opt_ctx.lineWidth = 1;
        for(var qy=0; qy<H; qy+=gridRes){
          for(var qx=0; qx<W; qx+=gridRes){
            var wA = cap09opt_pxToW(qx, qy);
            var lA = cap09opt_calcLoss(wA.w1, wA.w2);
            var wB = cap09opt_pxToW(qx + gridRes, qy);
            var lB = cap09opt_calcLoss(wB.w1, wB.w2);
            var wCc = cap09opt_pxToW(qx, qy + gridRes);
            var lCc = cap09opt_calcLoss(wCc.w1, wCc.w2);
            if((lA <= alvoL && lB >= alvoL) || (lA >= alvoL && lB <= alvoL)){
              cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(qx, qy); cap09opt_ctx.lineTo(qx + gridRes, qy); cap09opt_ctx.stroke();
            }
            if((lA <= alvoL && lCc >= alvoL) || (lA >= alvoL && lCc <= alvoL)){
              cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(qx, qy); cap09opt_ctx.lineTo(qx, qy + gridRes); cap09opt_ctx.stroke();
            }
          }
        }
      }

      cap09opt_ctx.strokeStyle = "rgba(255,255,255,0.35)";
      cap09opt_ctx.lineWidth = 1;
      cap09opt_ctx.beginPath();
      cap09opt_ctx.moveTo(0, 120); cap09opt_ctx.lineTo(W, 120);
      cap09opt_ctx.moveTo(160, 0); cap09opt_ctx.lineTo(160, H);
      cap09opt_ctx.stroke();

      cap09opt_ctx.font = "600 9px 'JetBrains Mono', monospace";
      cap09opt_ctx.fillStyle = "rgba(38,36,29,0.55)";
      cap09opt_ctx.textAlign = "center";
      for(var wv=-2; wv<=2; wv++){
        if(wv===0) continue;
        var px1 = cap09opt_wToPx(wv, 0);
        cap09opt_ctx.fillText(wv.toString(), px1.x, 132);
        var py1 = cap09opt_wToPx(0, wv*0.9);
        cap09opt_ctx.fillText(wv.toString(), 172, py1.y+3);
      }
      cap09opt_ctx.font = "700 10px 'Inter', sans-serif";
      cap09opt_ctx.fillText("w₁ →", 300, 134);
      cap09opt_ctx.save(); cap09opt_ctx.translate(150, 14); cap09opt_ctx.fillText("w₂ ↑", 0, 0); cap09opt_ctx.restore();

      var posGlobal = cap09opt_wToPx(-1.4, 0);
      var posLocal  = cap09opt_wToPx(1.3, 0);

      [ [posGlobal, "#1E8F6F", "Mínimo Global ★"], [posLocal, "#C1443A", "Mínimo Local ⚠️"] ].forEach(function(item){
        var p = item[0];
        cap09opt_ctx.fillStyle = "rgba(255,255,255,0.65)";
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(p.x, p.y, 8, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.fillStyle = item[1];
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(p.x, p.y, 4.5, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.strokeStyle = "#FFF"; cap09opt_ctx.lineWidth = 1.5; cap09opt_ctx.stroke();
        cap09opt_ctx.font = "700 9px 'JetBrains Mono', monospace";
        cap09opt_ctx.fillStyle = "#26241D";
        cap09opt_ctx.textAlign = "center";
        cap09opt_ctx.fillText(item[2], p.x, p.y - 12);
      });

      if(cap09opt_chkComp.checked && cap09opt_trajFantasma.length > 1){
        var corFantasma = cap09opt_algoritmo === "sgd" ? "#2F6F9F" : "#C1443A";
        cap09opt_ctx.save();
        cap09opt_ctx.setLineDash([5,4]);
        cap09opt_ctx.strokeStyle = corFantasma;
        cap09opt_ctx.globalAlpha = 0.55;
        cap09opt_ctx.lineWidth = 2;
        cap09opt_ctx.beginPath();
        for(var fi=0; fi<cap09opt_trajFantasma.length; fi++){
          var fp = cap09opt_wToPx(cap09opt_trajFantasma[fi].w1, cap09opt_trajFantasma[fi].w2);
          if(fi===0) cap09opt_ctx.moveTo(fp.x, fp.y); else cap09opt_ctx.lineTo(fp.x, fp.y);
        }
        cap09opt_ctx.stroke();
        cap09opt_ctx.restore();
        var fEnd = cap09opt_wToPx(cap09opt_trajFantasma[cap09opt_trajFantasma.length-1].w1, cap09opt_trajFantasma[cap09opt_trajFantasma.length-1].w2);
        cap09opt_ctx.fillStyle = corFantasma; cap09opt_ctx.globalAlpha = 0.7;
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(fEnd.x, fEnd.y, 4, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.globalAlpha = 1;
      }

      if(cap09opt_trajetoria.length > 1){
        var corPrincipal = cap09opt_algoritmo === "sgd" ? "#C1443A" : "#2F6F9F";
        cap09opt_ctx.strokeStyle = corPrincipal;
        cap09opt_ctx.lineWidth = 2.5;
        cap09opt_ctx.beginPath();
        for(var i=0; i<cap09opt_trajetoria.length; i++){
          var p = cap09opt_wToPx(cap09opt_trajetoria[i].w1, cap09opt_trajetoria[i].w2);
          if(i === 0) cap09opt_ctx.moveTo(p.x, p.y); else cap09opt_ctx.lineTo(p.x, p.y);
        }
        cap09opt_ctx.stroke();
        for(var j=0; j<cap09opt_trajetoria.length; j++){
          var pj = cap09opt_wToPx(cap09opt_trajetoria[j].w1, cap09opt_trajetoria[j].w2);
          cap09opt_ctx.fillStyle = corPrincipal;
          cap09opt_ctx.beginPath(); cap09opt_ctx.arc(pj.x, pj.y, 2, 0, 2*Math.PI); cap09opt_ctx.fill();
        }
      }

      var pInicio = cap09opt_wToPx(cap09opt_posInicial.w1, cap09opt_posInicial.w2);
      cap09opt_ctx.fillStyle = "#FBF7EE";
      cap09opt_ctx.beginPath(); cap09opt_ctx.arc(pInicio.x, pInicio.y, 7, 0, 2*Math.PI); cap09opt_ctx.fill();
      cap09opt_ctx.strokeStyle = "#26241D"; cap09opt_ctx.lineWidth = 1.5; cap09opt_ctx.stroke();
      cap09opt_ctx.fillStyle = "#26241D"; cap09opt_ctx.font = "700 8px 'JetBrains Mono', monospace";
      cap09opt_ctx.textAlign = "center"; cap09opt_ctx.textBaseline = "middle";
      cap09opt_ctx.fillText("S", pInicio.x, pInicio.y);

      // Ponto atual + seta do gradiente DESCENDENTE (CORRIGIDO)
      var ptAtual = cap09opt_wToPx(cap09opt_posW.w1, cap09opt_posW.w2);
      var gradAtual = cap09opt_calcGrad(cap09opt_posW.w1, cap09opt_posW.w2);
      
      // Mapeia 1 passo na direção de DESCIDA ( - gradiente )
      var ptDescida = cap09opt_wToPx(cap09opt_posW.w1 - gradAtual.g1 * 0.2, cap09opt_posW.w2 - gradAtual.g2 * 0.2);
      var angTela = Math.atan2(ptDescida.y - ptAtual.y, ptDescida.x - ptAtual.x);
      var magGrad = Math.hypot(gradAtual.g1, gradAtual.g2);

      if(magGrad > 0.01){
        cap09opt_desenharSeta(ptAtual.x, ptAtual.y, angTela, 22, "#F5B301");
      }

      cap09opt_ctx.fillStyle = "#26241D";
      cap09opt_ctx.beginPath(); cap09opt_ctx.arc(ptAtual.x, ptAtual.y, 6, 0, 2*Math.PI); cap09opt_ctx.fill();
      cap09opt_ctx.strokeStyle = "#FFFFFF"; cap09opt_ctx.lineWidth = 2; cap09opt_ctx.stroke();

      return magGrad;
    }

    function cap09opt_darPasso(){
      var lr = parseFloat(cap09opt_selLR.value);
      var estadoTmp = { w1: cap09opt_posW.w1, w2: cap09opt_posW.w2, m1: cap09opt_m.w1, m2: cap09opt_m.w2, v1: cap09opt_v.w1, v2: cap09opt_v.w2, t: cap09opt_tStep };
      cap09opt_passoGenerico(cap09opt_algoritmo, estadoTmp, lr);
      cap09opt_posW.w1 = estadoTmp.w1; cap09opt_posW.w2 = estadoTmp.w2;
      cap09opt_m.w1 = estadoTmp.m1||0; cap09opt_m.w2 = estadoTmp.m2||0;
      cap09opt_v.w1 = estadoTmp.v1||0; cap09opt_v.w2 = estadoTmp.v2||0;
      cap09opt_tStep = estadoTmp.t||0;

      cap09opt_trajetoria.push({ w1: cap09opt_posW.w1, w2: cap09opt_posW.w2 });
      cap09opt_passoAtual++;
      cap09opt_atualizar();
    }

    function cap09opt_atualizar(){
      var magGrad = cap09opt_desenharMapa();
      var loss = cap09opt_calcLoss(cap09opt_posW.w1, cap09opt_posW.w2);
      cap09opt_txtW1.textContent   = cap09opt_posW.w1.toFixed(2);
      cap09opt_txtW2.textContent   = cap09opt_posW.w2.toFixed(2);
      cap09opt_txtLoss.textContent = loss.toFixed(3);
      cap09opt_txtPasso.textContent = cap09opt_passoAtual;
      cap09opt_txtGrad.textContent = magGrad.toFixed(3);

      var lrVal = parseFloat(cap09opt_selLR.value);
      var conseguiuEscapar = cap09opt_posW.w1 < -0.5;
      var estaPresoLocal = cap09opt_posW.w1 > 0.5;
      var corPrincipal = cap09opt_algoritmo === "sgd" ? "#C1443A" : "#2F6F9F";
      cap09opt_legSolid.style.background = corPrincipal;
      var corFantasma = cap09opt_algoritmo === "sgd" ? "#2F6F9F" : "#C1443A";
      cap09opt_legGhost.style.backgroundImage = "repeating-linear-gradient(90deg,"+corFantasma+" 0 4px,transparent 4px 7px)";

      if(estaPresoLocal){
        cap09opt_txtStatus.textContent = "Região: Preso no Mínimo Local ⚠️";
        cap09opt_txtStatus.style.color = "#C1443A";
      } else if(conseguiuEscapar){
        cap09opt_txtStatus.textContent = "Região: Convergiu para Mínimo Global ★";
        cap09opt_txtStatus.style.color = "#1E8F6F";
      } else {
        cap09opt_txtStatus.textContent = "Região: Aclive / Transposição de Barreira";
        cap09opt_txtStatus.style.color = "#2F6F9F";
      }

      if(cap09opt_algoritmo === "sgd"){
        cap09opt_descOpt.innerHTML = "<b>SGD (Sem Momento):</b> a cada passo, o SGD olha apenas para o gradiente <i>local e instantâneo</i> — sem memória do que veio antes. Por isso, ao partir do lado direito, ele fica <b>preso no Mínimo Local</b>: não tem energia acumulada para subir o aclive até a barreira central. Compare com a linha tracejada azul (Adam) ao lado.";
      } else {
        if(conseguiuEscapar){
          cap09opt_descOpt.innerHTML = "<b>Adam (Sucesso):</b> o Adam acumula <i>momento</i> (uma média móvel dos gradientes recentes) e ajusta a taxa de cada peso adaptativamente. Com η=" + lrVal + ", esse impulso acumulado foi suficiente para vencer a barreira e alcançar o <b>Mínimo Global ★</b>. Note como a linha tracejada vermelha (SGD) fica presa antes disso.";
        } else if(estaPresoLocal && cap09opt_trajetoria.length > 8){
          cap09opt_descOpt.innerHTML = "<b>Adam (Retido no Mínimo Local):</b> mesmo acumulando momento, com η=" + lrVal + " o impulso não foi suficiente para transpor a elevação. <i>Isso mostra que nem mesmo o Adam garante escapar de poços profundos sem ajuste fino da taxa de aprendizado ou de uma inicialização melhor.</i>";
        } else {
          cap09opt_descOpt.innerHTML = "<b>Adam (Em movimento):</b> acumulando momento e ajustando o tamanho do passo adaptativamente conforme percorre o relevo...";
        }
      }
    }

    function cap09opt_resetar(){
      if(cap09opt_autoInterval) { clearInterval(cap09opt_autoInterval); cap09opt_autoInterval = null; cap09opt_btnAuto.textContent = "⏵ Executar Auto"; }
      cap09opt_posW = { w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 };
      cap09opt_trajetoria = [{ w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 }];
      cap09opt_m = { w1: 0, w2: 0 }; cap09opt_v = { w1: 0, w2: 0 }; cap09opt_tStep = 0;
      cap09opt_passoAtual = 0;
      cap09opt_trajFantasma = cap09opt_computarFantasma();
      cap09opt_atualizar();
    }

    cap09opt_canvas.addEventListener('click', function(evt){
      var rect = cap09opt_canvas.getBoundingClientRect();
      var scaleX = 320 / rect.width, scaleY = 240 / rect.height;
      var clickX = (evt.clientX - rect.left) * scaleX;
      var clickY = (evt.clientY - rect.top) * scaleY;
      var ptW = cap09opt_pxToW(clickX, clickY);
      cap09opt_posInicial = { w1: ptW.w1, w2: ptW.w2 };
      cap09opt_resetar();
    });

    cap09opt_btnSGD.addEventListener('click', function(){
      cap09opt_btnSGD.classList.add('cap09opt_active'); cap09opt_btnAdam.classList.remove('cap09opt_active');
      cap09opt_algoritmo = "sgd"; cap09opt_resetar();
    });
    cap09opt_btnAdam.addEventListener('click', function(){
      cap09opt_btnAdam.classList.add('cap09opt_active'); cap09opt_btnSGD.classList.remove('cap09opt_active');
      cap09opt_algoritmo = "adam"; cap09opt_resetar();
    });

    cap09opt_btnPasso.addEventListener('click', cap09opt_darPasso);
    cap09opt_btnReset.addEventListener('click', cap09opt_resetar);
    cap09opt_chkComp.addEventListener('change', cap09opt_atualizar);

    cap09opt_btnAuto.addEventListener('click', function(){
      if(cap09opt_autoInterval){
        clearInterval(cap09opt_autoInterval); cap09opt_autoInterval = null; cap09opt_btnAuto.textContent = "⏵ Executar Auto";
      } else {
        cap09opt_btnAuto.textContent = "⏸ Pausar";
        cap09opt_autoInterval = setInterval(cap09opt_darPasso, 120);
      }
    });

    cap09opt_selLR.addEventListener('change', cap09opt_resetar);

    cap09opt_resetar();
  }

  function cap09opt_tryInit(){
    var cap09opt_root = document.getElementById('sim-09-opt');
    if(cap09opt_root) cap09opt_init(cap09opt_root); else setTimeout(cap09opt_tryInit, 200);
  }
  cap09opt_tryInit();
})();
</script>
''')

### Arquitetura de uma CNN

Uma CNN para classificação de imagens combina as camadas apresentadas nas seções anteriores. Durante o **passo à frente** (*forward pass*), a imagem percorre sucessivamente as camadas convolucionais, as funções de ativação, as operações de *pooling*, a etapa de **Flatten**, as camadas totalmente conectadas e, por fim, a camada **Softmax**, que produz as probabilidades das classes. Durante o treinamento, essa previsão é comparada ao rótulo correto para calcular a função de perda, realizar a retropropagação e atualizar os parâmetros por meio de um algoritmo de otimização.

A @fig-09-cnn-arquitetura apresenta esse fluxo de processamento e treinamento.

::: {#fig-09-cnn-arquitetura}
![](imagens/fig-09-cnn-arquitetura.png){width=100% fig-align="center"}

Arquitetura simplificada de uma CNN para classificação de imagens, destacando o *forward pass* e as etapas de treinamento por meio da função de perda, da retropropagação e do algoritmo de otimização.
:::

Após o último bloco convolucional, a operação **Flatten** reorganiza os mapas de características em um vetor unidimensional, que alimenta as **camadas totalmente conectadas** (*fully connected layers*), responsáveis por combinar as características extraídas para produzir os escores (*logits*) de cada classe. A camada **Softmax** converte esses escores em uma distribuição de probabilidades, utilizada tanto para a classificação quanto para o cálculo da função de perda durante o treinamento.

A @fig-09-sim-09-arquitetura apresenta uma versão interativa dessa arquitetura, permitindo executar sucessivas etapas de treinamento e observar a redução da perda, a retropropagação dos gradientes e a atualização dos filtros da rede.

In [ ]:
#| label: fig-09-sim-09-arquitetura
#| fig-cap: "Simulador interativo da arquitetura de uma CNN: escolha uma das imagens de entrada 12×12 (casa, rosto feliz ou triste), clique em cada bloco do *pipeline* — Entrada, *Conv+ReLU*, *Pooling*, *Flatten*, *FC* e *Softmax* — e execute passos de treinamento reais (*forward pass* + retropropagação) para observar a perda e a acurácia evoluindo, os *kernels* sendo ajustados e o *Softmax* passando a apontar a classe correta."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-arquitetura" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-arquitetura .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-arquitetura .cn-grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-arquitetura .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-arquitetura .cn-navbtn2 {
      padding:0 10px; height:26px; border-radius:8px; border:1px solid #E4DCC8; background:#FAFAF7;
      color:#26241D; font-size:10.5px; font-weight:600; cursor:pointer; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-navbtn2:hover { background:#F1EAD7; }
    #sim-09-arquitetura .cn-playbtn {
      padding:0 12px; height:26px; border-radius:8px; border:1px solid #2F6F9F; background:#EAF2FA;
      color:#2F6F9F; font-size:10.5px; font-weight:700; cursor:pointer; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-playbtn:hover { background:#DCEEFB; }
    #sim-09-arquitetura .cn-kcell {
      width:30px;height:30px;border-radius:5px;border:1px solid #e5e7eb;display:flex;
      align-items:center;justify-content:center;font-size:8.5px;font-weight:700;
    }
    #sim-09-arquitetura .cn-ciclo { font-size:10.5px; transition: color .3s ease, background .3s ease; padding:3px 6px; border-radius:6px; }
    #sim-09-arquitetura .cn-ciclo.pulso { background:#FCE8E6; color:#C1443A; font-weight:700; }
    #sim-09-arquitetura .cn-graflabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:4px; text-align:center; letter-spacing:.3px; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulador: Arquitetura Completa de uma CNN</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Entrada (12×12) → Conv → Pool → FC → Softmax</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletor de Imagem de Entrada para manter a mesma didática do simulador anterior -->
    <div style="margin-bottom:12px;max-width:320px;">
      <div class="cn-grouplabel">IMAGEM DE ENTRADA DO PIPELINE (12×12)</div>
      <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
        <button id="cap09arch_btnCasa" class="cn-modebtn active">🏠 Casa</button>
        <button id="cap09arch_btnFeliz" class="cn-modebtn">😊 Feliz</button>
        <button id="cap09arch_btnTriste" class="cn-modebtn">😢 Triste</button>
      </div>
    </div>

    <!-- Fluxo das Camadas -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px;margin-bottom:14px;overflow-x:auto;">
      <div id="cap09arch_flow" style="display:flex;align-items:center;gap:2px;padding:4px 2px;min-width:680px;"></div>
    </div>

    <!-- Detalhe da Visualização da Camada Selecionada -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:20px;flex-wrap:wrap;align-items:center;margin-bottom:14px;">
      <div style="flex:0 0 auto;text-align:center;">
        <canvas id="cap09arch_canvasVis" width="280" height="200" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;box-shadow:0 1px 3px rgba(0,0,0,0.03);"></canvas>
        <div id="cap09arch_barsWrap" style="display:none;align-items:flex-end;gap:24px;height:140px;margin-top:10px;justify-content:center;padding:0 10px;"></div>
        <div id="cap09arch_kernelsPanel" style="display:none;margin-top:10px;"></div>
        <div id="cap09arch_legenda" style="font-size:10.5px;color:#8A8371;margin-top:8px;max-width:280px;line-height:1.4;"></div>
      </div>
      <div id="cap09arch_desc" style="flex:1;min-width:240px;font-size:12px;line-height:1.6;color:#374151;"></div>
    </div>

    <!-- Painel de Treinamento: Forward Pass + Retropropagação de verdade -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;">
      <div style="font-size:11.5px;font-weight:600;color:#4b5563;margin-bottom:10px;">🎯 Treinamento (Forward Pass + Retropropagação)</div>

      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;">
        <div style="min-width:230px;">
          <div id="cap09arch_cicloForward" class="cn-mono cn-ciclo" style="color:#374151;">Entrada ▸ Conv+ReLU ▸ Pool ▸ Flatten ▸ FC ▸ Softmax ▸ Previsão</div>
          <div id="cap09arch_cicloBackward" class="cn-mono cn-ciclo" style="color:#8A8371;margin-top:3px;">Perda ◂ Otimizador ◂ Retropropagação ◂ (a cada passo)</div>
        </div>
        <div>
          <div class="cn-graflabel">PERDA (LOSS)</div>
          <canvas id="cap09arch_canvasLoss" width="260" height="110" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div>
          <div class="cn-graflabel">ACURÁCIA</div>
          <canvas id="cap09arch_canvasAcc" width="260" height="110" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div style="min-width:170px;">
          <div id="cap09arch_lossTxt" class="cn-mono" style="font-size:11px;color:#7EE7C6;background:#1B2430;padding:6px 10px;border-radius:6px;">Loss: —</div>
          <div id="cap09arch_accTxt" class="cn-mono" style="font-size:11px;color:#FFD98E;background:#1B2430;padding:6px 10px;border-radius:6px;margin-top:5px;">Acurácia: —</div>
          <div id="cap09arch_stepTxt" class="cn-mono" style="font-size:10.5px;color:#8A8371;margin-top:5px;">Passo de treinamento: 0</div>
          <div style="display:flex;gap:5px;margin-top:8px;flex-wrap:wrap;">
            <button id="cap09arch_btnPassoUnico" class="cn-navbtn2">Passo Único</button>
            <button id="cap09arch_btnTreinar" class="cn-playbtn">▶ Treinar</button>
            <button id="cap09arch_btnReiniciarPesos" class="cn-navbtn2">↺ Pesos Novos</button>
          </div>
        </div>
      </div>

      <div id="cap09arch_notaTreino" style="font-size:10.5px;color:#8A8371;margin-top:10px;padding-top:8px;border-top:1px dashed #E9E3D3;line-height:1.5;">
        Os <i>kernels</i> e pesos começam <b>aleatórios</b> (não são mais os filtros fixos do simulador anterior). A cada passo, a rede faz o <i>forward pass</i> nas 3 imagens, calcula a perda (<i>cross-entropy</i>) e a <b>acurácia</b> (quantas das 3 imagens são classificadas corretamente), retropropaga o erro e ajusta todos os pesos (inclusive os <i>kernels</i> da convolução) via gradiente descendente. ⚠️ Como o "conjunto de treino" tem apenas 3 exemplos, isso demonstra o <b>mecanismo</b> do treinamento (perda caindo, acurácia subindo, pesos mudando) — não a capacidade de generalizar para imagens novas, que exigiria muito mais dados.
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function clamp01(v){ return Math.max(0, Math.min(1, v)); }
  // Transforma uma ativação ReLU (não-negativa, sem limite superior) em algo
  // sempre entre 0 e 1 apenas para fins de exibição em cor — os valores brutos
  // usados no forward/backward NÃO passam por essa saturação.
  function saturar(v){ return 1 - Math.exp(-Math.max(0, v)); }

  // Gerador pseudoaleatório determinístico (mesma semente = mesmo resultado
  // inicial), para que o comportamento do simulador seja reprodutível.
  function criarRng(seed){
    return function(){
      seed |= 0; seed = (seed + 0x6D2B79F5) | 0;
      var t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
      t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
      return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
    };
  }

  // Mesmas matrizes 12x12 em formato ASCII usadas no simulador da camada convolucional
  var IMAGENS_ASCII = {
    casa: [
      "............",
      "....XXXX....",
      "...XXXXXX...",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXX..XXXX.",
      ".XXXX..XXXX.",
      ".XXXX..XXXX.",
      "............"
    ],
    feliz: [
      "............",
      "....XXXX....",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXX.XX.XXX.",
      ".XXXXXXXXXX.",
      ".XX......XX.",
      ".XXX....XXX.",
      ".XXXXXXXXXX.",
      "..XXXXXXXX..",
      "............"
    ],
    triste: [
      "............",
      "....XXXX....",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXX.XX.XXX.",
      ".XXXXXXXXXX.",
      ".XXX....XXX.",
      ".XX......XX.",
      ".XXXXXXXXXX.",
      "..XXXXXXXX..",
      "............"
    ]
  };
  var ORDEM_CLASSES = ["casa", "feliz", "triste"];
  var ROTULOS = { casa: "🏠 Casa", feliz: "😊 Feliz", triste: "😢 Triste" };
  var ONE_HOTS = { casa: [1,0,0], feliz: [0,1,0], triste: [0,0,1] };

  function converterLinhas(linhas){
    return linhas.map(function(l){
      var res = [];
      for (var i = 0; i < l.length; i++) res.push(l[i] === 'X' ? 1.0 : 0.0);
      return res;
    });
  }
  var IMAGENS = {};
  ORDEM_CLASSES.forEach(function(id){ IMAGENS[id] = converterLinhas(IMAGENS_ASCII[id]); });

  function corMapa(v){
    var r = Math.round(255 - v*(255-47));
    var g = Math.round(255 - v*(255-111));
    var b = Math.round(255 - v*(255-159));
    return 'rgb('+r+','+g+','+b+')';
  }

  // ---------------------------------------------------------------------
  // Rede: Conv1 (4 filtros 3×3) → ReLU → MaxPool 2×2 → Flatten(100) →
  // Densa1 (16, ReLU) → Densa2/saída (3 logits) → Softmax.
  // Implementação manual de forward e backward (sem bibliotecas), pensada
  // para ficar pequena o bastante para caber num simulador didático.
  // ---------------------------------------------------------------------

  function inicializarParametros(rng){
    var K = [], bConv = [];
    for (var f = 0; f < 4; f++){
      var k = [];
      for (var r = 0; r < 3; r++){
        var row = [];
        for (var c = 0; c < 3; c++) row.push((rng() - 0.5) * 1.0);
        k.push(row);
      }
      K.push(k); bConv.push(0);
    }
    var W1 = [], b1 = [];
    for (var i = 0; i < 16; i++){
      var row1 = [];
      for (var j = 0; j < 100; j++) row1.push((rng() - 0.5) * 0.2);
      W1.push(row1); b1.push(0);
    }
    var W2 = [], b2 = [];
    for (var i2 = 0; i2 < 3; i2++){
      var row2 = [];
      for (var j2 = 0; j2 < 16; j2++) row2.push((rng() - 0.5) * 0.3);
      W2.push(row2); b2.push(0);
    }
    return { K: K, bConv: bConv, W1: W1, b1: b1, W2: W2, b2: b2 };
  }

  function relu(x){ return Math.max(0, x); }
  function reluDeriv(x){ return x > 0 ? 1 : 0; }
  function softmax(logits){
    var m = Math.max.apply(null, logits);
    var exps = logits.map(function(v){ return Math.exp(v - m); });
    var soma = exps.reduce(function(a,b){ return a+b; }, 0);
    return exps.map(function(v){ return v / soma; });
  }

  function forwardPassRede(p, img){
    var Z1 = [], A1 = [];
    for (var f = 0; f < 4; f++) {
      var zf = [], af = [];
      for (var r = 0; r < 10; r++) {
        var zr = [], ar = [];
        for (var c = 0; c < 10; c++) {
          var s = p.bConv[f];
          for (var kr = 0; kr < 3; kr++)
            for (var kc = 0; kc < 3; kc++)
              s += img[r+kr][c+kc] * p.K[f][kr][kc];
          zr.push(s); ar.push(relu(s));
        }
        zf.push(zr); af.push(ar);
      }
      Z1.push(zf); A1.push(af);
    }

    var P1 = [], argMax = [];
    for (var f2 = 0; f2 < 4; f2++) {
      var pf = [], amf = [];
      for (var r2 = 0; r2 < 5; r2++) {
        var pr = [], amr = [];
        for (var c2 = 0; c2 < 5; c2++) {
          var v00=A1[f2][2*r2][2*c2], v01=A1[f2][2*r2][2*c2+1];
          var v10=A1[f2][2*r2+1][2*c2], v11=A1[f2][2*r2+1][2*c2+1];
          var mv=v00, dr=0, dc=0;
          if (v01>mv){mv=v01;dr=0;dc=1;}
          if (v10>mv){mv=v10;dr=1;dc=0;}
          if (v11>mv){mv=v11;dr=1;dc=1;}
          pr.push(mv); amr.push({dr:dr,dc:dc});
        }
        pf.push(pr); amf.push(amr);
      }
      P1.push(pf); argMax.push(amf);
    }

    var flat = [];
    for (var f3 = 0; f3 < 4; f3++)
      for (var r3 = 0; r3 < 5; r3++)
        for (var c3 = 0; c3 < 5; c3++)
          flat.push(P1[f3][r3][c3]);

    var Z2 = [], A2 = [];
    for (var i = 0; i < 16; i++) {
      var s2 = p.b1[i];
      for (var j = 0; j < 100; j++) s2 += p.W1[i][j] * flat[j];
      Z2.push(s2); A2.push(relu(s2));
    }

    var logits = [];
    for (var o = 0; o < 3; o++) {
      var s3 = p.b2[o];
      for (var j2 = 0; j2 < 16; j2++) s3 += p.W2[o][j2] * A2[j2];
      logits.push(s3);
    }
    var probs = softmax(logits);

    return { Z1:Z1, A1:A1, P1:P1, argMax:argMax, flat:flat, Z2:Z2, A2:A2, logits:logits, probs:probs };
  }

  function backwardPassRede(p, cache, img, oneHot){
    var dLogits = cache.probs.map(function(v,i){ return v - oneHot[i]; });

    var gW2 = [], gb2 = dLogits.slice();
    for (var o = 0; o < 3; o++) {
      var row = [];
      for (var j = 0; j < 16; j++) row.push(dLogits[o] * cache.A2[j]);
      gW2.push(row);
    }
    var dA2 = [];
    for (var j = 0; j < 16; j++) {
      var s = 0;
      for (var o2 = 0; o2 < 3; o2++) s += p.W2[o2][j] * dLogits[o2];
      dA2.push(s);
    }
    var dZ2 = dA2.map(function(v,i){ return v * reluDeriv(cache.Z2[i]); });

    var gW1 = [], gb1 = dZ2.slice();
    for (var i = 0; i < 16; i++) {
      var row1 = [];
      for (var j2 = 0; j2 < 100; j2++) row1.push(dZ2[i] * cache.flat[j2]);
      gW1.push(row1);
    }
    var dFlat = [];
    for (var j3 = 0; j3 < 100; j3++) {
      var s2 = 0;
      for (var i2 = 0; i2 < 16; i2++) s2 += p.W1[i2][j3] * dZ2[i2];
      dFlat.push(s2);
    }

    var dP1 = []; var idx = 0;
    for (var f = 0; f < 4; f++) {
      var pf = [];
      for (var r = 0; r < 5; r++) {
        var pr = [];
        for (var c = 0; c < 5; c++) pr.push(dFlat[idx++]);
        pf.push(pr);
      }
      dP1.push(pf);
    }

    var dA1 = [];
    for (var f2 = 0; f2 < 4; f2++) {
      var af = [];
      for (var r2 = 0; r2 < 10; r2++) af.push(new Array(10).fill(0));
      for (var r3 = 0; r3 < 5; r3++)
        for (var c3 = 0; c3 < 5; c3++) {
          var am = cache.argMax[f2][r3][c3];
          af[2*r3+am.dr][2*c3+am.dc] += dP1[f2][r3][c3];
        }
      dA1.push(af);
    }

    var dZ1 = [];
    for (var f3 = 0; f3 < 4; f3++) {
      var zf = [];
      for (var r4 = 0; r4 < 10; r4++) {
        var row2 = [];
        for (var c4 = 0; c4 < 10; c4++)
          row2.push(dA1[f3][r4][c4] * reluDeriv(cache.Z1[f3][r4][c4]));
        zf.push(row2);
      }
      dZ1.push(zf);
    }

    var gK = [], gbConv = [];
    for (var f4 = 0; f4 < 4; f4++) {
      var gk = [[0,0,0],[0,0,0],[0,0,0]]; var gb = 0;
      for (var r5 = 0; r5 < 10; r5++)
        for (var c5 = 0; c5 < 10; c5++) {
          var d = dZ1[f4][r5][c5]; gb += d;
          for (var kr = 0; kr < 3; kr++)
            for (var kc = 0; kc < 3; kc++)
              gk[kr][kc] += d * img[r5+kr][c5+kc];
        }
      gK.push(gk); gbConv.push(gb);
    }

    var loss = -Math.log(Math.max(cache.probs[oneHot.indexOf(1)], 1e-9));
    return { gK:gK, gbConv:gbConv, gW1:gW1, gb1:gb1, gW2:gW2, gb2:gb2, loss:loss };
  }

  function zeros3(f,r,c){
    var a = [];
    for (var i=0;i<f;i++){ var b=[]; for(var j=0;j<r;j++){ b.push(new Array(c).fill(0)); } a.push(b); }
    return a;
  }

  // Um passo de treinamento em lote (as 3 imagens de uma vez): calcula o
  // forward+backward para cada uma, faz a média dos gradientes e atualiza
  // todos os parâmetros (kernels inclusive) via gradiente descendente.
  function treinarPassoLote(params, lr){
    var gK = zeros3(4,3,3), gbConv = [0,0,0,0];
    var gW1 = [], gb1 = new Array(16).fill(0);
    for (var i=0;i<16;i++) gW1.push(new Array(100).fill(0));
    var gW2 = [], gb2 = [0,0,0];
    for (var o=0;o<3;o++) gW2.push(new Array(16).fill(0));
    var totalLoss = 0;

    ORDEM_CLASSES.forEach(function(id){
      var cache = forwardPassRede(params, IMAGENS[id]);
      var grads = backwardPassRede(params, cache, IMAGENS[id], ONE_HOTS[id]);
      totalLoss += grads.loss;
      for (var f=0;f<4;f++){
        for (var kr=0;kr<3;kr++) for (var kc=0;kc<3;kc++) gK[f][kr][kc] += grads.gK[f][kr][kc];
        gbConv[f] += grads.gbConv[f];
      }
      for (var ii=0;ii<16;ii++){
        for (var jj=0;jj<100;jj++) gW1[ii][jj] += grads.gW1[ii][jj];
        gb1[ii] += grads.gb1[ii];
      }
      for (var oo=0;oo<3;oo++){
        for (var jj2=0;jj2<16;jj2++) gW2[oo][jj2] += grads.gW2[oo][jj2];
        gb2[oo] += grads.gb2[oo];
      }
    });

    var nB = ORDEM_CLASSES.length;
    for (var f2=0;f2<4;f2++){
      for (var kr2=0;kr2<3;kr2++) for (var kc2=0;kc2<3;kc2++) params.K[f2][kr2][kc2] -= lr*gK[f2][kr2][kc2]/nB;
      params.bConv[f2] -= lr*gbConv[f2]/nB;
    }
    for (var i2=0;i2<16;i2++){
      for (var j2=0;j2<100;j2++) params.W1[i2][j2] -= lr*gW1[i2][j2]/nB;
      params.b1[i2] -= lr*gb1[i2]/nB;
    }
    for (var o2=0;o2<3;o2++){
      for (var j3=0;j3<16;j3++) params.W2[o2][j3] -= lr*gW2[o2][j3]/nB;
      params.b2[o2] -= lr*gb2[o2]/nB;
    }
    return totalLoss / nB;
  }

  // Calcula a fração de acertos do lote de 3 imagens com os parâmetros
  // atuais: para cada imagem, roda o forward pass e verifica se a classe
  // de maior probabilidade (argmax do Softmax) coincide com a classe correta.
  function calcularAcuraciaLote(params){
    var acertos = 0;
    ORDEM_CLASSES.forEach(function(id){
      var cache = forwardPassRede(params, IMAGENS[id]);
      var maxProb = Math.max.apply(null, cache.probs);
      var idxPredito = cache.probs.indexOf(maxProb);
      var idxCorreto = ONE_HOTS[id].indexOf(1);
      if (idxPredito === idxCorreto) acertos += 1;
    });
    return acertos / ORDEM_CLASSES.length;
  }

  function construirPipeline(params, imgMatriz, imgId){
    var cache = forwardPassRede(params, imgMatriz);

    var conv1 = cache.A1.map(function(mapa){
      return mapa.map(function(row){ return row.map(saturar); });
    });
    var pool1 = cache.P1.map(function(mapa){
      return mapa.map(function(row){ return row.map(saturar); });
    });
    var flatten = cache.flat.map(saturar);
    var fc = cache.A2.map(saturar);
    var softmaxBars = ORDEM_CLASSES.map(function(id, i){
      return { rotulo: ROTULOS[id], valor: cache.probs[i] };
    });

    return { conv1: conv1, pool1: pool1, flatten: flatten, fc: fc, softmax: softmaxBars, probsCrus: cache.probs };
  }

  function initArch(root){
    if (!root || root.dataset.initArch) return;
    root.dataset.initArch = "1";

    var imgAtualId = "casa";
    var imgMatriz = IMAGENS[imgAtualId];

    var rngInicial = criarRng(42);
    var PARAMS = inicializarParametros(rngInicial);
    var historicoLoss = [];
    var historicoAcuracia = [];
    var passoTreino = 0;
    var autoplayInterval = null;

    var PIPE = construirPipeline(PARAMS, imgMatriz, imgAtualId);

    var btnCasa   = root.querySelector('#cap09arch_btnCasa');
    var btnFeliz  = root.querySelector('#cap09arch_btnFeliz');
    var btnTriste = root.querySelector('#cap09arch_btnTriste');

    var flowEl     = root.querySelector('#cap09arch_flow');
    var descEl     = root.querySelector('#cap09arch_desc');
    var barsWrapEl = root.querySelector('#cap09arch_barsWrap');
    var kernelsPanelEl = root.querySelector('#cap09arch_kernelsPanel');
    var legendaEl  = root.querySelector('#cap09arch_legenda');
    var canvas     = root.querySelector('#cap09arch_canvasVis');
    var ctx        = canvas.getContext('2d');
    var W = canvas.width, H = canvas.height;

    var canvasLoss = root.querySelector('#cap09arch_canvasLoss');
    var ctxLoss = canvasLoss.getContext('2d');
    var canvasAcc = root.querySelector('#cap09arch_canvasAcc');
    var ctxAcc = canvasAcc.getContext('2d');
    var lossTxt = root.querySelector('#cap09arch_lossTxt');
    var accTxt = root.querySelector('#cap09arch_accTxt');
    var stepTxt = root.querySelector('#cap09arch_stepTxt');
    var cicloBackward = root.querySelector('#cap09arch_cicloBackward');

    var btnPassoUnico    = root.querySelector('#cap09arch_btnPassoUnico');
    var btnTreinar       = root.querySelector('#cap09arch_btnTreinar');
    var btnReiniciarPesos = root.querySelector('#cap09arch_btnReiniciarPesos');

    var ETAPAS = [
      {
        id: "entrada", nome: "Entrada", forma: "12×12", tipo: "imagem",
        legenda: "Imagem em escala de cinza 12×12 com margem fixa de zeros.",
        desc: "A <b>imagem de entrada</b> é representada como uma matriz de pixels 12×12. É a mesma estrutura (casa, rosto feliz ou triste) do simulador anterior."
      },
      {
        id: "conv1", nome: "Conv1 + ReLU", forma: "10×10×4", tipo: "mapas", tam: 10, dadosKey: "conv1",
        legenda: "4 mapas de características 10×10, um por filtro aprendido.",
        desc: "A primeira <b>camada convolucional</b> aplica 4 <i>kernels</i> 3×3 <b>aprendidos por treinamento</b> — diferente do simulador anterior, aqui eles começam aleatórios e vão sendo ajustados a cada passo de treinamento (veja os valores abaixo do mapa). Nesta versão simplificada usamos apenas 1 bloco convolucional (N=1); redes reais costumam empilhar vários."
      },
      {
        id: "pool1", nome: "Pooling1", forma: "5×5×4", tipo: "mapas", tam: 5, dadosKey: "pool1",
        legenda: "Os 4 mapas reduzidos para 5×5 via Max-Pooling (2×2, stride 2).",
        desc: "A camada de <b>Max-Pooling</b> reduz a resolução espacial de 10×10 para 5×5, mantendo apenas a ativação máxima de cada janela 2×2 — o que reduz a dimensão dos dados e dá alguma tolerância a pequenos deslocamentos."
      },
      {
        id: "flatten", nome: "Flatten", forma: "100 valores", tipo: "vetor", dadosKey: "flatten",
        legenda: "Vetor linearizado com 4 × 5 × 5 = 100 elementos.",
        desc: "A operação <b>Flatten</b> 'achata' os 4 mapas 2D em um único vetor 1D de 100 valores, preparando a informação para entrar nas camadas densas."
      },
      {
        id: "fc", nome: "Camada Densa (FC)", forma: "16 neurônios", tipo: "vetor", dadosKey: "fc",
        legenda: "16 neurônios (com ReLU) combinando o vetor de características.",
        desc: "A <b>camada totalmente conectada</b> tem 16 neurônios com pesos também aprendidos, combinando todas as características locais extraídas antes. Uma segunda camada densa de saída (16→3, não desenhada separadamente aqui) produz os valores brutos (<i>logits</i>) que alimentam o Softmax."
      },
      {
        id: "softmax", nome: "Softmax", forma: "3 classes", tipo: "softmax",
        legenda: "Distribuição de probabilidade final, calculada a partir dos pesos atuais da rede.",
        desc: "A função <b>Softmax</b> converte os <i>logits</i> em probabilidades que somam 1.0 (100%). Estes valores são <b>calculados de verdade</b> a partir dos pesos atuais — antes de treinar, tendem a ficar próximos de 33%/33%/33%; depois de alguns passos de treinamento, devem convergir para a classe correta."
      }
    ];

    var etapaSelecionada = 0;

    function estiloBloco(el, selecionado){
      el.style.flex = '1';
      el.style.minWidth = '95px';
      el.style.textAlign = 'center';
      el.style.padding = '8px 4px';
      el.style.borderRadius = '8px';
      el.style.cursor = 'pointer';
      el.style.fontSize = '11px';
      el.style.fontWeight = '600';
      if (selecionado){
        el.style.border = '2px solid #2F6F9F';
        el.style.background = '#EAF2FA';
        el.style.color = '#2F6F9F';
      } else {
        el.style.border = '1px solid #E4DCC8';
        el.style.background = '#FFFFFF';
        el.style.color = '#374151';
      }
    }

    function montarFluxo(){
      flowEl.innerHTML = '';
      ETAPAS.forEach(function(etapa, idx){
        var bloco = document.createElement('div');
        bloco.id = 'cap09arch_bloco_' + etapa.id;
        estiloBloco(bloco, false);

        var linha1 = document.createTextNode(etapa.nome);
        var linha2 = document.createElement('span');
        linha2.textContent = etapa.forma;
        linha2.style.display = 'block';
        linha2.style.fontSize = '9px';
        linha2.style.fontWeight = '500';
        linha2.style.color = '#8A8371';
        linha2.style.marginTop = '2px';

        bloco.appendChild(linha1);
        bloco.appendChild(linha2);
        bloco.addEventListener('click', function(){ selecionar(idx); });
        flowEl.appendChild(bloco);

        if (idx < ETAPAS.length - 1){
          var seta = document.createElement('div');
          seta.textContent = '➔';
          seta.style.color = '#C5BC9D';
          seta.style.fontSize = '12px';
          seta.style.padding = '0 2px';
          flowEl.appendChild(seta);
        }
      });
    }

    function desenharImagemEntrada(){
      ctx.clearRect(0,0,W,H);
      var n = 12, tam = 13;
      var offX = Math.round((W - n*tam)/2), offY = Math.round((H - n*tam)/2);
      for (var r=0; r<n; r++){
        for (var c=0; c<n; c++){
          var val = imgMatriz[r][c];
          var g = Math.round(val * 255);
          ctx.fillStyle = "rgb(" + g + "," + g + "," + g + ")";
          ctx.fillRect(offX + c*tam, offY + r*tam, tam, tam);
          ctx.strokeStyle = "#D1D5DB";
          ctx.strokeRect(offX + c*tam, offY + r*tam, tam, tam);
        }
      }
    }

    function desenharMapas(mapasArr, tamEspacial){
      ctx.clearRect(0,0,W,H);
      var count = mapasArr.length;
      var cols = 2, rows = 2;
      var pad = 12;
      var thumb = 65;
      var totalW = cols*thumb + (cols-1)*pad;
      var totalH = rows*thumb + (rows-1)*pad;
      var offX = Math.round((W-totalW)/2), offY = Math.round((H-totalH)/2);
      var px = thumb/tamEspacial;

      var nomesFiltros = ["Filtro 1", "Filtro 2", "Filtro 3", "Filtro 4"];

      for (var f=0; f<count; f++){
        var col = f % cols, row = Math.floor(f/cols);
        var bx = offX + col*(thumb+pad);
        var by = offY + row*(thumb+pad);
        var mapa = mapasArr[f];
        for (var yy=0; yy<tamEspacial; yy++){
          for (var xx=0; xx<tamEspacial; xx++){
            ctx.fillStyle = corMapa(mapa[yy][xx]);
            ctx.fillRect(bx+xx*px, by+yy*px, px+0.5, px+0.5);
          }
        }
        ctx.strokeStyle = '#2F6F9F';
        ctx.lineWidth = 1;
        ctx.strokeRect(bx, by, thumb, thumb);

        ctx.fillStyle = "#5E5A4A";
        ctx.font = "9px Inter, sans-serif";
        ctx.fillText(nomesFiltros[f], bx, by - 3);
      }
    }

    function desenharVetor(vals){
      ctx.clearRect(0,0,W,H);
      var count = vals.length;
      var cols = count > 20 ? 10 : 4;
      var rows = Math.ceil(count/cols);
      var cellW = Math.min(22, (W - 40)/cols);
      var cellH = Math.min(22, (H - 40)/rows);
      var offX = Math.round((W - cols*cellW)/2);
      var offY = Math.round((H - rows*cellH)/2);

      for (var i=0; i<count; i++){
        var c = i % cols, r = Math.floor(i/cols);
        ctx.fillStyle = corMapa(vals[i]);
        ctx.fillRect(offX + c*cellW, offY + r*cellH, cellW-1, cellH-1);
        ctx.strokeStyle = "#E4DCC8";
        ctx.strokeRect(offX + c*cellW, offY + r*cellH, cellW-1, cellH-1);
      }
    }

    function desenharBarras(barras){
      barsWrapEl.innerHTML = '';
      barsWrapEl.style.display = 'flex';
      var alturaMax = 100;
      barras.forEach(function(b){
        var wrap = document.createElement('div');
        wrap.style.display = 'flex';
        wrap.style.flexDirection = 'column';
        wrap.style.alignItems = 'center';
        wrap.style.fontSize = '11px';
        wrap.style.color = '#374151';

        var bar = document.createElement('div');
        bar.style.width = '38px';
        bar.style.height = Math.max(1, Math.round(b.valor * alturaMax)) + 'px';
        bar.style.background = 'linear-gradient(#2F6F9F, #1E8F6F)';
        bar.style.borderRadius = '4px 4px 0 0';

        var legendaBar = document.createElement('div');
        legendaBar.style.marginTop = '6px';
        legendaBar.style.fontWeight = '600';
        legendaBar.textContent = b.rotulo;

        var valBar = document.createElement('div');
        valBar.className = 'cn-mono';
        valBar.style.fontSize = '10px';
        valBar.style.color = '#2F6F9F';
        valBar.textContent = (b.valor * 100).toFixed(1) + '%';

        wrap.appendChild(bar);
        wrap.appendChild(legendaBar);
        wrap.appendChild(valBar);
        barsWrapEl.appendChild(wrap);
      });
    }

    function desenharPainelKernels(){
      var html = '<div class="cn-grouplabel" style="text-align:left;">KERNELS APRENDIDOS (VALORES ATUAIS)</div>' +
        '<div style="display:flex;gap:10px;flex-wrap:wrap;justify-content:center;">';
      for (var f=0; f<4; f++){
        html += '<div style="display:grid;grid-template-columns:repeat(3,30px);gap:2px;">';
        for (var r=0; r<3; r++){
          for (var c=0; c<3; c++){
            var v = PARAMS.K[f][r][c];
            var cor = v > 0 ? "#E6F4EA" : (v < 0 ? "#FCE8E6" : "#F3F4F6");
            var corTxt = v > 0 ? "#1E8F6F" : (v < 0 ? "#C1443A" : "#374151");
            html += '<div class="cn-kcell cn-mono" style="background:' + cor + ';color:' + corTxt + ';">' + v.toFixed(1) + '</div>';
          }
        }
        html += '</div>';
      }
      html += '</div>';
      kernelsPanelEl.innerHTML = html;
    }

    function desenharGraficoLoss(){
      var Wc = canvasLoss.width, Hc = canvasLoss.height;
      ctxLoss.clearRect(0,0,Wc,Hc);
      ctxLoss.strokeStyle = "#E4DCC8"; ctxLoss.lineWidth = 1;
      ctxLoss.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoLoss.length < 2){
        ctxLoss.fillStyle = "#B8AE94";
        ctxLoss.font = "10px Inter, sans-serif";
        ctxLoss.textAlign = "center";
        ctxLoss.fillText("perda aparecerá aqui", Wc/2, Hc/2 + 3);
        return;
      }
      var maxN = 200;
      var dados = historicoLoss.slice(-maxN);
      var maxLoss = Math.max.apply(null, dados);
      maxLoss = Math.max(maxLoss, 0.05);
      var padL = 8, padR = 8, padT = 8, padB = 8;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxLoss.strokeStyle = "#2F5FA8";
      ctxLoss.lineWidth = 1.6;
      ctxLoss.beginPath();
      dados.forEach(function(v, i){
        var x = padL + (dados.length === 1 ? 0 : (i/(dados.length-1)) * plotW);
        var y = padT + (1 - v/maxLoss) * plotH;
        if (i===0) ctxLoss.moveTo(x,y); else ctxLoss.lineTo(x,y);
      });
      ctxLoss.stroke();
    }

    // Mesmo padrão visual do gráfico de perda, mas com eixo Y fixo em [0,1]
    // (a acurácia do lote é sempre 0, 1/3, 2/3 ou 1, já que só há 3 imagens).
    function desenharGraficoAcuracia(){
      var Wc = canvasAcc.width, Hc = canvasAcc.height;
      ctxAcc.clearRect(0,0,Wc,Hc);
      ctxAcc.strokeStyle = "#E4DCC8"; ctxAcc.lineWidth = 1;
      ctxAcc.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoAcuracia.length < 2){
        ctxAcc.fillStyle = "#B8AE94";
        ctxAcc.font = "10px Inter, sans-serif";
        ctxAcc.textAlign = "center";
        ctxAcc.fillText("acurácia aparecerá aqui", Wc/2, Hc/2 + 3);
        return;
      }
      var maxN = 200;
      var dados = historicoAcuracia.slice(-maxN);
      var padL = 8, padR = 8, padT = 8, padB = 8;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxAcc.strokeStyle = "#1E8F6F";
      ctxAcc.lineWidth = 1.6;
      ctxAcc.beginPath();
      dados.forEach(function(v, i){
        var x = padL + (dados.length === 1 ? 0 : (i/(dados.length-1)) * plotW);
        var y = padT + (1 - v) * plotH;
        if (i===0) ctxAcc.moveTo(x,y); else ctxAcc.lineTo(x,y);
      });
      ctxAcc.stroke();
    }

    function estimarClassePredita(probs){
      var maxV = Math.max.apply(null, probs);
      return ORDEM_CLASSES[probs.indexOf(maxV)];
    }

    function atualizarVisual(etapa){
      kernelsPanelEl.style.display = 'none';
      if (etapa.tipo === 'softmax'){
        canvas.style.display = 'none';
        desenharBarras(PIPE.softmax);
      } else {
        canvas.style.display = 'block';
        barsWrapEl.style.display = 'none';
        if (etapa.tipo === 'imagem') desenharImagemEntrada();
        else if (etapa.tipo === 'mapas') desenharMapas(PIPE[etapa.dadosKey], etapa.tam);
        else if (etapa.tipo === 'vetor') desenharVetor(PIPE[etapa.dadosKey]);

        if (etapa.id === 'conv1'){
          kernelsPanelEl.style.display = 'block';
          desenharPainelKernels();
        }
      }
      legendaEl.textContent = etapa.legenda || '';
    }

    function selecionar(idx){
      etapaSelecionada = idx;
      var etapa = ETAPAS[idx];
      ETAPAS.forEach(function(e){
        var el = root.querySelector('#cap09arch_bloco_' + e.id);
        if (el) estiloBloco(el, false);
      });
      var atual = root.querySelector('#cap09arch_bloco_' + etapa.id);
      if (atual) estiloBloco(atual, true);
      descEl.innerHTML = '<b>' + etapa.nome + '</b> — dimensão: <code class="cn-mono" style="background:#EAF2FA;color:#2F6F9F;padding:2px 6px;border-radius:4px;">' + etapa.forma + '</code><br><br>' + etapa.desc;
      atualizarVisual(etapa);
    }

    function recomputarPipeline(){
      PIPE = construirPipeline(PARAMS, imgMatriz, imgAtualId);
    }

    function trocarImagem(id, btn){
      [btnCasa, btnFeliz, btnTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      imgAtualId = id;
      imgMatriz = IMAGENS[id];
      recomputarPipeline();
      selecionar(etapaSelecionada);
    }

    function pulsarBackward(){
      cicloBackward.classList.add('pulso');
      setTimeout(function(){ cicloBackward.classList.remove('pulso'); }, 350);
    }

    function passoDeTreinamento(){
      var lr = 0.3;
      var loss = treinarPassoLote(PARAMS, lr);
      var acuracia = calcularAcuraciaLote(PARAMS);
      passoTreino += 1;
      historicoLoss.push(loss);
      historicoAcuracia.push(acuracia);
      recomputarPipeline();

      lossTxt.textContent = 'Loss: ' + loss.toFixed(4);
      accTxt.textContent = 'Acurácia: ' + Math.round(acuracia * 100) + '%';
      stepTxt.textContent = 'Passo de treinamento: ' + passoTreino;
      desenharGraficoLoss();
      desenharGraficoAcuracia();
      pulsarBackward();
      selecionar(etapaSelecionada);
      return { loss: loss, acuracia: acuracia };
    }

    function pararAutoplay(){
      if (autoplayInterval){
        clearInterval(autoplayInterval);
        autoplayInterval = null;
        btnTreinar.textContent = '▶ Treinar';
      }
    }

    btnCasa.addEventListener('click', function(){ trocarImagem('casa', btnCasa); });
    btnFeliz.addEventListener('click', function(){ trocarImagem('feliz', btnFeliz); });
    btnTriste.addEventListener('click', function(){ trocarImagem('triste', btnTriste); });

    btnPassoUnico.addEventListener('click', function(){
      pararAutoplay();
      passoDeTreinamento();
    });

    btnTreinar.addEventListener('click', function(){
      if (autoplayInterval){ pararAutoplay(); return; }
      btnTreinar.textContent = '⏸ Pausar';
      autoplayInterval = setInterval(function(){
        var resultado = passoDeTreinamento();
        if (resultado.loss < 0.002 && resultado.acuracia === 1){ pararAutoplay(); }
      }, 120);
    });

    btnReiniciarPesos.addEventListener('click', function(){
      pararAutoplay();
      PARAMS = inicializarParametros(criarRng(Date.now() % 2147483647));
      historicoLoss = [];
      historicoAcuracia = [];
      passoTreino = 0;
      lossTxt.textContent = 'Loss: —';
      accTxt.textContent = 'Acurácia: —';
      stepTxt.textContent = 'Passo de treinamento: 0';
      recomputarPipeline();
      desenharGraficoLoss();
      desenharGraficoAcuracia();
      selecionar(etapaSelecionada);
    });

    montarFluxo();
    desenharGraficoLoss();
    desenharGraficoAcuracia();
    selecionar(0);
  }

  function tryInitArch(){
    var root = document.getElementById('sim-09-arquitetura');
    if (root) initArch(root); else setTimeout(tryInitArch, 200);
  }
  tryInitArch();
})();
</script>
''')


### Como o Gradiente Ajusta os *Kernels* da Convolução

A compreensão do processo de aprendizado em uma Rede Neural Convolucional (CNN) requer a elucidação de um mecanismo fundamental: **como os coeficientes aleatórios de um filtro inicial se transformam em detectores precisos de bordas, texturas e padrões complexos.**

A resposta reside no princípio do **compartilhamento de pesos (*weight sharing*)**. Durante a etapa de propagação à frente (*forward pass*), o mesmo filtro de dimensão $3\times3$ desliza por toda a extensão da imagem de entrada. Consequentemente, cada peso do *kernel* — como o elemento $K[0][0]$ no canto superior esquerdo — é reutilizado múltiplas vezes ao longo das diferentes regiões espaciais do dado de entrada.

Durante a etapa de retropropagação (*backpropagation*), essa reutilização estabelece uma dinâmica direta: **cada posição espacial processada pelo filtro gera uma contribuição individual ("voto") para a atualização do respectivo peso.**

#### A Intuição por Trás do Cálculo

Seja $Z[r][c]$ o mapa de características pré-ativação na posição $(r,c)$ da janela deslizante, obtido pela correlação cruzada entre o *kernel* $K$ e a entrada $X$:

$$
Z[r][c] = \sum_{k_r} \sum_{k_c} K[k_r][k_c] \cdot X[r + k_r][c + k_c]
$$

Aplicando a regra da cadeia para determinar a contribuição de um peso específico $K[k_r][k_c]$ na função de perda $L$, obtêm-se as seguintes etapas:

1. **Erro Local ($dZ$):** Em cada posição $(r,c)$, calcula-se a derivada parcial da função de perda em relação à pré-ativação:
   $$dZ[r][c] = \frac{\partial L}{\partial Z[r][c]}$$
   que quantifica a responsabilidade dessa posição específica no erro total da rede ($L$).

2. **Contribuição do Peso:** Como $\frac{\partial Z[r][c]}{\partial K[k_r][k_c]} = X[r + k_r][c + k_c]$, a influência de um peso específico $K[k_r][k_c]$ no erro da posição $(r,c)$ é obtida multiplicando-se o erro local $dZ[r][c]$ pelo valor do pixel de entrada alinhado a esse peso no instante do cálculo:
   $$dZ[r][c] \cdot X[r + k_r][c + k_c]$$

3. **Acúmulo de Gradientes:** O gradiente final do peso corresponde à **soma das contribuições ("votos") de todas as posições** percorridas pela janela deslizante:

$$
\frac{\partial L}{\partial K[k_r][k_c]} = \sum_{(r,c)} dZ[r][c] \cdot X[r + k_r][c + k_c]
$$

Essa formulação assegura paridade direta entre a derivação analítica e os valores computados no simulador de inspeção do gradiente (@fig-09-sim-09-gradiente-kernel).

#### O Papel da Função ReLU como "Filtro de Relevância"

A aplicação da função de ativação **ReLU** ($\max(0, z)$) imediatamente após a convolução introduz uma propriedade de seletividade ao gradiente:

* **Ativação Positiva ($Z[r][c] > 0$):** A derivada da ReLU é $1$. O erro local é propagado integralmente ($dZ \neq 0$), permitindo que a posição contribua para a atualização dos pesos do *kernel*.
* **Ativação Inativa ($Z[r][c] \le 0$):** A derivada da ReLU é $0$. O erro local é anulado ($dZ = 0$), suprimindo a contribuição da posição para o gradiente final.

> **Nota didática:** A ReLU assegura que apenas as regiões espaciais que produziram respostas ativas durante a propagação à frente possuam capacidade de modificar os pesos do *kernel* no processo de retropropagação.

#### Atualização dos Pesos via Gradiente Descendente

Após a consolidação dos gradientes acumulados de todas as posições, a atualização do peso ocorre segundo o algoritmo do Gradiente Descendente Estocástico (SGD):

$$
K[k_r][k_c] \leftarrow K[k_r][k_c] - \eta \cdot \frac{\partial L}{\partial K[k_r][k_c]}
$$

em que $\eta$ denota a **taxa de aprendizado (*learning rate*)**. 

* Se a soma dos gradientes for **positiva**, o valor do peso é reduzido.
* Se a soma for **negativa**, o valor do peso é incrementado.

#### Explorando o Simulador Interativo

:::{.callout-note}

#### 🔗 Da Arquitetura Global à Inspeção do Gradiente {.unnumbered}
No simulador de arquitetura (@fig-09-sim-09-arquitetura), observa-se o erro $dZ$ derivado da retropropagação multicamada completa, originado na perda de entropia cruzada (*Softmax*) sobre as imagens de entrada $12\times12$.

Para viabilizar a verificação analítica do gradiente sem a sobrecarga de $100$ posições de convolução e retropropagação multicamada, o simulador de aprendizado do *kernel* (@fig-09-sim-09-gradiente-kernel) adota um modelo de inspeção reduzido ($6\times6$). Nesse cenário, simplifica-se o problema substituindo a classificação complexa por uma **metadado de calibração escalar**: ajusta-se o filtro para produzir uma resposta acumulada predefinida ($\text{alvo} = 9$) ao identificar um padrão específico (como uma borda a 45 graus). O mecanismo de acúmulo de gradientes ($dZ \cdot X$) permanece rigorosamente idêntico em ambas as formulações.
:::

Para inspecionar essa dinâmica em nível numérico, utiliza-se o simulador na @fig-09-sim-09-gradiente-kernel:

* **Imagem de Entrada ($X$):** Matriz $6\times6$.
* **Filtro Convolucional ($K$):** Matriz $3\times3$ (9 pesos).
* **Mapa de Saída ($Z$ / $A$):** Matriz $4\times4$ (16 posições da janela).
* **Função de Perda ($L$):** Definida por $L = \frac{1}{2}(S - \text{alvo})^2$, em que $S = \sum A[r][c]$ representa a soma global das ativações pós-ReLU.

> **O papel do $\text{alvo} = 9$:** O valor escalar $\text{alvo} = 9$ representa a "energia de ativação" ideal estipulada para a imagem com borda diagonal. Como o mapa $A$ possui 16 posições, esse valor equivale a buscar uma resposta média de $\frac{9}{16} \approx 0,56$ por pixel ativado. Quando $S > 9$, a rede identifica que o filtro está reagindo com intensidade excessiva ao padrão, gerando um erro $dZ > 0$ que força a redução dos pesos $K$. Quando $S < 9$, os pesos são incrementados para amplificar o sinal.

##### Roteiro Sugerido de Experimentação:

1. **Seleção de Peso:** Na grade $3\times3$, escolha o peso a ser analisado (ex.: $K[0][0]$).
2. **Varredura da Janela:** Utilize o botão **"▶ Avançar posição"** para acompanhar o deslocamento da janela pelas 16 posições espaciais. Note o destaque visual na célula do mapa de entrada que alinha o pixel $X$ ao peso selecionado.
3. **Análise do Voto Local:** Examine o produto do erro local pelo pixel de entrada ($dZ \cdot X$) no painel de cálculo da posição.
4. **Verificação do Histórico:** Acompanhe a consolidação dos 16 resultados parciais organizados nas quatro colunas de histórico, observando o acúmulo do gradiente final.
5. **Atualização do Kernel:** Clique em **"▶ Aplicar passo de gradiente descendente"** para visualizar a convergência da curva de perda e a adaptação do *kernel* aleatório ao padrão de entrada selecionado.

In [ ]:
#| label: fig-09-sim-09-gradiente-kernel
#| fig-cap: "Simulador interativo do cálculo do gradiente de um peso do *kernel* convolucional."
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-09-gradiente-kernel" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-gradiente-kernel .cap09kgrad_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-gradiente-kernel .cap09kgrad_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-gradiente-kernel .cap09kgrad_modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_modebtn.cap09kgrad_active { background:#26241D; color:#FBF7EE; }
    #sim-09-gradiente-kernel .cap09kgrad_lrbtn {
      padding:3px 8px; font-size:10px; font-weight:600; border:1px solid #E4DCC8; background:#FFFFFF;
      color:#5E5A4A; border-radius:6px; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-gradiente-kernel .cap09kgrad_lrbtn.cap09kgrad_active { background:#2F6F9F; color:#FFFFFF; border-color:#2F6F9F; }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn {
      padding:0 10px; height:26px; border-radius:8px; border:1px solid #E4DCC8; background:#FAFAF7;
      color:#26241D; font-size:10.5px; font-weight:600; cursor:pointer; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn:hover { background:#F1EAD7; }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn:disabled { opacity:0.4; cursor:not-allowed; }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn {
      padding:0 12px; height:26px; border-radius:8px; border:1px solid #2F6F9F; background:#EAF2FA;
      color:#2F6F9F; font-size:10.5px; font-weight:700; cursor:pointer; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn:hover { background:#DCEEFB; }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn:disabled { opacity:0.4; cursor:not-allowed; }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn {
      height:24px; border-radius:5px; border:1px solid #E4DCC8; background:#FFFFFF;
      color:#374151; cursor:pointer; font-weight:600;
    }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn:hover { background:#F1EAD7; }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn_ativo { background:#26241D !important; color:#FBF7EE !important; border-color:#26241D !important; }
    #sim-09-gradiente-kernel .cap09kgrad_linhavoto { font-size:8.5px; padding:2px 3px; border-bottom:1px dashed #EDE7D6; border-radius:4px; transition:background .15s ease; white-space:nowrap; text-align:center; }
    #sim-09-gradiente-kernel .cap09kgrad_graflabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:4px; text-align:center; letter-spacing:.3px; }
    #sim-09-gradiente-kernel .cap09kgrad_painel { background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:12px; }
    #sim-09-gradiente-kernel .cap09kgrad_cartao {
      flex:1; min-width:140px; background:#FFFFFF; border:1px solid #E4DCC8; border-radius:10px;
      padding:10px 12px; box-sizing:border-box;
    }
    #sim-09-gradiente-kernel .cap09kgrad_barraProgresso {
      width:100%; height:6px; background:#EDE7D6; border-radius:4px; overflow:hidden; margin-top:6px;
    }
    #sim-09-gradiente-kernel .cap09kgrad_barraProgressoFill {
      height:100%; background:#2F6F9F; border-radius:4px; transition:width .2s ease;
    }

    #sim-09-gradiente-kernel .cap09kgrad_labelinfo {
      position:relative; display:inline-flex; align-items:center; gap:4px; cursor:help;
    }
    #sim-09-gradiente-kernel .cap09kgrad_icone {
      width:12px; height:12px; min-width:12px; border-radius:50%; background:#E4DCC8; color:#5E5A4A;
      font-size:8px; font-weight:700; display:inline-flex; align-items:center; justify-content:center;
      font-family:'Inter',sans-serif;
    }
    #sim-09-gradiente-kernel .cap09kgrad_labelinfo:hover .cap09kgrad_icone { background:#2F6F9F; color:#FBF7EE; }
    #sim-09-gradiente-kernel .cap09kgrad_tooltip {
      visibility:hidden; opacity:0; position:absolute; top:calc(100% + 6px); left:0;
      width:240px; max-width:60vw; background:#26241D; color:#FBF7EE; font-size:10px; font-weight:400;
      line-height:1.55; padding:9px 11px; border-radius:8px; z-index:999; letter-spacing:0;
      box-shadow:0 6px 16px rgba(0,0,0,.2); transition:opacity .15s ease, visibility .15s ease;
      pointer-events:none; text-align:left;
    }
    #sim-09-gradiente-kernel .cap09kgrad_graflabel .cap09kgrad_tooltip { left:50%; transform:translateX(-50%); }
    #sim-09-gradiente-kernel .cap09kgrad_labelinfo:hover .cap09kgrad_tooltip { visibility:visible; opacity:1; }

    #sim-09-gradiente-kernel .cap09kgrad_grid_historico {
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 4px 6px;
      max-height: 140px;
      overflow-y: auto;
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 8px;
      padding: 6px;
      margin-top: 4px;
    }

    #sim-09-gradiente-kernel .cap09kgrad_eq_box {
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 6px;
      padding: 5px 8px;
      margin: 4px 0;
      display: block;
      text-align: left;
      font-family: 'JetBrains Mono', monospace;
      font-size: 10px;
      color: #26241D;
    }

    #sim-09-gradiente-kernel .cap09kgrad_matriz_details {
      margin-top: 8px;
      max-width: 175px;
      font-size: 10px;
      color: #5E5A4A;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_summary {
      font-weight: 600;
      font-size: 9.5px;
      color: #2F6F9F;
      cursor: pointer;
      user-select: none;
      outline: none;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_summary:hover {
      text-decoration: underline;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_expl {
      margin-top: 4px;
      padding: 6px;
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 6px;
      line-height: 1.4;
    }

    /* ---- Hover das células (contas detalhadas + destaque entre camadas) ---- */
    #sim-09-gradiente-kernel .cap09kgrad_cell_hoverable { cursor: help; }
    #sim-09-gradiente-kernel .cap09kgrad_hlPrincipal {
      outline: 2px solid #2F6F9F !important;
      outline-offset: -1px;
      box-shadow: 0 0 0 3px rgba(47,111,159,0.15) inset;
      position: relative;
      z-index: 2;
    }
    #sim-09-gradiente-kernel .cap09kgrad_hlSecundario {
      outline: 2px dashed #2F6F9F !important;
      outline-offset: -2px;
      position: relative;
      z-index: 1;
    }
    .cap09kgrad_hovertip {
      position: fixed;
      background: #26241D;
      color: #FBF7EE;
      font-family: 'JetBrains Mono', ui-monospace, monospace;
      font-size: 10px;
      line-height: 1.65;
      padding: 9px 11px;
      border-radius: 8px;
      box-shadow: 0 6px 18px rgba(0,0,0,.25);
      z-index: 99999;
      pointer-events: none;
      max-width: 260px;
      white-space: normal;
      display: none;
    }
    .cap09kgrad_hovertip b { color: #7EE7C6; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">🧮 Simulador: Gradiente de um Peso do Kernel</span>
    <span class="cap09kgrad_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">&part;Perda / &part;K[kr][kc] = &Sigma; dZ &middot; X</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletor de padrão e Taxa de Aprendizado -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;margin-bottom:12px;">
      <div style="max-width:320px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">PADRÃO DE ENTRADA (IMAGEM 6&times;6)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Escolhe qual imagem 6&times;6 alimenta a convolução.</span>
        </div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09kgrad_btnDiagonal" class="cap09kgrad_modebtn cap09kgrad_active">↘ Borda diagonal</button>
          <button id="cap09kgrad_btnVertical" class="cap09kgrad_modebtn">▍ Borda vertical</button>
        </div>
      </div>

      <div>
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">TAXA DE APRENDIZADO (&eta;)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Taxa de aprendizado. Ajuste para ver a diferença entre convergência suave (0.002) e colapso por overshooting (0.02).</span>
        </div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09kgrad_lr0005" class="cap09kgrad_lrbtn">0.0005 (Lenta)</button>
          <button id="cap09kgrad_lr002" class="cap09kgrad_lrbtn cap09kgrad_active">0.002 (Ideal)</button>
          <button id="cap09kgrad_lr02" class="cap09kgrad_lrbtn">0.02 (Alta)</button>
        </div>
      </div>
    </div>

    <div style="font-size:11.5px;line-height:1.6;color:#5E5A4A;background:#FAFAF7;border:1px solid #E9E3D3;border-radius:10px;padding:10px 14px;margin-bottom:14px;">
      Exemplo <b>reduzido</b>: imagem 6&times;6 e filtro 3&times;3 gerando mapas 4&times;4. Clique nas abas <b>"🔍 Como é calculado?"</b> abaixo de cada matriz para entender as contas passo a passo. <b>Passe o mouse sobre qualquer célula</b> de X, Z, A, dZ ou K para ver a conta exata daquele valor, com os elementos usados nas camadas relacionadas destacados com contorno tracejado/azul.
    </div>

    <div style="display:flex;gap:10px;flex-wrap:wrap;align-items:flex-start;">

      <!-- PAINEL 1: Seleção do Peso e Kernel K -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;min-width:160px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">1. PESO DO KERNEL<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Selecione qual peso do kernel você deseja analisar individualmente.</span>
        </div>
        <div id="cap09kgrad_seletorPeso"></div>
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo" style="margin-top:10px;">KERNEL ATUAL (K)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Valores do filtro 3&times;3. O peso selecionado é destacado em azul. Passe o mouse sobre um peso para ver onde ele é usado.</span>
        </div>
        <div id="cap09kgrad_gradeK"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Como é atualizado?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Regra do Gradiente:</b><br>
            <div class="cap09kgrad_eq_box">K &larr; K &minus; &eta; &middot; &nabla;K</div>
            • <b>&eta;</b> = taxa de aprendizado.<br>
            • <b>&nabla;K</b> = soma dos 16 votos dZ &times; X.
          </div>
        </details>
      </div>

      <!-- PAINEL 2: Entrada X -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ENTRADA X (6&times;6)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Imagem 6&times;6. Pixel azul = sobreposição com o peso K selecionado na janela atual. Passe o mouse sobre um pixel para ver em quais posições de Z ele é usado.</span>
        </div>
        <div id="cap09kgrad_gradeX"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Como funciona X?</summary>
          <div class="cap09kgrad_matriz_expl">
            Matriz de entrada. Na posição (r,c), o peso K multiplica o pixel:
            <div class="cap09kgrad_eq_box">X[r + k<sub>r</sub>][c + k<sub>c</sub>]</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 3: Saída Z (Pré-ativação) -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">PRÉ-ATIVAÇÃO Z (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Resultado da convolução antes do ReLU: Z = &Sigma; K &middot; X. Passe o mouse sobre uma célula para ver os 9 termos da soma, destacando a janela em X e todo o kernel K.</span>
        </div>
        <div id="cap09kgrad_gradeZ"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Como calcula Z?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Correlação cruzada:</b><br>
            Multiplicação ponto a ponto do filtro 3&times;3 sobre X:
            <div class="cap09kgrad_eq_box">Z[r][c] = &Sigma; K &middot; X</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 4: Ativação A (Pós-ReLU) -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ATIVAÇÃO A (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Resultado pós-ReLU: A = max(0, Z). Se Z &le; 0, a ativação é zerada. Passe o mouse sobre uma célula para destacar o Z correspondente.</span>
        </div>
        <div id="cap09kgrad_gradeA"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Como calcula A?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Função ReLU:</b><br>
            <div class="cap09kgrad_eq_box">A[r][c] = max(0, Z[r][c])</div>
            <b>Soma Global (S):</b><br>
            <div class="cap09kgrad_eq_box">S = &Sigma; A[r][c]</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 5: Erro dZ -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ERRO dZ (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Erro propagado: dZ = (S - alvo) &middot; I(Z > 0). Onde A=0, o erro dZ também é 0. Passe o mouse sobre uma célula para ver a conta completa, destacando o Z correspondente e todas as 16 células de A que formam S.</span>
        </div>
        <div id="cap09kgrad_gradeDZ"></div>
        <div id="cap09kgrad_textoS" class="cap09kgrad_mono" style="font-size:9.5px;color:#374151;margin-top:6px;"></div>
        <div id="cap09kgrad_textoLoss" class="cap09kgrad_mono" style="font-size:9.5px;color:#374151;margin-top:2px;"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Como calcula dZ, S e Loss?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>1. Perda (Loss L):</b><br>
            <div class="cap09kgrad_eq_box">L = &frac12; (S &minus; alvo)&sup2;</div>
            <b>2. Erro propagado dZ:</b><br>
            <div class="cap09kgrad_eq_box">dZ = (S &minus; alvo) &middot; deriv_ReLU(Z)</div>
          </div>
        </details>
      </div>

    </div>

    <!-- PAINEL SECUNDÁRIO: Cálculo dos Votos e Gradiente -->
    <div style="display:flex;gap:10px;flex-wrap:wrap;margin-top:12px;">
      <div class="cap09kgrad_painel" style="flex:1;min-width:340px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">2. CÁLCULO E SOMA DOS "VOTOS" DE CADA POSIÇÃO<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Cada posição (r,c) gera um voto = dZ[r][c] &times; X[r+kr][c+kc]. A soma de todos os 16 votos forma o gradiente do peso.</span>
        </div>
        <div id="cap09kgrad_formula" style="font-size:11.5px;color:#26241D;margin-bottom:8px;line-height:1.5;"></div>
        
        <div style="display:flex;gap:6px;flex-wrap:wrap;margin-bottom:10px;">
          <button id="cap09kgrad_btnVoltarPos" class="cap09kgrad_navbtn">⏮ Voltar posição</button>
          <button id="cap09kgrad_btnAvancar" class="cap09kgrad_playbtn">▶ Avançar posição</button>
          <button id="cap09kgrad_btnSomarTudo" class="cap09kgrad_navbtn">Somar tudo</button>
          <button id="cap09kgrad_btnReiniciarPos" class="cap09kgrad_navbtn">↺ Reiniciar posições</button>
        </div>

        <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px;">
          <div class="cap09kgrad_cartao">
            <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">CÁLCULO DESTA POSIÇÃO<span class="cap09kgrad_icone">?</span>
              <span class="cap09kgrad_tooltip">Exibe o erro local (dZ) e o pixel de entrada (X) multiplicados na posição atual da janela deslizante.</span>
            </div>
            <div id="cap09kgrad_calcAtual" class="cap09kgrad_mono" style="font-size:11px;color:#374151;line-height:1.6;"></div>
          </div>
          
          <div class="cap09kgrad_cartao">
            <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">SOMA ACUMULADA (GRADIENTE)<span class="cap09kgrad_icone">?</span>
              <span class="cap09kgrad_tooltip">O valor acumulado dos produtos dZ &times; X de todas as posições já percorridas. Quando atinge 16/16, este é o gradiente final do peso.</span>
            </div>
            <div id="cap09kgrad_somaAtual" class="cap09kgrad_mono" style="font-size:11px;color:#374151;line-height:1.6;"></div>
            <div class="cap09kgrad_barraProgresso"><div id="cap09kgrad_barraFill" class="cap09kgrad_barraProgressoFill" style="width:0%;"></div></div>
          </div>
        </div>

        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo" style="margin-top:2px;">HISTÓRICO DAS 16 POSIÇÕES (COLUNAS c=0, c=1, c=2, c=3)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Acompanhe a lista de todas as 16 posições organizadas em 4 colunas para corresponder ao movimento da janela sobre a imagem de saída.</span>
        </div>
        <div id="cap09kgrad_listaVotos" class="cap09kgrad_grid_historico"></div>
      </div>
    </div>

    <!-- PAINEL TERCIÁRIO: Atualização e Gráfico -->
    <div class="cap09kgrad_painel" style="margin-top:12px;">
      <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">3. USE O GRADIENTE PARA ATUALIZAR O KERNEL<span class="cap09kgrad_icone">?</span>
        <span class="cap09kgrad_tooltip">Aplica a regra do Gradiente Descendente (K &larr; K &minus; &eta; &middot; gradiente) para todos os 9 pesos.</span>
      </div>
      <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;">
        <div style="display:flex;gap:6px;flex-wrap:wrap;">
          <button id="cap09kgrad_btnAtualizarPesos" class="cap09kgrad_playbtn">▶ Aplicar passo de gradiente descendente</button>
          <button id="cap09kgrad_btnDesfazerPasso" class="cap09kgrad_navbtn">⏮ Desfazer passo</button>
          <button id="cap09kgrad_btnNovoKernel" class="cap09kgrad_navbtn">🎲 Novo kernel aleatório</button>
        </div>
        <div>
          <div class="cap09kgrad_graflabel cap09kgrad_labelinfo">PERDA AO LONGO DAS ATUALIZAÇÕES<span class="cap09kgrad_icone">?</span>
            <span class="cap09kgrad_tooltip">
              <b>Evolução do erro L = ½(S &minus; alvo)²:</b><br>
              • <b>Objetivo:</b> L &rarr; 0 (S &rarr; alvo).<br>
              • <b>Se travar em L = 40.5:</b> Ocorreu "overshooting" (salto exagerado). Os pesos ficaram muito negativos, gerando Z &le; 0 (morte da ReLU). Com S = 0, a perda trava em ½(0 &minus; 9)&sup2; = 40.5.
            </span>
          </div>
          <canvas id="cap09kgrad_canvasLoss" width="220" height="75" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div id="cap09kgrad_textoEpoca" class="cap09kgrad_mono" style="font-size:10.5px;color:#8A8371;"></div>
      </div>

      <div id="cap09kgrad_notaAtualizacao" style="font-size:10.5px;color:#374151;margin-top:8px;padding-top:6px;border-top:1px dashed #E9E3D3;line-height:1.6;text-align:left;"></div>
    </div>

  </div>
</div>

<script>
(function cap09kgrad_scope(){
  function cap09kgrad_clamp01(v){ return Math.max(0, Math.min(1, v)); }
  function cap09kgrad_relu(x){ return Math.max(0, x); }
  function cap09kgrad_reluDeriv(x){ return x > 0 ? 1 : 0; }

  function cap09kgrad_criarRng(seed){
    return function(){
      seed |= 0; seed = (seed + 0x6D2B79F5) | 0;
      var t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
      t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
      return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
    };
  }

  var cap09kgrad_PADROES = {
    diagonal: [
      [1,0,0,0,0,0],[1,1,0,0,0,0],[1,1,1,0,0,0],
      [1,1,1,1,0,0],[1,1,1,1,1,0],[1,1,1,1,1,1]
    ],
    vertical: [
      [1,1,1,0,0,0],[1,1,1,0,0,0],[1,1,1,0,0,0],
      [1,1,1,0,0,0],[1,1,1,0,0,0],[1,1,1,0,0,0]
    ]
  };
  var cap09kgrad_ALVOS = { diagonal: 9, vertical: 8 };

  function cap09kgrad_forward(K, X){
    var Z = [], A = [];
    for (var r = 0; r < 4; r++){
      var zr = [], ar = [];
      for (var c = 0; c < 4; c++){
        var s = 0;
        for (var kr = 0; kr < 3; kr++)
          for (var kc = 0; kc < 3; kc++)
            s += X[r+kr][c+kc] * K[kr][kc];
        zr.push(s); ar.push(cap09kgrad_relu(s));
      }
      Z.push(zr); A.push(ar);
    }
    var S = 0;
    for (var r2 = 0; r2 < 4; r2++)
      for (var c2 = 0; c2 < 4; c2++) S += A[r2][c2];
    return { Z: Z, A: A, S: S };
  }

  function cap09kgrad_inicializarKernelVivo(startSeed, X){
    var seedAtual = startSeed;
    var K, cache;
    for (var tentativas = 0; tentativas < 500; tentativas++){
      var rng = cap09kgrad_criarRng(seedAtual);
      K = [];
      for (var r = 0; r < 3; r++){
        var row = [];
        for (var c = 0; c < 3; c++) row.push(Number(((rng() - 0.5)).toFixed(2)));
        K.push(row);
      }
      cache = cap09kgrad_forward(K, X);
      if (cache.S > 0) return K;
      seedAtual++;
    }
    return K;
  }

  function cap09kgrad_calcularErro(cache, alvo){
    var dS = cache.S - alvo;
    var dZ = [];
    for (var r = 0; r < 4; r++){
      var row = [];
      for (var c = 0; c < 4; c++) row.push(dS * cap09kgrad_reluDeriv(cache.Z[r][c]));
      dZ.push(row);
    }
    var loss = 0.5 * dS * dS;
    return { dZ: dZ, loss: loss, dS: dS };
  }

  function cap09kgrad_gradientePeso(dZ, X, kr0, kc0){
    var votos = [];
    var soma = 0;
    for (var r = 0; r < 4; r++){
      for (var c = 0; c < 4; c++){
        var xVal = X[r+kr0][c+kc0];
        var dVal = dZ[r][c];
        var voto = dVal * xVal;
        soma += voto;
        votos.push({ r: r, c: c, x: xVal, dz: dVal, voto: voto, somaParcial: soma });
      }
    }
    return { votos: votos, gradiente: soma };
  }

  function cap09kgrad_gradienteKernelCompleto(dZ, X){
    var gK = [[0,0,0],[0,0,0],[0,0,0]];
    for (var kr = 0; kr < 3; kr++)
      for (var kc = 0; kc < 3; kc++)
        gK[kr][kc] = cap09kgrad_gradientePeso(dZ, X, kr, kc).gradiente;
    return gK;
  }

  function cap09kgrad_atualizarKernel(K, gK, lr){
    var novo = [[0,0,0],[0,0,0],[0,0,0]];
    for (var kr = 0; kr < 3; kr++)
      for (var kc = 0; kc < 3; kc++)
        novo[kr][kc] = K[kr][kc] - lr * gK[kr][kc];
    return novo;
  }

  function cap09kgrad_corValor(v, maxAbs){
    var m = maxAbs || 1;
    var t = cap09kgrad_clamp01(Math.abs(v) / m);
    if (v >= 0){
      var g = Math.round(230 - t*90);
      return "rgb(" + Math.round(235-t*120) + "," + g + "," + Math.round(220-t*90) + ")";
    } else {
      var r2 = Math.round(252 - t*20);
      return "rgb(" + r2 + "," + Math.round(232-t*130) + "," + Math.round(230-t*130) + ")";
    }
  }

  function cap09kgrad_initSim(root){
    if (!root || root.dataset.cap09kgradInit) return;
    root.dataset.cap09kgradInit = "1";

    var padraoAtual = "diagonal";
    var X = cap09kgrad_PADROES[padraoAtual];
    var alvo = cap09kgrad_ALVOS[padraoAtual];
    var lr = 0.002;

    var K = cap09kgrad_inicializarKernelVivo(7, X);
    var historicoKernel = [];

    var cacheForward = cap09kgrad_forward(K, X);
    var cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);

    var pesoSelKr = 0, pesoSelKc = 0;
    var posicaoIdx = 0;
    var votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
    var historicoLoss = [cacheErro.loss];
    var epocaAtual = 0;
    var ultimoGradiente = null;
    var ultimoKAntigo = null;

    // ---- Estado do sistema de hover (contas + destaque entre camadas) ----
    var celulasDestacadas = [];
    var elHoverTip = null;

    var elBtnDiagonal = root.querySelector('#cap09kgrad_btnDiagonal');
    var elBtnVertical = root.querySelector('#cap09kgrad_btnVertical');
    var elLr0005 = root.querySelector('#cap09kgrad_lr0005');
    var elLr002 = root.querySelector('#cap09kgrad_lr002');
    var elLr02 = root.querySelector('#cap09kgrad_lr02');
    var elGradeX = root.querySelector('#cap09kgrad_gradeX');
    var elGradeZ = root.querySelector('#cap09kgrad_gradeZ');
    var elGradeA = root.querySelector('#cap09kgrad_gradeA');
    var elGradeDZ = root.querySelector('#cap09kgrad_gradeDZ');
    var elGradeK = root.querySelector('#cap09kgrad_gradeK');
    var elSeletorPeso = root.querySelector('#cap09kgrad_seletorPeso');
    var elFormula = root.querySelector('#cap09kgrad_formula');
    var elListaVotos = root.querySelector('#cap09kgrad_listaVotos');
    var elCalcAtual = root.querySelector('#cap09kgrad_calcAtual');
    var elSomaAtual = root.querySelector('#cap09kgrad_somaAtual');
    var elBarraFill = root.querySelector('#cap09kgrad_barraFill');
    var elBtnVoltarPos = root.querySelector('#cap09kgrad_btnVoltarPos');
    var elBtnAvancar = root.querySelector('#cap09kgrad_btnAvancar');
    var elBtnSomarTudo = root.querySelector('#cap09kgrad_btnSomarTudo');
    var elBtnReiniciarPos = root.querySelector('#cap09kgrad_btnReiniciarPos');
    var elBtnAtualizarPesos = root.querySelector('#cap09kgrad_btnAtualizarPesos');
    var elBtnDesfazerPasso = root.querySelector('#cap09kgrad_btnDesfazerPasso');
    var elBtnNovoKernel = root.querySelector('#cap09kgrad_btnNovoKernel');
    var elTextoS = root.querySelector('#cap09kgrad_textoS');
    var elTextoLoss = root.querySelector('#cap09kgrad_textoLoss');
    var elTextoEpoca = root.querySelector('#cap09kgrad_textoEpoca');
    var elCanvasLoss = root.querySelector('#cap09kgrad_canvasLoss');
    var ctxLoss = elCanvasLoss.getContext('2d');
    var elNotaAtualizacao = root.querySelector('#cap09kgrad_notaAtualizacao');

    function montarSeletorPeso(){
      elSeletorPeso.innerHTML = '';
      elSeletorPeso.style.display = 'grid';
      elSeletorPeso.style.gridTemplateColumns = 'repeat(3, 30px)';
      elSeletorPeso.style.gap = '3px';
      for (var kr = 0; kr < 3; kr++){
        for (var kc = 0; kc < 3; kc++){
          (function(kr2, kc2){
            var bt = document.createElement('button');
            bt.className = 'cap09kgrad_pesobtn';
            bt.textContent = 'K[' + kr2 + '][' + kc2 + ']';
            bt.style.fontSize = '7.5px';
            bt.addEventListener('click', function(){ selecionarPeso(kr2, kc2); });
            elSeletorPeso.appendChild(bt);
          })(kr, kc);
        }
      }
    }

    function estiloCelula(el, ativo, corFundo, tam){
      var t = tam || '24px';
      el.style.width = t; el.style.height = t;
      el.style.display = 'flex'; el.style.alignItems = 'center'; el.style.justifyContent = 'center';
      el.style.fontSize = '7.5px'; el.style.fontWeight = '700'; el.style.borderRadius = '4px';
      el.style.background = corFundo;
      el.style.border = ativo ? '2px solid #2F6F9F' : '1px solid #E4DCC8';
      el.style.boxSizing = 'border-box';
    }

    function desenharGradeX(){
      elGradeX.innerHTML = '';
      elGradeX.style.display = 'grid';
      elGradeX.style.gridTemplateColumns = 'repeat(6, 24px)';
      elGradeX.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var janelaAtiva = posicaoIdx < 16;
      for (var rr = 0; rr < 6; rr++){
        for (var cc = 0; cc < 6; cc++){
          var cel = document.createElement('div');
          var v = X[rr][cc];
          var cor = v > 0 ? '#EFE9D8' : '#FFFFFF';
          var dentroJanela = janelaAtiva && rr >= r && rr <= r+2 && cc >= c && cc <= c+2;
          var ehPixelDoVoto = janelaAtiva && rr === r+pesoSelKr && cc === c+pesoSelKc;
          estiloCelula(cel, false, cor, '24px');
          if (dentroJanela){ cel.style.border = '1px solid #B8AE94'; }
          if (ehPixelDoVoto){ cel.style.border = '2px solid #2F6F9F'; cel.style.background = '#DCEEFB'; }
          cel.textContent = v; cel.style.color = '#5E5A4A';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeX.appendChild(cel);
        }
      }
    }

    function desenharGradeZ(){
      elGradeZ.innerHTML = '';
      elGradeZ.style.display = 'grid';
      elGradeZ.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeZ.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheForward.Z[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheForward.Z[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #2F6F9F'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeZ.appendChild(cel);
        }
      }
    }

    function desenharGradeA(){
      elGradeA.innerHTML = '';
      elGradeA.style.display = 'grid';
      elGradeA.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeA.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheForward.A[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheForward.A[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #2F6F9F'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeA.appendChild(cel);
        }
      }
    }

    function desenharGradeDZ(){
      elGradeDZ.innerHTML = '';
      elGradeDZ.style.display = 'grid';
      elGradeDZ.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeDZ.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheErro.dZ[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheErro.dZ[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #C1443A'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeDZ.appendChild(cel);
        }
      }
    }

    function desenharGradeK(){
      elGradeK.innerHTML = '';
      elGradeK.style.display = 'grid';
      elGradeK.style.gridTemplateColumns = 'repeat(3, 30px)';
      elGradeK.style.gap = '3px';
      var maxAbs = 0;
      for (var i=0;i<3;i++) for (var j=0;j<3;j++) maxAbs = Math.max(maxAbs, Math.abs(K[i][j]));
      maxAbs = maxAbs || 1;
      for (var kr = 0; kr < 3; kr++){
        for (var kc = 0; kc < 3; kc++){
          var cel = document.createElement('div');
          var ativo = (kr === pesoSelKr && kc === pesoSelKc);
          estiloCelula(cel, ativo, cap09kgrad_corValor(K[kr][kc], maxAbs), '30px');
          cel.textContent = K[kr][kc].toFixed(2); cel.style.color = '#374151';
          cel.dataset.kr = kr; cel.dataset.kc = kc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeK.appendChild(cel);
        }
      }
    }

    // ================= SISTEMA DE HOVER: contas detalhadas + destaque entre camadas =================

    function cap09kgrad_criarTooltipGlobal(){
      if (elHoverTip) return elHoverTip;
      elHoverTip = document.createElement('div');
      elHoverTip.className = 'cap09kgrad_hovertip';
      document.body.appendChild(elHoverTip);
      return elHoverTip;
    }

    function cap09kgrad_posicionarTooltip(evt){
      var tip = elHoverTip;
      if (!tip) return;
      var margem = 14;
      var x = evt.clientX + margem;
      var y = evt.clientY + margem;
      var larguraTip = tip.offsetWidth, alturaTip = tip.offsetHeight;
      if (x + larguraTip > window.innerWidth - 8) x = evt.clientX - larguraTip - margem;
      if (y + alturaTip > window.innerHeight - 8) y = evt.clientY - alturaTip - margem;
      tip.style.left = x + 'px';
      tip.style.top = y + 'px';
    }

    function cap09kgrad_mostrarTooltip(htmlConteudo, evt){
      var tip = cap09kgrad_criarTooltipGlobal();
      tip.innerHTML = htmlConteudo;
      tip.style.display = 'block';
      cap09kgrad_posicionarTooltip(evt);
    }

    function cap09kgrad_esconderTooltip(){
      if (elHoverTip) elHoverTip.style.display = 'none';
    }

    function cap09kgrad_limparDestaques(){
      celulasDestacadas.forEach(function(item){ item.el.classList.remove(item.classe); });
      celulasDestacadas = [];
    }

    function cap09kgrad_destacar(elementos, classe){
      elementos.forEach(function(el){
        if (!el) return;
        el.classList.add(classe);
        celulasDestacadas.push({ el: el, classe: classe });
      });
    }

    function cap09kgrad_celX(rr,cc){ return elGradeX.querySelector('[data-r="'+rr+'"][data-c="'+cc+'"]'); }
    function cap09kgrad_celZ(r,c){ return elGradeZ.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celA(r,c){ return elGradeA.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celDZ(r,c){ return elGradeDZ.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celK(kr,kc){ return elGradeK.querySelector('[data-kr="'+kr+'"][data-kc="'+kc+'"]'); }
    function cap09kgrad_todasCelA(){ return Array.prototype.slice.call(elGradeA.querySelectorAll('[data-r]')); }

    function cap09kgrad_hoverZ(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      var termos = [];
      for (var kr=0; kr<3; kr++){
        for (var kc=0; kc<3; kc++){
          var xv = X[r+kr][c+kc];
          var kv = K[kr][kc];
          termos.push({ kr:kr, kc:kc, x:xv, k:kv, t:kv*xv });
          cap09kgrad_destacar([cap09kgrad_celX(r+kr, c+kc)], 'cap09kgrad_hlSecundario');
          cap09kgrad_destacar([cap09kgrad_celK(kr, kc)], 'cap09kgrad_hlSecundario');
        }
      }
      var linhas = termos.map(function(t){
        return 'K['+t.kr+']['+t.kc+']&middot;X['+(r+t.kr)+']['+(c+t.kc)+'] = '+t.k.toFixed(2)+'&times;'+t.x+' = <b>'+t.t.toFixed(3)+'</b>';
      }).join('<br>');
      var html =
        '<b>Z['+r+']['+c+']</b> = &Sigma; K &middot; X (9 termos)<br>' +
        '<div style="margin:4px 0;border-top:1px dashed #4A473A;padding-top:4px;">' + linhas + '</div>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">total = <b>'+cacheForward.Z[r][c].toFixed(3)+'</b></div>' +
        '<div style="margin-top:4px;color:#B8AE94;">contorno tracejado = janela em X e pesos em K usados</div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverA(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlPrincipal');
      var zv = cacheForward.Z[r][c], av = cacheForward.A[r][c];
      var html =
        '<b>A['+r+']['+c+']</b> = max(0, Z['+r+']['+c+'])<br>' +
        '= max(0, '+zv.toFixed(3)+') = <b>'+av.toFixed(3)+'</b>' +
        (zv <= 0 ? '<br><span style="color:#FF9A93;">ReLU zerou este valor.</span>' : '');
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverDZ(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar(cap09kgrad_todasCelA(), 'cap09kgrad_hlSecundario');
      var zv = cacheForward.Z[r][c];
      var gate = cap09kgrad_reluDeriv(zv);
      var dS = cacheErro.dS;
      var dz = cacheErro.dZ[r][c];
      var html =
        '<b>dZ['+r+']['+c+']</b> = (S &minus; alvo) &middot; ReLU&prime;(Z['+r+']['+c+'])<br>' +
        'S = &Sigma; A (16 células, tracejado) = <b>'+cacheForward.S.toFixed(3)+'</b><br>' +
        'S &minus; alvo = '+cacheForward.S.toFixed(3)+' &minus; '+alvo+' = <b>'+dS.toFixed(3)+'</b><br>' +
        'ReLU&prime;(Z) = ' + (gate ? '1 (Z&gt;0)' : '0 (Z&le;0)') + '<br>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">dZ = '+dS.toFixed(3)+' &times; '+gate+' = <b>'+dz.toFixed(3)+'</b></div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverX(rr, cc, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      var contribs = [];
      for (var kr=0; kr<3; kr++){
        for (var kc=0; kc<3; kc++){
          var r = rr-kr, c = cc-kc;
          if (r>=0 && r<4 && c>=0 && c<4){
            contribs.push({ r:r, c:c, kr:kr, kc:kc, k:K[kr][kc] });
            cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlSecundario');
            cap09kgrad_destacar([cap09kgrad_celK(kr,kc)], 'cap09kgrad_hlSecundario');
          }
        }
      }
      var xv = X[rr][cc];
      var linhas = contribs.map(function(t){
        var termo = t.k*xv;
        return 'Z['+t.r+']['+t.c+'] usa K['+t.kr+']['+t.kc+']&times;X = '+t.k.toFixed(2)+'&times;'+xv+' = <b>'+termo.toFixed(3)+'</b>';
      }).join('<br>');
      var html =
        '<b>X['+rr+']['+cc+']</b> = '+xv+'<br>' +
        (contribs.length ?
          '<div style="margin:4px 0;border-top:1px dashed #4A473A;padding-top:4px;">Usado em '+contribs.length+' posição(ões) de Z:<br>'+linhas+'</div>'
          : '<span style="color:#B8AE94;">Fora do alcance de qualquer janela 3&times;3 sobre a saída atual.</span>');
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverK(kr, kc, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      for (var r=0; r<4; r++){
        for (var c=0; c<4; c++){
          cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlSecundario');
          cap09kgrad_destacar([cap09kgrad_celX(r+kr,c+kc)], 'cap09kgrad_hlSecundario');
        }
      }
      var gAtual = cap09kgrad_gradientePeso(cacheErro.dZ, X, kr, kc).gradiente;
      var html =
        '<b>K['+kr+']['+kc+']</b> = '+K[kr][kc].toFixed(3)+'<br>' +
        'Participa das 16 posições de Z (destacadas), cada uma multiplicando um pixel X diferente (também destacado).<br>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">Gradiente atual &nabla;K = &Sigma; dZ&middot;X = <b>'+gAtual.toFixed(3)+'</b></div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_configurarHover(container, seletor, manipulador){
      container.addEventListener('mouseover', function(e){
        var cel = e.target.closest(seletor);
        if (!cel || cel.parentElement !== container) return;
        manipulador(cel, e);
      });
      container.addEventListener('mousemove', function(e){
        var cel = e.target.closest(seletor);
        if (!cel || cel.parentElement !== container) return;
        cap09kgrad_posicionarTooltip(e);
      });
      container.addEventListener('mouseout', function(e){
        var cel = e.target.closest(seletor);
        if (!cel) return;
        var indoPara = e.relatedTarget;
        if (indoPara && cel.contains(indoPara)) return;
        cap09kgrad_limparDestaques();
        cap09kgrad_esconderTooltip();
      });
    }

    // ================= FIM DO SISTEMA DE HOVER =================

    function atualizarFormula(){
      elFormula.innerHTML =
        '<span class="cap09kgrad_mono">&part;L/&part;K[' + pesoSelKr + '][' + pesoSelKc + ']</span> = &Sigma;<sub>(r,c)</sub> ' +
        '<span class="cap09kgrad_mono">dZ[r][c] &middot; X[r+' + pesoSelKr + '][c+' + pesoSelKc + ']</span>';
    }

    function renderizarListaVotos(){
      elListaVotos.innerHTML = '';
      var elementoAtivo = null;
      var idxAtivo = posicaoIdx - 1;

      votosAtuais.forEach(function(v, idx){
        var linha = document.createElement('div');
        linha.className = 'cap09kgrad_linhavoto';
        var visivel = idx < posicaoIdx;
        linha.style.opacity = visivel ? '1' : '0.25';
        if (idx === idxAtivo){
          linha.style.background = '#EAF2FA';
          elementoAtivo = linha;
        }
        linha.innerHTML =
          '<span class="cap09kgrad_mono" style="color:#8A8371;">p(' + v.r + ',' + v.c + ')</span> ' +
          '<span class="cap09kgrad_mono">' + v.dz.toFixed(2) + '</span>&times;' +
          '<span class="cap09kgrad_mono">' + v.x + '</span>=' +
          '<span class="cap09kgrad_mono" style="color:' + (v.voto>=0 ? '#1E8F6F' : '#C1443A') + ';font-weight:700;">' + v.voto.toFixed(2) + '</span>';
        elListaVotos.appendChild(linha);
      });

      if (elementoAtivo){
        var alvoTop = elementoAtivo.offsetTop - (elListaVotos.clientHeight/2) + (elementoAtivo.clientHeight/2);
        elListaVotos.scrollTop = Math.max(0, alvoTop);
      } else {
        elListaVotos.scrollTop = 0;
      }
    }

    function atualizarCalcAtual(){
      if (posicaoIdx === 0){
        elCalcAtual.innerHTML =
          '<span style="color:#8A8371;">Clique em <b>"Avançar posição"</b> para ver a primeira conta.</span>';
        return;
      }
      var atual = votosAtuais[posicaoIdx - 1];
      var corVoto = atual.voto >= 0 ? '#1E8F6F' : '#C1443A';
      elCalcAtual.innerHTML =
        'posição <b>(' + atual.r + ',' + atual.c + ')</b><br>' +
        'dZ&nbsp;&nbsp;= <b>' + atual.dz.toFixed(3) + '</b><br>' +
        'X&nbsp;&nbsp;&nbsp;&nbsp;= <b>' + atual.x + '</b><br>' +
        '<span style="border-top:1px dashed #E4DCC8;display:block;margin:4px 0 2px;"></span>' +
        'voto = dZ &times; X = <b style="color:' + corVoto + ';">' + atual.voto.toFixed(3) + '</b>';
    }

    function atualizarSomaTexto(){
      var somaParcial = posicaoIdx > 0 ? votosAtuais[posicaoIdx-1].somaParcial : 0;
      var completo = posicaoIdx >= 16;

      elSomaAtual.innerHTML =
        'posições: <b>' + posicaoIdx + ' / 16</b><br>' +
        '<span style="border-top:1px dashed #E4DCC8;display:block;margin:4px 0 2px;"></span>' +
        'total = <b style="color:#2F6F9F;">' + somaParcial.toFixed(3) + '</b>' +
        (completo ? '<br><span style="color:#1E8F6F;font-weight:700;">✓ gradiente completo</span>' : '');

      elBarraFill.style.width = (posicaoIdx/16*100).toFixed(1) + '%';
      elBtnVoltarPos.disabled = (posicaoIdx === 0);
      elBtnAvancar.disabled = completo;
      elBtnSomarTudo.disabled = completo;
      elBtnDesfazerPasso.disabled = (historicoKernel.length === 0);
    }

    function desenharGraficoLoss(){
      var Wc = elCanvasLoss.width, Hc = elCanvasLoss.height;
      ctxLoss.clearRect(0,0,Wc,Hc);
      ctxLoss.strokeStyle = '#E4DCC8'; ctxLoss.lineWidth = 1;
      ctxLoss.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoLoss.length < 2){
        ctxLoss.fillStyle = '#B8AE94';
        ctxLoss.font = '10px Inter, sans-serif';
        ctxLoss.textAlign = 'center';
        ctxLoss.fillText('perda aparecerá aqui', Wc/2, Hc/2+3);
        return;
      }

      var maxLoss = Math.max.apply(null, historicoLoss);
      maxLoss = Math.max(maxLoss, 0.01);

      var padL = 30, padR = 8, padT = 10, padB = 14;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxLoss.strokeStyle = '#EDE7D6';
      ctxLoss.lineWidth = 1;
      ctxLoss.beginPath();
      ctxLoss.moveTo(padL, padT); ctxLoss.lineTo(Wc - padR, padT);
      ctxLoss.moveTo(padL, Hc - padB); ctxLoss.lineTo(Wc - padR, Hc - padB);
      ctxLoss.stroke();

      ctxLoss.strokeStyle = '#1E8F6F';
      ctxLoss.setLineDash([2, 2]);
      ctxLoss.beginPath();
      ctxLoss.moveTo(padL, Hc - padB); ctxLoss.lineTo(Wc - padR, Hc - padB);
      ctxLoss.stroke();
      ctxLoss.setLineDash([]);

      ctxLoss.fillStyle = '#8A8371';
      ctxLoss.font = '8px JetBrains Mono, monospace';
      ctxLoss.textAlign = 'right';
      ctxLoss.fillText(maxLoss.toFixed(1), padL - 3, padT + 3);
      ctxLoss.fillText('0.0', padL - 3, Hc - padB + 2);

      ctxLoss.strokeStyle = '#2F5FA8'; 
      ctxLoss.lineWidth = 1.6;
      ctxLoss.beginPath();
      historicoLoss.forEach(function(v, i){
        var x = padL + (historicoLoss.length === 1 ? 0 : (i / (historicoLoss.length - 1)) * plotW);
        var y = padT + (1 - v / maxLoss) * plotH;
        if (i === 0) ctxLoss.moveTo(x, y); else ctxLoss.lineTo(x, y);
      });
      ctxLoss.stroke();
    }

    function renderizarNotaAtualizacao(){
      var gAtual = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).gradiente;
      var kr = pesoSelKr, kc = pesoSelKc;
      var kAtual = K[kr][kc];
      var passoPrevisto = lr * gAtual;
      var kNovoPrevisto = kAtual - passoPrevisto;

      if (cacheForward.S <= 0){
        elNotaAtualizacao.innerHTML = 
          '<div style="background:#FDF2F2;border:1px solid #F87171;border-radius:8px;padding:8px 10px;color:#991B1B;">' +
          '<b>⚡ Colapso por ReLU Morta (S = 0.000):</b><br>' +
          'Todas as ativações Z &le; 0 foram zeradas pela ReLU. O erro dZ = 0, zerando o gradiente (&nabla;K = 0).<br>' +
          '• A perda travou em L = &frac12;&middot;(0 &minus; ' + alvo + ')&sup2; = ' + cacheErro.loss.toFixed(1) + '.<br>' +
          '• <b>Como resolver:</b> Reduza a taxa &eta;, clique em <b>"⏮ Desfazer passo"</b> ou em <b>"🎲 Novo kernel aleatório"</b>.' +
          '</div>';
        return;
      }

      if (ultimoGradiente === null){
        elNotaAtualizacao.innerHTML = 
          '<div style="text-align:left;">' +
          '<b>Estado Inicial do peso K[' + kr + '][' + kc + '] = ' + kAtual.toFixed(3) + ':</b><br>' +
          '• <b>Gradiente visível acumulado (&nabla;K):</b> <b style="color:#2F6F9F;">' + gAtual.toFixed(3) + '</b> (soma dos 16 votos dZ &times; X)<br>' +
          '• <b>Passo de ajuste previsto (&eta; &times; &nabla;K):</b> ' + lr + ' &times; (' + gAtual.toFixed(3) + ') = <b>' + passoPrevisto.toFixed(4) + '</b><br>' +
          '• <b>Novo Peso previsto:</b> K &larr; ' + kAtual.toFixed(3) + ' &minus; (' + passoPrevisto.toFixed(4) + ') = <b>' + kNovoPrevisto.toFixed(3) + '</b><br>' +
          '<br><span style="color:#8A8371;">Clique em <b>"▶ Aplicar passo de gradiente descendente"</b> para atualizar a matriz.</span>' +
          '</div>';
        return;
      }

      var gValAnterior = ultimoGradiente[kr][kc];
      var kAntigoVal = ultimoKAntigo[kr][kc];
      var passoAplicado = lr * gValAnterior;

      elNotaAtualizacao.innerHTML = 
        '<div style="text-align:left;">' +
        '<b>Última atualização aplicada ao peso K[' + kr + '][' + kc + ']:</b><br>' +
        '1. <b>Taxa (&eta;):</b> <span class="cap09kgrad_mono">' + lr + '</span> | <b>&nabla;K aplicado:</b> <span class="cap09kgrad_mono">' + gValAnterior.toFixed(3) + '</span><br>' +
        '2. <b>Passo executado:</b> ' + lr + ' &times; (' + gValAnterior.toFixed(3) + ') = <b>' + passoAplicado.toFixed(4) + '</b><br>' +
        '3. <b>Resultado:</b> K &larr; ' + kAntigoVal.toFixed(3) + ' &minus; (' + passoAplicado.toFixed(4) + ') = <b>' + kAtual.toFixed(3) + '</b><br>' +
        '<br><span style="color:#2F6F9F;"><b>Novo gradiente pronto na matriz atual:</b> &nabla;K = <b>' + gAtual.toFixed(3) + '</b></span>' +
        '</div>';
    }

    function renderizarTudo(){
      cap09kgrad_limparDestaques();
      cap09kgrad_esconderTooltip();
      desenharGradeX();
      desenharGradeZ();
      desenharGradeA();
      desenharGradeDZ();
      desenharGradeK();
      atualizarFormula();
      renderizarListaVotos();
      atualizarCalcAtual();
      atualizarSomaTexto();
      desenharGraficoLoss();
      renderizarNotaAtualizacao();

      var valS = cacheForward.S.toFixed(3);
      var valLoss = cacheErro.loss.toFixed(4);

      elTextoS.innerHTML = 'S = <b>' + valS + '</b> (alvo = ' + alvo + ')';
      elTextoLoss.innerHTML = 'L = &frac12;&middot;(' + valS + ' &minus; ' + alvo + ')&sup2; = <b>' + valLoss + '</b>';
      elTextoEpoca.textContent = 'Atualizações: ' + epocaAtual;
    }

    function selecionarPeso(kr, kc){
      pesoSelKr = kr; pesoSelKc = kc;
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, kr, kc).votos;
      var botoes = elSeletorPeso.querySelectorAll('.cap09kgrad_pesobtn');
      botoes.forEach(function(b){ b.classList.remove('cap09kgrad_pesobtn_ativo'); });
      var idxBotao = kr*3+kc;
      if (botoes[idxBotao]) botoes[idxBotao].classList.add('cap09kgrad_pesobtn_ativo');
      renderizarTudo();
    }

    function avancarPosicao(){
      if (posicaoIdx < 16) posicaoIdx += 1;
      renderizarTudo();
    }

    function voltarPosicao(){
      if (posicaoIdx > 0) posicaoIdx -= 1;
      renderizarTudo();
    }

    function somarTudoAutomatico(){
      posicaoIdx = 16;
      renderizarTudo();
    }

    function reiniciarPosicoes(){
      posicaoIdx = 0;
      renderizarTudo();
    }

    function atualizarPesosDoKernel(){
      var gK = cap09kgrad_gradienteKernelCompleto(cacheErro.dZ, X);
      historicoKernel.push({
        K: JSON.parse(JSON.stringify(K)),
        gK: ultimoGradiente,
        kAnt: ultimoKAntigo
      });

      ultimoKAntigo = JSON.parse(JSON.stringify(K));
      ultimoGradiente = gK;

      K = cap09kgrad_atualizarKernel(K, gK, lr);
      epocaAtual += 1;

      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;

      renderizarTudo();
    }

    function desfazerPasso(){
      if (historicoKernel.length === 0) return;
      var estadoAnt = historicoKernel.pop();
      K = estadoAnt.K;
      ultimoGradiente = estadoAnt.gK;
      ultimoKAntigo = estadoAnt.kAnt;
      historicoLoss.pop();
      epocaAtual -= 1;

      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function novoKernelAleatorio(){
      K = cap09kgrad_inicializarKernelVivo(Date.now() % 2147483647, X);
      historicoKernel = [];
      epocaAtual = 0;
      ultimoGradiente = null;
      ultimoKAntigo = null;
      historicoLoss = [];
      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function trocarPadrao(novoPadrao, botaoAtivo){
      [elBtnDiagonal, elBtnVertical].forEach(function(b){ b.classList.remove('cap09kgrad_active'); });
      botaoAtivo.classList.add('cap09kgrad_active');
      padraoAtual = novoPadrao;
      X = cap09kgrad_PADROES[padraoAtual];
      alvo = cap09kgrad_ALVOS[padraoAtual];

      var checagem = cap09kgrad_forward(K, X);
      if (checagem.S <= 0){
        K = cap09kgrad_inicializarKernelVivo(7, X);
      }

      historicoKernel = [];
      epocaAtual = 0;
      ultimoGradiente = null;
      ultimoKAntigo = null;
      historicoLoss = [];
      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function alterarLr(novaLr, btnAtivo){
      [elLr0005, elLr002, elLr02].forEach(function(b){ b.classList.remove('cap09kgrad_active'); });
      btnAtivo.classList.add('cap09kgrad_active');
      lr = novaLr;
      renderizarTudo();
    }

    elBtnDiagonal.addEventListener('click', function(){ trocarPadrao('diagonal', elBtnDiagonal); });
    elBtnVertical.addEventListener('click', function(){ trocarPadrao('vertical', elBtnVertical); });
    elLr0005.addEventListener('click', function(){ alterarLr(0.0005, elLr0005); });
    elLr002.addEventListener('click', function(){ alterarLr(0.002, elLr002); });
    elLr02.addEventListener('click', function(){ alterarLr(0.02, elLr02); });

    elBtnAvancar.addEventListener('click', avancarPosicao);
    elBtnVoltarPos.addEventListener('click', voltarPosicao);
    elBtnSomarTudo.addEventListener('click', somarTudoAutomatico);
    elBtnReiniciarPos.addEventListener('click', reiniciarPosicoes);
    elBtnAtualizarPesos.addEventListener('click', atualizarPesosDoKernel);
    elBtnDesfazerPasso.addEventListener('click', desfazerPasso);
    elBtnNovoKernel.addEventListener('click', novoKernelAleatorio);

    cap09kgrad_configurarHover(elGradeZ, '[data-r]', function(cel, e){
      cap09kgrad_hoverZ(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeA, '[data-r]', function(cel, e){
      cap09kgrad_hoverA(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeDZ, '[data-r]', function(cel, e){
      cap09kgrad_hoverDZ(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeX, '[data-r]', function(cel, e){
      cap09kgrad_hoverX(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeK, '[data-kr]', function(cel, e){
      cap09kgrad_hoverK(parseInt(cel.dataset.kr,10), parseInt(cel.dataset.kc,10), cel, e);
    });

    montarSeletorPeso();
    selecionarPeso(0,0);
  }

  function cap09kgrad_tentarIniciar(){
    var root = document.getElementById('sim-09-gradiente-kernel');
    if (root) cap09kgrad_initSim(root); else setTimeout(cap09kgrad_tentarIniciar, 200);
  }
  cap09kgrad_tentarIniciar();
})();
</script>
'''
)

::: {.callout-note}
#### 🧠 Síntese — Da convolução ao aprendizado de representações {.unnumbered}

Os simuladores desta seção demonstram, de forma sequencial, como uma CNN transforma uma imagem de entrada em uma estimativa probabilística e como seus parâmetros são otimizados durante o treinamento:

- **Convolução:** aplica filtros sobre a imagem para extrair características locais, gerando mapas de características por meio do **compartilhamento de pesos**.
- **ReLU:** introduz não linearidade ao sistema, permitindo a modelagem de relações complexas entre os dados.
- ***Pooling*:** reduz a resolução espacial dos mapas de características, diminuindo o custo computacional e conferindo invariância a pequenas translações locais.
- ***Flatten*:** reorganiza os mapas multidimensionais em um vetor unidimensional para alimentação das camadas subsequentes.
- **Camada totalmente conectada:** combina as características extraídas para produzir os escores brutos (*logits*) associados a cada classe.
- **Softmax:** converte os *logits* em uma distribuição de probabilidades normalizada.
- **Função de perda:** compara a distribuição prevista com a verdade de referência (*ground truth*), quantificando escalarmente o erro da rede.
- **Retropropagação:** aplica a regra da cadeia para calcular a derivada parcial (gradiente) da função de perda em relação a cada parâmetro treinável.
- **Otimizador:** atualiza os coeficientes dos filtros, pesos e vieses na direção oposta ao gradiente, reduzindo a perda a cada iteração.

Ao longo das iterações, os filtros convolucionais convertem-se de valores estocásticos em detectores especializados: as camadas iniciais aprendem primitivas visuais de baixo nível (como bordas e texturas), enquanto as camadas mais profundas consolidam essas representações em estruturas abstratas e semânticas.
:::

## Aplicações Práticas em VC

Após a consolidação teórica dos fundamentos das CNNs e a verificação visual de cada uma de suas operações elementares por meio dos simuladores interativos, torna-se essencial observar a integração dessas etapas em *pipelines* completos de programação. 

Nas seções a seguir, a teoria é traduzida em código executável em **PyTorch**, explorando as três tarefas fundamentais da VC: **classificação**, **detecção de objetos** e **segmentação semântica**. Essa progressão prática permite analisar desde a construção de uma arquitetura convolucional treinada do zero até a aplicação de estratégias avançadas de **transferência de aprendizado** (*transfer learning*) em modelos pré-treinados para conjuntos de dados sintéticos e reais.

### Classificação de Imagens com CNNs

A classificação de imagens é uma das aplicações mais tradicionais das CNNs. Nessa tarefa, o objetivo é atribuir um único rótulo à imagem de entrada, como identificar uma categoria de objeto, uma espécie de animal ou uma classe de diagnóstico. Para isso, a CNN transforma progressivamente os valores dos pixels em representações de maior nível de abstração, combinando camadas convolucionais, funções de ativação e operações de redução espacial até produzir uma distribuição de probabilidades entre as classes possíveis. Nesta seção, são apresentados a arquitetura básica de uma CNN classificadora, o fluxo de transformação dos dados ao longo da rede e o processo de treinamento para ajuste dos parâmetros aprendidos.

#### Treinamento de uma CNN do Zero em Dígitos

Para estabelecer uma comparação direta com as abordagens apresentadas no Capítulo 7, desenvolve-se nesta seção uma CNN treinada sobre o mesmo conjunto de dados de dígitos manuscritos (`load_digits`). A diferença fundamental reside na etapa de representação: enquanto os métodos clássicos dependem de *pixels* brutos ou de descritores calculados manualmente, como o *Histogram of Oriented Gradients* (*HOG*), a CNN aprende automaticamente os coeficientes dos filtros convolucionais durante o processo de otimização.

Os códigos a seguir (consolidados na @fig-09-cnn-treinamento) realizam a preparação dos dados, definem uma arquitetura convolucional simples em **PyTorch**, executam o laço de treinamento por meio do algoritmo *Adam* e geram as curvas de evolução da função de perda e da acurácia.

##### Bloco 1: Preparação e Estruturação dos Dados

A etapa inicial de qualquer *pipeline* de Aprendizado Profundo consiste na conversão e adequação dos dados de entrada para o formato exigido pelo *framework* de computação científica.

###### O Conceito de *Tensor*

Em Aprendizado Profundo, a estrutura fundamental de dados é o ***tensor***. Do ponto de vista computacional, um *tensor* consiste em um arranjo multidimensional de números generalizado para $n$ dimensões:

* Um *tensor* de ordem 0 é um escalar (um único valor).
* Um *tensor* de ordem 1 é um vetor (comprimento).
* Um *tensor* de ordem 2 é uma matriz (linhas e colunas).
* Um *tensor* de ordem 3 ou superior representa um volume ou hiper-arranjo de dados.

No contexto do **PyTorch**, a classe `torch.Tensor` estende a funcionalidade de arranjos numéricos multidimensionais (como os do **NumPy**) ao oferecer suporte a operações aceleradas em hardware por meio de *GPUs* (*Graphics Processing Units*) e suporte ao cálculo automático de derivadas (*autograd*), essencial para o algoritmo de retropropagação.

###### Análise do Código de Pré-processamento

1. **Carregamento e Normalização das Intensidades:**
   O conjunto `load_digits` possui $1.797$ amostras de dígitos manuscritos de $8 \times 8$ *pixels*, cujas intensidades originais variam na escala inteira de $0$ a $16$. A divisão por $16.0$ realiza a **normalização** dos dados para a faixa $[0.0, 1.0]$. Essa transformação em escala flutuante (`float32`) é indispensável em redes neurais para evitar a saturação das funções de ativação e estabilizar o cálculo dos gradientes no algoritmo de otimização.

2. **Divisão Estratificada (70% Treinamento / 30% Teste):**
   A função `train_test_split` separa $70\%$ das amostras para o ajuste dos parâmetros da rede e reserva $30\%$ para a avaliação do modelo em dados não vistos. O parâmetro `stratify=y` garante a amostragem estratificada, mantendo a proporção exata de cada uma das 10 classes de dígitos ($0$ a $9$) em ambos os conjuntos, prevenindo viés de distribuição.

3. **Adequação Dimensional para Convolução 2D (`unsqueeze`):**
   Nas CNNs, camadas convolucionais bidimensionais (`nn.Conv2d`) exigem que o *tensor* de entrada possua estritamente 4 dimensões na ordenação $(N, C, H, W)$:
   * $N$: número de amostras (*batch size*).
   * $C$: número de canais de cor ($1$ para escala de cinza, $3$ para *RGB*).
   * $H$: altura da imagem em *pixels* ($8$).
   * $W$: largura da imagem em *pixels* ($8$).

   Como o arranjo original possui formato $3\text{D}$ do tipo $(N, 8, 8)$, a chamada `.unsqueeze(1)` insere uma dimensão unitária especificamente no **índice 1** (a posição reservada para o canal de cor $C$), transformando a estrutura em um *tensor* $4\text{D}$ de formato $(N, 1, 8, 8)$, conforme exigido pelo PyTorch.

4. **Conversão dos Rótulos (`dtype=torch.long`):**
   Os rótulos das classes $y$ são convertidos em *tensors* inteiros de 64 *bits* (`torch.long`). Essa especificação de tipo é uma exigência da função de perda de Entropia Cruzada (`nn.CrossEntropyLoss`), que utiliza inteiros não negativos como índices para associar a classe correta aos *logits* de saída da rede.

::: {.callout-note appearance="simple"}
**Atenção às Dimensões:** A estrutura final é representada pelo **tensor `(N, 1, 8, 8)`**, onde `N` é o número de amostras (*batch size*), `1` é o canal de cor (escala de cinza) e `8×8` é a resolução espacial da imagem em *pixels*.
:::

A @fig-09-digits-amostra ilustra uma sequência de amostras do conjunto de treinamento após o pré-processamento e adequação dimensional aos *tensors* do **PyTorch**. Na etapa de exibição, a chamada `img.squeeze().numpy()` encadeia duas transformações: o método `.squeeze()` elimina a dimensão unitária redundante do canal de cor, reduzindo o *tensor* $3\text{D}$ de formato `(1, 8, 8)` para uma matriz $2\text{D}$ de `(8, 8)`; em seguida, o método `.numpy()` converte a estrutura do **PyTorch** em uma matriz nativa do **NumPy**, formato exigido pelas ferramentas de renderização gráfica como a `mm.show()`.

In [ ]:
#| label: fig-09-digits-amostra
#| fig-cap: "Amostras de dígitos do conjunto de treinamento após conversão para *tensores* PyTorch e normalização."
#| echo: true
#| output: true

# 1. Carregamento e pré-processamento dos dados
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0  # Normalização para a faixa [0, 1]
y = digits.target

# Divisão estratificada em conjuntos de treinamento (70%) e teste (30%)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Adequação à dimensão esperada pelo PyTorch: (N_amostras, Canais, Altura, Largura)
X_treino_t = torch.tensor(X_treino).unsqueeze(1)   # Dimensão: (N, 1, 8, 8)
y_treino_t = torch.tensor(y_treino, dtype=torch.long)
X_teste_t = torch.tensor(X_teste).unsqueeze(1)
y_teste_t = torch.tensor(y_teste, dtype=torch.long)

# Exibir uma amostra
n_amostras = 8
imgs = [img.squeeze().numpy() for img in X_treino_t[:n_amostras]]
imgs_titles = [str(label.item()) for label in y_treino_t[:n_amostras]]
mm.show(imgs, titles=imgs_titles, cols=n_amostras, figsize=(12, 2.5))

##### Bloco 2: Definição da Arquitetura Convolucional

A construção de modelos no **PyTorch** é estruturada a partir do paradigma de orientação a objetos, criando-se uma classe específica para representar a rede neural (neste exemplo, a classe `CNNDigitos`), a qual herda todas as funcionalidades da classe base `nn.Module`. O construtor `__init__` é responsável por instanciar as camadas e declarar seus parâmetros treináveis, enquanto o método `forward` estabelece a sequência numérica da propagação à frente (*forward pass*).

A @fig-09-cnn-digitos-esquema sintetiza as transformações espaciais dos *tensors* e o fluxo de dados ao longo da classe `CNNDigitos`.

::: {#fig-09-cnn-digitos-esquema}
![](imagens/fig-09-cnn-digitos-esquema.png){width=100% fig-align="center"}

Representação do fluxo de transformações dimensionais dos *tensors* ao longo da arquitetura *CNNDigitos*.
:::

1. **Construtor (`__init__`) e Instanciação dos Componentes:**
   * **Camada Convolucional 1 (`self.conv1`):** Aplica $8$ filtros $3 \times 3$ com `padding=1` sobre a entrada em escala de cinza ($1$ canal), preservando a resolução espacial de $8 \times 8$ *pixels*.
   * **Camada Convolucional 2 (`self.conv2`):** Processa os $8$ mapas de características recebidos da camada anterior aplicando $16$ filtros $3 \times 3$ com `padding=1`.
   * **Subamostragem (`self.pool`):** Instancia a operação de *Max-Pooling* com janela $2 \times 2$ e passo (*stride*) $2$, reduzindo a dimensão espacial (altura e largura) pela metade a cada aplicação.
   * **Camadas Totalmente Conectadas (`self.fc1` e `self.fc2`):** A primeira projeção densa recebe o *tensor* achatado de dimensão $16 \times 2 \times 2 = 64$ e produz $32$ características intermediárias. A segunda projeta essas $32$ características nos $10$ *logits* finais de saída.

2. **Propagação à Frente no Método `forward`:**
   * **Primeiro Bloco Convolucional:** O *tensor* de entrada de formato $(N, 1, 8, 8)$ passa por `conv1` + ReLU e é subamostrado por `pool`, resultando no formato $(N, 8, 4, 4)$.
   * **Segundo Bloco Convolucional:** O *tensor* $(N, 8, 4, 4)$ é processado por `conv2` + ReLU e reduzido por `pool` para o formato $(N, 16, 2, 2)$.
   * **Achatamento (*Flatten*):** O método `x.view(x.size(0), -1)` reconfigura a estrutura $3\text{D}$ em um vetor $1\text{D}$ de $64$ elementos por amostra, preservando a dimensão do lote $N$.
   * **Classificação:** O vetor de $64$ elementos alimenta `fc1` com ativação ReLU ($32$ neurônios) e finaliza em `fc2`, produzindo os $10$ *logits* não normalizados para o cálculo da função de perda.

In [ ]:
#| echo: true
#| output: true

# 2. Definição da Arquitetura Convolucional
class CNNDigitos(nn.Module):
    """
    Arquitetura convolucional compacta:
    2 camadas convolucionais com ReLU e Max-Pooling + 2 camadas densas.
    """
    def __init__(self, n_classes=10):
        super().__init__()
        
        # Conv1: 1 canal de entrada, 8 filtros 3x3 com padding 1 (saída: 8x8)
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        # Conv2: 8 canais de entrada, 16 filtros 3x3 com padding 1 (saída: 4x4)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)

        self.relu = nn.ReLU()

        # Max-Pooling 2x2 com passo (stride) 2
        self.pool = nn.MaxPool2d(2, 2)

        # Camadas totalmente conectadas (FC)
        self.fc1 = nn.Linear(16 * 2 * 2, 32)
        self.fc2 = nn.Linear(32, n_classes)

    def forward(self, x):
        # Primeiro bloco: Conv (8x8) -> ReLU -> Pool (4x4)
        x = self.pool(self.relu(self.conv1(x)))

        # Segundo bloco: Conv (4x4) -> ReLU -> Pool (2x2)
        x = self.pool(self.relu(self.conv2(x)))
        
        # Achatamento (Flatten): reconfigura a matriz 3D (16, 2, 2) em vetor 1D (64)
        x = x.view(x.size(0), -1)

        # Camada densa intermediária com ReLU
        x = self.relu(self.fc1(x))

        # Camada final de classificação (logits)
        return self.fc2(x)

###### Análise das Camadas e do Fluxo da Classe `CNNDigitos`

1. **Construtor (`__init__`) e Instanciação dos Componentes:**
   * **Camada Convolucional 1 (`self.conv1`):** Aplica $8$ filtros $3 \times 3$ com `padding=1` sobre a entrada em escala de cinza ($1$ canal), preservando a resolução de $8 \times 8$ *pixels*.
   * **Camada Convolucional 2 (`self.conv2`):** Processa os $8$ mapas de características recebidos aplicando $16$ filtros $3 \times 3$ com `padding=1`.
   * **Subamostragem (`self.pool`):** Instancia a operação de *Max-Pooling* com janela $2 \times 2$ e passo (*stride*) $2$, reduzindo as dimensões espaciais (altura e largura) pela metade a cada aplicação.
   * **Camadas Totalmente Conectadas (`self.fc1` e `self.fc2`):** A primeira projeção densa recebe o *tensor* achatado de dimensão $16 \times 2 \times 2 = 64$ e produz $32$ características intermediárias. A segunda projeta essas $32$ características nos $10$ *logits* de saída.

2. **Propagação à Frente no Método `forward`:**
   * **Primeiro Bloco:** O *tensor* $(N, 1, 8, 8)$ passa por `conv1` + ReLU e é reduzido por `pool` para $(N, 8, 4, 4)$.
   * **Segundo Bloco:** O *tensor* $(N, 8, 4, 4)$ passa por `conv2` + ReLU e é reduzido por `pool` para $(N, 16, 2, 2)$.
   * **Achatamento (*Flatten*):** O método `x.view(x.size(0), -1)` converte a estrutura $3\text{D}$ em um vetor $1\text{D}$ de $64$ elementos por amostra.
   * **Classificação:** O vetor de $64$ elementos alimenta `fc1` com ativação ReLU ($32$ neurônios) e finaliza em `fc2`, que produz os $10$ *logits* finais para o cálculo da perda de Entropia Cruzada.

##### Bloco 3: Instanciação e Parâmetros de Otimização

A etapa de configuração do aprendizado exige a instanciação da arquitetura definida e a escolha de dois componentes centrais: a **função de perda**, que quantifica o erro do modelo, e o **algoritmo de otimização**, responsável por ajustar os parâmetros em direção ao mínimo dessa função.

1. **Instanciação e Contagem de Parâmetros:**
   O modelo é criado a partir da instanciação do objeto `modelo_cnn` da classe `CNNDigitos`. A expressão `sum(p.numel() for p in modelo_cnn.parameters())` percorre todos os *tensors* de parâmetros treináveis da rede (pesos e vieses de cada camada) e calcula a cardinalidade total do modelo, quantificando sua capacidade de representação.

2. **Função de Perda (`nn.CrossEntropyLoss`):**
   A perda de Entropia Cruzada (*Cross-Entropy Loss*) é a escolha padrão para problemas de classificação multiclasse. No PyTorch, essa implementação combina internamente a aplicação da função *LogSoftmax* com a Perda de Log-Verossimilhança Negativa (*NLLLoss*). Por essa razão, a camada de saída da rede produz *logits* brutos, dispensando a aplicação explícita da função *Softmax* ao final do método `forward`.

3. **Otimizador Adaptativo (`optim.Adam`):**
   A atualização dos parâmetros utiliza o algoritmo *Adam* (*Adaptive Moment Estimation*), com taxa de aprendizado inicial $\eta = 0,01$ (`lr=1e-2`). O *Adam* combina os princípios do momento com a adaptação do tamanho do passo baseada na média móvel das derivadas de primeira e segunda ordens, ajustando individualmente a taxa de aprendizado de cada parâmetro da rede.

In [ ]:
#| echo: true
#| output: true

# 3. Inicialização do Modelo e Parâmetros de Otimização
modelo_cnn = CNNDigitos()
num_params = sum(p.numel() for p in modelo_cnn.parameters())
print(f"Parâmetros treináveis do modelo: {num_params}")

criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_cnn.parameters(), lr=1e-2)

##### Bloco 4: Laço de Treinamento e Avaliação

O treinamento de uma CNN ocorre de forma iterativa por meio do algoritmo de Gradiente Descendente Estocástico por *mini-batches* (*Mini-batch SGD*).

As curvas de aprendizado resultantes desse processo são apresentadas na @fig-09-cnn-treinamento, gerada ao final da execução.

1. **Fase de Treinamento (`modelo_cnn.train()`):**
   O laço principal executa o treinamento ao longo de $50$ épocas. Em cada época, ocorrem as seguintes etapas:
   * **Embaralhamento Estocástico:** A função `torch.randperm(n)` gera uma permutação aleatória dos índices das amostras, garantindo que a ordenação dos *mini-batches* varie a cada época para evitar vícios de amostragem.
   * **Divisão em *Mini-batches*:** O conjunto de treinamento é fatiado em lotes de $32$ amostras (`tam_lote = 32`).
   * **Zerar Gradientes (`otimizador.zero_grad()`):** Limpa os gradientes acumulados no *tensor* na iteração anterior, evitando a soma indesejada de derivadas entre lotes distintos.
   * **Passo à Frente e Perda:** O *forward pass* calcula as previsões `saida`, e a chamada `criterio(saida, y_treino_t[idx])` quantifica o erro do lote.
   * **Retropropagação (`perda.backward()`):** Aplica a regra da cadeia para calcular as derivadas parciais da perda em relação a cada parâmetro ($\frac{\partial L}{\partial w}$).
   * **Atualização dos Pesos (`otimizador.step()`):** Atualiza os parâmetros do modelo segundo as equações do otimizador *Adam*.

2. **Fase de Avaliação (`modelo_cnn.eval()`):**
   Ao final de cada época, o modelo é alterado para o modo de avaliação. O contexto `with torch.no_grad()` desativa temporariamente o motor de cálculo automático de derivadas (*autograd*), reduzindo o consumo de memória e acelerando a inferência sobre o conjunto de teste (`X_teste_t`). A operação `.argmax(dim=1)` extrai a classe de maior probabilidade para cada amostra, permitindo calcular a acurácia de teste.

3. **Visualização com a Biblioteca `morph`:**
   A função `mm.showTrainCurves` da biblioteca didática `morph` consolida o histórico de perda de treinamento e a acurácia de teste em um único painel gráfico, permitindo diagnosticar a convergência do modelo e monitorar a estabilidade do aprendizado ao longo das épocas.

In [ ]:
#| label: fig-09-cnn-treinamento
#| fig-cap: "Curvas de treinamento e avaliação da CNN na base de dígitos: evolução da perda de entropia cruzada no conjunto de treinamento e da acurácia no conjunto de teste ao longo de 50 épocas."
#| echo: true
#| output: true

# 4. Laço de Treinamento (Mini-batch SGD)
n = X_treino_t.size(0)                    # Número de amostras
tam_lote = 32                             # Tamanho do mini-batch
epocas = 50                               # Total de épocas
historico_perda, historico_acc = [], []   # Histórico de métricas

for epoca in range(epocas):                  # Repete por época
    modelo_cnn.train()                       # Modo treinamento
    perm = torch.randperm(n)                 # Embaralha amostras
    perda_epoca = 0.0                        # Acumula perdas
    for i in range(0, n, tam_lote):          # Percorre mini-batches
        idx = perm[i:i + tam_lote]           # Índices do lote
        otimizador.zero_grad()               # Zera gradientes
        saida = modelo_cnn(X_treino_t[idx])  # Propagação direta
        perda = criterio(saida, y_treino_t[idx]) # Calcula perda
        perda.backward()                         # Retropropagação
        otimizador.step()                        # Atualiza pesos
        perda_epoca += perda.item() * len(idx)   # Soma perda

    # Avaliação do modelo no conjunto de teste ao final de cada época
    modelo_cnn.eval()                            # Modo avaliação
    with torch.no_grad():                        # Sem gradientes
        pred_teste = modelo_cnn(X_teste_t).argmax(dim=1)  # Predições
        acc_teste = (pred_teste == y_teste_t).float().mean().item()  # Acurácia
    historico_perda.append(perda_epoca / n)      # Registra perda
    historico_acc.append(acc_teste)              # Registra acurácia

acc_final_cnn = historico_acc[-1]                # Última acurácia
print(f"Acurácia final da CNN no conjunto de teste: {acc_final_cnn:.4f}")  # Exibe resultado

final = mm.showTrainCurves(                      # Plota curvas
    historico_perda, historico_acc,
    titulo="Evolução do Treinamento da CNN — Base de Dígitos",
    subtitulo=f"Acurácia final no teste: {acc_final_cnn:.4f}",
)

##### Bloco 5: Visualização do Fluxo de Ativações

A inspeção da rede treinada permite observar a transformação progressiva do *tensor* de entrada ao longo das camadas da arquitetura `CNNDigitos`. A @fig-09-cnn-ativacoes ilustra as dimensões e as ativações intermediárias obtidas ao processar um exemplo real do dígito $3$.

1. **Seleção e Preparação da Amostra:**
   A semente estocástica é fixada com `torch.manual_seed(7)` para assegurar a reprodutibilidade dos resultados. A primeira ocorrência do dígito $3$ no conjunto de dados `load_digits` é isolada, normalizada para o intervalo $[0.0, 1.0]$ e reconfigurada como um *tensor* `x` de dimensão $(1, 1, 8, 8)$.

2. **Inspeção Intermediária com `mm.showNet`:**
   A função `mm.showNet` da biblioteca `morph` executa a propagação à frente (*forward pass*) do *tensor* `x` na instância `modelo_cnn` previamente treinada. Utilizando *hooks* de *forward*, a função intercepta o estado numérico das ativações nas camadas convolucionais (`nn.Conv2d`), de agrupamento (`nn.MaxPool2d`) e totalmente conectadas (`nn.Linear`), retornando-as no dicionário `acts`. As funções de ativação não linear (`nn.ReLU`) não são registradas como estágios independentes, pois sua aplicação ocorre diretamente sobre o *tensor* de saída da camada correspondente.

3. **Verificação dos Resultados:**
   A instrução `list(acts.keys())` exibe a sequência de identificadores das camadas monitoradas, permitindo confirmar a redução dimensional progressiva e a geração do *logit* de valor máximo no índice correspondente à classe $3$, conforme demonstrado na @fig-09-cnn-ativacoes.

In [ ]:
#| label: fig-09-cnn-ativacoes
#| fig-cap: "Fluxo de ativações da CNN treinada ao processar um exemplo real do dígito 3, do *dataset* *load_digits*: dimensões dos tensores camada a camada, da entrada ao *logit* de saída."
#| echo: true
#| output: true

torch.manual_seed(7)
digits = load_digits()
idx = np.where(digits.target == 3)[0][0]
img = digits.images[idx] / 16.0
x = torch.tensor(img, dtype=torch.float32).view(1, 1, 8, 8)

# Reutilização da instância do modelo previamente treinada
acts = mm.showNet(
    modelo_cnn,
    x,
    titulo="Fluxo de transformações dos tensors ao longo da arquitetura CNNDigitos",
    subtitulo=f"Exemplo real do dataset load_digits (classe verdadeira: {digits.target[idx]})",
)
print("Camadas capturadas:", list(acts.keys()))

##### Inspecionando o Grafo Computacional com `torchviz`

Enquanto `mm.showNet` prioriza a clareza didática — exibindo uma coluna por camada com parâmetros treináveis —, a biblioteca `torchviz` projeta o **grafo de autograd** exatamente como o PyTorch o constrói internamente para o cálculo de gradientes. A @fig-09-torchviz-grafo ilustra essa perspectiva ao representar a arquitetura `CNNDigitos`.

1. **Propagação à Frente Rastreada:** Com o modelo treinado em modo `eval()`, o *forward pass* sobre o *tensor* `x` do dígito $3$ é suficiente para que o motor de *autograd* registre todas as operações executadas, incluindo aquelas sem parâmetros treináveis, como a função de ativação `ReLU` e a reconfiguração dimensional `view`.

2. **Geração do Grafo (`make_dot`):** A função `make_dot(saida, params=...)` constrói o grafo a partir do *tensor* de saída, percorrendo retroativamente o histórico de operações até os nós folha (os parâmetros treináveis do modelo). Cada nó do diagrama representa uma operação do *backward pass* (como `ReluBackward` ou `AddmmBackward`), e não apenas um bloco conceitual do `nn.Module`.

3. **Exportação e Renderização (`.render`):** O método `.render(..., format="png", cleanup=True)` invoca o executável `dot` do **Graphviz** para compilar a imagem em formato PNG, eliminando automaticamente os arquivos intermediários de código-fonte.

A @fig-09-torchviz-grafo evidencia como esse grafo computacional, mesmo para uma arquitetura compacta, apresenta maior densidade que o painel do `mm.showNet`, pois detalha cada operação atômica responsável pelo fluxo de gradientes.

In [ ]:
#| label: fig-09-torchviz-grafo
#| fig-cap: "Grafo computacional da *CNNDigitos* gerado via *torchviz*, mostrando as operações de *forward* e os nós de gradiente (*backward*) associados a cada parâmetro treinável."
#| echo: true
#| output: true

# 1. Forward pass com rastreamento de gradiente habilitado
modelo_cnn.eval()
saida = modelo_cnn(x)  # Reaproveitamento do tensor x (dígito 3)

# 2. Grafo básico: fluxo de operações até a saída
grafo_simples = make_dot(saida, params=dict(modelo_cnn.named_parameters()))
caminho_simples = grafo_simples.render("cnn_digitos_grafo_simples", format="png", cleanup=True)

# 3. Exibição direta no ambiente Quarto/Jupyter
# O .render() retorna o caminho do arquivo PNG gerado; precisamos abri-lo como imagem
imagem_simples = np.array(Image.open(caminho_simples).convert("RGB"))
mm.show(imagem_simples, figsize=(8,16))

##### Visão Detalhada do Grafo Computacional com `torchviz`

Além da representação simplificada, a biblioteca `torchviz` permite expandir o grafo de *autograd* para inspecionar os detalhes internos de execução da rede `CNNDigitos`. A @fig-09-torchviz-grafo-detalhado apresenta essa estrutura expandida para o mesmo *tensor* de entrada `x`.

1. **Rastreamento com Atributos de Operação (`show_attrs=True`):** 
   A inclusão de atributos exibe as configurações hiperparamétricas associadas a cada nó computacional durante a propagação à frente (*forward pass*), tais como dimensões de *kernel* (`kernel_size`), passos (*stride*) e preenchimentos (*padding*) nas convoluções e subamostragens.

2. **Detecção de *Tensors* Salvos em Memória (`show_saved=True`):**
   O parâmetro força a exibição explícita dos *tensors* intermediários que o PyTorch retém na memória durante o *forward pass*. Esses dados são preservados porque serão estritamente necessários para o cálculo das derivadas parciais durante a etapa de retropropagação (*backward pass*).

3. **Geração e Compilação dos Grafos:**
   Enquanto `grafo_simples` gera uma visão direta do fluxo de gradientes, `grafo_detalhado` compila o grafo expandido no arquivo `cnn_digitos_grafo_detalhado.png` via executável `dot` do **Graphviz**.

Conforme observado na @fig-09-torchviz-grafo-detalhado, essa visualização minuciosa é útil para depurar o consumo de memória de vídeo (*VRAM*) e verificar como o motor do PyTorch aloca internamente cada nó da regra da cadeia.

In [ ]:
#| label: fig-09-torchviz-grafo-detalhado
#| fig-cap: "Grafo detalhado da classe CNNDigitos gerado via *torchviz*."
#| echo: true
#| output: true

# mesmo código anterior (make_dot + render)...

# 2. Grafo detalhado: exibição de dimensões e tensores salvos para o backward
grafo_detalhado = make_dot(
    saida,
    params=dict(modelo_cnn.named_parameters()),
    show_attrs=True,   # Exibe atributos das operações (ex.: kernel_size, stride)
    show_saved=True,   # Exibe tensores salvos na memória para a retropropagação
)

caminho_detalhado = grafo_detalhado.render("cnn_digitos_grafo_detalhado", 
                                           format="png", cleanup=True)

# 3. Exibição direta no ambiente Quarto/Jupyter
imagem_detalhado = np.array(Image.open(caminho_detalhado).convert("RGB"))
mm.show(imagem_detalhado, figsize=(8, 16))

##### Comparando com o Capítulo 7

A @fig-09-comparativo-cap7 reúne os resultados obtidos no mesmo conjunto de dados (`load_digits`), estabelecendo um paralelo direto entre as abordagens clássicas exploradas anteriormente e a CNN desenvolvida neste capítulo.

1. **Desempenho dos *Pixels* Brutos vs. Descritores Manuais:** Nos experimentos do Capítulo 7, o classificador $k\text{-NN}$ ($k=3$) atingiu uma acurácia de $98,4\%$ quando alimentado diretamente com os *pixels* brutos das imagens. Em contrapartida, a extração prévia de características via *Histogram of Oriented Gradients* (*HOG*) resultou em um desempenho significativamente inferior ($75,8\%$). Essa queda ocorre porque o *HOG* foi concebido para capturar gradientes de bordas em imagens de maior resolução; em matrizes de apenas $8 \times 8$ *pixels*, a resolução espacial é insuficiente para formar histogramas de orientação informativos.

2. **Equivalência da CNN e Aprendizado *End-to-End*:** A rede convolucional `CNNDigitos` alcança um desempenho competitivo de $97,6\%$, aproximando-se da acurácia do $k\text{-NN}$ com *pixels* brutos em uma base pequena e pré-alinhada. A grande vantagem conceitual reside no aprendizado de representação: em vez de depender de descritores projetados manualmente (*handcrafted features*) ou de manter todo o conjunto de dados em memória para a busca por vizinhos no momento da inferência, a CNN otimiza automaticamente seus próprios filtros convolucionais durante o treinamento, gerando um modelo compacto capaz de realizar a extração de características e a classificação de forma integrada (*end-to-end*).

In [ ]:
#| label: fig-09-comparativo-cap7
#| fig-cap: "Comparação de acurácia entre os classificadores clássicos do Capítulo 7 (pixels brutos e HOG com k-NN) e a CNN treinada neste capítulo, na mesma base de dígitos."
#| echo: true
#| output: true

import matplotlib.pyplot as plt

# Valores obtidos no Capítulo 7 (k-NN, k=3), reproduzidos para comparação direta
ACC_KNN_PIXELS_CAP7 = 0.9844
ACC_KNN_HOG_CAP7 = 0.7578

metodos = ["k-NN\n(pixels brutos)", "k-NN\n(HOG)", "CNN\n(este capítulo)"]
acuracias = [ACC_KNN_PIXELS_CAP7, ACC_KNN_HOG_CAP7, acc_final_cnn]

plt.figure(figsize=(5, 4))
cores = ["#6366f1", "#f97316", "#16a34a"]
plt.bar(metodos, acuracias, color=cores)
plt.ylim(0, max(acuracias) + 0.08)
plt.ylabel("Acurácia (conjunto de teste)")
plt.title("Cap. 7 vs. Cap. 9 — Base de Dígitos")

for i, v in enumerate(acuracias):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

::: {.callout-note}
###### 🧠 Por que funciona? — E por que a CNN nem sempre "ganha" {.unnumbered}

O resultado observado aqui **repete o padrão já visto no Capítulo 7**: a CNN, apesar de aprender automaticamente suas características, não supera necessariamente o $k\text{-NN}$ com *pixels* brutos nesta base específica. A explicação é a mesma: `load_digits` é uma base pequena (menos de $1.800$ exemplos), com imagens já centralizadas, normalizadas e de baixíssima resolução ($8 \times 8$) — condições em que a comparação direta de intensidades já é altamente informativa, e há poucos dados para que a rede aprenda filtros verdadeiramente superiores aos descritores simples.

O verdadeiro diferencial das CNNs aparece em cenários que descritores artesanais e classificadores simples não conseguem endereçar: imagens maiores e mais realistas, com milhares de categorias, variação substancial de pose, iluminação e fundo, e conjuntos de treinamento massivos — exatamente o regime em que os modelos apresentados na seção *"Aplicações em Larga Escala"*, mais adiante, foram treinados. A lição pedagógica que atravessa os Capítulos 7, 8 e 9 deste livro é consistente: **a sofisticação de um método deve ser proporcional à complexidade do problema** — usar uma CNN para um problema que um $k\text{-NN}$ resolve igualmente bem é desperdício de recursos computacionais, não uma virtude.

Essa mesma proporcionalidade vale para as ferramentas de inspeção usadas ao longo do capítulo. O `mm.showNet` foi construído para fins **didáticos** e funciona bem em redes rasas como a `CNNDigitos`, mas não escala para arquiteturas profundas: cada camada rastreada vira uma coluna na figura, e camadas convolucionais com centenas de canais geram mosaicos grandes demais para interpretação visual; além disso, os *hooks* armazenam todas as ativações em memória, e o *layout* assume um fluxo sequencial, não representando fielmente conexões residuais ou ramificações (como em *ResNets* ou módulos *Inception*). Assim, `showNet` deve ser entendido como uma lente pedagógica para redes pequenas — análoga ao papel de `mm.showBoundBox` na depuração visual de detecções — e não como substituto de ferramentas voltadas à produção, como *TensorBoard* ou *torchviz*.
:::

#### Transferência de Aprendizado

Treinar uma CNN do zero geralmente requer uma grande quantidade de dados rotulados e recursos computacionais significativos, pois o processo de treinamento precisa ajustar todos os parâmetros da rede. Em muitas aplicações, entretanto, apenas um conjunto reduzido de dados está disponível para a tarefa de interesse. Nessa situação, a **transferência de aprendizado** (*transfer learning*) reutiliza as representações aprendidas por um modelo previamente treinado em uma tarefa de origem com grande volume de dados, reduzindo o custo de treinamento e a necessidade de novas amostras.

Na VC, essa estratégia explora a organização hierárquica das CNNs. As camadas iniciais aprendem características visuais de baixo nível, como bordas, texturas, gradientes de intensidade e padrões de cores, que permanecem úteis em diferentes domínios. As camadas mais profundas combinam essas informações para formar representações progressivamente mais abstratas e especializadas, relacionadas às classes presentes na base de treinamento.

Esta seção investiga em quais condições a transferência de aprendizado produz bons resultados. O primeiro experimento mostra que um extrator pequeno e treinado em um domínio restrito pode levar à **transferência negativa** (*negative transfer*). O segundo demonstra por que modelos profundos pré-treinados em grandes bases de imagens alcançam elevado desempenho em novas tarefas. Por fim, o terceiro aplica essa estratégia a um problema de diagnóstico fitossanitário, ilustrando um cenário próximo de aplicações reais.

##### Experimento 1 — Limitações de um Extrator Pequeno e Especializado

O primeiro experimento mostra que a transferência de aprendizado nem sempre melhora o desempenho de um modelo. Para isso, o conjunto de dígitos manuscritos (`load_digits`) é dividido em dois domínios disjuntos:

- **Domínio A (origem):** dígitos $0$ a $4$, utilizados para treinar uma pequena CNN;
- **Domínio B (destino):** dígitos $5$ a $9$, reindexados para $0$ a $4$, formando uma nova tarefa com apenas $20$ amostras de treinamento.

O objetivo consiste em avaliar o efeito de reutilizar o extrator de características aprendido no Domínio A sem permitir sua adaptação ao Domínio B.

###### Bloco 1: Divisão dos Domínios, Escassez e Exibição das Amostras

Este bloco prepara o conjunto de dados para o experimento. Diferentemente do projeto anterior, que utilizou todos os dígitos em um único problema de classificação, a base é dividida em duas tarefas independentes: uma tarefa de origem (Domínio A) e uma tarefa de destino (Domínio B).

A @fig-09-transfer-amostras apresenta exemplos dos dois domínios após a separação das classes, a conversão para *tensors* do **PyTorch** e o pré-processamento.

1. **Separação das classes:** As máscaras booleanas `mask_A` e `mask_B` separam os exemplos de cada domínio. Em seguida, o código reindexa os rótulos do Domínio B (`y[mask_B] - 5`) para o intervalo $[0,4]`, permitindo que ambos os modelos utilizem cinco classes de saída.

2. **Escassez de dados:** O gerador `np.random.default_rng(0)` seleciona apenas $20$ amostras para o treinamento do Domínio B, aproximadamente quatro por classe, simulando um cenário em que o treinamento do zero tende a sofrer com *overfitting*.

3. **Conversão para *tensors*:** A função `para_tensor` converte as imagens para o formato $(N,1,8,8)$ e os rótulos para `torch.long`, compatíveis com as camadas `nn.Conv2d` e a função de perda.

4. **Visualização das amostras:** O código utiliza `.squeeze().numpy()` para converter os *tensors* em matrizes do **NumPy**. A @fig-09-transfer-amostras apresenta exemplos dos dois domínios e evidencia a reindexação aplicada aos rótulos do Domínio B.

In [ ]:
#| label: fig-09-transfer-amostras
#| fig-cap: "Amostras dos conjuntos de treinamento após pré-processamento e adequação dimensional aos tensores PyTorch: Domínio A (dígitos 0 a 4, tarefa de origem) e Domínio B (dígitos 5 a 9 reindexados para 0 a 4, tarefa de destino)."
#| echo: true
#| output: true

# 1. Divisão do dataset em dois domínios disjuntos
classes_A, classes_B = [0, 1, 2, 3, 4], [5, 6, 7, 8, 9]
mask_A, mask_B = np.isin(y, classes_A), np.isin(y, classes_B)

XA, yA = X[mask_A], y[mask_A]
XB, yB = X[mask_B], y[mask_B] - 5  # Reindexação dos rótulos para o intervalo [0, 4]

# Divisão em treino e teste para ambos os domínios
XA_tr, XA_te, yA_tr, yA_te = train_test_split(
    XA, yA, test_size=0.25, random_state=42, stratify=yA
)
XB_tr, XB_te, yB_tr, yB_te = train_test_split(
    XB, yB, test_size=0.25, random_state=42, stratify=yB
)

# Simulação de escassez extrema no domínio de destino: apenas 20 amostras de treino
rng = np.random.default_rng(0)
idx_poucos = rng.choice(len(XB_tr), size=20, replace=False)
XB_tr_poucos, yB_tr_poucos = XB_tr[idx_poucos], yB_tr[idx_poucos]

# Função auxiliar para conversão em tensores PyTorch
def para_tensor(Ximg, yarr):
    return torch.tensor(Ximg).unsqueeze(1), torch.tensor(yarr, dtype=torch.long)

XA_tr_t, yA_tr_t = para_tensor(XA_tr, yA_tr)
XA_te_t, yA_te_t = para_tensor(XA_te, yA_te)
XB_tr_t, yB_tr_t = para_tensor(XB_tr_poucos, yB_tr_poucos)
XB_te_t, yB_te_t = para_tensor(XB_te, yB_te)

# Exibição de amostras de ambos os domínios
n_amostras = 5
imgs_A = [img.squeeze().numpy() for img in XA_tr_t[:n_amostras]]
titles_A = [f"A: {label.item()}" for label in yA_tr_t[:n_amostras]]

imgs_B = [img.squeeze().numpy() for img in XB_tr_t[:n_amostras]]
titles_B = [f"B: {label.item()} (orig: {label.item()+5})" for label in yB_tr_t[:n_amostras]]

mm.show(
    imgs_A + imgs_B,
    titles=titles_A + titles_B,
    cols=n_amostras,
    figsize=(12, 4.5)
)

###### Bloco 2: Arquitetura Modular e Rotinas Genéricas

Para viabilizar a transferência de aprendizado, a arquitetura convolucional e o laço de treinamento foram refatorados em relação à classe `CNNDigitos` do projeto anterior.

1. **Modularização da Arquitetura (Diferença para `CNNDigitos`):**
   * No projeto anterior, a classe `CNNDigitos` declarava todas as camadas (`conv1`, `conv2`, `pool`, `fc1`, `fc2`) como membros diretos de uma única classe monolítica.
   * Aqui, a arquitetura é separada em dois componentes: a classe `ExtratorConv` encapsula o bloco espacial convolucional ($2$ convoluções $3 \times 3$, $2$ *Max-Poolings* $2 \times 2$ e o achatamento para $64$ elementos), enquanto a classe `CNNCompleta` instancia esse extrator em `self.extrator` e anexa a "cabeça" classificadora (`fc1` e `fc2`).
   * Essa separação é o que permite copiar o estado interno do extrator (`state_dict()`) de um modelo para outro de forma isolada.

2. **Ajuste no Número de Classes de Saída:**
   Enquanto `CNNDigitos` no projeto anterior possuía $10$ *logits* na camada de saída (`self.fc2 = nn.Linear(32, 10)`), a classe `CNNCompleta` recebe `n_classes=5` no construtor para adequar-se à divisão dos domínios $A$ e $B$.

3. **Flexibilização do Laço de Treinamento (`treinar`):**
   * No projeto anterior, o laço de treinamento iterava diretamente sobre os atributos globais do modelo (`modelo_cnn.parameters()`) e calculava métricas específicas em linha.
   * A função `treinar` abstrai esse processo e introduz o parâmetro opcional `parametros`. Se fornecido, o otimizador *Adam* atualiza **apenas** os parâmetros dessa lista, ignorando as camadas cujos gradientes foram desativados. Essa flexibilidade é crucial para executar o treinamento com congelamento parcial da rede.

4. **Isolamento da Avaliação (`calcular_acuracia`):**
   Assim como feito na fase de teste do projeto anterior, a função coloca o modelo em `eval()` e utiliza o contexto `torch.no_grad()` para desativar o *autograd*, calculando a acurácia via `.argmax(dim=1)`.

In [ ]:
#| echo: true
#| output: true

# Definição do bloco convolucional reaproveitável (mesma extração do projeto anterior)
class ExtratorConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return x.view(x.size(0), -1)

# Arquitetura modular combinando o extrator e a cabeça classificadora
class CNNCompleta(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.extrator = ExtratorConv()
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.extrator(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Rotina genérica de treinamento com otimização seletiva de parâmetros
def treinar(modelo, X_t, y_t, epocas, lr, tam_lote=16, parametros=None):
    # Parâmetros treináveis
    params = parametros if parametros is not None else modelo.parameters()  
    otim = optim.Adam(params, lr=lr)               # Otimizador Adam
    crit = nn.CrossEntropyLoss()                   # Função de perda
    n_amostras = X_t.size(0)                       # Número de amostras
    for _ in range(epocas):                        # Repete por época
        perm = torch.randperm(n_amostras)          # Embaralha amostras
        for i in range(0, n_amostras, tam_lote):   # Percorre mini-batches
            idx = perm[i:i + tam_lote]             # Índices do lote
            otim.zero_grad()                       # Zera gradientes
            perda = crit(modelo(X_t[idx]), y_t[idx])  # Calcula perda
            perda.backward()                       # Retropropagação
            otim.step()                            # Atualiza pesos

# Rotina de avaliação
def calcular_acuracia(modelo, X_t, y_t):
    modelo.eval()                                  # Modo avaliação
    with torch.no_grad():                          # Sem gradientes
        pred = modelo(X_t).argmax(dim=1)           # Classes preditas
    return (pred == y_t).float().mean().item()     # Retorna acurácia

###### Bloco 3: Pré-treinamento, Transferência e Análise Comparativa

Este bloco executa a comparação entre o ajuste do modelo a partir do zero e a aplicação da transferência com congelamento estático do extrator. Para dar total transparência ao experimento, os tamanhos dos conjuntos de treino e teste de ambos os domínios são impressos no terminal.

1. **Quantificação das Amostras por Domínio:**
   * **Domínio A (origem, dígitos $0$ a $4$):** Conta com $675$ amostras de treino ($75\%$) e $226$ de teste ($25\%$), fornecendo dados abundantes para que o `modelo_origem` aprenda o extrator convolucional até atingir $100\%$ de acurácia.
   * **Domínio B (destino, dígitos $5$ a $9$):** Possui $224$ amostras de teste no total, mas seu conjunto de treino é intencionalmente reduzido de $672$ para apenas **$20$ amostras** (`XB_tr_poucos`), criando um cenário severo de escassez de dados.

2. **Etapa 1: Pré-treinamento no Domínio A (Origem):**
   O `modelo_origem` é treinado do zero sobre as $675$ amostras dos dígitos $0$ a $4$. Durante $40$ épocas, o extrator convolucional ajusta seus filtros para identificar os traços característicos desses cinco primeiros dígitos, atingindo $100\%$ de acurácia no conjunto de teste ($226$ amostras).

3. **Etapa 2: Transferência de Pesos e Congelamento:**  
   * Cria-se o `modelo_transferencia` para resolver a tarefa do Domínio B (dígitos $5$ a $9$).
   * Os pesos aprendidos no Domínio A são copiados via:
  
     - `load_state_dict(modelo_origem.extrator.state_dict())`
  
   * **Congelamento:** O laço `for p in modelo_transferencia.extrator.parameters(): p.requires_grad = False` desativa o cálculo de gradientes nas camadas convolucionais.
   * **Treinamento Seletivo:** A chamada `treinar(...)` passa estritamente os parâmetros das camadas densas (`params_cabeca`), ajustando a cabeça de classificação com apenas as $20$ amostras de treino.

4. **Etapa 3: Treinamento do Zero no Domínio B (Controle Experimental):**
   O `modelo_do_zero` possui a mesma arquitetura, mas é treinado do zero sobre as mesmas $20$ amostras do Domínio B, sem qualquer reaproveitamento de pesos, pelas mesmas $40$ épocas.

5. **Análise dos Resultados (@fig-09-transfer-learning):**
   * **Com Transferência Congelada ($72,77\%$):** Ao reaproveitar o extrator treinado no Domínio A e congelar seus parâmetros, a rede alcança $72,77\%$ de acurácia no teste ($224$ amostras) ajustando apenas as camadas densas.
   * **Treinado do Zero ($76,79\%$):** O treinamento do zero supera a transferência congelada no conjunto de teste do Domínio B.
   * **Causa da Diferença:** Por se tratar de um modelo minúsculo (apenas $16$ filtros convolucionais em matrizes de $8 \times 8$), o extrator treinado no Domínio A tornou-se **hiperespecializado** nas formas geométricas dos dígitos $0$ a $4$. Ao congelar rigidamente esses poucos filtros, o modelo de destino ficou limitado a detectores inadequados para $5$ a $9$. A rede treinada do zero, mesmo com apenas $20$ amostras, conseguiu adaptar seus $16$ filtros diretamente aos traços do Domínio B.

In [ ]:
#| label: fig-09-transfer-learning
#| fig-cap: "Comparação de acurácia no conjunto de teste do Domínio B (dígitos 5 a 9) sob restrição de dados (20 exemplos de treino): demonstração do impacto do congelamento rígido e da transferência negativa em redes de baixa capacidade."
#| echo: true
#| output: true

# Fixar semente para reprodutibilidade
torch.manual_seed(42)

# Exibição do tamanho dos grupos de treino e teste
print("=== Detalhamento do Tamanho das Bases ===")
print(f"Domínio A (0-4) — Treino: {len(XA_tr_t)} amostras | Teste: {len(XA_te_t)} amostras")
print(f"Domínio B (5-9) — Treino completo: {len(XB_tr)} | Treino reduzido: {len(XB_tr_poucos)}",
      f"| Teste: {len(XB_te_t)} amostras\n")

# 1. Pré-treinamento na tarefa de origem (Domínio A: dígitos 0-4)
modelo_origem = CNNCompleta(n_classes=5)                       # Cria CNN

#######
treinar(modelo_origem, XA_tr_t, yA_tr_t, epocas=40, lr=1e-2)   # Treina modelo
         
acc_A = calcular_acuracia(modelo_origem, XA_te_t, yA_te_t)     # Mede acurácia
                          
print(f"Acurácia no domínio de origem A "                      # Exibe resultado
      f"(dígitos 0-4, {len(XA_te_t)} testes): " f"{acc_A:.4f}")

# 2. Transferência de Aprendizado (Extrator Congelado)
modelo_transferencia = CNNCompleta(n_classes=5)        # Cria CNN
modelo_transferencia.extrator.load_state_dict(         # Copia extrator
    modelo_origem.extrator.state_dict())

for p in modelo_transferencia.extrator.parameters():   # Percorre extrator
    p.requires_grad = False                            # Congela pesos

params_cabeca = list(modelo_transferencia.fc1.parameters())  # FC1
params_cabeca += list(modelo_transferencia.fc2.parameters()) # +FC2

#######
treinar(modelo_transferencia, XB_tr_t, yB_tr_t,              # Treina cabeça
         epocas=40, lr=1e-2, parametros=params_cabeca)

acc_transferencia = calcular_acuracia(modelo_transferencia, XB_te_t, yB_te_t) # Mede acurácia

# 3. Treinamento do Zero no Domínio B
modelo_do_zero = CNNCompleta(n_classes=5)                     # Cria CNN

#######
treinar(modelo_do_zero, XB_tr_t, yB_tr_t, epocas=40, lr=1e-2) # Treina modelo
         
acc_do_zero = calcular_acuracia(modelo_do_zero,  XB_te_t, yB_te_t) # Mede acurácia
                               

print(f"Domínio de destino B (dígitos 5-9), apenas {len(XB_tr_poucos)} ", 
      f"exemplos de treino ({len(XB_te_t)} testes):")
print(f"  Com transferência (extrator congelado): {acc_transferencia:.4f}")
print(f"  Treinando do zero (mesmos dados/épocas): {acc_do_zero:.4f}")

# Visualização comparativa
plt.figure(figsize=(4.5, 4))
plt.bar(["Do zero", "Transferência"], [acc_do_zero, acc_transferencia], 
        color=["#dc2626", "#16a34a"])
plt.ylim(0, max([acc_do_zero, acc_transferencia]) + 0.1)
plt.ylabel("Acurácia no domínio B (teste)")
plt.title(f"Efeito da Transferência ({len(XB_tr_poucos)} exemplos de treino)")

for i, v in enumerate([acc_do_zero, acc_transferencia]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

###### Bloco 4: Visualização do Fluxo de Ativações com `mm.showNet`

Para confirmar que a extração de características reutilizada preserva as transformações dimensionais estudadas no projeto anterior, utiliza-se novamente a função `mm.showNet` da biblioteca `morph`. A @fig-09-transfer-shownet exibe o fluxo de ativações do `modelo_transferencia` ao processar uma amostra do Domínio B (dígito $7$, reindexado para a classe $2$).

1. **Preservação do Fluxo Convolucional:** Como a arquitetura `ExtratorConv` replica as mesmas camadas de convolução e *pooling* da `CNNDigitos` do projeto anterior, as dimensões dos *tensors* intermediários mantêm-se em $(1, 8, 4, 4)$ no primeiro bloco.
2. **Inspeção da Cabeça Adaptada:** A diferença em relação ao projeto anterior surge na camada de saída (`fc2`): enquanto o modelo do projeto anterior projetava o vetor intermediário em $10$ *logits* (classes de $0$ a $9$), o modelo de transferência projeta o vetor em $5$ *logits* (classes de $0$ a $4$), capturando as probabilidades relativas do Domínio B.

In [ ]:
#| label: fig-09-transfer-shownet
#| fig-cap: "Fluxo de ativações e transformações dimensionais dos tensores no modelo de transferência de aprendizado ao processar uma amostra de teste do Domínio B (dígito 7, reindexado para classe 2)."
#| echo: true
#| output: true

# Seleciona a primeira amostra de teste do Domínio B
x_amostra_B = XB_te_t[0:1]  # Tensor de dimensão (1, 1, 8, 8)
classe_verdadeira = yB_te_t[0].item()
classe_original = classe_verdadeira + 5

# Inspeção do fluxo de ativações no modelo de transferência
acts_transfer = mm.showNet(
    modelo_transferencia,
    x_amostra_B,
    titulo="Fluxo de ativações no modelo de transferência (Domínio B)",
    subtitulo=f"Amostra do dígito {classe_original} (rótulo reindexado: {classe_verdadeira})",
)

print("Camadas capturadas no modelo de transferência:\n", list(acts_transfer.keys()))

###### Análise do Experimento 1

O modelo treinado do zero alcança acurácia superior ao modelo com transferência de aprendizado e extrator congelado. Esse resultado caracteriza um caso de **transferência negativa** (*negative transfer*) e decorre de três fatores:

1. **Baixa capacidade:** O extrator possui apenas $16$ filtros $3 \times 3$, insuficientes para aprender representações generalizáveis.

2. **Especialização no domínio:** O treinamento com os dígitos $0$ a $4$ produz filtros pouco discriminativos para os dígitos $5$ a $9$.

3. **Ausência de adaptação:** O congelamento impede que o extrator ajuste seus filtros à nova tarefa.

::: {.callout-note}
###### 💡 Provocação Pedagógica {.unnumbered}

Este experimento utiliza um extrator pequeno treinado em um domínio restrito. O resultado seria diferente se o extrator tivesse aprendido suas representações em uma base com milhões de imagens e grande diversidade de objetos?

:::

##### Experimento 2 — Quando a Transferência Realmente Funciona (*ResNet-18* Pré-treinada)

O segundo experimento repete a mesma estrutura do primeiro — poucos exemplos de treino, duas classes, comparação entre estratégias —, mas troca o extrator artesanal de $16$ filtros pela **_ResNet-18_**, uma arquitetura de $18$ camadas pré-treinada na *ImageNet* ($1,4$ milhão de imagens, $1.000$ categorias), e o *dataset* sintético de dígitos por fotografias reais do **Oxford-IIIT Pet Dataset** [@parkhi2012cats].

A tarefa: distinguir duas raças caninas — **Pug** e **Boxer** — a partir de apenas $15$ fotografias de treino por classe.

::: {.callout-tip}
###### 🐶 Por que este cenário? {.unnumbered}
O desafio aqui não é a semelhança visual entre as raças — Pug e Boxer têm portes e proporções bem distintos —, e sim a escassez de dados: apenas $30$ fotografias reais no total, sem qualquer imagem sintética. É o tipo de problema de baixo orçamento de dados que motiva, na prática, o uso de redes pré-treinadas: não há tempo nem recursos para fotografar e rotular milhares de cães antes de treinar um classificador do zero.
:::

###### Bloco 1: Carregamento do *Dataset* Real e Amostragem Escassa

1. **Fonte:** o *Oxford-IIIT Pet Dataset* [@parkhi2012cats] é carregado via `torchvision.datasets.OxfordIIITPet`, que baixa automaticamente as $7.349$ fotografias e seus rótulos de raça na primeira execução.
2. **Filtragem:** apenas as duas raças de interesse (`Pug`, `Boxer`) são mantidas.
3. **Escassez deliberada:** apenas $15$ fotografias de treino por classe ($30$ no total) são sorteadas — o restante compõe o conjunto de teste, usado exclusivamente para avaliação.

A @fig-09-pets-amostras exibe amostras de treino de cada raça.

In [ ]:
#| label: fig-09-pets-amostras
#| fig-cap: "Amostras reais de treino do *Oxford-IIIT Pet Dataset* (Parkhi et al., 2012, licença CC BY-SA 4.0): *Pug* e *Boxer*, raças escolhidas para simular um cenário real de escassez de dados."
#| echo: true
#| output: true
#| eval: false

RACAS_ALVO = ["Pug", "Boxer"]
N_TREINO_POR_CLASSE = 15
N_TESTE_POR_CLASSE = 20

# 1. Download do dataset completo (37 raças) — licença CC BY-SA 4.0
pets_completo = OxfordIIITPet(
    root="dados_pets", split="trainval", target_types="category", download=True
)
nomes_racas = pets_completo.classes
indices_alvo = [nomes_racas.index(r) for r in RACAS_ALVO]

# 2. Filtragem das duas raças de interesse, separadas por classe
por_classe = {idx: [] for idx in indices_alvo}
for img, lbl in pets_completo:
    if lbl in indices_alvo:
        por_classe[lbl].append(img)

# 3. Amostragem: poucas imagens de treino, mais imagens de teste
rng = random.Random(42)
imgs_treino, y_treino, imgs_teste, y_teste = [], [], [], []
for classe_idx, idx_original in enumerate(indices_alvo):
    imgs_raca = por_classe[idx_original][:]
    rng.shuffle(imgs_raca)
    imgs_treino += imgs_raca[:N_TREINO_POR_CLASSE]
    y_treino += [classe_idx] * N_TREINO_POR_CLASSE
    imgs_teste += imgs_raca[N_TREINO_POR_CLASSE : N_TREINO_POR_CLASSE + N_TESTE_POR_CLASSE]
    y_teste += [classe_idx] * N_TESTE_POR_CLASSE

print(f"Treino: {len(imgs_treino)} imagens | Teste: {len(imgs_teste)} imagens")

amostras_pil = imgs_treino[:4] + imgs_treino[N_TREINO_POR_CLASSE:N_TREINO_POR_CLASSE + 4]
amostras_exibicao = [np.array(img.convert("RGB")) for img in amostras_pil]  # PIL -> ndarray
titulos_exibicao = [RACAS_ALVO[0]] * 4 + [RACAS_ALVO[1]] * 4
mm.show(amostras_exibicao, titles=titulos_exibicao, cols=4, figsize=(11, 6))

###### Bloco 2: Três Estratégias sobre a Mesma Arquitetura

Para isolar o efeito da transferência de aprendizado, as três estratégias reutilizam **exatamente a mesma arquitetura** (*ResNet-18*), variando apenas a origem dos pesos e quais parâmetros permanecem treináveis:

1. **`do_zero`:** pesos aleatórios (`weights=None`) — equivalente a treinar a arquitetura da *ResNet-18* inteiramente do zero, como no Bloco 2 do Experimento 1.
2. **`congelado`:** pesos pré-treinados na *ImageNet*, com `requires_grad = False` em todas as camadas convolucionais — apenas a nova camada final é treinada.
3. **`fine_tuning`:** pesos pré-treinados na *ImageNet* como ponto de partida, mas **sem** congelamento — toda a rede se ajusta ao novo domínio, com taxa de aprendizado baixa para não destruir o conhecimento prévio.

Em todos os casos, a camada final `fc` é substituída por `nn.Linear(fc.in_features, 2)`, correspondente às duas raças de destino.

In [ ]:
#| label: sec-09-pets-arquitetura
#| echo: true
#| output: true
#| eval: false

transformacao_resnet = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def prepara_tensores(imgs, labels):
    X = torch.stack([transformacao_resnet(img.convert("RGB")) for img in imgs])
    y = torch.tensor(labels, dtype=torch.long)
    return X, y

X_tr, y_tr = prepara_tensores(imgs_treino, y_treino)
X_te, y_te = prepara_tensores(imgs_teste, y_teste)

def cria_modelo_pets(estrategia):
    pesos = None if estrategia == "do_zero" else models.ResNet18_Weights.DEFAULT
    modelo = models.resnet18(weights=pesos)
    if estrategia == "congelado":
        for p in modelo.parameters():
            p.requires_grad = False
    modelo.fc = nn.Linear(modelo.fc.in_features, len(RACAS_ALVO))
    return modelo

modelo_do_zero = cria_modelo_pets("do_zero")
modelo_congelado = cria_modelo_pets("congelado")
modelo_fine_tuning = cria_modelo_pets("fine_tuning")

###### Bloco 3: Treinamento Comparativo e Análise de Acurácia

Reutilizando as funções genéricas `treinar` e `calcular_acuracia`, definidas no Bloco 2 do Experimento 1, os três modelos são treinados sobre o mesmo conjunto de $30$ fotografias e avaliados no conjunto de teste (imagens nunca vistas durante o treino):

* O modelo `do_zero` tende a **superajustar** rapidamente às $30$ fotografias de treino, sem generalizar para o conjunto de teste — $30$ exemplos são drasticamente insuficientes para ajustar os $11$ milhões de parâmetros da *ResNet-18* a partir do zero.
* O modelo `congelado` já deve alcançar uma acurácia consideravelmente superior, pois reaproveita, sem qualquer ajuste, características visuais genéricas (bordas, texturas, contornos) aprendidas na *ImageNet* — apenas a nova camada linear precisa ser ajustada às $30$ fotografias.
* O modelo `fine_tuning` tende a igualar ou superar o extrator congelado, pois parte do mesmo conhecimento prévio, mas ainda permite um ajuste fino de toda a rede às particularidades visuais das raças.

A @fig-09-pets-comparativo resume os três resultados.

In [ ]:
#| label: fig-09-pets-comparativo
#| fig-cap: "Comparação de acurácia no conjunto de teste real (*Pug* vs. *Boxer*, 15 fotografias de treino por classe): do zero, extrator congelado e *fine-tuning* completo, todos sobre a mesma arquitetura ResNet-18."
#| echo: true
#| output: true
#| eval: false

torch.manual_seed(42)

configuracoes = [
    ("Do zero",              modelo_do_zero,      None, 2e-3),
    ("Extrator congelado",   modelo_congelado,    "fc", 1e-3),
    ("Fine-tuning completo", modelo_fine_tuning,  None, 1e-4),
]

resultados_pets = {}
for nome, modelo, alvo_params, taxa in configuracoes:
    parametros = modelo.fc.parameters() if alvo_params == "fc" else None
    treinar(modelo, X_tr, y_tr, epocas=15, lr=taxa, tam_lote=8, parametros=parametros)
    resultados_pets[nome] = calcular_acuracia(modelo, X_te, y_te)
    print(f"{nome}: {resultados_pets[nome]*100:.1f}%")

plt.figure(figsize=(5.5, 4))
cores = ["#dc2626", "#f59e0b", "#16a34a"]
plt.bar(resultados_pets.keys(), resultados_pets.values(), color=cores)
plt.ylim(0, 1.05)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Chute aleatório (50%)")
plt.ylabel("Acurácia no teste")
plt.title("Pug vs. Boxer — 15 fotos de treino/classe")
for i, v in enumerate(resultados_pets.values()):
    plt.text(i, v + 0.02, f"{v*100:.1f}%", ha="center")
plt.xticks(rotation=10)
plt.legend()
plt.tight_layout()
plt.show()

###### Bloco 4: Inspeção Qualitativa das Predições

Como no Experimento 1 e na seção seguinte sobre diagnóstico foliar, é instrutivo observar individualmente algumas predições do melhor modelo (tipicamente `fine_tuning` ou `congelado`) sobre fotografias reais de teste, comparando o rótulo previsto com a raça real.

In [ ]:
#| label: fig-09-pets-predicoes
#| fig-cap: "Predições do modelo com extrator pré-treinado (ResNet-18) em fotografias reais de teste: rótulo previsto vs. raça real, quatro amostras de cada classe."
#| echo: true
#| output: true
#| eval: false

melhor_modelo = modelo_fine_tuning  # ou modelo_congelado, conforme o resultado do Bloco 3
melhor_modelo.eval()

idx_amostras = list(range(4)) + list(range(N_TESTE_POR_CLASSE, N_TESTE_POR_CLASSE + 4))

imgs_pred, titulos_pred = [], []
with torch.no_grad():
    for idx in idx_amostras:
        entrada = X_te[idx].unsqueeze(0)
        pred_idx = melhor_modelo(entrada).argmax(dim=1).item()
        real_idx = y_te[idx].item()
        marcador = "✓" if pred_idx == real_idx else "✗"
        imgs_pred.append(np.array(imgs_teste[idx].convert("RGB")))  # PIL -> ndarray
        titulos_pred.append(f"{marcador} previsto: {RACAS_ALVO[pred_idx]}\n"
                             f"real: {RACAS_ALVO[real_idx]}")

mm.show(imgs_pred, titles=titulos_pred, cols=4, figsize=(12, 7))

In [ ]:
#| echo: true
#| output: true
 
# Limpeza explícita dos dados baixados e liberação de memória
if FLAG_LIMPAR_DADOS:
    if os.path.exists('./dados_pets'):
        shutil.rmtree('./dados_pets')
        print('🧹 Diretório de dados temporários ./dados_pets removido com sucesso.')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

##### Experimento 3 — Diagnóstico Fitossanitário com Transferência de Aprendizado

O experimento anterior mostrou que uma *ResNet-18* pré-treinada na **ImageNet** pode adaptar-se a uma nova tarefa utilizando poucas amostras. Agora, a mesma estratégia é aplicada a um problema de diagnóstico fitossanitário. A *ResNet-18* deve classificar imagens de folhas em três categorias: `["folha_saudavel", "folha_doente", "sintoma_desconhecido"]`.

- **`folha_saudavel`:** folha sem lesões visíveis.
- **`folha_doente`:** folha com manchas escuras que simulam uma doença fúngica.
- **`sintoma_desconhecido`:** folha com clorose amarelada, representando um padrão diferente da doença conhecida.

::: {.callout-tip}
###### 🌱 Por que este cenário? {.unnumbered}

O diagnóstico fitossanitário constitui uma importante aplicação da VC na agricultura de precisão. Um modelo pré-treinado na **ImageNet** pode reutilizar características como bordas, texturas e padrões de cor para aprender essa nova tarefa com poucas imagens.

:::

###### Bloco 1: Geração do *Dataset* Sintético

Este bloco gera um conjunto sintético com **30** imagens por classe para treinamento e **8** para validação, totalizando **90** e **24** imagens, respectivamente.

1. **Geração da folha:** A função `desenha_folha_base` cria o contorno da folha, variando tamanho, orientação e tonalidade de verde.

2. **Simulação da doença:** A função `aplica_manchas_doenca` adiciona manchas escuras irregulares que simulam lesões fúngicas.

3. **Simulação de outro sintoma:** A função `aplica_sintoma_desconhecido` adiciona regiões amareladas que representam um padrão distinto da doença conhecida.

4. **Visualização das amostras:** A @fig-09-folhas-amostras apresenta exemplos das três categorias do conjunto sintético.

In [ ]:
#| label: fig-09-folhas-amostras
#| fig-cap: "Amostras sintéticas do *dataset* de diagnóstico foliar: folha saudável, folha doente (manchas escuras) e sintoma desconhecido (clorose amarelada) com ruído sal e pimenta."
#| echo: true
#| output: true

CLASSES_FOLHA = ["folha_saudavel", "folha_doente", "sintoma_desconhecido"]


def desenha_folha_base(tam_img, rng):
    '''Desenha o contorno oval de uma folha verde com nervura central,
    com pequenas variações de tom, tamanho e orientação entre amostras.'''
    img = np.full((tam_img, tam_img, 3), 245, dtype=np.uint8)  # fundo claro
    cx, cy = tam_img // 2, tam_img // 2
    eixo_a = rng.randint(int(tam_img * 0.30), int(tam_img * 0.38))
    eixo_b = rng.randint(int(tam_img * 0.20), int(tam_img * 0.26))
    angulo = rng.uniform(-15, 15)
    verde = (rng.randint(40, 70), rng.randint(120, 160), rng.randint(40, 70))
    cv2.ellipse(img, (cx, cy), (eixo_a, eixo_b), angulo, 0, 360, verde, -1, cv2.LINE_AA)
    ang_rad = np.deg2rad(angulo)
    dx, dy = np.cos(ang_rad), np.sin(ang_rad)
    p1 = (int(cx - eixo_a * dx), int(cy - eixo_a * dy))
    p2 = (int(cx + eixo_a * dx), int(cy + eixo_a * dy))
    cv2.line(img, p1, p2, (25, 90, 25), 2, cv2.LINE_AA)  # nervura central
    return img, (cx, cy, eixo_a, eixo_b, angulo)


def aplica_manchas_doenca(img, centro_folha, rng, n_manchas=(4, 8)):
    '''Simula lesões foliares: manchas escuras de bordas irregulares
    (padrão típico de doenças fúngicas).'''
    cx, cy, eixo_a, eixo_b, _ = centro_folha
    for _ in range(rng.randint(*n_manchas)):
        raio = rng.randint(1, 3)
        px = cx + rng.randint(-int(eixo_a * 0.7), int(eixo_a * 0.7))
        py = cy + rng.randint(-int(eixo_b * 0.7), int(eixo_b * 0.7))
        cor_mancha = (rng.randint(50, 90), rng.randint(25, 45), rng.randint(10, 25))
        cv2.circle(img, (px, py), raio, cor_mancha, -1, cv2.LINE_AA)
        cv2.circle(img, (px, py), raio + 1, (120, 85, 30), 1, cv2.LINE_AA)  # halo
    return img


def aplica_sintoma_desconhecido(img, centro_folha, rng):
    '''Simula um padrão distinto (mosqueado amarelado/clorose), diferente
    das manchas escuras da doença conhecida.'''
    cx, cy, eixo_a, eixo_b, _ = centro_folha
    for _ in range(rng.randint(3, 5)):
        eixo_m = (rng.randint(1, 3), rng.randint(2, 3))
        px = cx + rng.randint(-int(eixo_a * 0.6), int(eixo_a * 0.6))
        py = cy + rng.randint(-int(eixo_b * 0.6), int(eixo_b * 0.6))
        cor_clorose = (rng.randint(200, 235), rng.randint(195, 225), rng.randint(50, 90))
        ang_m = rng.uniform(0, 180)
        cv2.ellipse(img, (px, py), eixo_m, ang_m, 0, 360, cor_clorose, -1, cv2.LINE_AA)
    return img


def aplica_ruido_sal_pimenta(img, prop_ruido=0.02, rng=None):
    '''Aplica ruído sal (pontos brancos) e pimenta (pontos pretos) aleatórios.
    prop_ruido: fração de pixels alterados (ex: 0.02 = 2% dos pixels).'''
    if prop_ruido <= 0:
        return img
    
    img_ruido = img.copy()
    num_pixels = int(prop_ruido * img.shape[0] * img.shape[1])
    n_sal = num_pixels // 2
    n_pimenta = num_pixels - n_sal

    # Aplica Sal (Branco - [255, 255, 255])
    for _ in range(n_sal):
        y = rng.randint(0, img.shape[0] - 1)
        x = rng.randint(0, img.shape[1] - 1)
        img_ruido[y, x] = [255, 255, 255]

    # Aplica Pimenta (Preto - [0, 0, 0])
    for _ in range(n_pimenta):
        y = rng.randint(0, img.shape[0] - 1)
        x = rng.randint(0, img.shape[1] - 1)
        img_ruido[y, x] = [0, 0, 0]

    return img_ruido


def gera_folha(classe_idx, tam_img=128, rng=None, prop_ruido=0.02):
    rng = rng or random.Random()
    img, geometria = desenha_folha_base(tam_img, rng)
    nome = CLASSES_FOLHA[classe_idx]
    
    if nome == "folha_doente":
        img = aplica_manchas_doenca(img, geometria, rng)
    elif nome == "sintoma_desconhecido":
        img = aplica_sintoma_desconhecido(img, geometria, rng)
        
    ruido_exp = rng.randint(-3, 3)  # leve variação de exposição
    img = np.clip(img.astype(np.int16) + ruido_exp, 0, 255).astype(np.uint8)
    
    # Aplicação do ruído Sal e Pimenta
    img = aplica_ruido_sal_pimenta(img, prop_ruido=prop_ruido, rng=rng)
    
    return img


def gera_conjunto(n_por_classe, tam_img=128, seed=0, prop_ruido=0.02):
    rng = random.Random(seed)
    imgs, labels = [], []
    for classe_idx in range(len(CLASSES_FOLHA)):
        for _ in range(n_por_classe):
          imgs.append(gera_folha(classe_idx, tam_img=tam_img, rng=rng, prop_ruido=prop_ruido))
          labels.append(classe_idx)
    return imgs, labels


N_POR_CLASSE_TREINO, N_POR_CLASSE_VAL = 30, 8

imgs_treino, labels_treino = gera_conjunto(n_por_classe=N_POR_CLASSE_TREINO, seed=42, 
                                           prop_ruido=0.02)
imgs_val, labels_val = gera_conjunto(n_por_classe=N_POR_CLASSE_VAL, seed=123, prop_ruido=0.02)

print(f"Treino: {len(imgs_treino)} imagens ({N_POR_CLASSE_TREINO} por classe) | "
      f"Validação: {len(imgs_val)} imagens ({N_POR_CLASSE_VAL} por classe)")

# Exibição de 2 amostras de cada classe (6 imagens no total)
amostras_exibir, titulos_exibir = [], []
for classe_idx, nome in enumerate(CLASSES_FOLHA):
    for k in range(2):
        idx = classe_idx * N_POR_CLASSE_TREINO + k
        amostras_exibir.append(imgs_treino[idx])
        titulos_exibir.append(nome)

mm.show(amostras_exibir, titles=titulos_exibir, cols=3, figsize=(10, 7))

::: {.callout-note}
###### 🧠 Armadilha Comum {.unnumbered}

A transferência de aprendizado exige que cada classe apresente padrões visuais distintos. Repetir a mesma imagem com rótulos diferentes impede que a camada classificadora aprenda uma fronteira de decisão, pois o extrator gera praticamente as mesmas características para todas as amostras.

Neste experimento, cada imagem é gerada de forma independente, com padrões visuais compatíveis com sua classe (folha saudável, lesões fúngicas ou clorose), fornecendo informações suficientes para o treinamento da camada classificadora.

:::

###### Bloco 2: Preparação dos Tensores e Adaptação da Arquitetura

Modelos como a *ResNet-18* exigem imagens coloridas de $224 \times 224$ *pixels* normalizadas segundo as estatísticas da *ImageNet* ($\mu = [0,485; 0,456; 0,406]$ e $\sigma = [0,229; 0,224; 0,225]$).

1. **Transformação de Entrada (`transforms.Compose`):** aplica-se o redimensionamento e a normalização padrão a cada imagem do *dataset* sintético, produzindo os tensores `X_treino`/`X_val` e os rótulos `y_treino`/`y_val` — cada exemplo é uma imagem genuinamente distinta, associada ao rótulo correto de sua classe.
2. **Congelamento do Extrator:** o laço `for p in modelo_resnet.parameters(): p.requires_grad = False` desativa os gradientes nas camadas convolucionais pré-treinadas.
3. **Nova Camada Final:** a camada `modelo_resnet.fc` é substituída por uma nova instância `nn.Linear(modelo_resnet.fc.in_features, n_classes_destino)`, recém-inicializada e com gradientes ativos por padrão. O número de características de entrada é obtido dinamicamente a partir da própria camada original (`in_features`, igual a $512$ na ResNet-18), em vez de fixado manualmente no código — prática recomendada, pois torna o trecho reutilizável para outras variantes da arquitetura sem alterações.

In [ ]:
#| eval: false
#| echo: true

# 1. Pipeline de transformações esperadas pela ResNet
transformacao_resnet = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def prepara_tensores(imgs, labels):
    tensores = [transformacao_resnet(T.functional.to_tensor(img)) for img in imgs]
    X = torch.stack(tensores)
    y = torch.tensor(labels, dtype=torch.long)
    return X, y


# 2. Conversão do dataset sintético (Bloco 1) em tensores normalizados
X_treino, y_treino = prepara_tensores(imgs_treino, labels_treino)
X_val, y_val = prepara_tensores(imgs_val, labels_val)

loader_treino = DataLoader(TensorDataset(X_treino, y_treino), batch_size=16, shuffle=True)
loader_val = DataLoader(TensorDataset(X_val, y_val), batch_size=16, shuffle=False)

# 3. Carregamento da ResNet-18 pré-treinada e congelamento do extrator
n_classes_destino = len(CLASSES_FOLHA)
modelo_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for parametro in modelo_resnet.parameters():
    parametro.requires_grad = False

# 4. Substituição da camada final para as 3 novas classes de destino
modelo_resnet.fc = nn.Linear(modelo_resnet.fc.in_features, n_classes_destino)

print(f"Nova camada final: {modelo_resnet.fc}")

###### Bloco 3: Laço de Treinamento e Avaliação

Com o extrator congelado e a nova camada de saída devidamente acoplada, executa-se o ajuste fino (*fine-tuning*) da nova cabeça de classificação.

1. **Otimização Focada:** o otimizador *Adam* recebe estritamente `modelo_resnet.fc.parameters()`, atualizando apenas a nova camada de saída — o restante da rede permanece congelado, conforme definido no Bloco 2.
2. **Execução do Laço:** a cada época, o modelo itera sobre os lotes de treino, calcula a perda por Entropia Cruzada e ajusta os pesos da camada final; em seguida, avalia-se a acurácia no conjunto de validação (imagens nunca vistas durante o treino).
3. **Curvas de Treinamento:** a @fig-09-resnet-treino acompanha a evolução da perda de treino e da acurácia de validação ao longo das épocas — como as três classes são visualmente distintas entre si, espera-se convergência genuína, bem acima do patamar de $33\%$ correspondente a um chute aleatório entre $3$ classes.

In [ ]:
#| label: fig-09-resnet-treino
#| fig-cap: "Curvas de treinamento do fine-tuning da ResNet-18 no *dataset* sintético de diagnóstico foliar: perda de treino e acurácia de validação ao longo das épocas."
#| eval: false
#| echo: true
#| output: true

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_resnet = modelo_resnet.to(dispositivo)

criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_resnet.fc.parameters(), lr=1e-3)

historico_perda, historico_acc = [], []
epocas = 10

for epoca in range(epocas):
    modelo_resnet.train()
    perda_acumulada, n_batches = 0.0, 0

    for X_batch, y_batch in loader_treino:
        X_batch, y_batch = X_batch.to(dispositivo), y_batch.to(dispositivo)

        otimizador.zero_grad()
        saidas = modelo_resnet(X_batch)
        perda = criterio(saidas, y_batch)
        perda.backward()
        otimizador.step()

        perda_acumulada += perda.item()
        n_batches += 1
    historico_perda.append(perda_acumulada / n_batches)

    modelo_resnet.eval()
    acertos, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch, y_batch = X_batch.to(dispositivo), y_batch.to(dispositivo)
            predicoes = modelo_resnet(X_batch).argmax(dim=1)
            acertos += (predicoes == y_batch).sum().item()
            total += y_batch.size(0)
    acc = acertos / total
    historico_acc.append(acc)

    print(f"Época {epoca+1}/{epocas} — perda: {historico_perda[-1]:.4f} — ",
          f" acurácia_val: {acc*100:.1f}%")

f=mm.showTrainCurves(
    historico_perda, historico_acc,
    titulo="Fine-Tuning da ResNet-18 — Diagnóstico Foliar",
    subtitulo="Apenas a nova camada linear (fc) é treinada; o extrator permanece congelado",
)

###### Bloco 4: Inspeção Qualitativa das Predições

Além da curva de acurácia agregada, é instrutivo observar individualmente algumas predições do modelo sobre o conjunto de validação, comparando o rótulo previsto com o rótulo real. A @fig-09-resnet-predicoes exibe duas amostras de cada classe.

In [ ]:
#| label: fig-09-resnet-predicoes
#| fig-cap: "Predições da ResNet-18 ajustada em amostras de validação: rótulo previsto vs. rótulo real, com duas amostras por classe."
#| eval: false
#| echo: true
#| output: true

modelo_resnet.eval()

# Duas amostras de cada classe no conjunto de validação
idx_amostras = [0, N_POR_CLASSE_VAL, 2 * N_POR_CLASSE_VAL,
                1, N_POR_CLASSE_VAL + 1, 2 * N_POR_CLASSE_VAL + 1]

imgs_pred, titulos_pred = [], []
with torch.no_grad():
    for idx in idx_amostras:
        entrada = X_val[idx].unsqueeze(0).to(dispositivo)
        pred_idx = modelo_resnet(entrada).argmax(dim=1).item()
        real_idx = y_val[idx].item()
        marcador = "✓" if pred_idx == real_idx else "✗"
        imgs_pred.append(imgs_val[idx])
        titulos_pred.append(f"{marcador} previsto: {CLASSES_FOLHA[pred_idx]}\n"+
                            f"real: {CLASSES_FOLHA[real_idx]}")

mm.show(imgs_pred, titles=titulos_pred, cols=3, figsize=(10, 7))

###### Análise do Experimento 3

A *ResNet-18* alcança alta acurácia mesmo utilizando um conjunto reduzido de imagens sintéticas. Esse resultado mostra que as representações aprendidas na **ImageNet** permanecem úteis em um domínio completamente diferente, exigindo apenas a adaptação da camada classificadora.

O experimento também ilustra uma situação comum em aplicações reais, nas quais a disponibilidade de dados rotulados é limitada. Nesses cenários, a transferência de aprendizado reduz o tempo de treinamento e permite obter modelos com bom desempenho mesmo sem treinar toda a rede.

::: {.callout-note}
###### 🧠 Síntese Comparativa — Quando a Transferência de Aprendizado Funciona? {.unnumbered}

Os três experimentos mostram que a transferência de aprendizado depende da capacidade de generalização do extrator de características.

- **Experimento 1:** um extrator pequeno, treinado em um domínio restrito, aprende representações pouco generalizáveis e pode produzir **transferência negativa**.

- **Experimento 2:** uma *ResNet-18* pré-treinada na **ImageNet** transfere representações gerais para uma tarefa de classificação de raças de cães e gatos, alcançando alta acurácia com poucas amostras.

- **Experimento 3:** a mesma estratégia adapta o modelo a um problema de diagnóstico fitossanitário, mostrando que um único extrator pode servir como base para diferentes domínios de aplicação.

:::

##### Comparativo das Abordagens de Transferência

A @tbl-comparativo-transferencia resume os resultados obtidos nos três experimentos.

| Aspecto | Experimento 1 | Experimento 2 | Experimento 3 |
| --- | --- | --- | --- |
| **Extrator** | CNN pequena | *ResNet-18* | *ResNet-18* |
| **Treinamento do extrator** | Dígitos ($0$–$4$) | **ImageNet** | **ImageNet** |
| **Capacidade de generalização** | Baixa | Alta | Alta |
| **Nova tarefa** | Dígitos ($5$–$9$) | Raças de cães e gatos | Diagnóstico fitossanitário |
| **Resultado** | Transferência negativa | Transferência positiva | Transferência positiva |

: Comparação entre os três cenários de transferência de aprendizado apresentados nesta seção. {#tbl-comparativo-transferencia}

### Detecção de Objetos

Os experimentos anteriores mostraram como a transferência de aprendizado adapta modelos pré-treinados para tarefas de classificação de imagens. O mesmo princípio também fundamenta arquiteturas de detecção de objetos, nas quais um extrator de características pré-treinado fornece representações visuais gerais, enquanto módulos especializados localizam e classificam os objetos na imagem.

As próximas seções apresentam a **Faster R-CNN** como exemplo de detector pré-treinado usado diretamente para inferência e, em seguida, um experimento completo de ajuste fino com a arquitetura **YOLO**.

#### Faster R-CNN: Detector Pré-treinado

A detecção de objetos estende o uso de modelos pré-treinados para uma tarefa mais complexa que a classificação. A **Faster R-CNN** utiliza uma CNN pré-treinada, como a *ResNet-50*, como **extrator de características** (*backbone*) e acrescenta módulos especializados para localizar e classificar objetos.

Sobre esse extrator, a arquitetura incorpora duas "cabeças" principais:

- ***Region Proposal Network* (RPN):** propõe regiões da imagem com alta probabilidade de conter objetos.
- **Cabeça classificadora:** refina essas regiões, atribui uma classe a cada objeto e ajusta suas caixas delimitadoras.

O código a seguir utiliza uma Faster R-CNN com pesos pré-treinados no **COCO** para detectar objetos em uma imagem, produzindo suas classes, coordenadas e pontuações de confiança.

::: {.callout-note}
##### 🔍 Onde está a transferência de aprendizado aqui? {.unnumbered}

Diferentemente dos experimentos anteriores, este exemplo **não realiza ajuste fino** (*fine-tuning*). O modelo executa apenas a inferência (`eval()`), reutilizando diretamente os pesos do *backbone*, da RPN e da cabeça classificadora treinados no **COCO**.

A adaptação para um novo domínio exigiria substituir a camada `box_predictor` por uma nova cabeça de classificação, compatível com as classes da aplicação, e treiná-la sobre um conjunto de imagens anotadas. Esse procedimento segue o mesmo princípio apresentado na seção de transferência de aprendizado e constitui o fluxo usual para aplicações específicas, como detecção de pragas, defeitos de fabricação ou veículos.
:::

1. **Categorias do COCO:** O código recupera os nomes das classes a partir das meta-informações dos pesos (`FasterRCNN_ResNet50_FPN_Weights.DEFAULT.meta["categories"]`). Embora essa lista contenha $91$ entradas por razões históricas do formato de anotação do COCO, apenas $80$ correspondem a categorias de objetos.

2. **Inferência:** A imagem carregada por `mm.read()` é convertida em *tensor* e processada pelo modelo em modo de avaliação (`eval()`). O código mantém apenas as detecções com confiança superior a $80\%$.

3. **Anotação da imagem:** Para cada objeto detectado, o código desenha a caixa delimitadora (`cv2.rectangle`) e escreve a classe predita e sua confiança (`cv2.putText`).

4. **Visualização:** A @fig-09-deteccao-pretreinada apresenta a imagem anotada com as detecções realizadas pelo modelo.

In [ ]:
#| label: fig-09-deteccao-pretreinada
#| fig-cap: "Resultado da inferência com o modelo Faster R-CNN pré-treinado no *dataset* COCO: identificação da classe, caixa delimitadora e pontuação de confiança sobrepostas."
#| eval: false
#| echo: true
#| output: true

# 1. Carregamento da imagem e dos nomes das categorias do COCO
url_imagem = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
url_imagem = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = mm.read(url_imagem)

pesos_coco = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
categorias_coco = pesos_coco.meta["categories"]  # Mapeamento índice -> nome da classe

# 2. Carregamento do modelo Faster R-CNN pré-treinado
modelo_detection = fasterrcnn_resnet50_fpn(weights=pesos_coco).eval()

# 3. Execução da inferência sem cálculo de gradientes
with torch.no_grad():
    predicao = modelo_detection([to_tensor(img)])[0]

# 4. Filtragem das detecções com confiança superior a 80%
limiar_confianca = 0.8
mascara_confianca = predicao["scores"] >= limiar_confianca

caixas_filtradas = predicao["boxes"][mascara_confianca].numpy()
scores_filtrados = predicao["scores"][mascara_confianca].numpy()
labels_filtrados = predicao["labels"][mascara_confianca].numpy()

img_com_caixas = img.copy()

# 5. Desenho das caixas delimitadoras e dos rótulos de classe
for box, score, label_idx in zip(caixas_filtradas, scores_filtrados, labels_filtrados):
    x1, y1, x2, y2 = box.astype(int)
    nome_classe = categorias_coco[label_idx]
    texto_rotulo = f"{nome_classe}: {score:.2f}"

    # Desenha o retângulo vermelho (RGB: 255, 0, 0) com espessura de 3 pixels
    cv2.rectangle(img_com_caixas, (x1, y1), (x2, y2), (255, 0, 0), 3)

    # Escreve a classe e a confiança acima da caixa delimitadora
    cv2.putText(
        img_com_caixas,
        texto_rotulo,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2,
        cv2.LINE_AA,
    )

# 6. Exibição gráfica da imagem resultante
mm.show(img_com_caixas, title="Faster R-CNN (COCO) — Detecção com Classe e Confiança")

#### Detecção de Objetos e Transferência de Aprendizado com YOLO

As seções anteriores aplicaram a transferência de aprendizado a problemas de **classificação de imagens**, em que o modelo associa um único rótulo à imagem inteira. Nesta seção, o mesmo princípio é estendido à **detecção de objetos**, tarefa que exige identificar simultaneamente **o que** está presente na imagem e **onde** cada objeto se encontra.

A subseção anterior apresentou a **Faster R-CNN** como exemplo de detector pré-treinado utilizado diretamente para inferência, sem qualquer adaptação ao novo domínio. Neste experimento, o modelo passa por uma etapa de **ajuste fino** (*fine-tuning*): parte-se de uma arquitetura **YOLO** (*You Only Look Once*) pré-treinada no conjunto **COCO** e adapta-se a rede para detectar e classificar objetos de um novo domínio.

Diferentemente da Faster R-CNN, que realiza a detecção em dois estágios, a família **YOLO** adota uma arquitetura de estágio único (*single-stage detector*), estimando, em uma única propagação pela rede, as caixas delimitadoras (*bounding boxes*), a confiança de cada detecção e a classe correspondente. Essa estratégia reduz o custo computacional e viabiliza aplicações em tempo real.

Como exemplo, o experimento utiliza um conjunto sintético de formas geométricas (triângulos, quadrados, estrelas, entre outras), com variações de cor, tamanho, rotação e degradação por ruído do tipo sal e pimenta.

##### Bloco 1: Geração do *Dataset* Sintético

O bloco a seguir gera um conjunto sintético para treinamento e avaliação do detector. Cada imagem contém entre um e três objetos pertencentes a uma das nove classes:

```python
CLASSES = [
    'Triangle', 'Square', 'Pentagon', 'Hexagon',
    'Heptagon', 'Circle', 'Ellipse', 'Star', 'Cross'
]
```

1. **Geração das formas:** As funções `poligono_regular`, `poligono_estrela` e `poligono_cruz` constroem as coordenadas dos objetos. A função `desenha_objeto` desenha cada forma com posição, tamanho, orientação e cor aleatórios e calcula sua *bounding box*.

2. **Anotação no formato YOLO:** A função `gera_imagem_ruidosa` gera entre um e três objetos por imagem e converte cada *bounding box* para o formato YOLO, representado pela classe e pelas coordenadas normalizadas do centro, largura e altura.

3. **Degradação da imagem:** A função `adiciona_ruido_sal_pimenta` adiciona ruído impulsivo, simulando imperfeições de aquisição.

4. **Visualização das amostras:** A @fig-09-yolo-amostras-iniciais apresenta exemplos do conjunto sintético com as *bounding boxes* sobrepostas pela função `mm.showBoundBox()`.

In [ ]:
#| echo: true
#| output: true

CLASSES = [
    'Triangle', 'Square', 'Pentagon', 'Hexagon',
    'Heptagon', 'Circle', 'Ellipse', 'Star', 'Cross'
]
N_LADOS = {'Triangle': 3, 'Square': 4, 'Pentagon': 5, 'Hexagon': 6, 'Heptagon': 7}

def poligono_regular(cx, cy, r, n_lados, rot_graus):
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + 2 * np.pi * np.arange(n_lados) / n_lados
    return np.stack([cx + r * np.cos(angs), cy + r * np.sin(angs)], axis=1)

def poligono_estrela(cx, cy, r_externo, rot_graus, n_pontas=5):
    r_interno = r_externo * 0.45
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + np.pi * np.arange(2 * n_pontas) / n_pontas
    raios = np.where(np.arange(2 * n_pontas) % 2 == 0, r_externo, r_interno)
    return np.stack([cx + raios * np.cos(angs), cy + raios * np.sin(angs)], axis=1)

def poligono_cruz(cx, cy, r, rot_graus, espessura_rel=0.35):
    w = r * espessura_rel
    base = np.array([
        (-w, -r), (w, -r), (w, -w), (r, -w), (r, w), (w, w),
        (w, r), (-w, r), (-w, w), (-r, w), (-r, -w), (-w, -w),
    ])
    theta = np.deg2rad(rot_graus)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return base @ R.T + np.array([cx, cy])

def desenha_objeto(img, classe_idx, cx, cy, tamanho, rotacao, cor):
    nome = CLASSES[classe_idx]
    if nome in N_LADOS:
        pts = poligono_regular(cx, cy, tamanho, N_LADOS[nome], rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Star':
        pts = poligono_estrela(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Cross':
        pts = poligono_cruz(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Circle':
        cv2.circle(img, (int(cx), int(cy)), int(tamanho), cor, -1)
        xs, ys = np.array([cx - tamanho, cx + tamanho]), np.array([cy - tamanho, cy + tamanho])
    else:  # Ellipse
        eixo = (int(tamanho), int(tamanho * 0.6))
        cv2.ellipse(img, (int(cx), int(cy)), eixo, rotacao, 0, 360, cor, -1)
        ang = np.deg2rad(rotacao)
        dx = np.hypot(eixo[0] * np.cos(ang), eixo[1] * np.sin(ang))
        dy = np.hypot(eixo[0] * np.sin(ang), eixo[1] * np.cos(ang))
        xs, ys = np.array([cx - dx, cx + dx]), np.array([cy - dy, cy + dy])
    return xs.min(), ys.min(), xs.max(), ys.max()

def adiciona_ruido_sal_pimenta(img, quantidade=0.05):
    img_ruidosa = img.copy()
    h, w, c = img_ruidosa.shape
    num_ruido = int(quantidade * h * w)
    
    # Sal (255, 255, 255)
    coords_sal = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_sal[0], coords_sal[1]] = [255, 255, 255]
    
    # Pimenta (0, 0, 0)
    coords_pimenta = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_pimenta[0], coords_pimenta[1]] = [0, 0, 0]
    
    return img_ruidosa

def gera_imagem_ruidosa(tam_img=160, n_objetos=(1, 3), taxa_ruido=0.01, rng=None):
    rng = rng or random.Random()
    img_limpa = np.full((tam_img, tam_img, 3), 255, dtype=np.uint8)
    anotacoes = []
    
    for _ in range(rng.randint(*n_objetos)):
        classe_idx = rng.randrange(len(CLASSES))
        tamanho = rng.randint(tam_img // 10, tam_img // 5)
        cx = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        cy = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        rotacao = rng.uniform(0, 360)
        cor = tuple(rng.sample(range(30, 226), 3))
        
        x0, y0, x1, y1 = desenha_objeto(img_limpa, classe_idx, cx, cy, tamanho, rotacao, cor)
        x0, y0 = max(x0, 0), max(y0, 0)
        x1, y1 = min(x1, tam_img), min(y1, tam_img)
        
        # Padrão YOLO: (classe, x_centro, y_centro, largura, altura) normalizados
        xc, yc = (x0 + x1) / 2 / tam_img, (y0 + y1) / 2 / tam_img
        w, h = (x1 - x0) / tam_img, (y1 - y0) / tam_img
        anotacoes.append((classe_idx, xc, yc, w, h)) 
        
    img_ruidosa = adiciona_ruido_sal_pimenta(img_limpa, quantidade=taxa_ruido)
    return img_ruidosa, anotacoes

In [ ]:
#| label: fig-09-yolo-amostras-iniciais
#| fig-cap: "Amostras iniciais do *dataset* sintético ruidoso de objetos geométricos com as caixas delimitadoras do formato YOLO sobrepostas."
#| echo: true
#| output: true

# Geração de 5 amostras para exibição inicial no topo do projeto
n_amostras_iniciais = 5
rng_demo = random.Random(42)

imgs_demo = []
titulos_demo = []

for idx in range(n_amostras_iniciais):
    img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_demo)
    
    # Gravação temporária da anotação para leitura nativa pelo mm.showBoundBox
    filename_temp = f"temp_label_{idx}.txt"
    with open(filename_temp, "w") as f:
        for c, xc, yc, w, h in anotacoes:
            f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")
            
    img_anotada = mm.showBoundBox(img_ruid, filename=filename_temp, fmt="yolo", show=False)
    imgs_demo.append(img_anotada)
    titulos_demo.append(f"Amostra {idx+1}")

# Exibição do painel de 5 amostras
mm.show(
    imgs_demo,
    titles=titulos_demo,
    cols=n_amostras_iniciais,
    figsize=(14, 3)
)

##### Bloco 2: Organização do *Dataset* e Criação do Arquivo `data.yaml`

Este bloco organiza o conjunto de dados no formato esperado pela biblioteca **Ultralytics YOLO**. As imagens e as anotações são distribuídas em diretórios separados para treinamento e validação, enquanto o arquivo `data.yaml` reúne as informações necessárias para o treinamento do detector.

```text
shapes_dataset/
├── data.yaml
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

1. **Geração do conjunto de dados:** O código cria **90** imagens para treinamento e **20** para validação. Para cada imagem, grava um arquivo `.txt` contendo uma linha por objeto, no formato YOLO (`classe`, `x_c`, `y_c`, `largura`, `altura`), com todas as coordenadas normalizadas.

2. **Organização dos arquivos:** As imagens são armazenadas em `images/train` e `images/val`, enquanto as anotações correspondentes são gravadas em `labels/train` e `labels/val`, preservando o mesmo nome de arquivo.

3. **Criação do arquivo `data.yaml`:** O código gera automaticamente o arquivo de configuração contendo o caminho do *dataset*, os diretórios de treinamento e validação e o mapeamento entre os índices numéricos e os nomes das nove classes.

In [ ]:
#| echo: true

base_dir = "shapes_dataset"
rng_global = random.Random(42)

for split, n_imgs in [("train", 90), ("val", 20)]:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)
    
    for i in range(n_imgs):
        img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_global)
        cv2.imwrite(f"{base_dir}/images/{split}/{i:04d}.jpg", img_ruid)
        
        with open(f"{base_dir}/labels/{split}/{i:04d}.txt", "w") as f:
            for c, xc, yc, w, h in anotacoes:
                f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")

with open(f"{base_dir}/data.yaml", "w") as f:
    f.write(
        f"path: {os.path.abspath(base_dir)}\n"
        "train: images/train\nval: images/val\nnames:\n"
    )
    for i, nome in enumerate(CLASSES):
        f.write(f"  {i}: {nome}\n")

print("Dataset ruidoso gerado com sucesso: 90 imagens de treino e 20 de validação.")

##### Bloco 3: Pré-processamento com Filtro de Mediana

Este bloco aplica um pré-processamento para reduzir o efeito do ruído do tipo sal e pimenta introduzido na geração do *dataset*. O **Filtro de Mediana** (`cv2.medianBlur`) remove esse tipo de degradação preservando melhor as bordas dos objetos do que filtros de suavização convencionais.

1. **Filtragem da imagem:** O código aplica um filtro de mediana com janela $3 \times 3$ à imagem ruidosa, reduzindo os *pixels* impulsivos sem alterar as anotações do conjunto de dados.

2. **Visualização comparativa:** A @fig-09-yolo-pre-processamento compara a imagem original e a imagem filtrada, mantendo as *bounding boxes* sobrepostas por meio da função `mm.showBoundBox()`.

In [ ]:
#| label: fig-09-yolo-pre-processamento
#| fig-cap: "Comparação entre a imagem original com ruído do tipo sal e pimenta e a imagem após a aplicação do Filtro de Mediana (3x3). As *bounding boxes* no formato YOLO permanecem inalteradas."
#| echo: true
#| output: true

# 1. Carregamento da primeira amostra ruidosa do dataset
caminho_img = f"{base_dir}/images/train/0000.jpg"
caminho_label = f"{base_dir}/labels/train/0000.txt"

img_ruidosa = mm.read(caminho_img)

# 2. Pré-processamento com Filtro de Mediana (janela 3x3)
img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)

# 3. Sobreposição das bounding boxes com mm.showBoundBox
img_ruid_anotada = mm.showBoundBox(img_ruidosa, filename=caminho_label, fmt="yolo", show=False)
img_filt_anotada = mm.showBoundBox(img_filtrada, filename=caminho_label, fmt="yolo", show=False)

# 4. Exibição comparativa com mm.show
mm.show(
    [img_ruid_anotada, img_filt_anotada],
    titles=[
        "1. Imagem Ruidosa Original (Sal e Pimenta)",
        "2. Pré-processada (Filtro de Mediana 3x3)"
    ],
    cols=2,
    figsize=(9, 4)
)

##### Bloco 4: Pré-processamento do *Dataset* e Ajuste Fino do YOLOv8

Este bloco aplica o Filtro de Mediana ao conjunto de imagens e realiza o ajuste fino (*fine-tuning*) do detector **YOLOv8n** pré-treinado no conjunto **COCO**. A filtragem reduz o efeito do ruído impulsivo introduzido na geração das imagens, enquanto o treinamento adapta os parâmetros da rede ao novo domínio de formas geométricas.

1. **Filtragem em lote:** O código percorre os diretórios `train` e `val` e aplica `cv2.medianBlur` com janela $3 \times 3$ em todas as imagens, mantendo as anotações YOLO originais.

2. **Ajuste fino do detector:** A rede `YOLO("yolov8n.pt")`, inicialmente treinada no COCO, é adaptada ao conjunto geométrico por meio da função `.train()`. O treinamento utiliza imagens com resolução $320 \times 320$ *pixels* durante $30$ épocas.

3. **Avaliação do modelo:** A função `.val()` calcula as métricas de detecção no conjunto de validação, incluindo **precisão** (*precision*), **revocação** (*recall*) e **mAP@50** (*mean Average Precision* com limiar de IoU igual a $0,5$).

In [ ]:
#| echo: true

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

base_dir = "shapes_dataset"
base_dir_filt = "shapes_dataset_filtrado"

# 1. Cria uma CÓPIA filtrada do dataset em uma pasta separada
#    (o dataset original em base_dir permanece ruidoso, intacto)
for split in ["train", "val"]:
    pasta_imgs_orig = f"{base_dir}/images/{split}"
    pasta_labels_orig = f"{base_dir}/labels/{split}"
    pasta_imgs_filt = f"{base_dir_filt}/images/{split}"
    pasta_labels_filt = f"{base_dir_filt}/labels/{split}"

    os.makedirs(pasta_imgs_filt, exist_ok=True)
    os.makedirs(pasta_labels_filt, exist_ok=True)

    for nome_arq in os.listdir(pasta_imgs_orig):
        if nome_arq.endswith(".jpg"):
            img_ruidosa = cv2.imread(f"{pasta_imgs_orig}/{nome_arq}")
            img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)
            # grava a versão filtrada na pasta NOVA, não sobrescreve a original
            cv2.imwrite(f"{pasta_imgs_filt}/{nome_arq}", img_filtrada)

    # copia os labels (não mudam com o filtro)
    for nome_arq in os.listdir(pasta_labels_orig):
        shutil.copy(f"{pasta_labels_orig}/{nome_arq}", f"{pasta_labels_filt}/{nome_arq}")

# 2. data.yaml apontando para o dataset FILTRADO
with open(f"{base_dir_filt}/data.yaml", "w") as f:
    f.write(
        f"path: {os.path.abspath(base_dir_filt)}\n"
        "train: images/train\nval: images/val\nnames:\n"
    )
    for i, nome in enumerate(CLASSES):
        f.write(f"  {i}: {nome}\n")

print("Pré-processamento concluído: dataset filtrado salvo em pasta separada.\n")

# Baixar modelo YOLOv8 pré-treinado (yolov8n.pt) se ainda não estiver presente
with contextlib.redirect_stdout(io.StringIO()), \
    contextlib.redirect_stderr(io.StringIO()):
    modelo_yolo = YOLO("yolov8n.pt")
print("Modelo YOLOv8 carregado.")

In [ ]:
#| echo: true

# 3. Callback customizado para imprimir apenas a época em execução
def on_train_epoch_start(trainer):
    epoch_atual = trainer.epoch + 1
    total_epochs = trainer.epochs
    # Escreve diretamente no stdout original (burlando o silenciador)
    sys.__stdout__.write(f"🔄 Processando Época {epoch_atual}/{total_epochs}...\n")
    sys.__stdout__.flush()

# Adiciona o callback ao modelo
modelo_yolo.add_callback("on_train_epoch_start", on_train_epoch_start)

# 4. Gerenciador de contexto para silenciar o lixo do Ultralytics (C/C++ e Python)
@contextlib.contextmanager
def silenciar_logs():
    logger = logging.getLogger("ultralytics")
    disabled_state = logger.disabled
    logger.disabled = True
    
    with open(os.devnull, "w") as fnull:
        old_stdout_fd = os.dup(1)
        old_stderr_fd = os.dup(2)
        try:
            os.dup2(fnull.fileno(), 1)
            os.dup2(fnull.fileno(), 2)
            with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
                yield
        finally:
            os.dup2(old_stdout_fd, 1)
            os.dup2(old_stderr_fd, 2)
            os.close(old_stdout_fd)
            os.close(old_stderr_fd)
            logger.disabled = disabled_state

# Execução do Treinamento
print("--- Iniciando Treinamento YOLOv8 ---")
with silenciar_logs():
    resultados_treino = modelo_yolo.train(
        data=f"{base_dir_filt}/data.yaml",   # <-- treina no dataset filtrado
        epochs=30,
        imgsz=320,
        batch=16,
        device=device,
        verbose=False,
        plots=False
    )
    metricas = modelo_yolo.val(verbose=False)

# 5. Métricas Finais
precision = metricas.results_dict["metrics/precision(B)"]
recall = metricas.results_dict["metrics/recall(B)"]
map50 = metricas.results_dict["metrics/mAP50(B)"]

print("\n--- Desempenho Otimizado do Modelo YOLOv8 ---")
print(f"Precisão: {precision*100:.2f}%")
print(f"Revocação (Recall): {recall*100:.2f}%")
print(f"mAP@50 (Acurácia de Detecção): {map50*100:.2f}%")

##### Bloco 5: Comparativo de Inferência: Imagem Ruidosa e Imagem Restaurada

Após o ajuste fino da YOLOv8 no conjunto restaurado, realiza-se uma comparação visual entre a detecção aplicada diretamente sobre uma imagem degradada pelo ruído sal e pimenta e a mesma imagem após o Filtro de Mediana.

O objetivo é observar como uma etapa simples de pré-processamento pode influenciar a qualidade das previsões de um detector já adaptado ao novo domínio.

1. **Inferência com a YOLO:** A função `modelo_yolo.predict()` executa a detecção nas duas versões da imagem, utilizando um limiar de confiança de $25\%$ (`conf=0.25`).

2. **Visualização das Predições:** A função `plot()` gera as imagens anotadas com as caixas delimitadoras e os rótulos previstos pelo modelo. A @fig-09-yolo-inferencia-comparativa apresenta a comparação entre os dois cenários.


In [ ]:
#| label: fig-09-yolo-inferencia-comparativa
#| fig-cap: "Comparação da inferência da YOLOv8 ajustada por transferência de aprendizado em uma imagem com ruído sal e pimenta e na versão restaurada pelo Filtro de Mediana. As caixas delimitadoras e classes previstas são exibidas sobre cada imagem."
#| eval: false
#| echo: true
#| output: true

# 1. Carregamento de uma amostra de teste original (sem o filtro salvo em lote)
caminho_teste = f"{base_dir}/images/val/0002.jpg"
img_ruidosa_teste = mm.read(caminho_teste)

# 2. Aplicação pontual do Filtro de Mediana (3x3) para comparação
img_filtrada_teste = cv2.medianBlur(img_ruidosa_teste, ksize=3)

# 3. Inferência com o modelo YOLOv8 treinado
pred_ruidosa = modelo_yolo.predict(img_ruidosa_teste, conf=0.25, verbose=False)[0]
pred_filtrada = modelo_yolo.predict(img_filtrada_teste, conf=0.25, verbose=False)[0]

# 4. Extração das matrizes anotadas pelo gerador do YOLO (conversão BGR -> RGB)
img_pred_ruid = cv2.cvtColor(pred_ruidosa.plot(), cv2.COLOR_BGR2RGB)
img_pred_filt = cv2.cvtColor(pred_filtrada.plot(), cv2.COLOR_BGR2RGB)

# 5. Exibição comparativa padronizada via mm.show
mm.show(
    [img_pred_ruid, img_pred_filt],
    titles=[
        f"Inferência na Imagem Ruidosa ({len(pred_ruidosa.boxes)} objetos)",
        f"Inferência na Imagem Filtrada ({len(pred_filtrada.boxes)} objetos)"
    ],
    cols=2,
    figsize=(10, 4)
)

In [ ]:
#| echo: true
#| output: true

# Limpeza explícita dos dados baixados e liberação de memória
if FLAG_LIMPAR_DADOS:
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
        print('🧹 Diretório de dados temporários {} removido com sucesso.'.format(base_dir))

    if os.path.exists(base_dir_filt):
        shutil.rmtree(base_dir_filt)
        print('🧹 Diretório de dados temporários {} removido com sucesso.'.format(base_dir_filt))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

::: {.callout-note}

###### 🧠 Um domínio muito mais distante que o dos dígitos {.unnumbered}

No experimento de transferência de aprendizado entre dígitos manuscritos (domínios A e B), a tarefa de origem e a de destino compartilhavam estatísticas visuais muito próximas: ambas eram traços em tons de cinza sobre fundo uniforme. Aqui, a distância entre domínios é bem maior — o YOLO foi pré-treinado em **fotografias naturais coloridas** da COCO (pessoas, animais, veículos, objetos do cotidiano), e a tarefa de destino consiste em **formas geométricas sintéticas, de cor sólida e contorno bem definido**, sem textura, iluminação ou fundo complexo.

Ainda assim, a transferência de aprendizado é vantajosa: as camadas iniciais de um detector treinado na COCO aprendem filtros genéricos — detectores de bordas, cantos e regiões de contraste — que continuam úteis para delimitar o contorno de um triângulo ou de uma estrela, mesmo que o conteúdo visual final seja bastante distinto. É por isso que permitir o ajuste fino (*fine-tuning*) de todas as camadas, combinado com um pré-processamento consistente entre treino e inferência para atenuar o ruído sal e pimenta, viabiliza taxas de acurácia elevadas na detecção de objetos do novo domínio.
:::

##### Usando um Conjunto de Dados Real

O *pipeline* acima foi construído inteiramente em torno do **formato de
anotação YOLO** (classe, $x_{centro}$, $y_{centro}$, largura, altura,
normalizados pela largura e altura da imagem) exatamente para que ele possa
ser reaproveitado sem alterações caso o leitor tenha acesso a um conjunto de
imagens reais anotado da mesma forma — por exemplo, um conjunto de imagens de
objetos geométricos fotografados ou renderizados, cada um com um arquivo
`.txt` correspondente no mesmo formato usado aqui. Para isso, bastaria:

1. Organizar as imagens reais em `shapes_dataset/images/train` e
   `shapes_dataset/images/val`, e os arquivos `.txt` de anotação
   correspondentes nas pastas `labels/train` e `labels/val` (um arquivo de
   anotação por imagem, mesmo nome-base, extensão `.txt`);
2. Ajustar o arquivo `data.yaml` caso o número ou os nomes das classes sejam
   diferentes;
3. Executar as mesmas células de treinamento, ajuste fino e visualização já
   apresentadas, sem qualquer outra modificação de código.

Essa separação entre **geração/organização dos dados** e **treinamento do
modelo** é, na prática, o motivo pelo qual formatos de anotação padronizados
(como o do YOLO) são tão amplamente adotados: eles permitem trocar o conjunto
de dados de entrada — sintético por real, um domínio por outro — mantendo
inalterado todo o restante do *pipeline* de transferência de aprendizado.

### Segmentação de Objetos

O mesmo princípio de transferência de aprendizado também fundamenta arquiteturas de segmentação de imagens, nas quais um extrator de características pré-treinado fornece representações visuais gerais, enquanto uma cabeça especializada realiza a classificação densa, pixel a pixel.

As próximas seções apresentam a **DeepLabV3** como exemplo de segmentador pré-treinado usado diretamente para inferência e, em seguida, a arquitetura **U-Net**, treinada do zero e comparada com uma linha de base morfológica clássica.

#### DeepLabV3: Segmentador Pré-treinado

Na segmentação semântica, o objetivo não se limita à localização dos objetos por meio de caixas delimitadoras (*bounding boxes*). A rede atribui uma classe a cada *pixel* da imagem, produzindo um mapa de rótulos com a mesma resolução da entrada. Arquiteturas como a **DeepLabV3**, com *backbone* **ResNet-50**, utilizam um extrator de características pré-treinado e uma cabeça especializada para realizar essa classificação densa.

1. **Carregamento e inferência:** O modelo `deeplabv3_resnet50(weights="DEFAULT")` carrega pesos pré-treinados no conjunto **Pascal VOC**, que define $21$ classes de segmentação. O código converte a imagem em *tensor*, adiciona a dimensão de *batch* (`unsqueeze(0)`) e executa a inferência.

2. **Mapa de classes:** A saída do modelo possui dimensão $(1, 21, H, W)$, contendo um valor para cada classe em cada *pixel*. A operação `.argmax(dim=1)` seleciona a classe de maior resposta em cada posição, gerando uma matriz bidimensional de rótulos com dimensões $(H, W)$.

3. **Visualização:** O código converte o mapa de rótulos para o formato esperado por `mm.show()`, que exibe o resultado da segmentação na @fig-09-segmentacao-pretreinada.

In [ ]:
#| label: fig-09-segmentacao-pretreinada
#| fig-cap: "Mapa de segmentação semântica gerado pelo modelo DeepLabV3 pré-treinado: classificação pixel a pixel representada em matriz 2D exibida."
#| eval: false
#| echo: true
#| output: true

# 1. Carregamento da imagem (retorna numpy.ndarray)
url_imagem = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
url_imagem = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = mm.read(url_imagem)

# 2. Carregamento do modelo DeepLabV3 pré-treinado no modo de avaliação
modelo_segmentacao = deeplabv3_resnet50(weights="DEFAULT").eval()

# 3. Execução da inferência sem cálculo de gradientes
with torch.no_grad():
    tensor_entrada = to_tensor(img).unsqueeze(0)  # Formato (1, C, H, W)
    saida = modelo_segmentacao(tensor_entrada)["out"]
    
    # Seleção da classe com maior probabilidade por pixel (argmax no eixo dos canais)
    mapa_classes = saida.argmax(dim=1).squeeze(0).byte().cpu().numpy()

# 4. Exibição do mapa de segmentação semântica
mm.show(
    [img,mapa_classes],
    title=["Imagem original","Segmentação Semântica (DeepLabV3)"]
)

#### Segmentação Semântica com Arquitetura U-Net

A subseção anterior apresentou um modelo pré-treinado (*DeepLabV3*) produzindo diretamente um rótulo de classe por *pixel*. Esta seção completa a sequência de projetos práticos — classificação e detecção  — abordando a segmentação semântica implementada e treinada do zero com a **U-Net**, a arquitetura de referência introduzida por @ronneberger2015u.

Na classificação o mapa de características final era achatado (*flatten*) em um vetor, descartando a informação espacial em favor de um único rótulo por imagem. A U-Net, por sua vez, produz uma saída com a **mesma resolução espacial da entrada**: um mapa bidimensional no qual cada *pixel* recebe sua própria classificação. Essa exigência — preservar detalhe espacial de alta resolução ao mesmo tempo em que se constrói contexto semântico em camadas profundas — motiva a arquitetura de codificador-decodificador com conexões de atalho (*skip connections*).

O cenário utilizado simula a segmentação de nódulos em exames médicos sintéticos: imagens em escala de cinza contêm uma região circular ("nódulo") sobreposta a um fundo, ambos contaminados por ruído gaussiano e com médias de intensidade muito próximas — um desafio intencional de baixo contraste, ideal para demonstrar o ganho do aprendizado espacial em relação à limiarização pontual.




##### Bloco 1: Gerador do Conjunto Sintético de Nódulos e Exibição Inicial

O gerador a seguir produz pares (imagem, máscara): a imagem contém uma região circular de intensidade ligeiramente superior à do fundo, ambas afetadas pelo mesmo desvio-padrão de ruído gaussiano. A máscara binária delimita exatamente a região do nódulo e serve como verdade de referência (*ground truth*).

1. **Construção do Nódulo:** A função `gera_imagem_com_nodulo` sobrepõe o nódulo ao fundo sobre uma matriz $64 \times 64$ e aplica ruído gaussiano.
2. **Visualização com `mm.show`:** A @fig-09-unet-dataset ilustra as duas primeiras amostras e suas respectivas máscaras.

In [ ]:
#| label: fig-09-unet-dataset
#| fig-cap: "Amostras do conjunto de dados sintético de nódulos: imagem em escala de cinza sob ruído e respectiva máscara binária de referência exibidass."
#| echo: true
#| output: true

TAM_IMG = 64


def gera_imagem_com_nodulo(
    tam=TAM_IMG,
    raio_min=7,
    raio_max=15,
    media_fundo=95,
    media_nodulo=118,
    sigma_ruido=26,
    rng=None,
):
    rng = rng or np.random.default_rng()
    fundo = rng.normal(media_fundo, sigma_ruido, (tam, tam))
    nodulo = rng.normal(media_nodulo, sigma_ruido, (tam, tam))
    mascara = np.zeros((tam, tam), dtype=np.uint8)

    raio = int(rng.integers(raio_min, raio_max))
    cx = int(rng.integers(raio + 4, tam - raio - 4))
    cy = int(rng.integers(raio + 4, tam - raio - 4))

    cv2.circle(mascara, (cx, cy), raio, 255, -1)
    imagem = np.clip(np.where(mascara > 0, nodulo, fundo), 0, 255).astype(
        np.uint8
    )
    return imagem, mascara


# Geração dos conjuntos de treinamento e validação
rng_dados = np.random.default_rng(42)
N_TREINO, N_VAL = 160, 40

imgs_treino, masks_treino = zip(
    *[gera_imagem_com_nodulo(rng=rng_dados) for _ in range(N_TREINO)]
)
imgs_val, masks_val = zip(
    *[gera_imagem_com_nodulo(rng=rng_dados) for _ in range(N_VAL)]
)

print(
    f"Conjunto de treino: {N_TREINO} imagens | Conjunto de validação: {N_VAL} imagens\n"
)

# Exibição de amostras iniciais 
mm.show(
    [imgs_treino[0], masks_treino[0], imgs_treino[1], masks_treino[1]],
    titles=["Imagem 1", "Máscara 1", "Imagem 2", "Máscara 2"],
    cols=4,
    figsize=(11, 3),
)

##### Bloco 2: Linha de Base Clássica (Filtragem, Otsu e Morfologia)

Antes de empregar a U-Net, avalia-se o desempenho de um *pipeline* morfológico clássico construído com a biblioteca `morph`: um **suavizador gaussiano** (`mm.blur`), uma **limiarização de Otsu** (`mm.threshold`) e uma **abertura morfológica** (`mm.open`) para eliminação de ruídos isolados.

1. **Métrica de Interseção sobre União (IoU):** A função `iou_mascaras` calcula o grau de sobreposição *pixel a pixel* entre a predição e a máscara real.
2. **Execução e Comparativo:** A @fig-09-unet-classico exibe o resultado da segmentação clássica em uma imagem de teste, demonstrando as limitações do limiar global sob baixo contraste.

In [ ]:
#| label: fig-09-unet-classico
#| fig-cap: "Linha de base clássica de segmentação: suavização, limiarização de Otsu e abertura morfológica exibidas."
#| echo: true
#| output: true

def iou_mascaras(predita, referencia):
    p, r = predita > 0, referencia > 0
    intersecao = np.logical_and(p, r).sum()
    uniao = np.logical_or(p, r).sum()
    return intersecao / uniao if uniao else 1.0


def segmenta_classico(imagem, elemento_estrutural):
    suavizada = mm.blur(imagem, 7)
    binaria = mm.threshold(suavizada)
    return mm.open(binaria, elemento_estrutural)


elemento_estrutural = mm.sedisk(5)
ious_classico = [
    iou_mascaras(segmenta_classico(img, elemento_estrutural), mask)
    for img, mask in zip(imgs_val, masks_val)
]
iou_classico_medio = float(np.mean(ious_classico))
print(
    f"IoU médio (linha de base clássica) na validação: {iou_classico_medio:.4f}\n"
)

predicao_classica_exemplo = segmenta_classico(imgs_val[0], elemento_estrutural)

mm.show(
    [imgs_val[0], masks_val[0], predicao_classica_exemplo],
    titles=["Imagem", "Máscara de Referência", "Predição Clássica"],
    cols=3,
    figsize=(9, 3.2),
)

##### Bloco 3: Construção da Arquitetura U-Net e Funções de Perda
 
A U-Net é uma arquitetura em formato de "U" (daí o nome), pensada especificamente para segmentação de imagens. Ela é formada por dois caminhos que trabalham em conjunto:
 
- 🔽 **Codificador (encoder):** desce pela imagem, reduzindo a resolução espacial a cada etapa enquanto extrai características cada vez mais abstratas (bordas → texturas → formas → contexto).
- 🔼 **Decodificador (decoder):** sobe de volta, reconstruindo a resolução original através de convoluções transpostas (*upsampling*), até gerar uma máscara do mesmo tamanho da imagem de entrada.
 
O elemento que torna a U-Net especial são as **conexões de atalho** (*skip connections*): elas levam os mapas de características do codificador diretamente para a etapa correspondente do decodificador, na mesma resolução. Isso evita que detalhes finos — como contornos e bordas — se percam durante a compressão espacial.
 
**Fluxo geral da arquitetura:**
 
```text
Entrada
  │
  ▼
Codificador (Conv → Conv → Pool) × 3
  │
  ├──── conexões de atalho reinjetam mapas de alta resolução  ────┐
  ▼                                                               │
Base (bottleneck)                                                 │
  │                                                               │
  ▼                                                               │
Decodificador (Upsample → Concat → Conv → Conv) × 3  ◄────────────┘
  │
  ▼
Saída 1×1 (logits)
```
 
**Componentes principais:**
 
1. **Bloco Convolucional Base (`BlocoConv`)**
   A unidade fundamental repetida em toda a rede. Aplica duas convoluções $3 \times 3$ em sequência, cada uma seguida de ativação ReLU, com *padding* que preserva as dimensões espaciais da entrada. É esse bloco que aparece tanto no codificador quanto no decodificador.
 
2. **Convolução Transposta (`nn.ConvTranspose2d`)**
   É a operação responsável pelo *upsampling* no decodificador: ao invés de reduzir a resolução espacial (como o `MaxPool2d` faz no codificador), ela a aumenta, aprendendo os pesos necessários para "desfazer" a compressão e recuperar gradualmente o tamanho original da imagem.
 
3. **Perda Combinada (BCE + Dice)**
   A função `perda_segmentacao` soma duas métricas complementares:
   - **Entropia Cruzada Binária (BCE):** avalia o acerto *pixel a pixel*.
   - **Coeficiente de Dice:** avalia a *sobreposição global* entre a máscara prevista e a real.
 
   Juntas, elas equilibram precisão local com fidelidade da forma segmentada como um todo.
 

In [ ]:
#| echo: true

class BlocoConv(nn.Module):

    def __init__(self, canais_entrada, canais_saida):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Conv2d(canais_entrada, canais_saida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(canais_saida, canais_saida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.rede(x)


class UNetCompacta(nn.Module):

    def __init__(self, canais_entrada=1, base=8):
        super().__init__()
        self.enc1 = BlocoConv(canais_entrada, base)
        self.enc2 = BlocoConv(base, base * 2)
        self.enc3 = BlocoConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)

        self.fundo = BlocoConv(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(
            base * 8, base * 4, kernel_size=2, stride=2
        )
        self.dec3 = BlocoConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(
            base * 4, base * 2, kernel_size=2, stride=2
        )
        self.dec2 = BlocoConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(
            base * 2, base, kernel_size=2, stride=2
        )
        self.dec1 = BlocoConv(base * 2, base)

        self.saida = nn.Conv2d(base, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        f = self.fundo(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(f), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.saida(d1)


def para_tensores(imagens, mascaras):
    X = (
        torch.tensor(np.stack(imagens), dtype=torch.float32).unsqueeze(1)
        / 255.0
    )
    Y = (
        torch.tensor(np.stack(mascaras), dtype=torch.float32).unsqueeze(1)
        / 255.0
    )
    return X, Y


def perda_dice(logits, alvo, eps=1e-6):
    probs = torch.sigmoid(logits)
    intersecao = (probs * alvo).sum(dim=(1, 2, 3))
    uniao = probs.sum(dim=(1, 2, 3)) + alvo.sum(dim=(1, 2, 3))
    dice = (2 * intersecao + eps) / (uniao + eps)
    return 1 - dice.mean()


def perda_segmentacao(logits, alvo):
    return nn.functional.binary_cross_entropy_with_logits(
        logits, alvo
    ) + perda_dice(logits, alvo)

##### Bloco 4: Treinamento da U-Net e Avaliação de Desempenho

O treinamento é executado por $35$ épocas utilizando o otimizador *Adam*. A cada época, monitora-se a Perda de Treinamento e o índice IoU médio no conjunto de validação.

1. **Laço de Treinamento:** Atualizam-se os parâmetros com *mini-batches* de $16$ amostras.
2. **Evolução Gráfica:** A @fig-09-unet-treinamento exibe o gráfico com o progresso da perda e do IoU.


In [ ]:
#| label: fig-09-unet-treinamento
#| fig-cap: "Evolução do treinamento da U-Net compacta: redução da perda e elevação do índice IoU médio no conjunto de validação ao longo de 35 épocas."
#| echo: true
#| output: true

modelo_unet = UNetCompacta()
print(
    f"Parâmetros treináveis da U-Net: {sum(p.numel() for p in modelo_unet.parameters())}"
)

X_treino_unet, Y_treino_unet = para_tensores(imgs_treino, masks_treino)
X_val_unet, Y_val_unet = para_tensores(imgs_val, masks_val)

otimizador_unet = optim.Adam(modelo_unet.parameters(), lr=1e-3)
n = X_treino_unet.size(0)
tam_lote = 16
epocas_unet = 35
historico_perda_unet, historico_iou_unet = [], []

for epoca in range(epocas_unet):
    if epoca % 5 == 0:  # Imprime a cada 5 épocas
        print(epoca + 1, "/", epocas_unet)
    modelo_unet.train()
    perm = torch.randperm(n)
    perda_epoca = 0.0

    for i in range(0, n, tam_lote):
        idx = perm[i : i + tam_lote]
        otimizador_unet.zero_grad()
        logits = modelo_unet(X_treino_unet[idx])
        perda = perda_segmentacao(logits, Y_treino_unet[idx])
        perda.backward()
        otimizador_unet.step()
        perda_epoca += perda.item() * len(idx)

    modelo_unet.eval()
    with torch.no_grad():
        predicao_val = (torch.sigmoid(modelo_unet(X_val_unet)) > 0.5).float()
        intersecao = (predicao_val * Y_val_unet).sum(dim=(1, 2, 3))
        uniao = ((predicao_val + Y_val_unet) > 0).float().sum(dim=(1, 2, 3))
        iou_epoca = (intersecao / uniao.clamp(min=1e-6)).mean().item()

    historico_perda_unet.append(perda_epoca / n)
    historico_iou_unet.append(iou_epoca)

iou_unet_final = historico_iou_unet[-1]
print(f"IoU médio final da U-Net na validação: {iou_unet_final:.4f}")

# Gráfico da evolução do treinamento
r = mm.showTrainCurves(historico_perda_unet, historico_iou_unet,
        titulo="Treinamento da U-Net — Segmentação de Nódulos Sintéticos",
        subtitulo="Perda no conjunto de treino e IoU médio no conjunto de validação \
            ao longo de 35 épocas")

##### Bloco 5: Comparativo Quantitativo e Qualitativo (Clássico vs. U-Net)

A comparação entre a abordagem clássica e a U-Net explicita a superioridade do aprendizado de representações em cenários de baixo contraste.

1. **Painel de Métricas:** O gráfico de barras na @fig-09-unet-comparativo contrasta o IoU médio de ambos os métodos na validação.
2. **Visualização Qualitativa:** A comparação visual em três amostras demonstra como as conexões de atalho recuperam o contorno do nódulo mesmo sob ruído acentuado.

In [ ]:
#| label: fig-09-unet-comparativo
#| fig-cap: "Comparativo quantitativo (IoU médio) e qualitativo entre a abordagem clássica e a U-Net treinada em três amostras de validação."
#| echo: true
#| output: true

# 1.  comparativo 
print(f"IoU médio (clássico Suavização + Otsu): {iou_classico_medio:.4f}")
print(f"IoU médio (U-Net): {iou_unet_final:.4f}")

# 2. Exibição qualitativa lado a lado em 3 amostras via mm.show
imgs_comparativas = []
titulos_comparativos = []

for i in range(3):
    with torch.no_grad():
        pred_unet = (
            (torch.sigmoid(modelo_unet(X_val_unet[i : i + 1])) > 0.5)
            .float()
            .squeeze()
            .numpy()
            * 255
        )

    pred_classico = segmenta_classico(imgs_val[i], elemento_estrutural)

    imgs_comparativas.extend(
        [imgs_val[i], masks_val[i], pred_classico, pred_unet.astype(np.uint8)]
    )

    t_prefix = f"Amostra {i+1}"
    titulos_comparativos.extend(
        [
            f"{t_prefix}: Imagem",
            f"{t_prefix}: Referência",
            f"{t_prefix}: Clássico",
            f"{t_prefix}: U-Net",
        ]
    )

mm.show(
    imgs_comparativas,
    titles=titulos_comparativos,
    cols=4,
    figsize=(11, 7.5),
)

::: {.callout-note}

###### 🧠 Por que a U-Net supera a limiarização fixa? {.unnumbered}

A limiarização de Otsu aplica um valor de corte global sobre a intensidade local. Quando a diferença de média entre o nódulo e o fundo é pequena diante do ruído gaussiano, essa regra comete erros sistemáticos nas bordas.

A U-Net contorna essa limitação ao combinar o contexto semântico amplo extraído pelo codificador com os detalhes espaciais finos preservados pelas conexões de atalho. Isso permite identificar a presença do nódulo e delimitar seus contornos com precisão, mesmo sob ruído intenso.
:::

### Engenharia de Dados para VC (*Roboflow*)

Uma U-Net pode ser treinada a partir de imagens anotadas em plataformas de engenharia de dados para Visão Computacional (VC), como o *Roboflow*. Essas plataformas permitem organizar conjuntos de dados, realizar anotações, aplicar etapas de pré-processamento e *data augmentation*, treinar modelos e exportar os dados em diferentes formatos. Nesta seção, entretanto, o exemplo retoma a **detecção de objetos geométricos**, utilizando conjuntos de dados já apresentados neste livro.

::: {.callout-important}

#### Dependência de conexão e chave de API {.unnumbered}

As células desta seção exigem conexão com a Internet e uma chave de API gratuita do *Roboflow* (`app.roboflow.com`). No *Workspace* do projeto, acesse **⚙ → Roboflow API** e copie a **Private API Key**.

Crie, nesta pasta, o arquivo `chave_roboflow.txt` contendo **somente a chave**, sem aspas. Um modelo desse arquivo está disponível em `chave_roboflow.txt.exemplo`.

Adicione `chave_roboflow.txt` ao arquivo `.gitignore`, pois ele contém uma credencial de acesso que não deve ser versionada nem compartilhada.
:::

#### Detecção de objetos utilizando *Roboflow*

O *Roboflow* permite realizar inferência sobre imagens locais, possibilitando
avaliar o desempenho do modelo hospedado na identificação dos objetos de
interesse.

Além da inferência, o *dataset* pode ser exportado no formato
`png-mask-semantic`, no qual cada imagem é acompanhada por uma máscara de
segmentação semântica. Nessa máscara, cada pixel representa a classe à qual
pertence o objeto correspondente. Os pares imagem-máscara são utilizados
como dados de treinamento para a `UNetCompacta`.


#### Conexão, *download* e verificação do *dataset*

O código estabelece conexão com o *Workspace* `mctest` e o projeto
`geometric-test00`, versão 6, e realiza o *download* do *dataset* para
`dados/datasetRoboFlow`. Em seguida, verifica o *shape* das imagens em cada
*split*. A função de verificação é reutilizada posteriormente na seção.

In [ ]:
#| echo: true
#| eval: false
from roboflow import Roboflow
from pathlib import Path
from PIL import Image
from collections import Counter

def contar_shapes(raiz, splits=("train", "valid", "test")):
    """Conta o shape (altura, largura, canais) das imagens por split."""
    for split in splits:
        pasta = raiz / split / "images"
        if not pasta.exists():
            print(f"{split}: pasta não encontrada")
            continue
        shapes = Counter()
        for arquivo in pasta.iterdir():
            if arquivo.is_file():
                with Image.open(arquivo) as img:
                    shapes[(img.height, img.width, len(img.getbands()))] += 1
        txt = ", ".join(f"{s}: {n}" for s, n in shapes.items())
        print(f"{split}: {txt}")

with open("chave_roboflow.txt") as f:
    api_key = f.read().strip()

# Projeto:
# https://app.roboflow.com/mctest/geometric-test00/models
# geometric-test00/6

rf = Roboflow(api_key=api_key)
projeto = rf.workspace("mctest").project("geometric-test00")
versao = projeto.version(6)  # versão utilizada

print(f"ID: {versao.version} | Nome: {versao.name} | "
      f"Imagens: {versao.images}")

raiz = Path("dados/datasetRoboFlow")
versao.download("yolov8", location=str(raiz))
print("Dataset:", raiz)

contar_shapes(raiz)

#### Baixando o *dataset* de forma reprodutível (alternativa)

Como alternativa ao *dataset* obtido pelo *Roboflow*, pode-se utilizar um
*dataset* disponibilizado em um repositório GitHub, também organizado nos
mesmos *splits* e contendo as mesmas classes de objetos. Os conjuntos de
dados, entretanto, não são idênticos: as imagens do GitHub possuem resolução
de `608×608` pixels, enquanto as imagens exportadas pelo *Roboflow* possuem
`640×640` pixels.

O código abaixo realiza o *download* do *dataset* do GitHub, caso ele ainda
não esteja disponível localmente, e reutiliza `contar_shapes` para verificar
as dimensões das imagens em cada *split*.

In [ ]:
#| echo: true
#| eval: false
import os

if not os.path.exists("dados/dataset"):
    cmd = (
        "git clone --no-checkout --depth 1 --filter=blob:none "
        "https://github.com/fzampirolli/pdi-vc.git tmp_repo && "
        "cd tmp_repo && git sparse-checkout set all/cap09/dados/dataset "
        "&& git checkout && cd .. && mkdir -p dados && "
        "cp -r tmp_repo/all/cap09/dados/dataset dados/dataset && "
        "rm -rf tmp_repo"
    )
    !{cmd}

versao_recente = projeto.versions()[-1]
print(f"Versão mais recente: {versao_recente.version.split('/')[-1]}")

contar_shapes(Path("dados/dataset"))

#### Comparando uma imagem de cada *dataset*

Os dois *datasets* possuem imagens com resoluções diferentes: `608×608` no
GitHub e `640×640` no *Roboflow*. Para uma comparação visual direta, as
imagens são redimensionadas para a mesma dimensão antes de serem exibidas
lado a lado (@fig-09-datasets-comparacao).

In [ ]:
#| echo: true
#| eval: false
#| label: fig-09-datasets-comparacao
#| fig-cap: "Uma imagem de cada *dataset*, redimensionadas para comparação"
import cv2

caminho1 = next((raiz / "train/images").iterdir())
caminho2 = next((Path("dados/dataset") / "train/images").iterdir())

img1 = mm.read(str(caminho1))
img2 = mm.read(str(caminho2))

# Redimensiona ambas para o mesmo tamanho (o menor entre as duas)
largura = min(img1.shape[1], img2.shape[1])
altura = min(img1.shape[0], img2.shape[0])
img1_r = cv2.resize(img1, (largura, altura))
img2_r = cv2.resize(img2, (largura, altura))

mm.show(
    [img1_r, img2_r],
    title=[f"Roboflow {img1.shape}", f"GitHub {img2.shape}"],
)

#### Inferência com o modelo treinado

O modelo treinado na versão 6 é utilizado para realizar a inferência sobre
uma imagem de teste local. A predição considera os limiares de confiança e
de sobreposição empregados pela supressão de não máximos (*Non-Maximum
Suppression*, NMS), e o resultado é apresentado na imagem anotada da
@fig-09-roboflow-detection.

In [ ]:
#| echo: true
#| eval: false
#| label: fig-09-roboflow-detection
#| fig-cap: "Resultado da inferência com o modelo do Roboflow"
# version.model está deprecated; usar version.models()
modelo = versao.models()[0]

img_path = "dados/dataset/test/images/00001.jpg"
pred = modelo.predict(img_path, confidence=40, overlap=30)
resp = pred.json()  # inclui as caixas detectadas em resp["predictions"]

altura, largura, _ = mm.read(img_path).shape
print("Dimensões da imagem de teste:", (altura, largura))

pred.save("resultado.jpg")  # imagem anotada

mm.show(
    mm.read("resultado.jpg"),
    title="Resultado da inferência com o modelo do Roboflow",
    figsize=(6, 6)
)

#### Avaliando as predições com IoU e classe

O *Roboflow* retorna cada caixa com as coordenadas do centro (`x`, `y`) em
*pixels* e a classe prevista, enquanto os rótulos locais
(`dados/dataset/test/labels/00001.txt`) seguem o formato YOLO, com centro e
dimensões normalizados no intervalo `[0, 1]`. Antes da comparação por meio de
`mm.IoU`, as caixas devem ser convertidas para o mesmo formato, com as
coordenadas do canto superior esquerdo e as dimensões expressas em
*pixels*.

Uma predição só é considerada correta quando a classe prevista coincide com
a classe da caixa real e sua *Intersection over Union* (IoU) é maior ou
igual ao limiar definido, adotando-se, neste exemplo, `0,5` (50%) como valor
padrão.

::: {.callout-important}

##### O `class_id` do *Roboflow* não corresponde ao índice das *labels* locais {.unnumbered}

Na exportação, o *Roboflow* reordena as classes em **ordem alfabética** no
`data.yaml`, independentemente da ordem utilizada no projeto original,
mantida no `data.yaml` do GitHub. Assim, `class_id = 0` corresponde a
`Circulo` no retorno da API, enquanto o mesmo índice corresponde a
`Triangulo` nas *labels* locais.

Por isso, a comparação deve ser feita pelo **nome da classe**
(`p["class"]`), convertendo-o posteriormente para o índice correspondente
na lista local. Os `class_id` não devem ser comparados diretamente.
:::


In [ ]:
#| echo: true
#| eval: false
# resp, largura e altura foram definidos na célula anterior

# Ordem das classes usada nas labels locais (dados/dataset/*/labels/*.txt)
CLASSES_LOCAIS = ["Triangulo", "Quadrado", "Pentagono", "Hexagono",
                   "Heptagono", "Circulo", "Elipse"]

def predicao_correta(pred: tuple, real: tuple, limiar: float = 0.5) -> bool:
    """True se mesma classe e mm.IoU(caixa_pred, caixa_real) >= limiar."""
    classe_pred, caixa_pred = pred
    classe_real, caixa_real = real
    return classe_pred == classe_real and mm.IoU(caixa_pred, caixa_real) >= limiar

def caixa_roboflow(p: dict) -> tuple:
    """Converte predição do Roboflow para (classe, (x, y, w, h))."""
    caixa = (p["x"] - p["width"] / 2, p["y"] - p["height"] / 2,
             p["width"], p["height"])
    # usa o nome da classe (não o class_id!) para casar com a ordem local
    classe = CLASSES_LOCAIS.index(p["class"])
    return (classe, caixa)

def carrega_labels_yolo(caminho_txt: str, largura: int, altura: int) -> list:
    """Lê rótulos YOLO (normalizados) e converte para (classe, (x, y, w, h))."""
    caixas = []
    with open(caminho_txt) as f:
        for linha in f:
            classe, xc, yc, w, h = map(float, linha.split())
            w_px, h_px = w * largura, h * altura
            x_px = xc * largura - w_px / 2
            y_px = yc * altura - h_px / 2
            caixas.append((int(classe), (x_px, y_px, w_px, h_px)))
    return caixas

caixas_pred = [caixa_roboflow(p) for p in resp["predictions"]]
caixas_real = carrega_labels_yolo(
    "dados/dataset/test/labels/00001.txt", largura, altura
)

acertos = sum(
    any(predicao_correta(cp, cr, limiar=0.5) for cr in caixas_real)
    for cp in caixas_pred
)
print(f"{acertos}/{len(caixas_pred)} predições corretas (classe + IoU >= 50%)")

Para visualizar o resultado, cada predição é desenhada sobre a imagem:
verde quando é um acerto (classe + IoU ≥ limiar) e vermelho quando é um
erro, como mostra a @fig-09-roboflow-acertos.

In [ ]:
#| echo: true
#| eval: false
#| label: fig-09-roboflow-acertos
#| fig-cap: "Predições corretas (verde) e incorretas (vermelho)"
import cv2

VERDE, VERMELHO = (0, 255, 0), (255, 0, 0)

img_acertos = mm.read(img_path).copy()

for cp in caixas_pred:
    classe_pred, (x, y, w, h) = cp
    correta = any(predicao_correta(cp, cr, limiar=0.5) for cr in caixas_real)
    cor = VERDE if correta else VERMELHO

    p1, p2 = (int(x), int(y)), (int(x + w), int(y + h))
    cv2.rectangle(img_acertos, p1, p2, cor, 2)
    cv2.putText(img_acertos, str(classe_pred), (p1[0], p1[1] - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, cor, 2)

mm.show(
    img_acertos,
    title=f"{acertos}/{len(caixas_pred)} predições corretas "
          f"(classe + IoU >= 50%)",
    figsize=(6, 6)
)

### Aplicações Geométricas e Integradas

O capítulo encerra-se integrando dois pilares da VC: a **geometria projetiva** (estudada nos Capítulos 6 e 8) e o **aprendizado profundo**. A combinação dessas abordagens sustenta aplicações práticas no mundo real, como ilustrado a seguir.

#### Realidade Aumentada com Marcadores e Homografia

Imagine uma câmera apontada para uma mesa onde alguém colou um pequeno marcador ArUco. Dependendo do ângulo da câmera, esse marcador aparece **rotacionado, inclinado, em perspectiva** — nunca perfeitamente quadrado. É justamente essa distorção que a homografia sabe "ler" e desfazer (ou, no nosso caso, replicar para uma nova imagem).

O fluxo completo de uma aplicação de RA baseada em marcadores segue três passos:

1. **Ambientação:** o marcador é inserido em uma cena real, sofrendo uma transformação de perspectiva (simulando o ângulo da câmera).
2. **Detecção:** o algoritmo localiza o marcador na cena e recupera as coordenadas exatas dos seus 4 cantos com `cv2.aruco.ArucoDetector`.
3. **Substituição:** com a homografia entre o marcador "ideal" e o marcador "detectado", projeta-se uma **nova imagem virtual** exatamente sobre a área do marcador — como se ele tivesse se transformado em uma janela para outro conteúdo, conforme mostra a @fig-09-realidade-aumentada.

::: {.callout-note}
**Detalhe técnico importante:** o ArUco precisa de uma margem branca ao redor do padrão (a "zona de silêncio") para o detector conseguir diferenciar o marcador do fundo. Por isso, o código abaixo adiciona uma borda com `cv2.copyMakeBorder` e usa interpolação `cv2.INTER_NEAREST` ao deformar a imagem, evitando que a rotação borre os quadradinhos pretos e brancos e impeça a detecção.
:::

In [ ]:
#| label: fig-09-realidade-aumentada
#| fig-cap: "Realidade aumentada baseada em marcador ArUco: marcador inserido em cena real e rotacionada, detectado e substituído por imagem virtual via homografia."
#| echo: true
#| output: true

# 1. Geração do marcador ArUco sintético, já com margem branca (quiet zone)
# → essa margem é essencial para o detector conseguir "enxergar" o marcador
dic = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_bruto = cv2.aruco.generateImageMarker(dic, 7, 200)
aruco_gray = cv2.copyMakeBorder(
    aruco_bruto, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255
)  # 200 -> 280px, com 40px de moldura branca em cada lado
aruco = cv2.cvtColor(aruco_gray, cv2.COLOR_GRAY2BGR)
lado = aruco.shape[0]  # 280

# 2. Cena real de fundo, usando uma imagem de exemplo do skimage.data
fundo = cv2.cvtColor(skdata.coffee(), cv2.COLOR_RGB2BGR)
cena = cv2.resize(fundo, (640, 480))

# Cantos do marcador "de frente" (src) e sua posição rotacionada/inclinada na cena (dst)
src = np.float32([[0, 0], [lado, 0], [lado, lado], [0, lado]])
dst = np.float32([[190, 160], [420, 70], [470, 330], [150, 370]])  # rotação + perspectiva

# 3. Projeta o marcador (com bordas nítidas) sobre a cena real
H_cena, _ = cv2.findHomography(src, dst)
mask_cena = cv2.warpPerspective(
    np.full((lado, lado), 255, np.uint8), H_cena, (640, 480),
    flags=cv2.INTER_NEAREST,
)
cena[mask_cena > 0] = cv2.warpPerspective(
    aruco, H_cena, (640, 480), flags=cv2.INTER_NEAREST
)[mask_cena > 0]

# 4. Detecção do marcador dentro da cena (como uma câmera faria)
det = cv2.aruco.ArucoDetector(dic, cv2.aruco.DetectorParameters())
corners, ids, _ = det.detectMarkers(cena)

assert ids is not None and len(corners) > 0, \
    "Marcador não detectado — confira iluminação/contraste da cena."

# 5. Imagem virtual (outra imagem do skimage.data) que vai "substituir" o marcador
# Repare: usamos o quadrado INTERNO do marcador (sem a margem) como área de projeção,
# então a homografia da imagem virtual usa 'src' original (200x200), não o 'lado' com borda
src_interno = np.float32([[0, 0], [200, 0], [200, 200], [0, 200]])
virtual = cv2.resize(
    cv2.cvtColor(skdata.camera(), cv2.COLOR_RGB2BGR), (200, 200)
)

# Os cantos detectados correspondem ao marcador COM a margem (280x280),
# então recalculamos H_ra usando 'src' com margem, para manter a projeção coerente
H_ra, _ = cv2.findHomography(src, corners[0][0])

# A homografia detectada é aplicada à imagem virtual (redimensionada para 'lado')
virtual_grande = cv2.resize(virtual, (lado, lado))
ra = cena.copy()
mask_ra = cv2.warpPerspective(
    np.full((lado, lado), 255, np.uint8), H_ra, (640, 480), flags=cv2.INTER_NEAREST
)
ra[mask_ra > 0] = cv2.warpPerspective(
    virtual_grande, H_ra, (640, 480), flags=cv2.INTER_NEAREST
)[mask_ra > 0]

# Exibição: cena com marcador vs. cena com Realidade Aumentada aplicada
mm.show(
    [cv2.cvtColor(cena, cv2.COLOR_BGR2RGB), cv2.cvtColor(ra, cv2.COLOR_BGR2RGB)],
    titles=["Marcador na Cena Real (rotacionado)", "Sobreposição Virtual via Homografia"],
    cols=2,
    figsize=(9, 4),
)

**Resumo do *pipeline*:**

| Etapa | O que faz | Função-chave |
|---|---|---|
| 1. Geração | Marcador ArUco com margem branca | `generateImageMarker` + `copyMakeBorder` |
| 2. Ambientação | Insere marcador distorcido na cena | `findHomography` + `warpPerspective` |
| 3. Detecção | Localiza marcador e retorna cantos | `ArucoDetector.detectMarkers` |
| 4. Substituição | Projeta imagem virtual sobre o marcador | `findHomography` + `warpPerspective` |

> 💡 **Lição:** detector "não encontrar nada" é comum em VC. Pergunte sempre: **"dei contraste e espaço suficientes?"** — vale para ArUco, QR codes e reconhecimento facial.

#### Fotogrametria e Referência de Escala

A **fotogrametria** permite estimar dimensões físicas de objetos a partir de imagens digitais. Para isso, utiliza-se um objeto de referência com dimensões conhecidas, posicionado na mesma cena do objeto de interesse. Esse procedimento estabelece uma relação entre distâncias medidas em *pixels* e suas correspondentes dimensões no mundo real.

Considere, por exemplo, um cartão de crédito, cujas dimensões seguem o padrão internacional ISO/IEC 7810. Como sua largura é exatamente **8,56 cm**, basta determinar quantos *pixels* essa largura ocupa na imagem para calcular o fator de conversão entre *pixels* e centímetros. Se o cartão corresponder a 140 *pixels*, então cada *pixel* representará aproximadamente **0,061 cm**. Esse mesmo fator de escala pode ser aplicado para estimar as dimensões de qualquer outro objeto localizado no mesmo plano da cena, conforme ilustrado na @fig-09-fotogrametria.

O procedimento pode ser dividido em duas etapas principais:

1. **Segmentação e *bounding box*:** localizar, na imagem, tanto o objeto de referência quanto o objeto de interesse, utilizando técnicas como segmentação por cor, limiarização, detecção de contornos ou métodos de detecção de objetos.
2. **Conversão para dimensões físicas:** calcular a razão $\mathrm{cm/pixel}$ a partir da largura conhecida do objeto de referência e utilizá-la para converter as medidas do objeto de interesse de *pixels* para centímetros.

::: {.callout-note}
**Condição para medições confiáveis**

A conversão entre *pixels* e centímetros pressupõe que o objeto de referência e o objeto de interesse estejam aproximadamente no mesmo plano e à mesma distância da câmera. Nessas condições, a escala permanece praticamente constante em toda a imagem. Diferenças de profundidade, inclinação da câmera ou distorções da lente podem introduzir erros nas medidas estimadas.
:::

In [ ]:
#| label: fig-09-fotogrametria
#| fig-cap: "Medição de dimensões físicas reais utilizando um cartão de referência de escala conhecida (8,56 cm) renderizada. O retângulo cinza representa o cartão de referência e o retângulo azul representa o objeto alvo. As dimensões estimadas do objeto são exibidas em centímetros."
#| echo: true
#| output: true

COR_REFERENCIA = (200, 200, 200)  # Cartão de referência (cinza)
COR_OBJETO = (60, 60, 220)  # Objeto alvo (vermelho)

# Desenho da cena sintética
cena_medicao = np.full((300, 500, 3), 255, dtype=np.uint8)
cv2.rectangle(
    cena_medicao, (30, 200), (30 + 140, 200 + 88), COR_REFERENCIA, -1
)
cv2.rectangle(cena_medicao, (250, 100), (250 + 220, 100 + 150), COR_OBJETO, -1)


def caixa_delimitadora_por_cor(imagem_bgr, cor_bgr, tolerancia=40):
    diferenca = np.abs(imagem_bgr.astype(int) - np.array(cor_bgr)).sum(axis=2)
    mascara = (diferenca < tolerancia).astype(np.uint8) * 255
    contornos, _ = cv2.findContours(
        mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    maior_contorno = max(contornos, key=cv2.contourArea)
    return cv2.boundingRect(maior_contorno)


x_ref, y_ref, w_ref_px, h_ref_px = caixa_delimitadora_por_cor(
    cena_medicao, COR_REFERENCIA
)
x_obj, y_obj, w_obj_px, h_obj_px = caixa_delimitadora_por_cor(
    cena_medicao, COR_OBJETO
)

# Cálculo de escala física
LARGURA_REFERENCIA_CM = 8.56
razao_cm_por_px = LARGURA_REFERENCIA_CM / w_ref_px
largura_obj_cm = w_obj_px * razao_cm_por_px
altura_obj_cm = h_obj_px * razao_cm_por_px

# Desenho das caixas e medições estimadas
resultado = cena_medicao.copy()
cv2.rectangle(
    resultado, (x_ref, y_ref), (x_ref + w_ref_px, y_ref + h_ref_px), (0, 180, 0), 2
)
cv2.rectangle(
    resultado, (x_obj, y_obj), (x_obj + w_obj_px, y_obj + h_obj_px), (0, 180, 0), 2
)

cv2.putText(
    resultado,
    f"{LARGURA_REFERENCIA_CM:.2f} cm",
    (x_ref, y_ref - 8),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.55,
    (0, 120, 0),
    2,
)

cv2.putText(
    resultado,
    f"{largura_obj_cm:.1f} x {altura_obj_cm:.1f} cm",
    (x_obj, y_obj - 8),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (0, 120, 0),
    2,
)

mm.show(
    [
        cv2.cvtColor(cena_medicao, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB),
    ],
    titles=[
        "Cena Original",
        "Medição por Referência de Escala (Centímetros)",
    ],
    cols=2,
    figsize=(9, 4),
)


**Resumo do *pipeline*:**

| Etapa | O que faz | Função-chave |
|---|---|---|
| 1. Cena sintética | Desenha o cartão de referência e o objeto alvo com cores distintas | `cv2.rectangle` |
| 2. Segmentação | Isola cada objeto por cor e extrai seu contorno | `cv2.findContours` |
| 3. Bounding Box | Obtém a caixa delimitadora (posição e tamanho em pixels) de cada objeto | `cv2.boundingRect` |
| 4. Escalonamento | Converte pixels em centímetros usando a largura conhecida do cartão | Regra de três: $\text{cm/pixel} = \dfrac{8{,}56}{w_{ref\_px}}$ |
| 5. Anotação | Desenha as caixas e exibe as medidas estimadas sobre a imagem | `cv2.rectangle` + `cv2.putText` |

> 💡 **Aplicação no mundo real:** essa é exatamente a técnica usada por *apps* de *e-commerce* que estimam o tamanho de um produto a partir de uma foto ao lado de um cartão, por sistemas agrícolas que medem frutas em esteiras, e até por perícias forenses que calculam dimensões de vestígios em cenas de crime — tudo com a mesma ideia: **uma régua conhecida dentro da própria foto.**

## Resumo

Este capítulo, que encerra a Parte II do livro, apresentou:

* **Convolução e *pooling* aprendidos:** a mesma operação matemática de
  convolução do Capítulo 3, mas com *kernels* tratados como parâmetros
  ajustados por treinamento, em vez de definidos manualmente;

* **Compartilhamento de pesos e hierarquia de características** como
  propriedades que tornam as CNNs eficientes e capazes de aprender
  representações cada vez mais abstratas em camadas sucessivas;

* **Treinamento de uma CNN do zero**, com desempenho comparável — e não
  necessariamente superior — aos classificadores clássicos do Capítulo 7
  em uma base pequena e simples, reforçando que a escolha do método deve
  ser proporcional à complexidade real do problema;

* **Transferência de aprendizado**, demonstrada experimentalmente como uma
  estratégia eficaz para tarefas com poucos dados rotulados, reaproveitando
  um extrator de características já treinado em uma tarefa relacionada, e
  suas limitações, evidenciadas pela transferência negativa entre domínios
  muito distintos;

* **Aplicações em larga escala** com modelos pré-treinados de classificação,
  detecção (*Faster R-CNN*) e segmentação (*DeepLabV3*), seguindo o mesmo
  princípio de transferência de aprendizado em escala industrial;

* **Transferência de aprendizado aplicada à detecção de objetos**, ajustando
  um YOLO pré-treinado no COCO para localizar e classificar objetos
  geométricos sintéticos, ilustrando o mesmo princípio de congelamento
  parcial em um domínio de origem e destino ainda mais distantes entre si;

* **Segmentação semântica com U-Net**, implementada e treinada do zero sobre
  um conjunto sintético de baixo contraste, superando uma linha de base
  clássica de limiarização graças às conexões de atalho entre codificador e
  decodificador;

* **Engenharia de dados para Visão Computacional com *Roboflow***, utilizando
  um *dataset* e um modelo pré-treinado para realizar inferência na detecção
  de objetos geométricos, além de comparar conjuntos de dados com diferentes
  resoluções, mas com as mesmas classes de objetos;

* A integração de **geometria computacional** (homografia e calibração) e
  **aprendizado profundo** em duas aplicações reais que encerram o livro:
  **realidade aumentada** e **fotogrametria**.

## 🤖 Uso do Gemini Notebook como Tutor Complementar

Nesta edição, o uso do **Gemini Notebook** é incentivado como ferramenta
complementar de aprendizagem. Baseado em inteligência artificial, o sistema
utiliza exclusivamente os documentos fornecidos pelo autor como fonte de
conhecimento, produzindo respostas alinhadas ao conteúdo e à abordagem
adotada ao longo deste capítulo.

::: {.callout-important appearance="default" icon=false}

### 🎓 Estude com o Tutor Inteligente {.unnumbered}

[🚀 ACESSAR Gemini Notebook: CAPÍTULO 09](https://notebooklm.google.com/notebook/88495fa2-9139-45da-8d1b-9a9b2e609d36)

#### ⚠️ Aviso sobre Conteúdo Gerado por IA {.unnumbered}

Embora seja uma ferramenta valiosa de apoio aos estudos, o Gemini Notebook pode
eventualmente produzir respostas incompletas, imprecisas ou incorretas.
Recomenda-se validar as informações consultando o material do capítulo,
livros, artigos científicos e outras fontes acadêmicas confiáveis. Sempre
que possível, execute e experimente os exemplos práticos apresentados ao
longo do texto para consolidar a compreensão dos conceitos.
:::


## Lista de Exercícios

Os exercícios a seguir consolidam os conceitos apresentados neste capítulo por
meio de adaptações, experimentos e extensões dos algoritmos desenvolvidos ao
longo do texto, utilizando as bibliotecas `PyTorch`, `ultralytics` e a
biblioteca didática `morph`.

1. **(10%)** Investigue o impacto da profundidade em uma arquitetura
   convolucional. Partindo da rede de duas camadas do Projeto Prático 1,
   adicione uma terceira camada convolucional com 32 filtros antes das
   camadas totalmente conectadas. Treine a nova arquitetura mantendo o mesmo
   número de épocas e a mesma divisão dos dados. Compare a acurácia no
   conjunto de teste e o número total de parâmetros treináveis em relação à
   rede original, discutindo se o aumento de profundidade trouxe benefício
   mensurável para imagens de dimensão $8 \times 8$.

2. **(15%)** Avalie o limiar de dados necessários no domínio de destino para
   que o treinamento de uma CNN do zero se torne competitivo com a
   **transferência de aprendizado**. Variando o número de amostras de treino
   disponíveis no domínio B para $\{5, 10, 20, 40, 80\}$, meça a acurácia de
   teste para ambas as estratégias. Apresente os resultados em um gráfico de
   linhas e determine a partir de qual volume de dados o treinamento do zero
   atinge desempenho equivalente ao extrator pré-treinado.

3. **(15%)** Investigue a estratégia de **ajuste fino parcial**
   (*fine-tuning*) em comparação ao congelamento total de pesos. No cenário
   de transferência de aprendizado entre domínios de dígitos, descongele a
   segunda camada convolucional (`conv2`) do extrator para que ela seja
   atualizada juntamente com a cabeça de classificação durante o treinamento
   no domínio B. Compare a acurácia obtida com o congelamento total e com o
   treinamento do zero, discutindo o compromisso entre capacidade de
   adaptação e risco de sobreajuste (*overfitting*).

4. **(20%)** Avalie a influência da profundidade do congelamento (*freeze*) no
   desempenho de detectores **YOLO** submetidos à transferência de
   aprendizado. Utilizando o conjunto de dados sintético de formas
   geométricas, execute o ajuste fino variando o parâmetro de congelamento do
   *backbone* para $\{0, 5, 10, 15\}$. Registre a métrica $\text{mAP}_{50}$
   no conjunto de validação para cada configuração, apresente os dados em uma
   tabela e discuta se o congelamento parcial é vantajoso quando os domínios
   de origem (COCO) e de destino (formas geométricas) são significativamente
   distintos.

5. **(20%)** Estude a importância das **conexões de atalho** (*skip
   connections*) na arquitetura **U-Net** para segmentação semântica.
   Implemente uma variação `UNetSemAtalhos` que funcione como um
   *autoencoder* convolucional tradicional, removendo as concatenações entre
   os estágios do codificador e do decodificador. Treine ambos os modelos
   sobre a mesma base de nódulos sintéticos, compare o IoU médio no conjunto
   de validação e apresente visualmente a diferença na precisão das bordas
   segmentadas por cada método.

6. **(20%) — Desafio: segmentação semântica com *Roboflow*.** Utilize um
   projeto do *Roboflow* do tipo *instance segmentation*, contendo as sete
   classes de formas geométricas utilizadas neste capítulo. Exporte o
   *dataset* no formato `coco-segmentation` e desenvolva um procedimento para
   converter os polígonos armazenados nos arquivos
   `_annotations.coco.json` em máscaras semânticas multiclasse, nas quais
   cada pixel recebe o índice da classe correspondente e o valor `0`
   representa o fundo. Utilize as imagens e máscaras resultantes para treinar
   a `UNetCompacta`. Avalie o IoU médio no conjunto de teste e compare
   visualmente as máscaras preditas com as anotações originais. Discuta as
   principais dificuldades encontradas na conversão das anotações COCO para
   máscaras e os efeitos de objetos sobrepostos ou pertencentes a diferentes
   classes.

7. **(Bônus – 10%)** Desenvolva um sistema interativo que combine
   **detecção de objetos (YOLO)** com **medição por referência de escala
   (fotogrametria)**. Treine o detector para identificar duas classes em uma
   cena: um "Cartão de Referência" (dimensão conhecida de
   $8{,}56\text{ cm} \times 5{,}39\text{ cm}$) e um "Objeto Alvo". Ao
   realizar a inferência em uma nova imagem, utilize a dimensão em pixels da
   *bounding box* do cartão detectado para converter as dimensões da caixa do
   objeto alvo em centímetros. Exiba a imagem processada com os rótulos de
   classe, a probabilidade de confiança e as dimensões físicas estimadas
   sobrepostas.

8. **(Bônus – 10%)** Desenvolva um sistema de **estimativa de profundidade por
   visão estéreo** a partir de duas imagens da mesma cena obtidas de posições
   diferentes, simulando um par de câmeras estéreo. Considere que a distância
   entre as duas posições de captura (*baseline*) seja conhecida.

   Utilize um dos métodos de **detecção de objetos** apresentados no capítulo,
   como YOLO, para localizar os objetos de interesse nas duas imagens. Para
   cada detecção, estabeleça a correspondência entre o mesmo objeto nas duas
   posições e determine sua **disparidade**. A partir da disparidade, do
   *baseline* e dos parâmetros da câmera, utilize a geometria estéreo para
   estimar a distância de cada objeto em relação às câmeras.

   Como extensão da biblioteca didática `morph`, modifique o método
   `showBoundBox` para que, além da **classe** e da **confiança da detecção**,
   apresente sobre cada *bounding box* a **distância estimada do objeto**.
   O resultado deve permitir visualizar, diretamente nas imagens, a classe, a
   acurácia (confiança) e a profundidade de cada objeto detectado.

   Apresente as duas imagens com as detecções, as correspondências entre os
   objetos, a imagem de disparidade e uma representação da profundidade
   estimada. Discuta como a distância entre as câmeras, a precisão da
   detecção e da correspondência, a resolução das imagens e a posição do
   objeto na cena influenciam a qualidade da estimativa.

   Para validação, utilize pelo menos um objeto cuja distância à câmera seja
   conhecida. Compare a profundidade estimada com o valor real e reporte o
   **erro absoluto** e o **erro relativo**. Discuta também as limitações do
   método quando um objeto não é corretamente detectado nas duas imagens ou
   quando a correspondência entre as regiões observadas é ambígua.

## Encerramento da Parte II

Este capítulo conclui a Parte II do livro e encerra a sequência de conteúdos iniciada no Capítulo 6, dedicada à representação, detecção, descrição e correspondência de características em imagens. Ao longo desses capítulos, foram apresentados métodos clássicos de VC baseados em **características projetadas manualmente**, como Sobel, LBP, HOG, ORB e Haar Cascade, bem como métodos fundamentados em **características aprendidas automaticamente**, representados pelas CNNs.

Os exemplos e experimentos desenvolvidos evidenciam que nenhuma dessas abordagens é universalmente superior. A escolha da técnica mais adequada depende das características do problema, da disponibilidade de dados para treinamento, dos requisitos de precisão e das restrições computacionais da aplicação. Em problemas bem estruturados e com poucos dados, descritores clássicos frequentemente oferecem soluções simples e eficientes. Em contrapartida, tarefas mais complexas tendem a se beneficiar da capacidade de aprendizado proporcionada pelas CNNs.

Diversas direções de estudo podem aprofundar os conceitos apresentados nesta parte do livro, entre as quais se destacam:

- **Arquiteturas modernas de CNN**, como ResNet, EfficientNet e *Vision Transformers*, que ampliam a capacidade de representação e o desempenho em tarefas de classificação e reconhecimento visual;
- **Detecção e segmentação de objetos**, com destaque para as famílias YOLO e para modelos de segmentação baseados em *prompts*, como o *Segment Anything*;
- **Reconstrução tridimensional e SLAM** (*Simultaneous Localization and Mapping*), que utilizam múltiplas imagens para estimar a geometria da cena e a trajetória de câmeras em movimento;
- **Modelos generativos de imagens**, como redes adversariais generativas (GANs) e modelos de difusão, capazes de sintetizar imagens realistas a partir de exemplos ou descrições textuais.

Os fundamentos desenvolvidos ao longo da Parte II constituem a base para essas e outras áreas avançadas da VC, nas quais a representação adequada das informações visuais permanece como elemento central para a análise e a compreensão de imagens.

## Referências do Capítulo {.unnumbered}

Os conceitos e algoritmos apresentados neste capítulo foram fundamentados em referências clássicas e contemporâneas da literatura de Aprendizado Profundo aplicado à VC:

* @mcculloch1943logical, para a proposição da primeira abstração matemática e lógica do neurônio artificial, fundamentando as bases conceituais do processamento neural computacional.
* @rosenblatt1958perceptron, para a formulação original do **Perceptron**, modelo precursor do neurônio artificial utilizado nas arquiteturas modernas de aprendizado profundo.
* @goodfellow2016deep e @lecun2015deep, para os fundamentos de redes neurais, convolução, funções de ativação e treinamento de modelos profundos.
* @bishop2006pattern, para os conceitos rigorosos de reconhecimento de padrões, probabilidade, estimativa de máxima verossimilhança e métodos estatísticos aplicados ao aprendizado de máquina.
* @ronneberger2015u para a arquitetura **U-Net**, utilizada na segmentação semântica com conexões de atalho entre codificador e decodificador.
* @redmon2016you, para a arquitetura **YOLO** (*You Only Look Once*), utilizada nos experimentos de detecção de objetos com transferência de aprendizado.
* @ren2015faster, para a arquitetura **Faster R-CNN**, empregada como modelo pré-treinado de detecção de objetos.
* @he2016deep, para a arquitetura **ResNet**, base de diversos extratores de características pré-treinados utilizados neste capítulo.
* @chen2018encoder, para a arquitetura **DeepLabV3**, utilizada como modelo pré-treinado de segmentação semântica.
* @kirillov2023segment, para o modelo **Segment Anything (SAM)**, mencionado como direção de estudo para segmentação promptable.
* @notebooklm2025, referente à ferramenta **Gemini Notebook**, utilizada na elaboração do infográfico de síntese do capítulo e disponibilizada como apoio complementar ao estudo.